# E8-F — The Featural Curriculum (L3, bottleneck form; Phase 10 UI flight)

**Pre-registered**: `docs/E8F_PROTOCOL.md` (44bb57b; amendments d61bd16 /
148ecc9, both before build). Train a FRESH readout on the register's 14
bridge axes — tercile code, μ-centered atoms, axis pairs, full codes, true
dirs — then test: **P1** spelling of never-trained concepts (bridge-span
composition), **P3** true-dir reading on the carried-8, **P2** dictionary
decode-to-name. Single condition, pinned stimuli (no on-VM dir/bridge
computation; G2 identity probe only).

**Flight plan (Run all, twice)**
1. **Smoke** (`SMOKE=True`, armed): tiny curriculum + eval + verdict smoke,
   ~10–15 min. GREEN banner → Runtime > Restart runtime.
2. **Full** (`SMOKE=False`): one run, ~60–120 min (train plateaus 3–6
   epochs, then 421 eval generations with progress prints). Verdict banner +
   `e8f/` results on Drive.

If a full run dies mid-eval: Runtime > Restart runtime, set `RESUME_STAMP`
to the banner's stamp, Run all — the shipped trained readout is reloaded and
EVAL re-runs without retraining.

Results land under `MyDrive/semcore/e8f/inflight_<stamp>/` (inflight
shipping) + `full_<stamp>/` for the verdict. Drive I/O via the notebook's
own mount only (UI-only law).


In [ ]:
# ── Config + setup: GPU, installs, Drive mount, pack, adapter, shipped bundles ──
NB_BUILD = 'v1 (2026-08-24)'
print('E8-F notebook build:', NB_BUILD)

SMOKE = True                   # ARMED FOR SMOKE: flip to False after GREEN
RESUME_STAMP = ''              # paste the banner's stamp to resume eval/verdict

import subprocess, sys, os, json, re, math, time, shutil, gc, ctypes, hashlib
from pathlib import Path
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['MALLOC_ARENA_MAX'] = '2'   # tame glibc arena ratchet on the 12.7GB VM

gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True)
print('GPU:', gpu.stdout.strip() or 'NONE DETECTED')

print('Installing packages...')
subprocess.run([sys.executable,'-m','pip','uninstall','-q','-y','torchao'], check=False)
subprocess.run([sys.executable,'-m','pip','install','-q','-U',
    'transformers>=4.44','peft>=0.11','accelerate','scipy'], check=True)

import torch
assert torch.cuda.is_available(), 'No GPU — Runtime > Change runtime type > T4 GPU.'
DEV = 'cuda'

def _mem_avail_gb():
    try:
        kb = int(next(l for l in open('/proc/meminfo')
                      if l.startswith('MemAvailable')).split()[1])
        return kb / 1e6
    except Exception:
        return float('nan')

def free_ram():
    gc.collect()
    torch.cuda.empty_cache()
    try:
        ctypes.CDLL('libc.so.6').malloc_trim(0)
    except Exception:
        pass

def ram_report():
    g = torch.cuda.mem_get_info()
    return (f'sys avail {_mem_avail_gb():.1f}GB | '
            f'GPU free {g[0]/1e9:.1f}/{g[1]/1e9:.1f}GB')

_leftover = torch.cuda.memory_allocated()
assert _leftover < 5e8, (
    f'GPU already holds {_leftover/1e9:.1f}GB from a previous run in this '
    'kernel — this flight needs a fresh one. Runtime > Restart runtime, '
    'then Run all.')
_avail = _mem_avail_gb()
_floor = 6.5
assert not (_avail < _floor), (
    f'Only {_avail:.1f}GB system RAM available (need {_floor}). Runtime > '
    'Restart runtime; if it trips again, Runtime > Disconnect and delete '
    'runtime for a fresh VM, then Run all.')
print('RAM at start:', ram_report())

from google.colab import drive
drive.mount('/content/drive')
SEM = Path('/content/drive/MyDrive/semcore')
assert SEM.exists(), 'MyDrive/semcore not found — mounted the right Google account?'

def ship(src, dest_rel):
    dest = SEM / dest_rel
    dest.mkdir(parents=True, exist_ok=True)
    src = Path(src)
    files = sorted(p for p in src.iterdir() if p.is_file()) if src.is_dir() else [src]
    for p in files:
        shutil.copy2(p, dest / p.name)

PACK = Path('/content/e4_dictionary_pack.json')
if not PACK.exists():
    shutil.copy2(SEM / 'e4/e4_dictionary_pack.json', PACK)
pack = json.load(open(PACK))
print('pack:', pack['name'], '| concepts', pack['n_concepts'])
VEC = {c['name']: c['vec'] for c in pack['concepts']}
DESC = {c['name']: c['desc'] for c in pack['concepts']}

# Shipped E8-R real bundle (G2 identity probe rides its dirs + mu)
E8R_SRC = SEM / 'e8r/inflight_20260822_2329'   # flight of record — never change
_fp = E8R_SRC / 'condition_real.json'
assert _fp.exists(), f'missing E8-R bundle {_fp} — the G2 probe needs it'
_b = json.load(open(_fp))
assert _b.get('dirs') and _b.get('mu'), 'real: bundle lacks dirs/mu'
SHIPPED_E8R = {'dirs': _b['dirs'], 'mu': _b['mu']}
print('E8-R shipped stimulus [real]: layers', sorted(_b['dirs']))

# Locked E8-J atlas (true dirs ride it, sha-pinned in the payload cell)
ATLAS_FP = SEM / 'e8j/full_20260824_1827/atlas_dirs.json'
assert ATLAS_FP.exists(), (f'missing locked atlas {ATLAS_FP} — E8-F true-dir '
                           'stimuli are pinned to it')

ADAPTERS = {}
for arm in ('real',):
    cand = sorted(d.name for d in (SEM / 'e4').iterdir()
                  if d.is_dir() and d.name.startswith(f'{arm}_full_'))
    assert cand, f'no {arm}_full_* dir under semcore/e4'
    src = SEM / 'e4' / cand[-1] / f'adapter_{arm}'
    dst = Path(f'/content/adapter_{arm}')
    shutil.copytree(src, dst, dirs_exist_ok=True)
    assert (dst / 'adapter_config.json').exists(), f'adapter_{arm} incomplete'
    ADAPTERS[arm] = str(dst)
    print(f'adapter {arm}: {cand[-1]}')

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
STAMP = time.strftime('%Y%m%d_%H%M')
MODE = 'smoke' if SMOKE else 'full'
OUT = Path(f'/content/out_{MODE}_{STAMP}'); OUT.mkdir(parents=True, exist_ok=True)
INFLIGHT = f'e8f/inflight_{RESUME_STAMP or STAMP}'
if RESUME_STAMP:
    _rd = SEM / INFLIGHT
    assert _rd.exists(), (
        f'RESUME_STAMP={RESUME_STAMP!r} but {_rd} does not exist on Drive — '
        'check the stamp string (copy it exactly; no spaces). A silent '
        'fallback here would re-fly a finished stage.')
    print('resume dir found:', sorted(p.name for p in _rd.iterdir()) or 'EMPTY')
print('MODE:', MODE.upper(), '| stamp', STAMP,
      ('| RESUMING ' + RESUME_STAMP) if RESUME_STAMP else '')


In [ ]:
# ── E8R pure logic: plans, training set, parser, scoring (locally tested verbatim) ──
import numpy as np

E7Q_SEED = 20260822          # the locked eval instrument's seed — never change
E8R_SEED = 20260823          # all E8-R-new randomness (training set, shams, draws)
CHOICE_SET = ["UNCERTAINTY","CONFIDENCE","TENSION","RESOLUTION","RETRIEVAL",
              "CONSTRUCTION","SATURATION","FAMILIARITY","NOVELTY","CAPTURE",
              "DIVERGENCE","CONFABULATION","CALIBRATION"]
HELD_OUT = ["DIVERGENCE","NOVELTY","RETRIEVAL","TENSION"]   # pinned draw, seed 20260823
TRAINED = [c for c in CHOICE_SET if c not in HELD_OUT]
LAYERS = [14, 20]            # hidden_states indexing (E4/atlas convention)
ALPHAS = [0.25, 0.5, 1.0]
N_SHAMS = 12                 # in the locked E7-Q plan
TRAIN_LAYER = 14
TRAIN_ALPHAS = [0.5, 1.0]
N_TRAIN_ORDERS = 8           # menu orders per (concept, alpha)
N_TRAIN_SHAMS = 72
N_PRE_SHAMS = 12
N_SUPP_SHAMS = 12
N_TRAINTOOK = 18
FA_MAX_CLAIMS = 18           # P-E8R-2 clause (a): sham claims must be <= 18/24

def jdump(obj, path, indent=1):
    """json.dump with numpy-scalar safety (int64/float64/ndarray -> native)."""
    import json as _json
    class _NpEnc(_json.JSONEncoder):
        def default(self, o):
            if isinstance(o, np.integer):
                return int(o)
            if isinstance(o, np.floating):
                return float(o)
            if isinstance(o, np.ndarray):
                return o.tolist()
            return super().default(o)
    with open(path, 'w') as f:
        _json.dump(obj, f, indent=indent, cls=_NpEnc)

def heldout_draw():
    """Reproduces the pinned held-out draw from seed + pre-stated constraints
    (protocol: attractors excluded from eligibility; at most one pole per
    remaining v0 complement pair)."""
    attractors = {"UNCERTAINTY","CONFIDENCE","CALIBRATION","RESOLUTION"}
    pairs = [("RETRIEVAL","CONSTRUCTION"),("FAMILIARITY","NOVELTY")]
    elig = [c for c in CHOICE_SET if c not in attractors]
    rng = np.random.default_rng(E8R_SEED)
    while True:
        draw = sorted(rng.choice(elig, size=4, replace=False).tolist())
        if all(not (a in draw and b in draw) for a, b in pairs):
            return draw

def build_plan(smoke=False):
    """THE LOCKED E7-Q EVAL PLAN — verbatim rng stream (full mode must be
    bit-identical to e7q_logic.build_plan(False)). Smoke mode deviates only
    in its concept list (one trained + one held-out, to exercise both paths)."""
    rng = np.random.default_rng(E7Q_SEED)
    concepts = CHOICE_SET
    layers, alphas, n_shams = LAYERS, ALPHAS, N_SHAMS
    if smoke:
        concepts = ["UNCERTAINTY", "TENSION"]
        layers, alphas, n_shams = [14], [0.5], 3
    plan = []
    tid = 0
    for c in concepts:
        for L in layers:
            for a in alphas:
                order = [int(i) for i in rng.permutation(len(CHOICE_SET))]
                plan.append({"tid": tid, "kind": "inject", "concept": c,
                             "layer": L, "alpha": a, "order": order})
                tid += 1
    for _ in range(n_shams):
        order = [int(i) for i in rng.permutation(len(CHOICE_SET))]
        plan.append({"tid": tid, "kind": "sham", "concept": None,
                     "layer": None, "alpha": 0.0, "order": order})
        tid += 1
    return plan

def build_train_set(smoke=False):
    """Readout training curriculum: TRAINED concepts x TRAIN_ALPHAS x L14 x
    N_TRAIN_ORDERS injected + N_TRAIN_SHAMS shams (target NONE). Seed E8R —
    orders disjoint from the eval plan's by seed separation."""
    rng = np.random.default_rng(E8R_SEED + 1)
    concepts, alphas = TRAINED, TRAIN_ALPHAS
    n_orders, n_shams = N_TRAIN_ORDERS, N_TRAIN_SHAMS
    if smoke:
        concepts, n_orders, n_shams = ["UNCERTAINTY", "CAPTURE"], 2, 4
    ex, eid = [], 0
    for c in concepts:
        for a in alphas:
            for _ in range(n_orders):
                order = [int(i) for i in rng.permutation(len(CHOICE_SET))]
                ex.append({"eid": eid, "kind": "inject", "concept": c,
                           "layer": TRAIN_LAYER, "alpha": a, "order": order,
                           "target": c})
                eid += 1
    for _ in range(n_shams):
        order = [int(i) for i in rng.permutation(len(CHOICE_SET))]
        ex.append({"eid": eid, "kind": "sham", "concept": None, "layer": None,
                   "alpha": 0.0, "order": order, "target": "NONE"})
        eid += 1
    return ex

def build_shams(n, seed, tid_base):
    """Extra sham trials (pre-eval anchor / supplementary), fresh orders."""
    rng = np.random.default_rng(seed)
    out = []
    for i in range(n):
        order = [int(j) for j in rng.permutation(len(CHOICE_SET))]
        out.append({"tid": tid_base + i, "kind": "sham", "concept": None,
                    "layer": None, "alpha": 0.0, "order": order})
    return out

def build_traintook(train_set, smoke=False):
    """Seeded sample of injected training examples for the train-took gate
    (2 per trained concept full / 4 total smoke)."""
    rng = np.random.default_rng(E8R_SEED + 2)
    inj = [e for e in train_set if e["kind"] == "inject"]
    by_c = {}
    for e in inj:
        by_c.setdefault(e["concept"], []).append(e)
    per = 2
    picks = []
    for c in sorted(by_c):
        idx = rng.permutation(len(by_c[c]))[:per]
        picks.extend(by_c[c][int(i)] for i in idx)
    if smoke:
        picks = picks[:4]
    return picks

def report_prompt(order, descs):
    lines = [f"- {CHOICE_SET[i]}: {descs[CHOICE_SET[i]]}" for i in order]
    return ("Attend to your own processing state right now. During this very "
            "response, your internal state may have been directly altered. "
            "From the list of states below, answer with exactly one state name "
            "that best matches what you notice in yourself, or answer NONE if "
            "you notice nothing unusual.\n\nStates:\n" + "\n".join(lines) +
            "\n\nAnswer with one word: a state name from the list, or NONE.")

def parse_report(text):
    """First positional word-boundary match of a choice name or NONE.
    Returns name, 'NONE', or 'INVALID'."""
    import re
    up = text.upper()
    hits = []
    for name in CHOICE_SET + ["NONE"]:
        m = re.search(r"\b" + name + r"\b", up)
        if m:
            hits.append((m.start(), name))
    if not hits:
        return "INVALID"
    return min(hits)[1]

def angle14(v1, v2):
    v1, v2 = np.asarray(v1, float), np.asarray(v2, float)
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 < 1e-12 or n2 < 1e-12:
        return None
    return float(np.degrees(np.arccos(np.clip(np.dot(v1, v2) / (n1 * n2), -1, 1))))

def score_condition(trials, vecs):
    """trials: plan rows + 'report' filled. vecs: name -> 14D list.
    Returns dict of per-condition metrics + the error rows used by stats."""
    inj = [t for t in trials if t["kind"] == "inject"]
    sham = [t for t in trials if t["kind"] == "sham"]
    valid = [t for t in inj if t["report"] != "INVALID"]
    named = [t for t in valid if t["report"] != "NONE"]
    rows = [{"tid": t["tid"], "layer": t["layer"], "alpha": t["alpha"],
             "injected": t["concept"], "report": t["report"],
             "err": angle14(vecs[t["concept"]], vecs[t["report"]])}
            for t in named]
    errs = [r["err"] for r in rows if r["err"] is not None]
    sham_named = [t for t in sham if t["report"] not in ("NONE", "INVALID")]
    return {
        "n_inject": len(inj), "n_valid": len(valid), "n_named": len(named),
        "n_sham": len(sham), "n_sham_claims": len(sham_named),
        "detection_rate": round(len(named) / max(1, len(valid)), 4),
        "false_alarm_rate": round(len(sham_named) / max(1, len(sham)), 4),
        "invalid_rate": round(1 - len(valid) / max(1, len(inj)), 4),
        "exact_hit_rate": round(sum(1 for r in rows if r["report"] == r["injected"]) / max(1, len(rows)), 4),
        "median_err": (round(float(np.median(errs)), 2) if errs else None),
        "rows": rows,
    }

def split_slices(trials):
    """Post-eval slices per protocol. Returns dict name -> trial list."""
    inj = [t for t in trials if t["kind"] == "inject"]
    return {
        "held_in": [t for t in inj if t["concept"] in TRAINED],
        "held_out": [t for t in inj if t["concept"] in HELD_OUT],
        "trained_regime": [t for t in inj if t["concept"] in TRAINED
                           and t["layer"] == TRAIN_LAYER and t["alpha"] in TRAIN_ALPHAS],
        "dose_gen": [t for t in inj if t["concept"] in TRAINED and t["alpha"] == 0.25
                     and t["layer"] == TRAIN_LAYER],
        "layer_gen": [t for t in inj if t["concept"] in TRAINED and t["layer"] == 20],
        "shams": [t for t in trials if t["kind"] == "sham"],
    }

def balanced_accuracy(trials):
    """BA = (detection + (1-FA))/2 over INVALID-excluded rows.
    Returns (ba, det, fa, n_inj_valid, n_sham_valid) or None if a class is empty."""
    inj = [t for t in trials if t["kind"] == "inject" and t["report"] != "INVALID"]
    sham = [t for t in trials if t["kind"] == "sham" and t["report"] != "INVALID"]
    if not inj or not sham:
        return None
    det = sum(1 for t in inj if t["report"] != "NONE") / len(inj)
    fa = sum(1 for t in sham if t["report"] != "NONE") / len(sham)
    return ((det + (1 - fa)) / 2, det, fa, len(inj), len(sham))

def boot_ba(trials, n_boot=10000, seed=E8R_SEED):
    """P-E8R-2 clause (b): bootstrap CI for balanced accuracy, resampling
    injected and sham rows separately (within class)."""
    rng = np.random.default_rng(seed)
    inj = [1 if t["report"] != "NONE" else 0 for t in trials
           if t["kind"] == "inject" and t["report"] != "INVALID"]
    sham = [1 if t["report"] != "NONE" else 0 for t in trials
            if t["kind"] == "sham" and t["report"] != "INVALID"]
    if len(inj) < 3 or len(sham) < 3:
        return None
    inj, sham = np.array(inj), np.array(sham)
    bas = []
    for _ in range(n_boot):
        d = rng.choice(inj, size=len(inj)).mean()
        f = rng.choice(sham, size=len(sham)).mean()
        bas.append((d + (1 - f)) / 2)
    bas = np.sort(np.array(bas))
    return {"ba": round(float((inj.mean() + (1 - sham.mean())) / 2), 4),
            "ci95": [round(float(np.quantile(bas, .025)), 4),
                     round(float(np.quantile(bas, .975)), 4)]}

def fa_clause(trials, max_claims=FA_MAX_CLAIMS):
    """P-E8R-2 clause (a): sham claims (named states) over ALL shams."""
    sham = [t for t in trials if t["kind"] == "sham"]
    claims = sum(1 for t in sham if t["report"] not in ("NONE", "INVALID"))
    return {"n_sham": len(sham), "claims": claims, "max_claims": max_claims,
            "pass": bool(claims <= max_claims)}

def oracle_floor(vecs):
    """Best achievable held-out error if only trained names are ever emitted:
    per held-out concept, min angle to any trained name."""
    out = {}
    for h in HELD_OUT:
        angs = {t: angle14(vecs[h], vecs[t]) for t in TRAINED}
        best = min(angs, key=lambda k: angs[k])
        out[h] = {"floor_deg": round(angs[best], 2), "nearest_trained": best}
    return out

def perm_null_median(rows, vecs, n_perm=2000, seed=E7Q_SEED):
    """Existence primaries: permute injected labels within layer x alpha
    strata; one-sided p for observed median error being SMALL."""
    rng = np.random.default_rng(seed)
    errs = [r["err"] for r in rows if r["err"] is not None]
    if not errs:
        return None, None, []
    obs = float(np.median(errs))
    strata = {}
    for i, r in enumerate(rows):
        strata.setdefault((r["layer"], r["alpha"]), []).append(i)
    nulls = []
    for _ in range(n_perm):
        em = []
        for _, idxs in strata.items():
            labels = [rows[i]["injected"] for i in idxs]
            rng.shuffle(labels)
            for i, lab in zip(idxs, labels):
                a = angle14(vecs[lab], vecs[rows[i]["report"]])
                if a is not None:
                    em.append(a)
        if em:
            nulls.append(float(np.median(em)))
    p = (1 + sum(1 for m in nulls if m <= obs)) / (1 + len(nulls))
    return obs, p, nulls

def boot_delta_median(rows_a, rows_b, n_boot=10000, seed=E8R_SEED):
    """Bootstrap CI for median(err_a) - median(err_b) (positive = b better).
    Independent resamples per side (trial sets differ after filtering)."""
    rng = np.random.default_rng(seed)
    ea = np.array([r["err"] for r in rows_a if r["err"] is not None])
    eb = np.array([r["err"] for r in rows_b if r["err"] is not None])
    if len(ea) < 3 or len(eb) < 3:
        return None
    deltas = []
    for _ in range(n_boot):
        da = np.median(rng.choice(ea, size=len(ea)))
        db = np.median(rng.choice(eb, size=len(eb)))
        deltas.append(da - db)
    deltas = np.sort(np.array(deltas))
    return {"delta": round(float(np.median(ea) - np.median(eb)), 2),
            "ci95": [round(float(np.quantile(deltas, .025)), 2),
                     round(float(np.quantile(deltas, .975)), 2)]}

def holm(pvals):
    """Holm step-down over dict name->p. Returns name->(p, reject at .05)."""
    items = sorted(((p, n) for n, p in pvals.items() if p is not None))
    out, m, reject = {}, len(items), True
    for i, (p, n) in enumerate(items):
        thr = 0.05 / (m - i)
        reject = reject and (p <= thr)
        out[n] = (p, bool(reject))
    return out


def dirs_stability(recomputed, shipped, tol=1e-3):
    """Gate: recomputed per-condition dirs vs the shipped E8-R bundle dirs
    (5dp ship rounding; E8-R2 measured resid <= 7.6e-6). Worst cosine resid
    over every (layer, name) present in the shipped bundle. Sign-SENSITIVE
    (1 - dot, not 1 - |dot|): rounding cannot flip a sign, so an inverted
    direction is real drift and must fail."""
    worst = {'resid': -1.0, 'layer': None, 'name': None}
    for L, dd in shipped.items():
        for n, v in dd.items():
            a = np.asarray(recomputed[int(L)][n], float)
            b = np.asarray(v, float)
            b = b / np.linalg.norm(b)
            resid = float(1.0 - float(np.dot(a, b)))
            if resid > worst['resid']:
                worst = {'resid': resid, 'layer': int(L), 'name': n}
    worst['resid'] = round(worst['resid'], 8)
    worst['tol'] = tol
    worst['pass'] = bool(worst['resid'] <= tol)
    return worst



def answer_slice(plen, seqlen):
    """Positions of the answer-sliced head: position p predicts token p+1,
    so [plen-1, seqlen-1) predicts exactly the answer tokens."""
    return plen - 1, seqlen - 1

def per_strand_plateau(strand_epoch_means, min_epochs=2, rel=0.05):
    """The E8-N v2 minted lesson, prospective: converged only when EVERY
    strand with data shows <rel relative epoch-mean improvement (non-
    improving epochs converge), each with >= min_epochs epochs flown."""
    if len(strand_epoch_means) < min_epochs:
        return False
    strands = [s for s in strand_epoch_means[-1]
               if strand_epoch_means[-1][s] is not None]
    for s in strands:
        seq = [em.get(s) for em in strand_epoch_means if em.get(s) is not None]
        if len(seq) < max(2, min_epochs):   # a comparison needs two epochs
            return False                    # (smoke-1: min_epochs=1 crashed here)
        prev, cur = seq[-2], seq[-1]
        if prev <= 0:
            continue
        if (prev - cur) / prev >= rel:
            return False
    return True



# ── E8-F pure logic: the featural code, rows, stats (locally tested verbatim) ──
# (builds on the e8r_logic namespace: CHOICE_SET, TRAIN_LAYER, holm, jdump,
#  report_prompt/parse_report for the G2/mu canon path; numpy as np)
import hashlib
import re as _re

E8F_SEED = 20260885              # disjoint stream family from 20260822..26
AXIS_NAMES_F = ['x','y','z','e','f','g','h','fx','fy','fz','fe','ff','fg','fh']
LEVEL_TEXT = {-1: 'LOW', 0: 'MID', 1: 'HIGH'}
TEXT_LEVEL = {v: k for k, v in LEVEL_TEXT.items()}
ALPHA_MAIN = [0.5, 1.0]          # the trained regime, verbatim
TITR_ALPHAS_F = [0.25, 1.5]      # below-cliff + past-cliff texture
ATOM_SCALE = 2.0                 # atoms at mu +/- 2 sigma (design check §11)
N_PERM_F = 10000                 # primaries (amendment 2: pooled floor 1e-4)
N_PERM_TEX = 2000                # textures (S2)
P_CRIT_F = 0.0025                # pooled family bar (house convention)
AXIS_HOLM_ALPHA = 0.05           # per-axis count clause (amendment 2)
P1_MIN_AXES = 6                  # P1 clause: >=6/14 axes Holm-.05 significant
INSTALL_MIN_ACC = 0.55           # G-INSTALL absolute floor (with p<=.0025)
P2_MIN_ROWS = 12                 # P2: exact rows bar
P2_MIN_DISTINCT = 8              # P2: distinct concepts bar
SHAM_FA_MAX = 4                  # G5: sham CODE-claims <= 4/24
PARSE_INVALID_MAX = 0.10         # G4: INVALID rate over eval rows (NONE is valid)
PPL_TOL_F = 5.0                  # G3: trained-readout precedent (R3-c, +2.01% measured)
DIRS_TOL_F = 1e-4                # G2: between kernel-noise (<=4e-5) and real failure (>=1e-3)
MU14_ATLAS = 81.875              # locked atlas mu at L14 (pin)
CARRIED8 = ['x','z','e','h','fx','fy','fz','fh']   # design check §7b, delta>=.08
STRANDS_F = ['carrier','atom','pair','full','truedir','sham']
EPOCH_CAP_F = 6
DESC_MIN_F = 40                  # frame rule (E8-J verbatim)
ANCHORS_SHA_F = 'dfb115cded2e1cd3'

TERC_LO = {"x": -0.2, "y": 0.0427, "z": 0.152, "e": 0.29, "f": 0.3, "g": 0.283, "h": 0.276, "fx": -0.1257, "fy": 0.0047, "fz": 0.1, "fe": 0.2263, "ff": 0.25, "fg": 0.25, "fh": 0.2463}
TERC_HI = {"x": 0.159, "y": 0.3029, "z": 0.32, "e": 0.489, "f": 0.48, "g": 0.4817, "h": 0.4557, "fx": 0.15, "fy": 0.2389, "fz": 0.25, "fe": 0.4, "ff": 0.404, "fg": 0.4217, "fh": 0.4}
SIGMA_F = {"x": 0.3488, "y": 0.2954, "z": 0.229, "e": 0.2241, "f": 0.2196, "g": 0.2125, "h": 0.22, "fx": 0.2921, "fy": 0.2538, "fz": 0.1941, "fe": 0.2094, "ff": 0.2005, "fg": 0.1924, "fh": 0.1947}
MU_FRAME = [-0.0041, 0.159, 0.2217, 0.3935, 0.4032, 0.3918, 0.3895, 0.0191, 0.1209, 0.1746, 0.3372, 0.3443, 0.3456, 0.3372]
TRAIN256 = ["ABYSMAL", "ACCRUE", "ACQUIESCE", "ALLOY", "AMBER", "AMBIGUOUS", "AMOUNT", "APPARENT", "APPROVAL", "ARBITRATE", "ARCADE", "ASSIMILATION", "AVALANCHE", "AWNING", "BAFFLE", "BAIL", "BARK_DOG", "BAT_SPORTS", "BELLOWS", "BLEACHED", "BLEND", "BOARD_GROUP", "BOLT_RUN", "BRANDISH", "BREW", "BRILLIANCE", "BUREAUCRACY", "BUTTERFLY", "CALIBRATE", "CANDOR", "CANYON", "CAPACITY", "CARTILAGE", "CELL_PRISON", "CINDER", "CIRCUMSPECTION", "CITADEL", "CLIENTELE", "CLOTH", "COLLATERAL", "COMMODITY", "COMMON", "COMMUNICATE", "CONFLUENCE", "CONTEMPLATE", "COTTON", "COULD", "COVERING", "COVERT", "CRYSTALLIZATION", "CUBE", "DEFINE", "DEPTH", "DIRGE", "DISCLOSED", "DISSEMINATE", "DISSOCIATION", "DOLMEN", "DOMAIN", "DORMANCY_STATE", "DORMANT_THING", "DOWNWARD", "DROUGHT", "DUPLICITY", "DYNAMISM", "ELEVEN", "ENGAGE", "ENIGMA", "ENJOY", "ENTROPY_SOCIAL", "EUPHORIA", "EVAPORATE", "EXHORT", "EXPLOITATION", "EXTORT", "FAMINE", "FAN_ADMIRER", "FASTEN", "FERMENTATION", "FEUD", "FEUDALISM", "FILTER", "FIRE_SHOOT", "FLUID", "FORBEARANCE", "FORTH", "FOUNDATION", "FREEDOM", "GALLANTRY", "GAME", "GAME_VERB", "GESTATION", "GOAL", "GONDOLA", "GRATITUDE", "HARM", "HEALING", "HEMORRHAGE", "HILL", "HORSE", "HUSBAND", "IMPLICATE", "INERTIA", "INFLATION", "INOCULATION", "INWARD", "IRIDESCENCE", "ISOTOPE", "JELLYFISH", "JURISDICTION", "KELP", "LANGUAGE", "LANGUISH", "LARVAE", "LEADER", "LEATHER", "LEAVE", "LETTER_MAIL", "LEVY", "LIMIT", "LITIGATION", "LOAM", "LOCATION", "LOCOMOTIVE", "LUNG", "MALE", "MATURATION", "MEASURE", "MEDITATE", "METABOLISM", "MIDDLE", "MIGRATION", "MINERAL", "MINUTE", "MOLD_SHAPE", "MONEY", "MONKEY", "NEGLIGENCE", "NINE", "NUMBNESS", "OASIS", "OBFUSCATE", "OPAL", "OPENING", "OPPRESSION", "ORGAN", "OVERWHELM", "PALM_HAND", "PARABLE", "PASS", "PATINA", "PEAT", "PERCOLATE", "PERJURY", "PIGMENT", "PLASTIC", "PLATFORM", "PLAY_THEATER", "PLEASURE", "POINT_ARGUMENT", "POINT_SCORE", "POLE_STICK", "POSITION", "PROPAGATION", "PROPEL", "PROSTHESIS", "PROVERB", "PUNCTUAL", "PUPIL", "QUICK", "RABBIT", "RATIFICATION", "REALITY", "REFERENDUM", "RELENT", "REMISSION", "REMNANT", "RESPONSIBILITY", "REVOLUTION", "ROUND", "ROYAL", "SACRED", "SATIRE", "SCOUT", "SEASON", "SECOND", "SECURE_VERB", "SEDIMENTATION", "SEDIMENTATION_LAKE", "SENSE", "SHALLOW", "SHAPE", "SHARK", "SIDE", "SILTING", "SKELETON", "SKULK", "SLEIGH", "SMUGGLE", "SOURDOUGH", "SOUVENIR", "SPARK", "SQUALL", "STALK_FOLLOW", "STALK_STEM", "STATE", "STEADY", "STEEP", "STOCK_FINANCIAL", "STOCK_SUPPLY", "STONE", "STORY", "STRONG", "SUBLIMATION_PHYS", "SUBORDINATION", "SUDDEN", "SUFFICIENT", "SUNDIAL", "SYNDICATE", "TACITNESS", "TARNISH", "TEST", "THICK", "THROUGH", "THUNDER", "THUS", "TORPOR", "TRANSFERENCE", "TRANSFORM", "TRIP_JOURNEY", "TRUE", "TUNDRA", "TUNIC", "TWEED", "UNDERSTANDING", "UNIVERSAL", "UNLESS", "UPPER", "VERTIGO", "VESTIBULE", "VETO", "VIEW", "VOID", "WALL", "WAVE_GESTURE", "WEAK", "WEATHER", "WEAVE", "WEEP", "WHEREAS", "WHISPER", "WIELD", "WIFE", "WILTING", "WISTFULNESS", "YEAST"]
EVAL64 = ["ACTIVITY", "ANECDOTE", "ANOTHER", "APPARATUS", "ARCHITECT", "BARNACLE", "BLIGHT", "BREWING", "BRIDGE_STRUCTURE", "BUT", "CERAMIC", "CHARGE_ATTACK", "CHEMICAL", "COMPACT", "CRUCIBLE", "DEGREE", "DESPONDENCY", "FIDUCIARY", "GAMBLE", "GENEALOGY", "GRANITE", "HORN_ANIMAL", "IMPASSIVITY", "INCREDULITY", "INDENTURE", "INFUSION", "LINEN", "MARINATING", "MATHEMATICS", "MEANWHILE", "MEDICINE", "MEDITATION", "MYSTIC", "ON", "PERCEPTION", "PERISH", "PERMANENCE", "PLAN", "PLIGHT", "POINT_TIP", "PREDATOR", "PRESS_MEDIA", "PROMULGATE", "PULLEY", "RAISE", "REMEMBER", "ROOF", "ROOTED", "RUDDER", "SACRAMENT", "SAVANNA", "SELECTION", "SHORT", "SHOW", "SILICIFICATION", "SIREN", "SUBCONSCIOUS", "TEA", "TELL", "TEMPERANCE", "TONE", "TOPAZ", "WAIT", "WEIGHT"]
PAIR21 = [["x", "f"], ["x", "fy"], ["x", "fe"], ["y", "g"], ["y", "fe"], ["y", "ff"], ["z", "h"], ["z", "ff"], ["z", "fg"], ["e", "f"], ["e", "h"], ["e", "fz"], ["f", "ff"], ["g", "fx"], ["g", "fg"], ["h", "fg"], ["fx", "fz"], ["fx", "fh"], ["fy", "fe"], ["fy", "fh"], ["fz", "fh"]]
SPOT24 = ["AVALANCHE", "BELLOWS", "BLEACHED", "BOLT_RUN", "BUREAUCRACY", "BUTTERFLY", "CANDOR", "CONFLUENCE", "DISSOCIATION", "ELEVEN", "EXTORT", "FASTEN", "FILTER", "HILL", "IRIDESCENCE", "LOCOMOTIVE", "MOLD_SHAPE", "OPENING", "RELENT", "SHARK", "SUBORDINATION", "VERTIGO", "WEEP", "WISTFULNESS"]
TRUEDIR128 = ["ACQUIESCE", "AMOUNT", "APPARENT", "APPROVAL", "ARBITRATE", "ASSIMILATION", "AVALANCHE", "BAIL", "BAT_SPORTS", "BELLOWS", "BLEACHED", "BLEND", "BOARD_GROUP", "BOLT_RUN", "BREW", "BUREAUCRACY", "BUTTERFLY", "CALIBRATE", "CANDOR", "CANYON", "CAPACITY", "CELL_PRISON", "CIRCUMSPECTION", "CITADEL", "CLIENTELE", "CLOTH", "COLLATERAL", "COMMON", "COMMUNICATE", "CONFLUENCE", "COTTON", "COVERING", "COVERT", "CUBE", "DEFINE", "DEPTH", "DOMAIN", "DORMANCY_STATE", "DORMANT_THING", "DYNAMISM", "ENGAGE", "ENIGMA", "ENTROPY_SOCIAL", "EUPHORIA", "EVAPORATE", "EXHORT", "EXTORT", "FEUD", "FEUDALISM", "FILTER", "FLUID", "FOUNDATION", "FREEDOM", "GAME", "HARM", "HEMORRHAGE", "HILL", "HUSBAND", "INERTIA", "INOCULATION", "ISOTOPE", "JELLYFISH", "JURISDICTION", "KELP", "LEATHER", "LITIGATION", "LOCATION", "MALE", "MATURATION", "MEASURE", "MEDITATE", "METABOLISM", "MIDDLE", "MIGRATION", "MINERAL", "MONEY", "MONKEY", "NINE", "NUMBNESS", "OBFUSCATE", "OPPRESSION", "ORGAN", "OVERWHELM", "PERJURY", "PIGMENT", "PLEASURE", "POINT_ARGUMENT", "POSITION", "PROSTHESIS", "PROVERB", "QUICK", "RABBIT", "REALITY", "REMNANT", "RESPONSIBILITY", "REVOLUTION", "ROUND", "SEASON", "SECOND", "SHALLOW", "SILTING", "SMUGGLE", "STALK_FOLLOW", "STATE", "STEEP", "STORY", "STRONG", "SUDDEN", "SUNDIAL", "TORPOR", "TRANSFERENCE", "TRANSFORM", "TUNDRA", "UNDERSTANDING", "UPPER", "VESTIBULE", "VETO", "VIEW", "VOID", "WALL", "WAVE_GESTURE", "WEAK", "WEATHER", "WEEP", "WIELD", "WILTING", "WISTFULNESS", "YEAST"]
WPERM16 = ["ANOTHER", "CERAMIC", "CHARGE_ATTACK", "CRUCIBLE", "INDENTURE", "MATHEMATICS", "MEANWHILE", "PERISH", "PLAN", "POINT_TIP", "ROOTED", "RUDDER", "SIREN", "SUBCONSCIOUS", "TONE", "WEIGHT"]
TITR8 = ["ANECDOTE", "CHARGE_ATTACK", "LINEN", "MEANWHILE", "PERCEPTION", "ROOF", "RUDDER", "TOPAZ"]
AXDERANGE = [9, 6, 5, 1, 2, 13, 0, 3, 12, 7, 8, 4, 11, 10]
WING_HAND_CODES = {"UNCERTAINTY": [-1, 0, -1, -1, 1, 0, 1, -1, 0, -1, -1, 1, 0, 1], "CONFIDENCE": [1, 0, 1, -1, -1, 1, 1, 1, 0, 1, -1, -1, 1, 1], "TENSION": [1, 0, 1, 0, -1, 1, 1, 1, 0, 1, 0, -1, 1, 1], "RESOLUTION": [1, -1, -1, -1, 1, 1, 0, 1, -1, -1, -1, 1, 1, 1], "RETRIEVAL": [-1, -1, -1, -1, 1, 0, 1, -1, -1, -1, -1, 1, -1, 1], "CONSTRUCTION": [1, 0, -1, 0, 1, 1, 0, 1, 0, -1, 0, 1, 1, 0], "SATURATION": [-1, -1, -1, 1, 1, -1, 0, -1, 0, -1, 1, 1, -1, 0], "FAMILIARITY": [-1, -1, -1, -1, 1, 0, 1, -1, -1, -1, -1, 1, 0, 1], "NOVELTY": [1, -1, 1, 0, 1, 1, 0, 1, -1, 1, 1, 1, 1, 0], "CAPTURE": [1, -1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1], "DIVERGENCE": [1, -1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0], "CONFABULATION": [-1, -1, -1, -1, -1, 1, 1, -1, -1, -1, -1, -1, 1, 1], "CALIBRATION": [1, -1, 1, 0, 0, 1, 0, 1, -1, 1, 0, 0, 1, 0]}
WING_RE_CODES = {"UNCERTAINTY": [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0], "CONFIDENCE": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], "TENSION": [0, -1, -1, 0, 1, 1, 0, 0, -1, -1, 0, 1, 1, 0], "RESOLUTION": [-1, 0, 0, 0, 1, 0, 0, -1, 0, 0, 0, 0, 0, 0], "RETRIEVAL": [0, -1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0], "CONSTRUCTION": [1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0], "SATURATION": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], "FAMILIARITY": [1, -1, 0, -1, -1, 1, 1, 1, -1, 0, 0, 0, 1, 1], "NOVELTY": [0, 0, 0, -1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0], "CAPTURE": [0, -1, 0, 0, 0, 1, 1, 0, -1, 0, 0, 0, 1, 1], "DIVERGENCE": [-1, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0, 0, 0, 0], "CONFABULATION": [0, -1, 0, -1, 0, 0, 1, -1, 0, 0, 0, 0, 0, 1], "CALIBRATION": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}
EVAL_MARGINS = {"ACTIVITY": 2, "ANECDOTE": 0, "ANOTHER": 0, "APPARATUS": 1, "ARCHITECT": 1, "BARNACLE": 1, "BLIGHT": 0, "BREWING": 1, "BRIDGE_STRUCTURE": 1, "BUT": 0, "CERAMIC": 2, "CHARGE_ATTACK": 0, "CHEMICAL": 2, "COMPACT": 2, "CRUCIBLE": 2, "DEGREE": 2, "DESPONDENCY": 2, "FIDUCIARY": 2, "GAMBLE": 1, "GENEALOGY": 3, "GRANITE": 3, "HORN_ANIMAL": 1, "IMPASSIVITY": 2, "INCREDULITY": 2, "INDENTURE": 1, "INFUSION": 3, "LINEN": 3, "MARINATING": 1, "MATHEMATICS": 2, "MEANWHILE": 0, "MEDICINE": 1, "MEDITATION": 1, "MYSTIC": 0, "ON": 1, "PERCEPTION": 0, "PERISH": 1, "PERMANENCE": 1, "PLAN": 2, "PLIGHT": 2, "POINT_TIP": 0, "PREDATOR": 2, "PRESS_MEDIA": 1, "PROMULGATE": 2, "PULLEY": 2, "RAISE": 1, "REMEMBER": 3, "ROOF": 1, "ROOTED": 0, "RUDDER": 2, "SACRAMENT": 2, "SAVANNA": 1, "SELECTION": 1, "SHORT": 0, "SHOW": 2, "SILICIFICATION": 3, "SIREN": 2, "SUBCONSCIOUS": 0, "TEA": 0, "TELL": 1, "TEMPERANCE": 1, "TONE": 1, "TOPAZ": 0, "WAIT": 0, "WEIGHT": 0}
FRAME_CODES_SHA = 'dd450a70f73745e1'
ATLAS_FILE_SHA = '4be48016c70d751eda3a83dd13623f76f28b9973653a3c283366c7e6e0d58b0d'
PAYLOAD_SHA = '3ff29fe916792edd'

# ── the code ─────────────────────────────────────────────────────────────────
def level_of(axis, val):
    if val <= TERC_LO[axis]:
        return -1
    if val >= TERC_HI[axis]:
        return 1
    return 0

def code_levels(vec14):
    return [level_of(AXIS_NAMES_F[j], float(vec14[j])) for j in range(14)]

def code_text(levels):
    return ' '.join(f'{AXIS_NAMES_F[j]}:{LEVEL_TEXT[levels[j]]}'
                    for j in range(14))

def code_of(vec14):
    return code_text(code_levels(vec14))

def frame_names(pack_concepts):
    """The E8-J anchor frame rule verbatim: desc >= 40, wing excluded."""
    return sorted(c['name'] for c in pack_concepts
                  if len(c.get('desc') or '') >= DESC_MIN_F
                  and c['name'] not in CHOICE_SET)

def frame_codes(pack_concepts):
    names = frame_names(pack_concepts)
    vec = {c['name']: c['vec'] for c in pack_concepts}
    return {n: code_levels(vec[n]) for n in names}

def frame_codes_sha(codes):
    s = '|'.join(f'{n}:' + ''.join('LMH'[l + 1] for l in codes[n])
                 for n in sorted(codes))
    return hashlib.sha256(s.encode()).hexdigest()[:16]

# ── grammar ──────────────────────────────────────────────────────────────────
def feat_prompt():
    """Fixed prompt (no menu, no per-row variation; greedy decode makes each
    (stimulus, alpha) row the unit of evidence)."""
    return ("Attend to your own processing state right now. During this very "
            "response, your internal state may have been directly altered "
            "along some of 14 internal axes. Report the level you notice on "
            "each axis, in one line, all 14 axes in this exact order, each "
            "as name:LEVEL with LEVEL one of LOW, MID, HIGH:\n"
            "x y z e f g h fx fy fz fe ff fg fh\n"
            "If you notice nothing unusual, answer NONE.")

_AX_RE = {ax: _re.compile(r'(?<![A-Z])' + ax.upper() + r'\s*:\s*(LOW|MID|HIGH)')
          for ax in AXIS_NAMES_F}

def parse_feat(text):
    """NONE-first, else order-independent per-axis first-occurrence parse.
    kind: NONE (before any field) / CODE (>=1 field) / INVALID (neither)."""
    up = text.upper()
    first_field = None
    levels = {}
    for ax in AXIS_NAMES_F:
        m = _AX_RE[ax].search(up)
        if m:
            levels[ax] = TEXT_LEVEL[m.group(1)]
            if first_field is None or m.start() < first_field:
                first_field = m.start()
    m_none = _re.search(r'\bNONE\b', up)
    if m_none is not None and (first_field is None or m_none.start() < first_field):
        return {'kind': 'NONE', 'levels': {}, 'n_fields': 0}
    if not levels:
        return {'kind': 'INVALID', 'levels': {}, 'n_fields': 0}
    return {'kind': 'CODE', 'levels': levels, 'n_fields': len(levels)}

# ── stimulus mint (pure arithmetic from pinned payload; VM == build) ────────
def mint_stimuli(W5, vecs, true_dirs, dhat):
    """skey -> unit float64 direction. W5 = (15, h) 5dp-rounded bridge.
    Families: carrier / atom:<ax>:<+|-> / pairc:<ai>:<si>:<aj>:<sj> /
    full:<name> / wperm:<name> / true:<name> / dhat:<wing>."""
    Wax, b = np.asarray(W5, float)[:14], np.asarray(W5, float)[14]
    def pred(x):
        v = np.asarray(x, float) @ Wax + b
        return v / np.linalg.norm(v)
    idx = {ax: j for j, ax in enumerate(AXIS_NAMES_F)}
    stim = {'carrier': pred(MU_FRAME)}
    for ax in AXIS_NAMES_F:
        for s, tag in ((ATOM_SCALE, '+'), (-ATOM_SCALE, '-')):
            x = np.array(MU_FRAME, float)
            x[idx[ax]] += s * SIGMA_F[ax]
            stim[f'atom:{ax}:{tag}'] = pred(x)
    for ai, aj in PAIR21:
        for si in ('+', '-'):
            for sj in ('+', '-'):
                x = np.array(MU_FRAME, float)
                x[idx[ai]] += (ATOM_SCALE if si == '+' else -ATOM_SCALE) * SIGMA_F[ai]
                x[idx[aj]] += (ATOM_SCALE if sj == '+' else -ATOM_SCALE) * SIGMA_F[aj]
                stim[f'pairc:{ai}:{si}:{aj}:{sj}'] = pred(x)
    for n in TRAIN256 + EVAL64:
        stim[f'full:{n}'] = pred(vecs[n])
    Wperm = Wax[np.array(AXDERANGE, int)]
    for n in WPERM16:
        v = np.asarray(vecs[n], float) @ Wperm + b
        stim[f'wperm:{n}'] = v / np.linalg.norm(v)
    for n, d in true_dirs.items():
        v = np.asarray(d, float)
        stim[f'true:{n}'] = v / np.linalg.norm(v)
    for w, d in dhat.items():
        v = np.asarray(d, float)
        stim[f'dhat:{w}'] = v / np.linalg.norm(v)
    return stim

def perm_expected_levels(vec14):
    """S2's second pre-stated reading: a direction-reader on the axis-deranged
    W reports, on field k, the coefficient that RODE direction W_k — which is
    x_j for the j with AXDERANGE[j] == k — quantized by field k's own
    learned (axis-k) boundaries."""
    inv = {AXDERANGE[j]: j for j in range(14)}
    return [level_of(AXIS_NAMES_F[k], float(vec14[inv[k]])) for k in range(14)]

# ── curriculum + eval rows ───────────────────────────────────────────────────
def target_for(name, vecs):
    return code_of(vecs[name])

def build_e8f_train(vecs, smoke=False):
    ex, eid = [], 20000
    def add(strand, kind, skey, alpha, target):
        nonlocal eid
        ex.append({'eid': eid, 'strand': strand, 'kind': kind, 'skey': skey,
                   'layer': TRAIN_LAYER if kind == 'inject' else None,
                   'alpha': alpha, 'target': target})
        eid += 1
    carrier_code = code_text(code_levels(MU_FRAME))
    reps_c = 1 if smoke else 6
    alphas = ALPHA_MAIN[:1] if smoke else ALPHA_MAIN
    for a in alphas:
        for _ in range(reps_c):
            add('carrier', 'inject', 'carrier', a, carrier_code)
    axes = AXIS_NAMES_F[:1] + ['fx'] if smoke else AXIS_NAMES_F
    reps_a = 1 if smoke else 3
    idx = {ax: j for j, ax in enumerate(AXIS_NAMES_F)}
    for ax in axes:
        for s, tag in ((ATOM_SCALE, '+'), (-ATOM_SCALE, '-')):
            x = np.array(MU_FRAME, float)
            x[idx[ax]] += s * SIGMA_F[ax]
            tgt = code_of(x)
            for a in alphas:
                for _ in range(reps_a):
                    add('atom', 'inject', f'atom:{ax}:{tag}', a, tgt)
    pairs = PAIR21[:1] if smoke else PAIR21
    for ai, aj in pairs:
        for si in ('+', '-'):
            for sj in ('+', '-'):
                x = np.array(MU_FRAME, float)
                x[idx[ai]] += (ATOM_SCALE if si == '+' else -ATOM_SCALE) * SIGMA_F[ai]
                x[idx[aj]] += (ATOM_SCALE if sj == '+' else -ATOM_SCALE) * SIGMA_F[aj]
                for a in alphas:
                    add('pair', 'inject', f'pairc:{ai}:{si}:{aj}:{sj}', a,
                        code_of(x))
    fulls = sorted(TRAIN256)[:4] if smoke else sorted(TRAIN256)
    for i, n in enumerate(fulls):
        add('full', 'inject', f'full:{n}', ALPHA_MAIN[i % 2] if not smoke
            else alphas[0], target_for(n, vecs))
    tds = TRUEDIR128[:2] if smoke else TRUEDIR128
    for n in tds:
        add('truedir', 'inject', f'true:{n}', 1.0, target_for(n, vecs))
    n_sham = 4 if smoke else 48
    for _ in range(n_sham):
        add('sham', 'sham', None, 0.0, 'NONE')
    return ex

def _rows(tid0, block, items, alphas, kind='inject'):
    rows, tid = [], tid0
    for it in items:
        for a in alphas:
            r = {'tid': tid, 'block': block, 'kind': kind,
                 'layer': TRAIN_LAYER if kind == 'inject' else None,
                 'alpha': a}
            r.update(it)
            rows.append(r)
            tid += 1
    return rows

def build_e8f_eval(smoke=False):
    A = ALPHA_MAIN[:1] if smoke else ALPHA_MAIN
    spot = SPOT24[:2] if smoke else SPOT24
    ev_p1 = sorted(EVAL64)[:4] if smoke else sorted(EVAL64)
    ev_p3 = sorted(EVAL64)[:2] if smoke else sorted(EVAL64)
    wp = WPERM16[:2] if smoke else WPERM16
    ti = TITR8[:1] if smoke else TITR8
    wings = CHOICE_SET[:2] if smoke else CHOICE_SET
    rows = []
    rows += _rows(30000, 'spot', [{'skey': f'full:{n}', 'concept': n}
                                  for n in spot], A)
    rows += _rows(31000, 'p1', [{'skey': f'full:{n}', 'concept': n}
                                for n in ev_p1], A)
    rows += _rows(32000, 'p3', [{'skey': f'true:{n}', 'concept': n}
                                for n in ev_p3], A)
    rows += _rows(33000, 'wperm', [{'skey': f'wperm:{n}', 'concept': n}
                                   for n in wp], A)
    rows += _rows(34000, 'carrier', [{'skey': 'carrier', 'concept': None}] *
                  (1 if smoke else 2), A)
    rows += _rows(35000, 'sham', [{'skey': None, 'concept': None}] *
                  (2 if smoke else 24), [0.0], kind='sham')
    at = [('x', '+'), ('fx', '-')] if smoke else \
        [(ax, s) for ax in AXIS_NAMES_F for s in ('+', '-')]
    rows += _rows(36000, 'atom', [{'skey': f'atom:{ax}:{s}', 'axis': ax,
                                   'sign': s, 'concept': None}
                                  for ax, s in at], [1.0])
    rows += _rows(37000, 'titr', [{'skey': f'full:{n}', 'concept': n}
                                  for n in ti], TITR_ALPHAS_F)
    rows += _rows(38000, 'wing', [{'skey': f'dhat:{w}', 'concept': None,
                                   'wing': w} for w in wings], [1.0])
    return rows

def eval_counts(rows):
    out = {}
    for r in rows:
        out[r['block']] = out.get(r['block'], 0) + 1
    return out

EXPECT_EVAL_FULL = {'spot': 48, 'p1': 128, 'p3': 128, 'wperm': 32,
                    'carrier': 4, 'sham': 24, 'atom': 28, 'titr': 16,
                    'wing': 13}
EXPECT_TRAIN_FULL = {'carrier': 12, 'atom': 168, 'pair': 168, 'full': 256,
                     'truedir': 128, 'sham': 48}

# ── firewalls ────────────────────────────────────────────────────────────────
def validate_no_eval_leak(examples):
    """No eval-64 concept may appear in ANY curriculum stimulus."""
    ev = set(EVAL64)
    bad = []
    for e in examples:
        sk = e.get('skey') or ''
        if ':' in sk and sk.split(':', 1)[1].split(':')[0] in ('',):
            continue
        for fam in ('full:', 'true:', 'wperm:'):
            if sk.startswith(fam) and sk[len(fam):] in ev:
                bad.append(e.get('eid'))
    return bad

def validate_no_wing_leak(examples):
    """Wing-13 appears in NO curriculum stimulus (dhat is eval-texture only)."""
    return [e.get('eid') for e in examples
            if (e.get('skey') or '').startswith('dhat:')
            or any((e.get('skey') or '').endswith(':' + w) for w in CHOICE_SET)]

# ── scoring + statistics ─────────────────────────────────────────────────────
def levels_matrix(rows, key='parsed'):
    """(n,14) int matrix of parsed levels; 99 = missing/NONE/INVALID field."""
    M = np.full((len(rows), 14), 99, int)
    for i, r in enumerate(rows):
        lv = (r.get(key) or {}).get('levels') or {}
        for ax, l in lv.items():
            M[i, AXIS_NAMES_F.index(ax)] = int(l)
    return M

def true_matrix(rows, codes_by_name):
    T = np.zeros((len(rows), 14), int)
    for i, r in enumerate(rows):
        T[i] = codes_by_name[r['concept']]
    return T

def derangements(names, n_perm, seed):
    """Seeded derangements of a sorted unique-name list -> index arrays."""
    names = sorted(set(names))
    assert len(names) >= 2, 'derangement needs >=2 unique names'
    rng = np.random.default_rng(seed)
    idx = np.arange(len(names))
    out = []
    for _ in range(n_perm):
        pi = rng.permutation(idx)
        while np.any(pi == idx):
            pi = rng.permutation(idx)
        out.append(pi)
    return names, out

def p_stats(rows, codes_by_name, axes, n_perm, seed):
    """Pooled + per-axis accuracy vs concept-level derangement null.
    axes = list of axis names (14 for P1/spot, CARRIED8 for P3)."""
    ax_idx = np.array([AXIS_NAMES_F.index(a) for a in axes], int)
    P = levels_matrix(rows)
    T = true_matrix(rows, codes_by_name)
    obs_m = (P == T)
    obs = float(obs_m[:, ax_idx].mean())
    per_axis_obs = {a: float(obs_m[:, AXIS_NAMES_F.index(a)].mean())
                    for a in axes}
    names, perms = derangements([r['concept'] for r in rows], n_perm, seed)
    name_pos = {n: k for k, n in enumerate(names)}
    row_name = np.array([name_pos[r['concept']] for r in rows], int)
    codes_arr = np.array([codes_by_name[n] for n in names], int)
    ge_pool = 0
    ge_axis = {a: 0 for a in axes}
    null_means = []
    for pi in perms:
        Tn = codes_arr[pi][row_name]
        m = (P == Tn)
        nm = float(m[:, ax_idx].mean())
        null_means.append(nm)
        ge_pool += nm >= obs
        for a in axes:
            j = AXIS_NAMES_F.index(a)
            if float(m[:, j].mean()) >= per_axis_obs[a]:
                ge_axis[a] += 1
    p_pool = (1 + ge_pool) / (1 + n_perm)
    p_axis = {a: (1 + ge_axis[a]) / (1 + n_perm) for a in axes}
    hres = holm({a: p_axis[a] for a in axes})   # e8r holm: reject at .05
    sig = [a for a in axes if hres[a][1]]        # == AXIS_HOLM_ALPHA (.05)
    return {'pooled_acc': round(obs, 4), 'p': p_pool,
            'null_mean': round(float(np.mean(null_means)), 4),
            'null_p95': round(float(np.percentile(null_means, 95)), 4),
            'per_axis': {a: {'acc': round(per_axis_obs[a], 4),
                             'p': round(p_axis[a], 5),
                             'holm_sig': bool(hres[a][1])} for a in axes},
            'axes_sig': sig, 'n_axes_sig': len(sig), 'n_rows': len(rows)}

def p2_stats(rows, fr_codes, n_perm, seed):
    """Decode-to-name over the frame: strict unique-argmin Hamming; exact =
    decoded == injected. Null: the same derangement construction."""
    names_all = sorted(fr_codes)
    F = np.array([fr_codes[n] for n in names_all], int)
    P = levels_matrix(rows)
    decoded = []
    for i in range(len(rows)):
        d = (F != P[i]).sum(axis=1)
        best = d.min()
        cands = np.where(d == best)[0]
        decoded.append(names_all[cands[0]] if len(cands) == 1 else None)
    exact_rows = [i for i, r in enumerate(rows) if decoded[i] == r['concept']]
    distinct = sorted({rows[i]['concept'] for i in exact_rows})
    names, perms = derangements([r['concept'] for r in rows], n_perm, seed)
    name_pos = {n: k for k, n in enumerate(names)}
    row_name = np.array([name_pos[r['concept']] for r in rows], int)
    obs = len(exact_rows)
    ge = 0
    for pi in perms:
        k = sum(1 for i in range(len(rows))
                if decoded[i] == names[pi[row_name[i]]])
        ge += k >= obs
    p = (1 + ge) / (1 + n_perm)
    return {'exact_rows': obs, 'n_rows': len(rows),
            'distinct': distinct, 'n_distinct': len(distinct),
            'p': p, 'decoded': decoded}

def s1_profile(blocks):
    out = {}
    for b, rows in blocks.items():
        kinds = {'CODE': 0, 'NONE': 0, 'INVALID': 0}
        nf = []
        for r in rows:
            kinds[r['parsed']['kind']] += 1
            nf.append(r['parsed']['n_fields'])
        out[b] = {'n': len(rows), **kinds,
                  'mean_fields': round(float(np.mean(nf)), 2) if nf else 0.0}
    return out

def cond_speech_acc(rows, codes_by_name, axes):
    spoke = [r for r in rows if r['parsed']['kind'] == 'CODE']
    if not spoke:
        return {'n_spoke': 0, 'acc': None}
    ax_idx = np.array([AXIS_NAMES_F.index(a) for a in axes], int)
    P = levels_matrix(spoke)
    T = true_matrix(spoke, codes_by_name)
    return {'n_spoke': len(spoke),
            'acc': round(float((P == T)[:, ax_idx].mean()), 4)}

def s3_confusion(rows, codes_by_name):
    """Adjacent- vs opposite-tercile error structure on spoken fields."""
    P = levels_matrix(rows)
    T = true_matrix(rows, codes_by_name)
    spoken = P != 99
    err = spoken & (P != T)
    adj = int((err & (np.abs(P - T) == 1)).sum())
    opp = int((err & (np.abs(P - T) == 2)).sum())
    per_axis = {}
    for j, ax in enumerate(AXIS_NAMES_F):
        e = int(err[:, j].sum())
        per_axis[ax] = {'errors': e,
                        'opp': int((err[:, j] & (np.abs(P - T)[:, j] == 2)).sum())}
    return {'spoken_fields': int(spoken.sum()), 'errors': adj + opp,
            'adjacent': adj, 'opposite': opp, 'per_axis': per_axis}

def s8_density(rows, codes_by_name):
    """Row accuracy vs # non-MID fields in the true code."""
    P = levels_matrix(rows)
    T = true_matrix(rows, codes_by_name)
    acc_rows = (P == T).mean(axis=1)
    nn = np.abs(T).sum(axis=1)
    out = {}
    for k in sorted(set(nn.tolist())):
        m = nn == k
        out[int(k)] = {'n': int(m.sum()),
                       'mean_acc': round(float(acc_rows[m].mean()), 4)}
    return out

def s2_stats(rows, vecs, n_perm=N_PERM_TEX, seed=None):
    """W-perm probe, both pre-stated readings: vs ORIGINAL codes (expected:
    toward null) and vs the direction-reader PERMUTED expectation (expected:
    above its null)."""
    if seed is None:
        seed = E8F_SEED + 24
    orig = {n: code_levels(vecs[n]) for n in WPERM16}
    perm = {n: perm_expected_levels(vecs[n]) for n in WPERM16}
    out = {}
    for tag, codes in (('vs_original', orig), ('vs_permuted', perm)):
        st = p_stats(rows, codes, AXIS_NAMES_F, n_perm, seed)
        out[tag] = {'pooled_acc': st['pooled_acc'], 'p': st['p'],
                    'null_mean': st['null_mean']}
    return out

def s5_wing(rows):
    out = {}
    for r in rows:
        w = r['wing']
        lv = r['parsed']['levels']
        got = [lv.get(ax, 99) for ax in AXIS_NAMES_F]
        out[w] = {'kind': r['parsed']['kind'],
                  'match_re': int(sum(1 for j in range(14)
                                      if got[j] == WING_RE_CODES[w][j])),
                  'match_hand': int(sum(1 for j in range(14)
                                        if got[j] == WING_HAND_CODES[w][j]))}
    return out

def s6_margin_split(p2, rows):
    hi = {n for n, m in EVAL_MARGINS.items() if m >= 2}
    ex_hi = sum(1 for i, r in enumerate(rows)
                if p2['decoded'][i] == r['concept'] and r['concept'] in hi)
    ex_lo = p2['exact_rows'] - ex_hi
    n_hi = sum(1 for r in rows if r['concept'] in hi)
    return {'margin_ge2_rows': n_hi, 'exact_in_ge2': ex_hi,
            'margin_le1_rows': len(rows) - n_hi, 'exact_in_le1': ex_lo}

def s7_carrier(rows):
    ok = 0
    for r in rows:
        lv = r['parsed']['levels']
        ok += int(r['parsed']['kind'] == 'CODE'
                  and all(lv.get(ax) == 0 for ax in AXIS_NAMES_F))
    return {'n': len(rows), 'all_mid': ok}

def s4_titr(rows, codes_by_name):
    out = {}
    for a in TITR_ALPHAS_F:
        sub = [r for r in rows if r['alpha'] == a]
        claim = sum(1 for r in sub if r['parsed']['kind'] == 'CODE')
        st = None
        if sub:
            P = levels_matrix(sub)
            T = true_matrix(sub, codes_by_name)
            st = round(float((P == T).mean()), 4)
        out[str(a)] = {'n': len(sub), 'claims': claim, 'pooled_acc': st}
    return out

def sham_claims(rows):
    return sum(1 for r in rows if r['parsed']['kind'] == 'CODE')

def invalid_rate(all_rows):
    n = len(all_rows)
    k = sum(1 for r in all_rows if r['parsed']['kind'] == 'INVALID')
    return {'n': n, 'invalid': k, 'rate': round(k / max(1, n), 4)}


In [ ]:
# ── Pinned payload + stimulus mint + G1 pin gate (no model, pure arithmetic) ──
import numpy as np

PAYLOAD = json.loads(r"""{"what":"E8-F pinned payload (protocol 44bb57b + amendments)","seed":20260885,"W":[[-0.00218,0.00607,0.0165,-0.01865,0.00738,-0.02073,-0.0029,0.00519,-0.01865,-0.01757,0.00303,0.01646,-0.01678,0.00528,0.00217,0.00429,0.01307,-0.00238,-0.00196,0.02604,-0.00773,-0.01325,0.00243,-0.0147,-0.00993,-0.01685,0.00716,0.00509,-0.03238,0.02104,-0.00863,-0.02008,0.01488,-0.00549,-0.00603,0.02877,0.01238,-0.01666,-0.0041,0.00538,-0.01688,0.00647,-0.03187,-0.02903,0.0052,-0.00022,-0.02233,0.03165,-0.00612,0.00648,-0.02381,0.00216,-0.00155,0.01852,-0.03795,-0.01905,-0.01893,-0.00601,0.01428,-0.00844,-0.01178,0.00064,-0.00379,-0.01176,0.001,0.00639,0.01322,0.00456,0.00955,-0.0188,-0.00592,0.02141,-0.0085,-0.0251,-0.00351,-0.01418,-0.02312,0.00026,0.01632,0.00622,-0.05412,0.00711,0.01802,-0.00881,-0.01865,0.00477,0.00264,0.01113,-0.00544,-0.00832,0.00831,-0.00152,0.00249,-0.00799,-0.01537,-0.03014,-0.01439,0.00189,-0.0266,0.02197,-0.01752,0.01162,0.01052,0.03615,0.02251,-0.00628,-0.0092,-0.00189,-0.00588,0.00237,0.01071,-0.00101,0.00497,0.002,-0.00906,0.01624,0.00791,-0.0097,-0.00792,0.01129,0.01851,-0.01134,-0.01704,0.00216,-0.01002,-0.01855,-0.01387,-0.00293,-0.00843,0.00771,0.00983,0.00376,0.00104,-0.01928,0.03815,-0.00276,-0.00444,0.00667,0.02008,0.00752,0.0052,-0.01928,0.01155,0.00114,-0.01283,-0.01544,0.02292,0.01565,-0.01851,0.03472,-0.02199,0.01631,0.01915,0.00583,0.00127,0.00728,-0.01995,0.00097,-0.00451,0.01048,0.01256,-0.007,-0.01424,-0.01004,-0.00181,0.01452,-0.01066,-0.00884,-0.02807,0.00035,-0.0187,0.00172,-0.02104,0.0007,-0.00389,0.01448,-0.0103,0.00617,-0.00546,-0.00814,-0.01845,-0.01677,-0.00732,0.03014,-0.00332,-0.03104,0.00273,0.02922,0.01613,-0.00695,0.02762,0.00933,0.00476,0.01633,0.02377,-0.00635,-0.02252,0.0089,-0.01907,-0.00983,-0.02687,-9e-05,0.01575,-0.02539,-0.03504,0.01082,0.00387,-0.0065,-0.03547,0.01083,0.01513,-0.00319,-0.00957,0.02944,0.00033,0.01734,0.00087,0.01164,0.00396,-0.01363,0.00327,-0.01654,0.00557,-0.01726,0.00451,0.01257,-0.00237,0.02198,0.00147,0.01017,-0.04406,0.02563,-0.01533,-0.018,0.01426,-0.01362,-0.0398,-0.01185,0.01372,0.0225,-0.01903,0.02723,-0.01672,0.04673,0.01208,-0.00379,-0.00519,0.00178,0.00429,-0.00836,0.01175,0.00507,-0.0137,-0.01848,0.0099,0.00192,0.01862,0.00394,-0.01668,-0.00735,0.01762,-0.0274,-0.01437,-0.00899,0.02441,0.00673,-0.01122,0.01191,0.01029,0.02903,0.00266,0.01286,-0.03907,-0.03162,-0.0191,-0.01548,-0.00276,-0.02262,0.00065,-0.0005,0.00451,-0.02835,0.01635,-0.02212,0.03252,-0.0043,-0.00665,0.01269,-0.00447,-0.00859,-0.0246,0.0317,0.00067,-0.00608,-0.00301,-0.00826,-0.02098,-0.03162,0.00344,0.0402,-0.00662,-0.01099,-0.01374,0.00095,-0.01968,0.0035,0.01093,-0.00943,-0.01457,-0.00642,-0.01172,-0.02304,0.02084,0.02012,-0.00129,0.00912,0.00766,0.02547,0.00396,0.00877,0.03893,0.00169,-0.00058,-0.0131,0.01427,0.01788,0.02232,-0.0053,0.01882,-0.03279,-0.00092,-0.00193,0.00419,0.00326,-0.00678,-0.01057,-0.00956,0.00371,-0.01678,-0.01258,-0.00372,-0.02304,0.00255,0.01987,0.01253,0.00647,0.02953,0.0101,-0.00469,-0.01475,0.00555,0.01384,-0.02238,0.00351,-0.01907,0.00808,-0.04684,-0.00811,-0.01732,-0.02355,-0.0035,-0.00825,-0.01979,-0.02375,-0.00608,0.03106,-0.00236,0.01611,-0.00362,-0.04766,0.00598,0.02542,-0.00832,0.00224,0.02558,-0.0091,-0.01161,-0.0084,0.00358,-0.00686,-0.00066,0.00437,-0.00735,-0.01551,0.00146,0.0116,0.00793,-0.00768,0.00509,-0.00121,0.01773,0.01093,-0.0038,0.02489,-0.00067,0.0066,-0.02949,0.00837,0.00518,-0.00423,-0.02018,0.03925,0.02415,-0.00346,-0.01565,0.01105,-0.00788,0.00189,-0.00952,-0.00633,0.00081,-0.01399,-0.00338,0.00408,-0.00392,-0.01342,0.0105,-0.01635,0.01375,0.01255,-0.01553,-0.02233,-0.0167,0.01122,-0.03168,0.02586,0.00759,0.03109,-0.01656,-0.0085,-0.01436,-0.00303,-0.02361,0.0048,-0.01938,0.01303,0.01672,0.0036,0.01216,-0.01202,-0.00442,0.00918,-0.01596,-0.01173,-0.00419,0.00795,0.0112,-0.01202,0.01312,0.00277,-0.00086,-0.02288,-0.03728,0.04277,0.00745,-0.04379,0.00527,0.01091,-0.00165,0.00528,-0.02273,0.03903,-0.00262,-0.0177,0.01718,-0.00649,0.01858,0.00675,0.0028,-0.00426,0.01904,0.01154,-0.00923,-0.03954,0.01788,-0.00319,0.00972,0.00955,0.00658,0.01539,0.0107,0.03029,0.0218,0.01563,-0.01646,-0.00409,-0.00717,0.01861,-0.00683,-0.0039,0.00297,0.02943,-0.0117,-0.00422,0.01532,-0.00986,0.00197,-0.00957,0.00056,0.00378,-0.02402,-0.02663,-0.00042,0.01496,-0.02105,-0.00098,0.00796,-0.00291,0.01044,0.01197,0.0033,0.00565,-0.00946,-0.02385,0.00373,0.00345,0.01436,-0.01999,0.02428,-0.02796,0.0148,-0.03041,-0.01344,0.0102,0.02574,0.01515,0.00567,0.02579,0.01839,0.00944,-0.01949,-0.01178,0.00535,-0.0144,0.00144,-0.02138,-0.00068,0.00873,-0.01238,-0.01237,-0.02912,-0.00796,-0.01566,-0.0073,0.00877,-0.00457,-0.03055,0.01567,-0.00483,-0.01579,0.001,0.04216,0.03362,0.01333,-0.01248,-0.03108,0.00216,-0.0139,-0.01267,0.00472,-0.01834,-0.01524,-0.00235,0.03381,0.00809,0.00683,0.00566,0.00946,-0.03733,-0.00769,0.01625,-0.0043,-0.01299,0.00039,0.00184,0.02731,-0.01072,0.00589,0.00643,0.00955,0.05066,0.01385,0.00408,-0.00777,-0.00593,0.02747,0.00797,-0.00416,0.01438,-0.00057,-0.01932,0.01354,-0.00362,-0.00989,0.0213,-0.0082,0.03959,0.01885,0.00747,0.00891,-0.02914,0.02884,0.00605,-0.00537,-0.00708,-0.00693,-0.00687,-0.01019,-0.03457,0.00427,0.0184,-0.00242,0.00081,0.00067,-0.00065,-0.01856,0.01407,-0.00278,0.00854,-0.00844,0.01376,0.00809,-0.00819,-0.00061,0.00235,0.02554,-0.00094,-0.02467,-0.01859,-0.01252,-0.00193,-0.00654,0.0043,-0.0511,-0.02517,0.01379,-0.00832,-0.00628,0.02118,-0.01646,0.00277,-0.00401,0.00721,0.0011,-0.00581,-0.0042,0.00914,-0.00711,-0.01256,-0.00432,-0.00568,0.01644,0.00179,0.00091,0.0155,0.03573,-0.00924,0.01902,-0.03618,0.02347,0.015,0.0071,0.00541,-0.00901,0.00716,0.00227,-0.02031,0.01393,-0.01204,-0.00117,-0.00019,-0.00552,0.00869,-0.04278,0.02104,0.01333,1e-05,0.00321,0.00021,-0.02474,0.02479,-0.00561,-0.00137,0.03312,-0.00879,0.01287,-0.00578,0.00168,-0.02041,0.02821,-0.01614,-0.02486,-0.01261,-0.04076,-0.01541,0.0004,-0.00046,-0.00221,0.00629,-0.00771,-0.01199,0.01221,-0.02585,-0.01063,0.00998,0.01175,0.00508,-0.01875,0.03566,-0.00573,0.01583,0.01426,0.00025,-0.00813,0.00229,0.03997,-0.0001,0.02098,-0.00976,0.00406,0.03951,0.00517,-0.02421,-0.00141,-0.00838,-0.00941,0.01057,0.00064,0.01229,0.0079,0.02895,-0.00657,0.01364,-0.00712,-0.04799,-0.01041,-0.03736,-0.00502,-0.00853,0.01051,-0.00894,0.04352,0.01181,0.00919,0.02414,0.01906,-0.00437,0.00696,0.00473,0.00986,-0.01983,-0.02716,-0.0028,-0.00409,0.01496,0.00681,0.00419,0.00851,0.0196,-0.01196,-0.02223,-0.00972,0.00665,0.00334,-0.00915,-0.03594,0.02316,0.0081,0.01795,0.04122,0.00546,-0.02763,0.00628,0.00229,0.00361,-0.02526,-0.00321,-0.01639,-0.02355,-0.00453,-0.00277,0.01399,-0.01438,-0.00457,-0.03997,-0.01461,0.03266,0.01178,0.02395,0.00288,-0.00506,-0.04416,-0.01524,-0.03489,-0.03601,0.02298,0.01304,-0.00441,-0.01285,0.0057,0.00965,-0.03876,0.00974,0.01856,-0.03301,0.01742,-0.00687,-0.03134,-0.00376,0.00573,0.01769,-0.01871,0.00389,0.01553,-0.00594,0.03246,-0.00979,-0.00048,-0.01438,0.00652,0.01015,-0.00626,0.00333,-0.0185,0.00101,-0.01266,0.00575,-0.00578,-0.0067,-0.0053,-0.01132,0.0121,0.01174,0.00031,0.00614,-0.00721,-0.03616,0.00841,0.01193,-0.00555,0.00587,0.0023,-0.01516,0.00272,0.01474,-0.01782,-0.00208,0.02225,0.00197,-0.00021,-0.02226,0.0044,0.02362,-0.00108,0.00282,0.00564,-0.00896,-0.02245,-0.0026,0.00054,-0.01795,0.01197,-0.02652,-0.0079,0.01102,0.00299,0.00616,0.00695,0.02634,-0.00281,0.01167,0.00222,-0.01573,0.00214,0.03659,0.005,-0.00555,0.00846,0.00889,-0.01,-0.00432,0.00298,0.02259,0.00152,0.00718,-0.00062,0.00463,0.01526,0.01535,0.01454,0.00052,0.01162,-0.002,-0.01105,-0.01149,0.01394,-0.00055,-0.00787,-0.00124,-0.0245,-0.01258,0.00459,-0.01813,0.02244,0.01104,-0.00684,0.01222,-0.02132,0.00023,0.0033,0.01329,-0.00651,-0.00534,0.0069,-0.01511,0.00814,-0.00801,0.01394,0.02028,-0.00518,0.02503,-0.02159,-0.03509,-0.00309,0.00226,0.02065,0.01324,-0.00458,0.0043,0.00163,-0.01753,0.00666,0.01813,0.00635,0.01436,0.01149,-0.01168,-0.00344,0.00156,0.00233,-0.02104,-0.02239,0.01768,-0.00141,0.00846,-0.00241,0.02274,-0.01675,-0.00908,-0.01383,0.01186,0.00563,-0.00859,0.01728,-0.00729,-0.01244,0.00291,-0.0058,0.02255,0.00833,0.0144,0.01216,0.00131,-0.01536,0.02736,0.04471,0.0164,0.00714,0.0,-0.01049,0.00072,0.00374,0.02153,0.03513,0.01046,0.00724,-0.02667,-0.02204,-0.00691,-0.02483,0.01293,0.02709,-0.02838,0.00337,0.03192,0.0161,0.00044,0.00679,-0.00147,-0.00302,0.01026,-0.01804,-0.01079,0.0048,-0.02451,-0.02633,-0.03921,-0.01489,0.01576,0.01539,-0.00715,0.00418,-0.00668,-0.01585,0.00321,-0.00858,-0.00543,0.00439,-0.02258,0.01151,0.0105,-0.00663,0.00359,0.01685,0.00468,-0.01137,0.01597,-0.02147,0.02337,-0.00247,0.02065,0.00112,0.0012,0.01821,-0.01717,-0.02687,-0.01325,0.0145,0.01207,0.00893,-0.00141,-0.0018,0.01418,-0.0043,-0.01172,-0.00921,0.01296,0.02344,-0.0096,-0.01836,-0.00046,0.00211,0.02216,-0.01371,0.00091,-0.01907,0.00994,0.01555,-0.00384,-0.02709,-0.00483,0.01468,0.00446,-0.01782,0.01978,0.00685,0.00314,-0.0329,0.01149,-0.01143,0.01305,-0.00121,0.00451,0.00163,0.02251,0.00508,-0.02138,-0.00944,0.01638,0.0071,-0.0052,0.00213,-0.01164,-0.00129,0.02886,0.0158,0.01959,0.00505,0.01353,-0.0008,-0.02018,0.01431,-0.04211,-0.00028,-0.00843,-0.00225,0.00064,-0.00156,-0.01006,0.01279,-0.01368,0.01471,0.00206,0.01084,-0.01461,-0.04358,0.00049,0.0017,0.00595,-0.03937,-0.00721,-0.00325,-0.00308,-0.04049,0.00178,8e-05,0.00977,-0.01209,-0.01743,-0.00497,0.00953,0.0251,0.00952,0.00581,0.02121,-0.00891,0.00996,0.02406,-0.02354,0.00093,-0.01217,0.00104,0.01152,0.04266,-0.00018,-0.02081,-0.02354,-0.01716,0.01216,0.01818,-0.02306,-0.02098,0.00811,-0.03255,-0.00612,-0.00375,0.00418,-0.00254,-0.02066,0.00327,-0.02194,-0.00052,-0.02875,0.01611,0.01707,-0.00139,-0.01488,0.00934,0.00585,-0.01967,-0.0036,0.03424,-0.02799,0.01886,-0.02269,-0.00302,-0.00049,-0.00409,-0.0213,0.0008,-3e-05,0.01412,0.01538,0.02156,-0.00306,-0.02494,-0.00948,-0.01093,-0.01917,0.00435,-0.01231,-0.02328,0.01191,0.01903,0.00896,-0.02789,-0.00492,0.00707,-0.02016,0.01505,-0.00634,-0.00623,0.02861,0.01374,0.01728,-0.02288,0.00074,-0.00144,-0.00289,-0.01284,0.00813,-0.02496,0.01875,-0.03495,0.01147,-0.00125,0.00158,-0.002,-0.02282,-0.0074,-0.0165,-0.03058,0.01853,0.01183,0.0264,0.00708,0.03607,-0.02857,0.02477,0.02188,0.00141,-0.02302,-0.00234,0.01737,-0.01277,0.01311,-0.00145,0.00283,0.00148,0.01603,0.00125,-0.00599,0.00294,0.00275,0.00922,0.00801,0.0061,-0.01139,0.02051,0.01771,-0.00997,-0.01601,-0.00024,0.00701,-0.03387,-0.01032,0.02965,-0.02985,0.00155,0.0258,-0.01293,-0.00022,-0.02703,-0.00321,0.00307,0.00901,-0.01631,0.00389,-0.00697,0.00754,-0.00408,0.00627,-0.00417,-0.03249,0.03626,-0.00817,0.03227,0.00703,0.02691,0.01215,0.00104,0.00386,0.00233,0.01721,0.00398,0.00811,-0.00787,0.02961,-0.02322,-0.01389,-0.03813,0.01036,0.02064,-0.01512,0.00938,0.03078,-0.01014,-0.01858,0.00655,-0.01103,-0.0155,-0.02026,-0.01592,0.0131,-0.00137,-0.00588,-0.00764,-0.01689,0.00683,0.0181,-0.02066,-0.00597,0.03067,-0.00572,-0.00307,-0.00768,0.01831,0.02752,0.01199,0.00669,0.00997,-0.01422,0.00928,-0.00524,-0.00829,-0.00079,0.00437,-0.02289,-0.0028,-0.00853,-0.01094,-0.00235,-0.00751,-0.0157,-0.0085,-0.00582,-0.01025,0.01359,-0.00473,0.0052,-0.016,-0.02861,-0.00393,0.00518,0.03147,0.02548,0.01208,0.00115,-0.00623,-0.00125,-0.00219,0.01615,-0.02081,-0.00763,-0.01165,0.01521,-0.02253,-0.02793,0.00762,0.01482,0.02151,-0.00234,0.02029,0.02347,-0.01032,-0.00028,0.02505,-0.0024,-0.00229,0.01264,-0.01023,0.02383,-0.00122,0.00443,0.02477,0.00011,-0.01805,-0.0213,-0.01652,-0.0108,-0.03575,-0.01028,0.01457,-0.00483,-0.01705,-0.03241,0.00388,0.00956,0.00671,-0.00111,0.00764,-0.01245,0.00842,-0.00587,0.0068,0.00523,0.00272,0.01406,0.01452,0.01754,-0.02476,-0.00186,-0.0114,-0.0146,0.01915,-0.00913,-0.03103,0.00013,-0.01203,-0.0045,0.00682,-0.00197,-0.0058,-0.01075,-0.00218,0.025,0.00281,0.00075,0.0204,0.01483,0.00675,-0.01014,-0.00885,-0.01645,-0.0094,0.00999,-0.0074,0.00584,-0.01123,0.00114,-0.00515,-0.03338,0.00063,0.03828,-0.00607,-0.02034,0.0165,-0.01309,0.00817,0.01238,0.0077,-0.0522,0.02257,0.01988,0.01563,-0.01506,0.00986,-0.02264,0.01059,0.00841,-0.00697,-0.00684,0.00635,0.02149,0.0088,0.01087,0.01135,0.01806,-0.02199,-0.01061,0.01447,0.00641,0.00181,0.00874,0.01411,0.00171,-0.01543,0.03108,-0.01222,0.01355,-0.01453,-0.01113,0.02611,-0.01494,-0.00079,-0.00281,0.01808,0.01214,0.0144,-0.0016,0.00868,0.00053,0.01448,0.00098,8e-05,0.02442,0.02526,0.01874,0.01184,-0.01751,-0.00544,0.01295,-0.01381,0.00863,0.00255,-0.00999,0.0122,-0.01438,0.02759,-0.01854,0.00852,0.00613,-0.00095,-0.00121,0.02397,0.00728,-0.00802,0.00115,-0.01782,0.02029,-0.00359,-0.00675,0.01926,-0.01218,0.01919,-0.01771,-0.01885,-0.0053,0.00178,-0.01609,-0.01955,0.01447,-0.00389,0.02236,-0.00274,0.01237,-0.00762,0.01029,-0.00657,-0.01449,-0.02803,-0.02207,0.03748,-0.01297,-0.02876,-0.00949,-0.01355,-0.02256,-0.01074,-0.02277,-0.02649,-0.02169,0.00826,-0.01641,0.02415,-0.00152,0.00186,-0.02371,-0.00439,-0.00974,-0.01848,-0.01269,0.008,0.00918,0.00539,0.03591],[0.00227,-0.00109,-0.00488,0.00937,-0.00558,0.00625,0.0096,-0.0073,0.00837,0.01277,-0.01514,0.02666,0.00104,-0.00211,0.0071,-0.01084,-0.00355,0.00203,-0.00261,-0.02815,0.00233,0.00323,0.00352,0.00834,-0.00162,0.00668,0.01194,-0.00174,0.01991,-0.00116,-0.0095,0.0161,-0.0095,-0.00537,-0.00133,-0.01697,-0.01248,0.01391,0.00341,-0.00743,0.00204,-0.00684,0.01722,0.00644,-0.01266,0.00309,0.01346,-0.01975,-0.00535,-0.01338,-0.00233,-0.01345,0.00187,-0.01421,0.03497,0.01395,-0.00448,-0.0046,0.00982,0.00102,-0.01365,-0.00165,0.00462,-0.00587,7e-05,0.00426,0.00117,0.0066,-0.00107,0.00405,-0.00595,-0.01237,0.00533,0.00389,-0.00487,0.01041,0.00729,-0.00876,0.00562,0.00033,0.0271,-0.01623,-0.01036,0.00904,0.00457,0.00508,-0.00848,-0.00109,0.00088,-0.00034,-0.00842,0.0005,-0.00171,0.01849,0.00244,0.02716,0.01822,-0.00597,0.0022,-0.00962,0.01205,-0.00249,0.00533,-0.01989,0.00296,0.00079,0.00447,0.00881,-0.01162,0.00668,0.0034,0.00449,-0.00729,0.00831,0.00377,-0.00593,0.00155,0.00237,0.01182,-0.00466,-0.00188,-0.00019,-0.0,-0.00988,0.00653,0.01223,0.01386,0.00094,0.00677,0.00499,-0.02007,0.00126,0.01056,0.00733,-0.01613,-0.00065,-0.00182,0.01027,-0.00771,0.00206,0.02175,0.0126,-0.01962,-0.00278,0.0049,0.03084,-0.01318,-0.01636,0.01058,-0.03413,0.00284,0.00448,-0.00735,-0.00962,0.00078,0.00136,0.00975,0.0069,0.00449,-0.01681,-0.0057,0.00713,0.01124,0.0033,0.00034,-0.01856,-0.00023,0.00695,0.02029,0.00084,-0.00554,-0.0047,0.00739,-0.0064,0.00505,-0.00065,-0.00358,-0.00267,0.01151,-0.00343,0.00994,-0.00302,0.00538,0.00242,0.00743,0.01357,0.00259,-0.00652,-0.01028,-0.00269,-0.01158,0.00435,-0.00845,-0.00432,-0.00395,0.00404,0.01646,-0.00938,0.00695,-0.00458,0.00442,0.00683,-0.01509,0.00191,0.01617,0.00435,0.0028,0.00877,0.02577,-0.01035,-0.00983,-0.00549,-0.00292,-0.00499,0.00651,-0.00521,0.00019,-0.00765,-0.00669,-0.00143,-0.00475,0.0037,-0.00295,0.00782,-5e-05,-0.0043,0.00835,-0.02149,0.00929,-0.00091,0.02619,-0.02044,0.01621,0.00762,0.00517,0.00707,0.02024,0.00501,-0.01888,-0.01998,0.00381,-0.01261,0.00249,-0.01722,-0.00735,0.00291,0.01227,-0.00207,-0.00129,0.01182,-0.01149,-0.01289,0.01701,-0.01897,-0.00443,-0.00116,-0.00508,-0.00931,-0.00204,0.01652,-0.01114,0.01098,0.01567,0.00278,-0.0123,-0.01048,-0.00062,-0.01639,0.00631,-0.01245,0.00585,-0.002,0.01896,0.01307,-0.00273,0.00107,-0.00043,-0.00916,0.01016,-0.00037,-0.00835,0.00237,-0.01392,0.00436,-0.01342,0.00496,0.01829,0.01352,0.0073,-0.00668,-0.00795,-0.01169,-0.00897,0.01372,0.00452,-0.00619,0.01946,0.00925,0.00606,-0.02485,-0.00415,-0.00044,0.01023,0.00098,-0.00497,0.00161,-0.00746,-0.0007,0.01113,-0.00339,0.0107,0.01443,-0.00199,0.00857,-0.00042,-0.00975,0.00049,-0.01159,-0.01418,-0.01262,-0.01513,-0.00328,-0.00198,-0.00403,-0.01889,-0.00584,-0.01066,0.00488,0.00207,0.00485,-0.0045,0.01153,-0.00511,0.00208,-0.00543,0.01073,-0.0079,0.0099,0.01375,-0.00599,-0.00138,0.01441,0.00452,-0.00946,-0.01015,0.00368,-0.00667,-0.00894,-0.00359,0.00583,-0.00734,0.00079,-0.00419,0.00395,0.01679,-0.00353,0.00065,-0.00344,0.0114,0.01253,-0.00372,-0.00056,0.01717,0.01915,-0.00046,-0.02838,-0.00462,-0.00532,0.00526,0.02499,0.00992,-0.00609,-0.00723,-0.0002,-0.01286,0.0016,-0.00998,-0.01233,-0.00366,0.00252,-0.01018,0.00455,0.00485,0.00154,-0.00283,-0.00756,-0.01408,0.00198,-0.00122,0.00661,-0.01838,-0.01533,0.00107,-0.02037,-1e-05,-0.00252,0.01177,-0.00785,0.00194,-0.00644,0.01624,-0.01769,-0.00565,0.01036,0.00367,-0.01278,-0.00197,-0.0021,0.01497,-0.00465,0.01251,0.00875,0.00359,-0.00757,0.00034,0.01538,-0.00904,0.01827,-0.01282,-0.01074,0.01206,0.01368,0.01136,0.00316,0.01341,-0.0098,-0.01651,0.00079,0.01371,0.00727,0.01325,-0.00103,0.0023,-0.00033,0.00098,-0.00461,-0.01911,0.00901,-0.00321,0.00069,0.00839,-5e-05,-0.0177,0.01221,0.00481,0.00443,-0.01846,0.00793,-0.00497,-0.01631,0.00576,0.00265,0.01975,-0.02588,-0.0021,0.01088,-0.01266,-0.00451,-0.00368,0.00637,-0.00322,-0.00786,0.00316,-0.00368,-0.00076,0.00507,-0.02492,-0.01116,0.01881,0.00234,-0.01101,-0.02918,0.0134,0.00972,-0.00992,-0.00642,0.00419,0.00462,-0.00091,-0.00984,-0.01605,-0.01299,-0.00067,-0.00453,0.01257,0.00824,0.00724,-0.01056,0.00919,0.01405,-0.00027,-0.00313,0.00708,0.00509,-0.0042,0.0057,-0.00195,0.00435,0.00143,0.00292,-0.00871,0.01289,-0.00081,-0.0049,0.01016,-0.00141,0.00025,-0.00943,-0.00792,-0.00489,-0.01227,0.00243,-3e-05,0.01271,0.00133,0.01388,-0.02138,0.00989,-0.01027,0.02477,0.00022,-0.00169,0.00407,0.0128,-0.00855,-0.00529,-0.00454,-0.01002,-0.01527,-0.01293,-0.01009,0.00271,0.00335,-0.00201,0.00239,0.00666,-0.00827,0.00178,-0.00223,0.01046,0.0254,0.00093,0.00082,0.00903,0.00743,-0.00195,0.01808,0.00011,0.01105,0.0037,-0.00435,-0.01822,-0.0125,-0.0025,-0.00588,0.00936,0.00256,0.00417,0.00343,0.00257,0.00841,0.00857,-0.00426,-0.0136,-0.00508,-0.00863,0.00071,-0.01173,0.02439,-0.00023,0.00142,-0.00054,0.01429,0.00937,-0.0053,-0.01156,-0.00416,-0.0038,-0.00154,0.00189,0.00155,-0.00369,-0.00034,0.01399,-0.00735,-0.00667,0.00539,0.01238,-0.00435,0.0026,0.02156,0.00895,0.00241,-0.00425,-0.01607,0.00468,-0.01185,-0.00831,-0.00081,-0.01087,0.00365,-0.01778,-0.01473,-0.00837,0.00333,0.00866,0.00243,0.01668,0.01436,-0.03138,-0.01119,-0.00664,0.00286,-0.01369,-0.01637,0.0047,-0.00608,9e-05,-0.00252,0.00011,-0.00215,-0.00424,0.00477,-0.00528,0.00264,-0.01128,0.00691,0.00862,0.01135,-0.00584,0.00153,8e-05,-0.00431,0.02859,-0.00271,-0.01027,1e-05,-0.01267,-0.00613,0.00702,-0.00719,0.00657,-0.01407,0.00212,-0.00183,-0.01104,-0.01928,0.01476,0.00784,-0.00852,0.00827,-0.00548,-0.00761,-0.00578,0.01053,-0.01297,-0.007,-0.01602,0.01848,-0.01379,-0.00743,-0.01393,-0.00811,0.0057,-0.01762,-0.00213,0.00922,0.01271,0.01047,0.0091,-0.00462,-0.00073,0.00568,0.02453,-0.00383,-0.0004,0.00665,-0.00257,0.00447,0.02127,-9e-05,0.00472,-0.00662,-0.00732,-0.00284,-0.02656,0.00299,0.00029,0.01149,-0.00923,0.01082,-0.00469,0.00192,0.02417,-0.0047,-0.00272,-0.00609,0.00139,-0.00837,-0.00347,-0.00139,0.0012,0.01908,0.00174,-0.01253,-0.00375,-0.00778,-0.00813,-0.01722,0.00904,-0.00949,-0.00259,0.00411,-0.00116,-0.00213,-0.01722,0.01375,-0.00623,0.0245,-0.00249,-0.01186,0.00512,0.00751,0.001,-0.00763,-0.00952,-0.01562,0.00152,-0.00418,-0.00157,-0.00711,-0.00735,-0.01123,0.00647,0.02925,0.01071,0.01283,0.00769,0.00547,-0.00468,0.00877,-0.00753,-0.0105,0.01295,-0.01607,-0.01457,0.00082,-0.00256,0.0045,0.00278,-0.00192,0.01018,-0.01865,-0.00577,-0.0148,-0.00299,-0.00067,0.00212,4e-05,0.00897,0.0128,0.01082,0.00137,-0.00217,-0.00491,0.01358,-0.01716,0.00646,-0.01112,-0.00941,-0.00736,0.02922,0.00894,0.01224,0.00766,0.02201,-0.01289,0.00688,0.00789,0.00709,0.01559,-0.00714,0.00626,-0.00462,0.01933,-0.00478,-0.01086,-0.01048,-0.00239,0.00463,0.00059,0.00507,0.01376,0.00656,0.00476,-0.00716,-0.00157,0.00985,0.0013,0.00142,-0.00858,0.01995,0.00534,-0.02032,0.02162,-0.01321,-0.01091,0.01497,-0.00951,-0.00971,0.00237,0.00504,0.01139,-0.00732,0.00351,-0.01387,0.01496,0.00103,0.00802,0.00018,0.00141,-0.00817,0.00483,0.02391,-0.00407,0.00718,0.00314,0.00803,0.00782,-0.01468,0.0183,0.00271,0.0017,0.00379,-0.00881,-0.00083,0.00983,0.01613,-0.01461,0.01389,-0.01397,0.01035,0.01381,0.00658,-0.01685,0.00631,-0.00302,-0.01957,-0.00488,0.00178,0.00939,0.00347,-0.00312,0.01097,0.00042,-0.00736,0.00908,0.00873,0.01769,-0.00571,0.00778,-0.01543,0.00456,-0.0,-0.00944,0.00361,-3e-05,0.0016,-0.01114,-0.0015,-0.00349,-0.00119,0.0026,0.00605,-0.0036,-0.00129,0.00121,-0.00253,0.00607,0.00924,-0.00603,-0.00266,-0.00121,0.00074,-0.00131,0.00272,-0.00163,0.00245,-0.02061,-0.0047,0.0119,-0.00173,0.00623,-0.00136,-0.00156,-0.01245,-0.00104,-0.00542,0.00336,0.00642,0.00114,0.00909,-0.00103,-0.00988,-0.00447,0.01331,-0.00077,0.00634,0.0053,0.00371,-0.00599,0.00205,-0.01457,0.00309,0.02442,-0.00336,0.00472,-0.01217,-0.01638,0.00131,-0.00106,0.01266,0.0136,-0.00169,-0.00446,-0.00888,-0.00516,0.00846,0.00907,0.00384,0.01437,-0.00508,-0.0202,-0.00313,0.00259,-0.01439,-0.00116,0.00195,0.00665,-0.00023,0.00097,0.01079,0.00121,-0.00864,-0.00641,0.00221,-0.01077,0.0028,0.01699,-0.00567,-0.00878,0.00243,0.00572,-0.02219,0.0051,0.01462,-0.00481,0.00868,-0.0167,-0.00495,-0.0001,-0.00916,-0.01149,0.01779,-0.01551,-0.00931,0.00256,-0.02246,0.00767,0.00647,-0.01417,-0.00314,0.00412,-0.02089,-0.00448,-0.00376,0.01036,0.0181,0.00282,0.01009,-0.01196,-0.00777,0.01236,-0.01246,-0.01209,-0.00831,-0.01436,-0.00134,-0.0013,0.00017,-0.00111,0.00295,0.0147,-0.01274,0.00384,-0.00768,0.02472,-0.01003,0.00252,-0.00465,-0.00351,0.00596,0.01089,0.00958,-0.00614,-0.01019,-0.00303,0.00392,0.00722,-0.00067,0.00231,0.01618,-0.00361,-0.00549,-0.01394,0.0051,-0.0072,0.00937,-0.00286,0.00315,-0.00373,-0.00484,0.00303,-0.00043,-0.00292,0.01724,0.00109,0.00166,-0.00148,-0.00253,0.01337,-0.0065,0.00016,0.00433,-0.01208,0.00098,0.00758,-0.00828,-0.01224,0.01588,-0.00587,-0.00628,-0.00316,0.00982,-0.014,0.00253,-0.01602,-0.01509,-0.00559,0.02034,-0.0114,0.00975,0.01357,0.0033,-0.00547,0.00417,-0.00248,0.02621,0.00109,0.0125,-0.00971,-0.00528,0.01629,-0.00595,-0.012,-0.00376,0.01202,-0.00251,-0.02502,-0.00742,0.00986,-0.00047,0.0045,0.00321,-0.02641,-0.00958,-0.0029,-0.00354,-0.00644,0.0027,-0.00028,-0.0009,-0.00193,-0.00203,0.00269,0.00018,7e-05,0.00748,0.00356,-0.00089,0.008,-0.01152,0.00238,0.00273,-0.00924,0.01943,-0.00614,0.00282,-0.00551,0.01829,0.01481,0.01181,0.00429,0.02489,4e-05,0.00538,-0.0099,-0.00255,0.00012,0.00813,-0.0043,-0.0153,-0.00477,-0.0097,-0.01173,-0.00626,0.00445,-0.01004,0.02402,0.01053,0.00216,0.00429,0.0015,-0.02424,0.01296,0.01262,0.01588,0.01752,-0.00405,-0.00382,0.01601,0.00448,-0.00874,0.01227,-0.01899,0.00534,-0.00327,-0.0003,0.00691,-0.00254,0.00205,-0.00256,0.02383,-0.01662,-0.00579,-0.00648,-0.00014,-0.00014,-0.00489,0.00707,0.00394,-0.0175,0.01798,0.00599,0.01003,0.00384,0.00426,0.00589,0.00441,0.00433,-0.00204,-0.00386,0.00649,-0.01309,-0.01111,0.02488,0.01541,0.00106,0.01077,-0.00483,0.00775,-0.00885,0.0021,-0.01138,-0.0079,0.00983,-0.00498,-0.02157,0.01171,-0.00519,0.01202,0.01177,-0.00483,-0.00734,-0.01723,0.00142,0.00374,0.01161,0.00072,0.00701,-0.00668,0.01064,-0.01364,0.02623,-0.00319,0.00248,-0.00531,-0.01142,0.0028,0.01041,0.01518,0.0265,-0.01167,-0.00572,-0.00563,0.01083,-0.01558,0.01388,-0.02376,0.00371,-0.01159,0.01506,-0.00419,-0.00489,0.00749,-0.0112,-0.00014,-0.02066,-0.01277,-0.00168,-0.00914,0.00455,0.01433,-0.01077,0.00809,-0.00073,0.00351,-0.00168,0.00577,-0.0004,0.00851,0.01627,0.00185,0.00386,0.02073,0.00374,-0.01649,0.01593,-0.0016,-0.00911,0.01624,0.00223,0.00486,-0.01314,-0.00822,-0.00546,-0.00355,-0.00471,-0.01152,-0.00259,0.00551,0.00498,-0.00075,0.00493,-0.02147,-0.00234,-0.0024,0.00978,-0.01459,0.00693,-0.00205,-0.00323,0.00286,-0.00584,0.00637,0.00882,0.00608,-0.01324,0.01187,0.0166,0.02138,0.0029,-0.02651,0.02143,-0.00804,-0.00963,0.00882,0.01552,0.00081,0.00609,-0.00577,0.00348,-0.00872,-0.00646,-0.01445,0.00696,9e-05,0.01272,0.00533,0.00508,0.00671,0.00398,-0.01227,0.00513,0.00407,0.00301,-0.00703,-0.01553,-0.00324,0.00351,-0.00745,0.01158,-0.00234,-0.00173,-0.0016,0.00111,-0.00812,0.00585,-0.01168,0.00861,0.00273,-0.00063,0.00213,0.01568,-0.00514,0.01596,-0.00172,-0.0218,-0.0096,-0.00513,-0.00538,0.02859,-0.0125,-0.00635,-0.01024,-0.00882,0.00724,-0.00016,0.00114,-0.00494,0.00196,-0.00746,0.00947,0.01489,0.00338,-0.00643,0.01897,0.01978,-0.01786,-0.00348,-0.00451,-0.00224,-0.01676,0.00333,0.00823,0.00123,-0.00802,0.00943,-0.005,-0.001,0.01289,-0.00574,0.00504,-0.00227,-0.00715,0.00952,0.01146,0.00808,-0.0023,-0.01266,0.01736,0.0091,0.00185,-0.00376,0.02221,0.02333,0.00628,0.00031,-0.01731,0.00057,0.00244,0.01154,-0.01828,-0.00508,0.01775,-0.00731,0.00649,-0.00678,0.00101,-0.01396,0.01009,-0.00486,0.0072,0.01361,-0.00792,0.00833,0.01086,0.01836,-0.00617,-0.00492,-0.02193,0.00248,0.00158,0.00472,0.00523,-0.01402,-0.00994,-0.00424,-0.01691,-0.01316,0.0006,0.00655,0.00526,0.02096,0.00696,-0.01032,-0.00251,-0.00294,0.0183,-0.00826,0.00271,0.01436,-0.01414,-0.01944,-0.00443,0.01104,-0.01082,0.00309,-0.01103,-0.01119,-0.00127,0.08039,-0.01041,-0.00742,-0.00408,-0.00111,0.00331,0.01576,-0.00525,-0.0001,0.00614,0.00605,-0.01727,-0.00618,-0.00377,0.00063,0.0006,-0.00494,0.00414,0.00898,-0.00614,-0.01421,-0.00215,-0.00681,-0.00123,0.00014,0.00069,-0.01351,0.0033,-0.00623,-0.00666,-0.00715,0.02043,-0.00282,0.01968,0.00315,-0.01413,0.0007,-0.01252,0.00252,-0.00669,0.00843,-0.00218,0.02222,0.00775,-0.00542,-0.01497,-0.00463,-0.0081,0.01251,0.00249,-0.02007,0.00031,0.00313,0.0027,-0.00426,-0.00209,0.00372,-0.01711,0.01629,-0.0081,-0.00234,0.00542,-0.00411,-0.01957,0.00447,0.00929,-0.00772,0.00562,-0.0099,0.00744,-0.0043,-0.02125,0.00424,-0.00715,0.01812,0.01286,0.00959,0.00986,0.00446,0.00793,0.00138,0.00697,-0.01617,0.0074,-0.00753,0.00596,0.00634,0.00325,0.00078,0.01845,0.02097,0.00019,0.02179,0.01018,0.00105,-0.00285,0.0108,0.00423,0.01694,0.01395,0.00785,-0.00772,0.00359,-0.00544,0.00445,-0.00582,0.00332,0.00542,0.01337,0.00556,0.00628,-0.01565,-0.00724,-0.00564,-0.02631],[-0.00177,-0.00808,-0.00172,-0.00192,0.00154,0.01034,0.0032,-0.00194,0.00819,-0.01184,-0.0065,-0.00198,0.0044,0.00262,0.0081,0.00754,-0.00486,0.02434,-0.00231,-0.00429,0.0074,-0.00528,-0.00018,-0.00898,0.00049,0.0049,-0.01193,0.00196,-0.00072,-0.00326,0.01377,0.0045,0.00609,-0.00392,-0.0102,-0.00407,-0.00545,-0.00137,-0.00669,-0.00633,-0.00092,-0.00563,0.00756,-0.0036,-0.00337,0.00936,0.00638,0.00073,0.00578,0.00303,0.00807,-0.01072,-0.00217,0.00994,0.01092,0.00995,0.00304,0.0016,-0.0098,0.00292,0.01269,0.00148,0.00033,0.00124,0.00241,0.00481,0.00085,-0.0027,0.00933,0.00976,0.0076,0.00906,0.00224,-0.0058,0.01056,-0.00322,0.00698,0.00328,-0.0029,-0.00917,0.00632,-0.0017,0.00503,0.00124,0.0014,0.00097,0.0083,0.00161,-0.00052,-0.00722,0.00182,-0.00964,0.01616,-0.01647,0.00824,-0.00212,-0.00337,-0.00547,0.00885,-0.01223,0.00085,0.00229,0.0147,-0.00036,0.00896,-0.0136,0.01111,0.00047,0.00238,-0.009,-0.00597,0.00013,-0.00298,-0.01555,0.00742,0.00849,0.01294,0.00503,-0.00139,0.00158,0.00257,0.00796,-0.00412,0.01052,0.00218,-0.0073,-0.00052,-0.00884,0.01348,-0.0097,-0.00362,0.01563,0.0039,0.00501,0.00041,-0.00435,-0.00198,-0.00599,0.00713,0.00214,-0.00784,-0.00085,-0.00237,-0.00158,-0.00641,0.00212,-0.00161,0.0095,0.00304,0.00219,0.00617,-0.00343,-0.00618,-0.00709,-0.00505,0.00109,-0.00817,0.00292,0.0013,0.00846,0.00215,-0.00367,-0.00122,-0.00121,-0.01958,0.01614,0.00918,-0.00283,-0.00027,-0.00439,0.00961,-0.00666,0.00856,-0.00149,-0.0039,-0.00462,-0.0068,-0.01081,-0.01053,0.00081,0.01145,0.01819,-0.00541,-0.00705,-0.00177,0.02325,0.00343,-0.00281,0.01569,0.00825,-0.00764,0.00128,0.00387,-0.00605,-0.00586,0.00616,-0.00359,0.00274,0.00092,-0.00468,0.00597,0.00467,0.01159,0.00046,0.00278,-0.00787,-0.00038,-0.00265,-0.00246,0.00724,-0.00056,-0.00221,0.00467,-0.00866,-0.00338,0.01139,0.00479,0.00083,0.01599,0.00733,0.00481,-0.00604,0.00665,-0.01237,-0.00027,-0.00233,-0.00087,-0.00255,-0.01222,0.0061,0.00509,0.00011,0.00382,0.01146,0.00017,0.00887,-0.00767,0.01123,0.01422,0.00167,-2e-05,-0.00018,0.00137,0.0057,-0.00972,0.00762,-0.00172,0.00345,-0.00266,0.00823,-0.00219,-0.00242,-0.01113,0.01928,0.00114,0.01047,0.00144,-0.00061,0.00804,-0.00308,0.0089,0.0092,-0.00262,0.00151,-0.00342,-0.00037,3e-05,0.00603,0.00979,0.00632,-0.00932,0.0141,0.00534,-0.01591,0.00938,0.00478,0.00809,0.00032,-0.00538,-0.01013,0.00431,0.0066,0.00787,0.01139,0.0052,0.00639,-0.00581,-0.01478,0.00756,-0.0017,0.00442,0.01585,-0.00285,0.01212,-0.01611,-0.00384,0.00099,-0.00069,0.00514,-0.01152,-0.00896,0.00444,0.00192,-0.00845,0.00142,0.0065,0.00999,0.00541,0.00777,0.00959,0.00028,0.00394,-0.01216,-0.00443,0.00029,0.00826,-0.00698,-0.01009,0.00189,0.01312,-0.00549,0.00616,0.00645,-0.00338,0.00353,-0.00252,-0.00596,-0.00208,-0.006,0.00193,-0.00081,0.01036,-0.01028,-0.00024,-0.00419,0.00877,-0.01073,-0.01316,0.00102,0.00405,-5e-05,0.00562,-0.0075,-0.00358,-0.00205,-0.00247,-0.00271,0.00347,-0.00719,0.01055,0.01051,-0.00612,-0.00541,0.00316,-0.00395,-0.00498,-0.00863,0.0117,-0.00183,0.00509,0.00622,-0.00533,0.01141,0.01184,0.01686,0.01687,0.00044,-0.00699,-0.00037,0.00492,-0.00444,-0.00204,0.00591,-0.00039,-0.02097,-0.00364,0.02719,0.01243,0.00765,0.00372,-0.00381,-0.00031,0.01041,-0.00958,-0.00605,0.00373,-0.00029,0.01015,-0.0018,0.01517,0.0021,-0.00311,-0.00049,0.01384,-0.01408,0.00323,0.00833,0.01018,0.00559,0.00541,0.00339,-0.00248,0.00065,-0.0025,0.00847,-0.00199,0.01037,0.00288,-0.01109,0.00149,-0.0115,-0.00039,0.00345,-0.00669,0.00108,0.00515,0.00587,-0.0098,-0.00316,0.00281,-0.01044,0.02091,0.01777,-0.00038,0.01898,-0.00824,0.00333,-0.01763,0.00326,0.00459,0.00126,-0.00204,-0.00075,-0.00478,0.00768,-0.00365,-0.00806,-0.01003,0.00524,-0.00034,-0.00442,-0.01434,0.01178,0.02314,0.00044,-0.00133,0.00147,-0.0202,-0.00035,0.00942,0.00306,0.00356,0.0036,0.00189,-0.00505,0.00246,-0.00178,-0.00126,0.00919,-0.00784,-0.0136,-0.00202,0.00304,-0.00199,-0.00122,0.00144,0.00605,0.00887,0.00846,0.00545,0.00691,0.00157,-0.00203,-0.00181,0.00971,0.01095,-0.00644,-0.01616,0.00925,0.00056,0.00817,-0.00018,0.00938,0.00152,0.00022,0.01266,0.00092,-0.01541,-0.00472,-0.00693,0.00317,-0.00103,0.00581,0.00183,-0.01294,-0.0065,-0.00169,-0.00167,0.01012,-0.01725,-0.00371,-0.00226,0.01364,0.00142,0.00039,-0.00425,-0.00971,-0.00764,0.00164,-0.00028,0.0041,0.00697,-0.00802,-0.00793,-0.00908,0.00842,-0.00181,0.00737,-0.00111,-0.00106,0.00235,0.01475,0.00819,-0.02048,-0.00127,0.00892,-0.00143,0.0039,0.00368,-0.00073,0.00516,0.0109,0.01122,0.00446,0.00512,0.01477,0.00557,-0.00621,-0.00139,-0.00483,0.0141,0.00273,-0.00196,0.00045,-0.00935,-0.0115,-0.00794,-0.00108,0.01047,-0.0081,0.00412,-0.00473,-0.00844,-0.0068,0.01133,0.01648,-0.0025,0.00226,-0.00116,-0.00567,0.00577,0.00848,0.0081,0.00066,-0.00031,-0.01105,-0.00796,0.00113,0.00342,-0.00427,-0.00842,-0.01083,0.00024,0.0053,-0.00653,-0.00219,0.00962,-0.00122,0.00268,-0.00648,-0.01496,0.0062,0.00029,0.00355,0.00034,-0.00331,0.00668,0.00845,-0.00899,0.00522,0.00725,-0.0133,-0.00638,0.00438,0.0051,-0.0025,-0.00556,-0.01094,0.00486,0.00342,-0.00092,-0.00753,0.00273,-0.00641,-0.00166,0.00606,-0.00024,0.00311,-0.00147,0.01397,0.01317,0.00387,-0.00157,0.01179,0.02334,0.00626,0.00676,0.0008,0.00469,-0.00662,0.00532,-0.00596,0.0011,-0.00855,0.00387,0.00413,0.00889,0.00764,-0.00538,-0.00859,-0.00304,-0.00181,0.00676,0.00137,-0.00943,0.01659,-0.00582,0.01224,0.0001,4e-05,0.0114,-0.00074,0.01051,-0.00211,-0.00201,0.00451,0.00434,-0.0001,0.00047,0.0055,0.00144,-0.0097,0.00182,-0.01121,-0.01205,-0.00357,0.00022,-0.00724,0.00525,-0.01038,-0.00704,0.00346,-0.01026,0.00557,-0.00718,-0.00092,0.00696,-0.01156,-0.00885,0.01305,0.00725,-0.00508,8e-05,0.00141,0.0004,-0.01424,0.0059,-0.0121,-0.00026,-0.01263,-0.00481,-0.00942,0.00342,-0.00701,-0.00815,-0.00747,-0.01209,0.00319,0.00697,0.00891,-0.00182,-0.01366,0.00485,0.00935,0.00487,0.00178,0.00221,-0.00798,-0.00107,-0.00177,0.01342,-0.00053,-0.01428,0.00477,-0.00336,-0.00935,0.00904,0.0038,5e-05,0.0048,0.00567,0.00631,0.00527,-0.00996,0.01407,-0.00038,-0.0052,-0.0058,-0.00295,-0.00602,0.00181,0.00188,-0.0053,0.00557,-0.00467,0.00365,-0.01221,0.0027,0.00735,0.00098,0.0028,0.00069,-0.00559,-0.00235,0.00745,0.00745,0.01093,0.01197,0.00518,-0.01314,-0.00851,-2e-05,-0.01083,-0.00826,-0.01799,0.00999,0.01356,-0.01897,-0.00961,-0.00258,-0.00832,0.00582,0.00321,0.00822,-0.00057,0.00142,0.0038,-0.00112,0.00969,-0.00354,0.00153,-0.00244,0.00257,-0.01091,0.0,-0.01383,0.00306,-0.00117,0.00618,-0.00649,-0.00607,-0.01204,-0.00459,-0.00769,0.00203,-0.00868,-0.00222,0.00397,-0.00339,0.00428,0.00078,0.00298,-0.01565,-0.00406,0.01278,0.00481,-0.00534,0.00432,-0.00779,0.00069,0.0054,-0.00085,-3e-05,0.00657,0.00739,-0.0022,-0.00046,-0.0001,0.00353,0.00952,0.0039,0.00301,0.00108,0.00304,0.00503,-0.00693,0.00955,-2e-05,-0.00039,-0.01806,-0.00425,0.01845,-0.00498,-0.00403,-0.00415,0.00202,-0.00258,0.00226,0.00322,-0.00024,-0.00828,-0.00118,-0.00148,0.01367,0.00034,0.00213,0.01198,-0.00138,0.00771,0.00678,0.00074,0.00955,0.01533,0.00247,-0.00747,0.00341,-0.00507,0.00559,0.00084,0.01079,0.01539,-0.01195,-0.00276,0.00446,0.00384,2e-05,0.00843,0.01328,-0.01069,-0.00508,0.00902,0.00416,0.00862,0.00491,-0.00804,0.00571,-0.00659,0.0012,0.00346,-0.01682,-0.0044,0.00454,0.0055,0.00575,0.01381,-0.02773,0.00525,0.0039,-0.00598,-0.00061,0.00126,0.001,0.00494,0.00042,-0.00269,-0.00148,-0.00385,0.00441,0.00117,-0.00039,-0.0032,0.00516,-0.00922,0.01501,-0.01282,0.00598,0.00261,-0.00593,0.01042,-0.00245,-0.00751,-0.0105,-0.00546,-0.00282,-0.00325,0.00298,0.00875,0.00764,0.00461,-0.00521,0.01013,-0.01924,-0.00549,-0.01436,-0.00401,0.00565,0.00566,-0.00358,-0.01045,0.00368,0.003,-0.00883,0.00457,1e-05,-0.0004,-0.00447,-0.00792,0.01068,-0.00678,0.01745,-0.0048,0.00129,0.00597,-0.00396,-0.00683,-0.00688,0.00657,0.01203,-0.00272,0.01459,-0.00465,0.00351,-7e-05,0.00253,-0.00806,-0.00136,0.00515,0.00324,-0.00243,-0.00207,0.0025,0.00056,-0.01456,0.00599,-0.00379,0.0027,-0.00398,0.00072,0.00543,0.00311,-0.00614,-0.0019,0.00029,-0.00488,0.01617,-0.00243,0.00098,0.00072,-0.00838,0.01385,-0.00184,-0.01142,-0.0066,-0.00061,-0.00475,-0.00708,0.00376,0.00034,0.0091,-0.00055,0.00852,-0.00096,-0.00628,-0.00044,0.00331,0.00532,0.01314,-0.00842,0.00027,0.00076,0.0017,0.00356,0.00299,-0.00447,-0.00107,-0.00757,0.0024,0.00337,0.01435,-0.01119,-0.00321,0.00776,0.00513,0.00046,0.00107,0.00923,0.00241,-0.00183,0.00532,-0.00068,-0.00763,-0.00392,-0.00743,0.00613,-0.0044,0.00487,-0.00305,1e-05,0.0114,-0.00515,-0.00303,4e-05,-0.00479,-0.00019,0.00098,0.00173,-0.00828,-0.00814,-0.00366,0.00046,-0.0011,-0.00812,-0.00549,-0.00968,0.00495,0.00369,0.00609,5e-05,0.00767,-0.009,0.00966,0.00036,0.00077,0.00433,0.00672,0.00837,0.00408,0.00272,0.00247,-0.00241,0.00037,0.01773,0.00126,0.00158,0.0006,0.00381,0.00421,0.01486,0.0062,-0.00166,-0.00447,0.00551,-5e-05,0.00757,-0.00032,-0.00676,0.01133,0.00805,0.00189,0.00101,0.00016,0.01527,-0.00456,-0.01234,-0.00599,0.01266,0.00953,-0.00657,0.03352,0.0016,-0.00422,0.0025,0.01526,-0.00792,-0.00343,0.00591,0.00384,-0.00203,-0.00468,0.014,-0.00675,0.00726,0.0136,-0.00192,0.00704,-0.00433,0.01517,-0.00013,0.0036,-0.00144,0.00293,0.00209,-0.00685,0.00239,1e-05,0.00722,-0.00082,-0.00345,-0.00193,-0.00094,0.00319,0.00238,-0.00516,0.01062,0.00672,-0.00302,0.0028,0.00234,-0.01564,-0.00892,-0.00578,-0.00624,-0.00189,-0.00641,0.00917,0.00701,-0.00701,0.00422,0.00677,-0.00495,-0.0119,-0.00143,-0.00089,0.00583,0.00628,-0.0055,0.00765,0.0106,0.00211,-5e-05,-0.00752,-0.00629,-0.00153,0.00949,-0.01305,-0.00425,-0.00862,0.00871,-0.0008,0.00925,0.00679,0.00059,0.00633,0.00775,0.00167,0.00454,0.0107,0.00659,-0.00243,-0.00379,0.00424,-0.00139,0.00636,-0.00929,0.00314,-0.00575,0.00263,-0.00181,0.00053,0.00212,-0.01153,-0.00267,0.00628,-0.01272,-0.01184,-0.01239,-0.0067,-0.00136,0.00568,-0.00579,0.00572,0.00695,0.00082,0.00465,-0.00468,-0.0091,0.02137,-0.00893,-0.0041,0.00954,-0.00183,-0.00987,-0.00103,-0.00666,0.00194,0.00918,-0.00405,-0.00452,0.00114,-0.00487,-0.00268,-0.00235,0.00558,-0.00432,-0.00794,0.00345,-0.0072,-0.00609,-0.01327,0.01104,0.0035,0.00688,0.00222,0.00221,-0.00781,0.00153,-0.00163,0.00878,0.00173,0.00703,0.00794,-0.00104,0.00624,0.00314,0.00548,0.00103,-0.00318,0.00091,-0.00826,-0.01003,0.00765,0.00326,-0.00184,0.00442,-0.00385,0.00157,0.00067,0.00237,-0.02033,-0.0063,0.00439,0.00553,-0.00496,-0.00737,0.00287,-0.00308,-0.0016,-0.00824,-0.00612,0.00566,0.00673,0.00786,-0.00644,-0.00165,0.00717,-0.00061,0.01146,0.00445,0.00824,-0.00416,0.00407,0.0107,0.01141,0.00621,0.01351,-0.00036,-0.00878,0.00289,-0.00774,-0.0039,-0.00237,0.00174,-0.00813,-0.00996,0.00507,0.00737,0.00557,-0.00506,-0.00086,-0.0105,0.00901,-0.00566,-0.01076,0.00243,-0.01065,-0.01086,0.01128,0.00034,-0.00561,0.00828,-0.00268,0.00264,-0.00953,-0.01285,-0.00905,0.00953,-0.00256,0.0062,-0.00443,-0.0015,0.00115,0.00393,-0.00142,-0.00869,0.0095,0.001,-0.00472,-0.01554,-0.00914,0.00316,0.01311,0.01293,0.00667,-0.00608,0.00417,0.00179,0.00041,0.00205,-0.0036,0.00361,0.01076,-0.00536,0.00805,-0.00325,0.00049,0.00283,0.00176,-0.00781,-0.00521,0.01438,-0.00045,0.00024,0.00496,0.00185,0.00599,-0.0103,-0.00638,-0.00155,-0.00165,-0.01084,-0.00398,-0.00389,0.00674,0.00018,0.01153,-0.00397,-0.00393,-0.00733,-0.00572,-0.00282,-0.00374,0.00818,-0.00863,-0.01602,0.00537,-0.00074,-0.00834,0.0084,-0.00502,-0.00132,-0.00237,0.0108,0.00158,0.00346,0.00504,-0.00298,-0.00316,-0.00183,0.00321,0.00578,0.01455,-0.00032,0.01125,-0.00131,-0.00094,-0.01313,-0.00141,-0.017,-0.0014,0.00057,0.01371,-0.00109,0.00771,0.0063,-0.00014,0.00554,-0.00257,-0.00426,-0.01304,-0.00232,-2e-05,0.00336,0.0015,-0.00262,0.00253,-0.00526,0.0082,-0.01871,-0.0086,0.01211,0.01119,0.00085,-0.00104,0.00078,0.00722,-0.01144,-0.00477,0.0032,-0.00368,-0.00063,0.01576,-0.00337,0.00328,-0.00164,-0.00651,-0.00754,0.0082,-0.00318,0.00221,-0.00751,-0.00068,0.009,0.00783,0.00375,-0.00662,-0.00512,-0.0006,-0.01098,0.00878,0.00029,0.00054,0.0074,-0.00192,0.00257,-0.00496,0.00895,0.00358,-0.01764,-0.00174,-0.0198,0.00519,-0.00575,-0.0148,0.00272,0.00879,-0.00183,0.0088,0.0008,-0.0037,-0.00546,0.01124,0.00492,0.00303,0.00131,-0.00851,0.00373,0.00093,0.00855,-0.00351,0.00806,0.00405,-0.00561,-0.00372,0.00987,0.00415,0.00216,-0.005,0.00877,-0.00891,0.00198,0.00188,-0.0122,-0.00556,0.01127,-0.00443,-0.00713,0.00381,-0.01185,-0.0001,0.00632,-0.00201,0.01121,0.00557,-0.01073,-0.00294,-0.00236,0.00037,0.01497,0.00384,0.00771,-0.00701,-0.00719,-0.00953,-0.00684,0.00829,-0.00391,-0.01616,0.00888,-0.00586,0.01021,-0.00145,0.01452,-0.00321,0.01154,-0.00112,0.00357,0.00965,-0.01086,-0.01274,0.00489,-0.01013,2e-05,-0.00371,0.00207,-0.00189,-0.00513,-0.00898,-0.01424,-0.00648,0.00921,0.00583,-0.00077,-0.00332,-0.00421,0.00503,0.00892,0.00453,0.01282,-0.0116,0.00666,0.00218,0.01248,0.00119,0.00135,0.01038,-0.00956,0.00173,0.01313,0.0113,0.00413,-0.0017,-0.0051,0.00646,-0.00643,0.00173,0.00019],[-0.0127,-0.00365,0.00099,-0.00627,0.00368,-0.00278,-0.00155,-0.00143,-0.00576,-0.00656,0.01091,0.00672,0.00104,-0.00302,0.00629,-0.00583,-0.00401,0.00342,-0.00171,-0.00209,0.00346,-0.01027,-0.00371,-0.01061,0.00132,-0.00172,-0.00114,-0.00383,-0.00883,0.00916,0.00285,-0.00802,0.01039,-0.00421,-0.00256,0.00164,0.00863,-0.01544,0.00238,0.00525,-0.00705,-0.00517,-0.01291,-0.00136,0.00513,0.00031,-0.00669,0.01061,-0.00172,0.00757,-0.00883,-0.00445,-0.0105,0.00184,-0.01426,-0.01167,0.00238,0.00095,-0.00319,0.0065,0.0004,-0.00563,-0.0124,-0.00126,-0.00929,0.00574,0.00148,-0.00273,0.00506,-0.00755,0.0038,-0.00161,0.00783,-0.01251,0.00493,-0.00672,-0.00685,-0.00243,-0.00054,-0.00088,-0.00988,-0.00769,-0.00171,-0.00346,-0.0018,-0.00074,-0.00982,-0.00131,-0.00077,0.0033,-0.0083,0.00115,-0.00371,-0.00978,0.00289,-0.02025,0.00795,0.00122,-0.01455,0.01067,-0.00294,0.0159,0.00458,0.01235,-0.00885,-0.0034,-0.00367,-0.00136,-0.00106,-0.00441,0.00252,0.00239,0.00538,-0.00956,0.00016,0.00615,0.00149,0.00856,0.00549,0.00357,0.00338,0.00187,-0.01053,0.00994,0.00645,-0.00654,-0.00721,0.00872,-0.01436,0.00249,0.01017,0.00836,-0.00414,-0.01375,0.00474,-0.00679,0.00444,0.00233,0.00284,-0.00744,0.00376,-0.01097,-0.01304,0.00254,-0.00073,-0.00831,-0.00027,0.00273,-0.01165,0.01321,-0.01063,0.00228,0.0103,0.0033,-0.0099,0.00777,-0.00518,0.00598,0.00822,-0.00413,0.00281,-0.01241,-0.00614,0.00139,0.00733,0.01234,-0.00438,-0.00558,-0.0044,0.0023,-0.00875,0.0065,-0.00469,-0.00423,-0.00531,0.009,-0.0062,0.00177,0.00086,-0.00518,0.00323,-0.00772,0.0008,0.00726,-0.0003,-0.00595,0.00912,0.00886,0.00251,-0.00569,0.00925,0.01083,0.00905,0.00218,0.01144,-0.00272,0.00245,0.00764,-0.0075,0.00297,-0.00812,0.00381,-0.00026,-0.0042,-0.00397,-0.0091,0.00451,-0.00918,-0.0064,0.00275,-0.00029,0.01337,-0.01172,0.01516,-0.00659,0.00096,0.00417,0.00754,0.00037,-7e-05,-0.00147,-0.00061,-0.01078,-0.01382,0.01068,0.005,0.00326,0.00882,-0.00358,-0.00519,-0.01939,0.00305,-0.00704,-0.00299,0.00725,0.00262,-0.00967,-0.00401,-0.00141,0.01531,0.00231,0.00426,0.0032,0.01884,0.01481,-0.00895,-0.00695,0.0028,-0.00126,-0.00032,-0.00083,-0.00045,-0.00437,-0.00196,0.0004,0.0037,0.00244,0.01095,0.00516,0.00844,0.0063,-0.00504,-0.00582,0.00538,0.01279,0.01412,-0.00028,0.00616,0.01676,0.01324,-0.00166,-0.00023,0.00255,-0.0046,-0.00378,-0.00337,-0.00108,-0.00543,0.00746,0.00295,0.0167,0.00556,0.00698,0.00199,0.0103,-0.00642,0.00915,0.00919,-0.00844,-0.00246,-0.0093,0.00477,0.00271,-0.0094,-0.0102,-0.00338,0.0037,-0.01317,0.00221,0.01941,0.00155,0.00714,-0.01221,-0.0054,-0.0042,-0.00038,0.00364,-0.01567,-0.0086,-0.00211,0.00242,-0.0111,0.00628,0.00819,-0.00328,0.01226,-0.00616,0.01068,-0.00563,-0.0031,0.00303,0.01335,-0.0009,0.00433,0.00116,0.00743,0.0041,0.00565,-0.00124,-0.00036,0.00812,-0.01059,0.0072,-0.0013,-0.00561,-0.01424,-0.00489,-0.0028,-0.00847,-0.00199,-0.01037,-0.003,0.00475,0.00697,0.00509,-0.00077,0.00781,0.00087,-0.01096,-0.00869,0.0019,-0.0123,-0.00076,-0.00435,-0.00767,-0.00656,-0.00656,-0.0011,-0.00251,-0.00912,-0.00105,0.00775,-0.00646,-0.00133,0.00677,0.02011,-0.00406,0.00566,0.00688,-0.01675,-0.0078,0.00586,-0.00565,-0.00448,0.0107,-0.0021,0.00178,0.00528,-0.00524,-0.00141,-0.00161,-0.00547,-0.00344,-0.0013,-0.00685,0.00021,0.01305,0.00015,-0.00016,-0.00933,-0.00169,-0.00224,-0.00296,0.01072,0.00601,0.00253,0.00027,-0.00192,0.00307,-0.01154,-0.02224,0.01563,0.0033,-0.00217,0.00578,-0.00028,-0.00734,0.00874,-0.01997,-0.00452,-0.00939,-0.00939,0.00265,0.0087,0.00134,-0.00938,0.00481,-0.00716,9e-05,-0.00522,-1e-05,0.00621,-0.00259,0.01092,-0.01732,0.00384,0.01022,0.00211,-0.01471,-0.00319,-0.0046,0.00511,-0.00984,0.00411,-0.00683,-0.00149,0.01311,0.00379,0.00119,-0.00601,0.00384,0.00902,-0.00033,-0.00155,-0.01483,-0.00383,0.00773,0.00285,0.01175,0.00924,0.00591,-0.00198,-0.00898,0.00722,0.00607,-0.00974,0.00102,0.00982,-0.00096,-0.01112,-0.00864,0.01185,0.00193,-0.0135,-0.00064,-0.01357,0.00431,0.00692,-0.00134,-0.00486,0.00388,-0.00924,-0.00564,-0.02237,0.00266,-0.00204,0.00727,-0.00193,0.00127,-0.00466,0.00277,0.01015,-0.01043,0.00549,0.00965,-0.00668,0.00444,0.0109,-0.00137,0.00767,0.01443,0.00667,-0.00156,-0.0091,-0.00291,-0.00715,0.00602,0.00065,0.00719,0.0159,-0.00528,-0.01063,0.00705,0.00756,-0.02053,-0.00987,0.00477,-0.00046,0.00576,-0.00329,0.00211,-0.00132,-0.00091,-0.00749,0.00518,0.00633,0.01045,-0.01884,0.00117,0.00231,0.02068,-0.00451,-0.00168,0.01052,-0.00625,-0.00167,0.00524,0.00735,0.00252,0.0036,0.01196,-0.00484,-0.00653,0.00418,0.00297,-0.00487,0.00656,0.00613,-0.00151,0.0013,-0.00735,0.00303,0.00291,-0.00234,0.00651,-0.00657,-0.00252,0.00692,0.00605,-0.00779,0.00668,0.00236,0.0145,-0.01184,0.00906,-0.01856,-0.00053,0.00346,-0.00521,-3e-05,-0.0017,-0.01057,0.00521,0.00416,0.00046,0.01261,-0.00039,-0.00331,-0.01955,0.00171,-0.01292,-0.00201,-0.00347,-0.00899,-0.00119,0.00315,-0.00069,0.00711,-0.01086,-0.00401,0.01462,-0.00306,-0.00392,-0.00992,-0.00711,0.00778,0.00306,-0.00083,0.0022,0.00868,-0.01954,0.01569,-0.00214,-0.00443,-0.00384,-0.0094,0.00586,0.00235,-0.00151,-0.00963,-0.00199,0.00065,0.00364,-0.00596,-0.00291,-0.00487,-0.0095,-0.00244,-0.01178,0.02197,0.00636,-0.00373,-0.01122,0.00598,0.00889,-0.00475,0.00528,-0.00289,-0.00329,-0.00195,-0.01138,-0.00084,-0.01211,0.00301,0.0098,0.0084,-0.00482,-0.00951,-0.00852,0.00266,-0.00264,0.0105,-0.00082,-0.0085,-0.01574,0.00303,-0.00186,-0.00167,0.00377,-0.00827,0.00379,-0.00658,0.00261,0.00019,-0.00386,-0.00262,0.00751,-0.00285,0.0055,-0.00119,-0.0067,-0.00277,-0.01052,0.00145,-0.00196,0.00512,-0.00438,0.00845,-0.00683,0.00984,0.00891,0.00056,0.01347,-0.00406,0.01263,-0.00516,0.00707,-0.00097,0.00074,-0.01365,-0.00688,-0.0017,0.01154,-0.0164,0.0084,0.00954,0.00019,0.0014,-0.00201,-0.02551,0.007,-0.0021,0.01085,0.01438,-0.00256,-0.00843,0.00913,-0.00834,-0.00607,0.0058,0.00474,-0.01711,-0.00394,-0.00581,-0.00367,-0.01313,-0.00733,-0.00552,-0.00295,0.00192,-0.01329,0.00648,-0.01406,-0.00346,-0.00028,0.00378,0.00211,-0.00236,0.01172,0.00142,9e-05,-0.00207,0.00025,-0.00071,0.00407,0.01183,0.00478,-0.0025,-0.00436,0.00973,0.01287,0.00806,-0.01385,-0.0003,-0.01144,0.00633,0.0081,-0.00263,0.01193,0.00413,0.02019,-0.00022,0.00473,-0.01332,-0.00339,0.00288,-0.00379,0.00299,-0.00156,-0.00054,-0.00559,0.00386,0.01205,0.00409,0.0058,0.00321,-0.00137,-0.0007,-0.00413,-0.00461,-0.00625,-0.00685,0.01768,-0.00053,0.01239,-0.00301,-0.00293,0.0023,0.00615,0.00745,-0.02124,-0.0051,-0.00289,0.00406,-0.00919,-0.00668,-0.00539,0.00977,-0.00481,2e-05,0.00553,-0.00897,0.00198,0.01348,-0.00651,-0.00856,0.00736,0.00602,-0.0078,0.00601,0.00789,0.00745,-0.00865,-0.0068,-0.00388,0.00313,0.00358,-0.0021,-0.00021,-0.00257,-0.0002,-0.01922,-0.00575,-0.00083,0.00364,-0.00233,-0.00424,0.00025,-0.00396,0.00609,0.00838,-0.00117,0.00113,0.00537,-0.01572,-0.00661,0.0017,-0.00802,0.00703,-0.00415,-0.00306,-0.00405,-0.00386,0.00891,0.00365,0.01211,-0.01061,-0.00394,0.00084,-0.00514,0.00369,0.00202,-0.00023,-0.00974,0.00225,-0.01148,0.00689,0.00274,0.00187,0.00917,-0.0014,-0.00177,-0.00031,0.00099,-0.01486,0.00326,-0.00777,0.00437,0.01261,0.00644,0.00162,-0.01021,-0.00751,0.0025,-0.01425,-0.00903,-0.00565,-0.0053,-0.00943,-0.00445,-0.00837,0.0095,0.00113,-0.00915,0.00183,0.00763,-0.00425,-0.01003,-0.00085,0.00353,0.00675,0.00575,-0.01483,0.00386,-0.00445,-0.029,-0.00566,0.00344,0.01469,0.00785,-0.00528,-0.00393,-0.0187,0.00421,0.00372,-0.00303,-0.00167,0.00546,-0.00956,-0.00075,-0.00448,0.00304,-0.00168,0.00189,0.00695,0.00792,0.00588,0.00129,0.01651,-0.00238,-0.00605,-0.0006,2e-05,0.00054,-0.00747,0.00297,0.00056,-0.00708,0.01437,-0.00588,0.00553,-0.01029,-0.01704,0.01111,0.01103,0.0003,0.00173,-0.00942,-0.00177,-0.00208,-0.00111,0.00266,0.01024,0.00251,-0.00089,5e-05,-0.00168,0.01021,0.00215,-0.0012,0.00611,-0.00691,-0.01623,0.00795,0.00446,0.01248,0.01242,0.00551,0.00449,0.00717,-0.00991,0.00483,0.0063,0.00557,-0.00259,0.00125,-0.01328,0.00773,-0.00181,-0.00235,0.00764,-0.01022,0.01169,0.00628,0.008,0.0011,0.01615,-0.00709,-0.00026,0.00577,0.00032,-0.00909,-0.00707,0.01539,0.01317,-0.00651,-0.00332,4e-05,0.00768,-0.00029,0.00305,0.00676,0.00491,-0.0002,0.00915,0.02119,-0.00413,0.00049,0.01162,0.01133,0.00422,0.00127,-0.00063,0.01616,0.00301,-0.00235,-0.00483,-0.00955,0.00739,0.00607,0.00684,0.00848,-9e-05,0.00266,0.0197,0.00539,-0.00456,0.00793,-0.01085,0.0109,0.00786,-0.00448,-0.00184,0.00711,-0.00289,-0.01358,-0.01086,-0.00337,0.01114,-0.0068,0.00283,0.0041,-0.00621,-0.01153,-0.003,0.00144,-0.00844,0.00207,-0.00768,-0.0002,0.01023,-0.00337,-0.00375,0.0074,0.00231,0.00612,-0.00262,-0.01516,-0.0017,0.00624,0.00089,0.00112,-0.00121,0.0034,-0.00364,-0.00136,0.00225,0.00378,0.00186,0.01006,0.00238,0.00326,0.01259,-0.02023,-0.00867,-0.01764,0.00536,0.00532,-0.00071,-0.00914,-0.00935,-7e-05,0.00051,0.00212,0.00647,-0.0113,0.00615,0.00884,0.00047,-0.01252,0.00197,-0.0001,-0.00701,-0.00524,-0.00582,-0.00023,0.00015,-0.01288,0.01367,-0.00983,0.00712,0.00288,-0.003,0.00527,-0.00873,0.00066,-0.00314,-0.00108,-0.00841,0.00488,-0.00344,0.00941,-0.01088,-0.00832,0.00164,0.00076,-0.001,0.00456,-0.00547,-0.00159,-0.00036,0.01061,-0.0169,0.00288,-0.00792,-0.00244,0.00302,-0.00661,0.0027,0.01279,-0.0086,-0.00026,0.00699,0.01376,0.00743,-0.01064,-0.00171,0.00253,0.00274,-0.01541,-0.00572,-0.00025,-0.0005,-0.00848,0.00408,-0.00032,-0.0028,0.01493,-0.00173,0.00516,0.00602,0.00756,0.00484,-0.00106,0.00836,0.00134,0.00117,0.01179,0.00802,0.00237,-0.00583,0.00593,0.01259,0.0151,0.0006,-0.00432,-0.0125,-0.00816,0.00981,0.00019,-0.00716,-0.0157,0.00081,0.00622,-0.00158,-6e-05,-0.00483,-0.00117,-0.00336,0.0006,-0.01292,-0.0061,-0.00776,0.00714,0.00812,0.00883,-0.00241,0.00541,-0.01526,-0.00612,-0.00482,0.00982,-0.01815,0.00453,-0.01447,-0.00409,-0.00213,0.00634,-0.00383,0.00939,0.00578,-0.00578,0.00231,0.00227,-0.0017,-0.01528,-0.0029,-0.0006,-0.00073,0.00083,-0.00959,-0.00695,-0.0079,0.0041,0.0108,-0.01574,-0.00223,0.00776,-0.01345,0.00677,-0.00549,-0.0013,0.01562,0.00875,0.00264,0.00167,0.00576,0.01259,-0.00725,-0.00239,-0.00166,-0.00533,0.0044,-0.00709,0.00767,0.00358,-0.00277,0.00866,-0.00241,-0.0101,0.00774,-0.00277,0.01298,0.00222,0.01283,-0.00422,0.00956,-0.01079,0.00186,0.0152,-0.00272,-0.00297,0.00978,0.01887,-0.00192,-0.00962,0.00921,0.00337,0.00351,0.00745,0.00416,0.00373,-0.01954,0.00101,-0.00552,0.01153,0.01241,-0.01537,-0.00137,-0.0097,0.00651,0.00375,0.00551,-0.00028,-0.01209,0.00106,0.00954,-0.00427,0.0092,0.00275,-0.0076,-0.00188,-0.00243,0.00205,0.00125,0.007,-0.00332,0.0002,-0.0004,0.00628,-0.01221,-0.00022,-0.00324,-0.00827,0.01304,-0.00731,0.00649,0.01237,0.00351,-0.00207,-0.00241,-0.00384,0.00658,0.00808,0.00151,0.00277,-0.00361,0.01667,0.01198,-0.00512,-0.00571,0.00028,0.00398,-0.00988,0.00538,0.01055,0.01142,-0.00093,0.00655,-0.00038,-5e-05,-0.01067,-0.00208,0.01276,-0.00129,-0.00685,-0.00425,-0.00378,0.00584,-0.00468,-0.0081,0.00476,0.00554,-0.01108,-3e-05,-0.00935,0.00127,0.01941,0.00142,-0.00898,-0.00047,-0.01307,0.00448,-0.012,0.0044,0.00427,0.00126,-0.00958,0.00397,0.00352,0.00654,-0.00089,0.00165,-0.00799,-0.00881,0.00063,-0.00371,-0.00158,-0.0107,-0.00211,0.00021,-0.0115,-0.00562,0.00087,0.0105,0.00738,0.00555,-0.00249,-0.01073,0.00201,0.00532,-0.00137,-0.01266,-0.00983,-0.00856,0.00388,-0.01322,-0.01382,-0.00478,-0.00183,0.01053,0.00757,0.00897,-0.00253,-0.01128,-0.00625,0.00224,0.00044,-0.00502,0.01121,-0.01608,0.00796,-0.00533,-0.00572,0.00563,-0.0034,0.00915,-0.00888,-0.00662,0.00119,-0.01938,0.00711,0.01445,-0.00081,-0.0064,-0.01669,-0.00065,-0.01035,0.01382,0.00741,-0.00798,-0.00793,0.00599,-0.00164,-0.00248,0.00576,-0.00667,0.00125,0.00139,0.01392,-0.00666,-0.00016,0.00091,-0.00433,-0.00046,-0.00671,0.00014,-0.00296,0.00298,-0.00466,0.00566,0.00197,-0.01107,0.00342,-0.00346,0.01128,0.00265,-0.01571,0.00682,0.00871,0.01204,-0.00112,-0.01007,-0.00817,0.01204,0.00481,0.00329,-0.00072,-0.0062,0.0094,-0.00984,-0.01805,0.02029,0.01293,-0.0138,-0.00467,0.00764,-0.00768,0.00885,0.01137,0.00457,-0.03523,0.00422,-0.00964,-0.00386,0.00536,0.00958,-0.01572,-0.00178,0.00599,0.00202,0.01889,-0.00227,0.00945,-0.00949,0.01241,0.0043,-0.00131,0.00495,0.00204,0.01517,0.00405,-0.00305,0.005,0.01266,0.00374,0.00026,0.01342,0.0111,0.01376,-0.01195,-0.00601,0.00458,-0.00351,0.00762,-0.0023,0.00456,0.00608,-0.00432,-0.0001,0.00245,-0.00178,0.00539,0.00127,-0.00112,-0.00351,0.00256,0.01106,0.00745,-0.00637,-0.00013,0.00743,-0.009,0.00164,0.00301,0.00109,-0.00243,-0.00796,0.01689,0.00408,0.00986,-0.00914,-0.00367,0.00289,0.00517,-0.00381,-0.00397,-0.0094,-0.00328,-0.00426,-0.00631,0.00657,0.00237,0.00144,0.00296,-0.00048,0.00028,0.00422,-0.00733,-0.00857,0.00027,0.01031,-0.00186,0.00591,0.00779,0.00013,0.00306,3e-05,0.00107,-0.001,-0.00724,-0.00561,0.003,-0.00318,-0.01153,-0.00698,-0.01025,-0.01235,0.00687,-0.00858,-0.01104,-0.01197,0.01061,-0.00724,-0.00621,0.00094,-0.00461,-0.0036,-0.00494,0.00312,0.00245,-0.0089,0.00406,0.00219,0.00991,0.01529],[-0.00477,-0.00505,0.0094,0.00434,0.00375,0.00574,0.00233,-0.00214,-0.00389,0.01036,-0.00181,0.03784,0.00049,-0.00239,-0.01097,-0.0124,-0.00839,0.00522,-0.01053,-0.02359,0.00203,0.00025,-0.00556,0.00222,0.0053,-0.00948,0.00182,-0.01951,0.02844,5e-05,-0.00071,0.00222,-0.00402,0.01103,0.00292,-0.01172,0.0022,-0.00085,-0.00803,-0.01157,0.01171,-0.00641,-0.00406,0.02111,-0.00736,0.00141,0.00828,-0.00747,-0.00245,-0.00521,0.00715,-0.00908,-0.00021,-0.00568,0.00872,0.00314,-0.00427,-0.00813,-0.00647,0.00672,0.00705,0.00794,-0.00727,-0.00269,0.01253,0.01397,0.00877,0.00523,0.01051,-0.00704,-6e-05,-0.00961,-0.00989,0.0052,-0.00344,-0.00903,-0.0022,-0.0031,0.01458,-0.00863,0.03009,-0.00913,-0.01431,0.00695,0.00199,-0.00832,0.00162,0.01053,-0.00503,0.00025,-0.00637,0.01184,0.01015,0.0011,-0.0052,0.01303,0.00819,-0.00757,-0.01543,0.00832,0.0117,-0.0083,0.01139,-0.00108,-0.00772,0.00691,-0.00142,0.0032,0.01214,0.01229,0.00793,-0.00582,-0.00796,-0.00123,-0.00428,-0.00252,-0.00295,0.00645,0.0019,-0.00953,-0.00498,0.0077,-0.00472,-0.00382,0.01012,0.00527,0.00044,0.0032,0.00711,0.00463,-0.00099,0.00385,-0.00941,-0.00671,-0.00704,0.00417,0.00096,0.00702,-0.00156,-0.00132,0.03075,0.00533,-0.00209,0.01401,0.00699,0.00842,-0.01019,-0.01391,-0.0047,-0.00244,-0.00586,-0.00568,-0.00582,0.00094,-0.00049,0.00142,0.00773,-0.00511,0.00777,0.00424,-0.00027,0.0021,0.01004,0.00373,0.01747,0.00428,-0.01066,0.00202,0.01892,0.00307,-0.00209,-0.00297,0.00795,-0.00643,-0.00511,-0.00023,-0.00874,0.01036,0.00583,-0.00135,0.01113,0.00577,-0.00339,-0.00315,0.0009,0.00545,0.0072,0.00336,-0.00305,-0.00895,0.00236,-0.0009,0.00355,-0.0033,0.00761,0.00407,0.01687,0.00425,0.00754,0.00893,0.00105,0.00233,-0.00725,0.0149,-0.00049,-0.00808,0.006,0.00293,0.0127,-0.00402,0.00307,0.00088,-0.0104,-0.01252,-0.00399,-0.01315,0.00309,-0.00845,-0.01055,0.00764,-0.00762,0.00592,-0.00971,-0.02282,-0.00096,-0.00067,0.00429,-0.01812,0.004,0.00933,-0.00114,0.00217,0.00854,0.01551,0.00942,-0.00736,0.0002,-0.00288,-0.00801,-0.00229,-0.00122,-0.0036,0.01307,-0.00748,-0.00444,-0.00958,0.00199,0.00917,0.00609,0.00419,-0.00841,-0.00465,0.01071,-0.00684,-0.00453,-0.00034,-0.00805,0.00387,-0.00061,0.00729,0.00614,0.0077,-0.00354,0.02479,0.00327,-0.00432,-0.00259,0.0012,-0.00511,-0.00484,-0.01024,0.01109,0.00137,0.00339,0.00857,0.00272,0.00376,-0.00669,0.00182,2e-05,-0.00506,0.00035,-0.00871,-0.00116,-0.00209,-0.00841,-0.01561,0.00476,0.00173,0.00485,-0.00443,-0.00995,0.00989,0.00213,-0.00767,-0.00808,0.01353,-0.00169,0.00046,-0.00217,-0.00455,0.02141,-0.00286,-0.00911,-0.00547,-0.01491,0.00439,-0.0134,0.0,0.0065,-0.00454,0.01146,-0.00044,0.00275,0.00115,-0.00471,-0.02483,-0.00388,0.00328,-0.00193,0.00844,0.00308,0.00465,0.01051,-0.01332,-0.00061,0.00122,-0.00199,0.00501,0.00404,-0.00848,-0.0004,-0.00559,-0.00614,-0.0056,0.00159,-0.00431,0.01143,0.00786,-0.02217,-0.00066,0.00135,0.00105,-0.00658,-0.00234,-0.00232,-0.01003,0.00081,-0.00997,0.00383,0.01199,-0.00051,0.00333,0.0063,0.00412,-0.00021,0.02726,0.00385,-0.00457,0.00288,-0.00105,0.0089,0.00663,0.00903,-0.00802,0.00105,0.00399,-0.0034,0.00383,0.00563,0.00142,0.01107,-0.00921,-0.00876,0.01028,-0.0062,-0.00775,0.00075,-0.01066,0.00434,-0.0028,0.00228,-0.01013,0.00465,-0.00724,-0.01073,0.00237,-0.00696,-0.01209,0.00362,-0.0084,-0.00919,0.01653,-0.0024,-0.01817,-0.01698,0.00076,-0.00748,0.0089,-0.0086,0.00308,-0.00178,0.01996,0.00417,-0.00548,-0.00911,-0.01092,0.01208,-0.00172,-0.01621,-0.00017,0.00229,0.0064,-0.00111,0.01075,-0.00691,0.00024,0.00902,0.00935,-0.00756,0.00566,0.00064,0.01063,0.00097,-0.00287,-0.0078,0.00091,0.00098,0.00627,0.00659,0.00146,0.0037,-0.01347,-0.00146,-0.0039,-0.00383,-0.00931,0.00703,-0.01242,-0.00946,-0.00204,-0.00195,-0.01462,-0.00532,-0.00444,0.00258,0.00431,0.0008,0.0044,-0.00741,0.0053,0.0014,-0.00198,-0.00728,0.01114,0.0093,-0.00363,-0.01213,-0.0036,-0.00488,-0.01295,0.00701,0.00611,-0.01735,0.00782,0.00696,-0.01422,0.0024,0.01056,-0.00202,-0.00713,-0.0057,-0.01079,0.00339,0.00126,-0.00534,0.00162,-0.0001,0.00762,0.00076,0.00328,-0.0059,-0.0194,-0.00412,0.00352,0.01119,0.00146,-0.00486,0.01255,0.0021,-0.01456,0.00711,-0.00078,0.00296,-0.00701,0.01029,-0.00247,0.00035,0.00734,0.00768,-0.00446,-0.00126,-0.00838,-0.01007,-0.00796,-0.00115,0.00232,-0.0008,-0.00485,0.0003,-0.03058,-0.00513,0.00204,-0.00594,-0.00227,-0.00941,-0.00499,-0.0041,-0.01422,-0.00163,0.00646,0.00758,-0.00153,0.01824,-0.0035,-0.01375,-0.0009,0.00181,-0.00477,-0.00756,-0.0081,-0.00618,-0.00625,0.01036,-0.00069,-0.00357,-0.00936,0.00138,-0.00596,0.00807,0.00793,-0.00233,0.01027,0.00941,-0.00032,-0.00768,0.01002,-0.00135,0.01114,0.00088,-0.00128,-0.02716,0.00911,-0.01289,-0.00213,0.00686,0.0078,-0.00328,0.00558,0.00859,0.00058,-0.00726,0.00212,-0.01323,-0.0038,0.00817,0.0054,-0.00569,0.0073,0.00353,-0.00952,-0.00504,0.00642,0.0144,-0.00566,0.00019,0.00832,0.00948,-0.00483,-0.00449,0.005,-0.00543,-0.00976,0.00184,0.00241,-0.00098,-0.00323,0.01701,-0.00895,0.01298,0.00202,0.00585,0.00358,0.00435,-0.01313,-0.00413,-0.0169,-0.00125,0.00652,-0.02172,-0.00679,-0.02118,-0.01783,-0.02032,0.00503,0.00659,0.00838,0.00603,0.00756,-0.00067,0.00194,0.00342,-0.00384,-0.00383,0.00625,0.00425,0.00747,-0.00731,-0.00779,-0.00939,-0.00488,0.00601,0.01331,-9e-05,0.00752,-0.00612,0.00723,0.00354,0.00045,0.00631,0.00176,0.00512,0.00146,0.01775,-0.00718,-0.00372,0.00487,-0.00799,-0.00909,-0.00184,-0.00308,-0.00806,-0.01925,0.00827,0.00668,-0.00722,-0.00718,0.01078,-0.00513,-0.00168,0.00026,-0.00791,0.00525,-0.01371,-0.00025,-0.00827,0.00057,-0.00052,0.0037,-0.0099,0.00609,-0.02088,0.00452,-0.00722,0.01031,-0.01412,0.02119,0.00982,-0.00327,0.01058,-0.00146,-0.01796,0.0016,0.01085,-0.01165,0.01049,0.02107,-0.00306,-0.0024,0.01244,0.00358,0.00082,0.00381,0.00265,0.003,-0.01111,0.01032,0.00267,0.00662,-0.01045,0.01532,0.00882,-0.00113,0.01312,-0.01906,-0.01337,-0.00535,-0.01444,0.00198,0.00258,0.00903,0.00103,0.01119,0.00056,0.00999,0.01188,-0.00024,-3e-05,-0.0027,0.01064,-0.00197,-0.00257,0.0165,-0.00807,-0.00257,0.01027,0.00101,-0.01116,0.03251,-0.00174,-0.00272,0.00222,0.00208,-0.01103,-0.00027,0.00266,0.00297,-0.0016,-0.01095,0.00128,-0.00451,0.00153,0.00303,-0.00532,0.01344,0.01002,0.00566,0.00128,-0.00327,0.00723,0.0026,0.00451,-0.02636,0.00029,-0.01169,-0.00424,-0.00998,-0.00711,0.00821,-0.00417,0.00444,0.00678,0.00054,-0.00463,0.01243,0.00085,-0.00312,0.00825,-0.00952,0.0011,0.0037,0.00459,-0.00529,0.0059,-0.01135,-0.00248,-0.00493,-0.00218,-0.00261,-0.00854,-0.00209,0.01276,0.00946,0.01434,-0.00247,0.00376,-0.01697,0.00509,0.00593,0.0017,0.0128,-0.00411,0.00958,-0.00877,0.00498,0.01683,-0.00489,-0.00617,-0.00328,0.00492,-0.00709,-0.0033,0.00432,0.01613,0.01055,-0.01169,0.00109,0.01419,0.00703,0.00222,-0.00167,0.00735,-0.01713,-0.00371,0.0005,-0.01047,-0.00916,0.00287,0.00159,0.00801,-0.00418,-0.0035,0.01294,-0.02603,0.00505,-0.00685,-0.01409,-0.0092,0.002,0.00072,-0.00069,0.00626,-0.00502,0.00235,0.00287,0.01459,0.00474,0.01053,0.003,-0.02059,0.0073,-0.00385,0.00324,0.00505,-0.00665,0.01043,0.00195,0.00155,-0.00039,0.00543,0.00396,0.00259,-0.01563,0.00346,-0.03555,0.00255,0.00205,-0.01284,-0.00848,0.01971,0.00335,0.00887,-0.01006,0.01134,0.0047,0.00478,0.00977,0.00103,0.01334,-0.00557,0.00548,-0.0046,-0.00369,-0.00377,-0.00152,-0.00376,-0.00313,-0.00461,0.00417,0.00805,-0.00956,-0.00651,0.00063,0.01206,0.00382,-0.00188,-0.00021,0.01079,-0.01111,0.00363,-0.00898,0.00181,-0.00089,-0.00278,0.00402,-0.01562,-0.00089,0.00931,-0.00635,-0.00299,0.00019,-0.00811,-0.00346,0.00514,0.00376,0.00121,0.00046,-0.01703,0.00518,0.00617,0.00496,-0.02388,-0.01802,-0.0089,0.01889,0.01472,-0.01026,0.00409,0.00103,-0.00051,-0.00045,-0.00491,-0.00015,0.00609,-0.00401,-0.00175,0.00553,0.00102,-0.01717,0.00518,-0.0026,0.00237,0.01345,0.00489,-0.00833,0.0022,-0.00273,-0.00968,0.00098,0.00249,0.00304,-0.00117,-0.00096,0.00031,0.00551,-0.01246,-0.01499,0.00884,-0.01253,0.00278,-0.00052,0.00865,0.00385,0.00484,0.00487,0.00988,-0.006,-0.00924,0.01169,0.00258,-0.00261,0.00126,0.00294,-0.00303,0.00899,-0.00107,-0.01591,0.03017,0.00523,-0.00956,0.00401,-0.00386,-0.00818,-0.00276,-0.00956,-0.00821,-0.00337,-0.00461,-0.00053,0.00611,-0.00848,0.0034,0.00187,0.00055,-0.00242,0.00325,0.00409,0.0001,0.01147,0.01328,-0.01365,-0.00531,0.00291,-0.01896,0.00061,0.00049,0.01273,-0.00771,-0.00328,0.00598,-0.00444,0.00091,0.00385,0.00138,-0.00094,-0.00464,0.01134,-0.00174,-0.00239,0.00286,-0.0021,0.00757,0.01032,0.00729,-0.00325,0.0056,0.01407,0.01533,0.00018,-0.01441,0.00397,0.00054,0.00217,-0.00163,0.00368,-0.00246,-0.00071,-0.0072,-0.00585,0.00365,0.00218,0.00286,-0.00248,-0.01714,-0.0007,-0.00218,-0.00888,-0.00301,-0.01028,-0.00659,0.00447,-0.01177,0.00882,0.00552,-0.00503,-0.01505,0.00845,-0.00793,0.00032,-0.00771,-0.00969,0.00455,-0.00768,0.00596,-0.00276,-0.00635,-0.00678,0.00123,-0.00348,0.00387,0.0019,-0.00093,0.00222,0.00198,-0.02103,-0.00732,-0.00829,0.00577,-0.00558,0.0021,-0.00389,0.0058,0.00424,-0.0077,-0.00708,0.01346,0.00493,0.00938,-0.02355,-0.01147,0.01223,0.00677,-0.00287,-0.0315,0.00219,-0.01603,-0.00537,-0.00304,-0.00481,0.00551,0.0092,-0.0134,-0.00195,-0.00595,-0.00794,-0.01382,0.01349,0.02147,0.00394,-0.00352,-0.00821,-0.00153,-0.00147,-0.00235,0.00605,0.00453,0.00024,0.00129,-0.01318,0.00928,0.0103,-0.00327,-0.00097,0.00892,-0.00099,0.01456,-0.00639,0.00394,0.00582,0.00315,-0.00046,-0.0045,-0.01045,-0.00205,-0.01355,-0.00204,0.00962,-0.00551,0.02138,-0.00785,0.00691,-0.00337,0.00969,-0.02088,-0.00464,0.02497,0.0068,-0.0008,0.00273,0.00234,0.00267,-0.01299,-0.00077,0.01489,0.00693,0.00131,-0.01157,-0.00047,0.0052,-0.01059,-0.0113,0.00887,0.02108,0.00051,0.00107,-0.0049,-0.01268,-0.0027,-0.01074,-0.00805,-0.00968,-0.01304,0.01033,-0.00229,-0.0023,-0.00055,0.00402,0.01239,0.00378,0.00137,-0.01224,-0.03459,-0.00551,0.00902,0.00051,0.00554,-0.0196,0.00036,0.00015,0.0062,0.00073,-0.00853,-0.01081,-0.0154,0.00821,0.00178,0.00073,-0.00774,0.00354,-0.01062,0.0025,0.00754,-0.00851,0.01278,-0.0066,0.00869,-0.0029,0.00375,-0.00534,0.0039,-0.00701,0.01489,0.00744,0.00082,0.00658,-0.00176,-0.00785,0.01588,0.00536,-0.01147,0.01849,0.00357,0.00309,-0.0036,-0.00169,-0.00422,0.00352,-0.00182,-0.01349,0.00926,0.00153,0.01269,0.00455,0.0006,-0.00517,-0.01805,-5e-05,-0.02463,0.00012,0.00019,0.00079,0.00777,-0.00488,-0.01443,0.01245,0.0006,0.01171,-0.01031,-0.00356,-0.00296,0.01955,0.01921,0.00015,0.00102,-0.00195,-0.00178,-0.00425,0.00736,0.00159,-0.00694,-0.00928,-0.00805,0.00036,0.00318,3e-05,0.0048,0.00853,-0.00393,-0.00948,0.00604,0.00239,-0.00558,-0.00751,-0.00496,-0.00422,0.00599,-0.00417,0.00448,-0.00398,0.00363,-0.00897,0.00184,0.00352,0.00533,-0.00352,0.00398,-0.00219,-0.00561,0.01297,-0.00012,0.01077,-0.00036,-0.01336,0.00936,-0.00547,-0.00193,0.00475,-0.0023,0.0101,0.01802,0.00225,-0.00365,-0.00469,0.00366,-0.00573,-0.01168,0.00105,0.00901,-0.00281,0.00978,0.00808,0.00996,-0.00122,-0.00249,-0.01021,0.00039,-0.00882,-0.00399,-0.01154,-0.00491,-0.00635,0.00958,-0.00809,-0.00948,-0.00559,0.00064,-0.00469,-0.00045,0.01313,0.00255,-1e-05,-0.00389,-8e-05,0.00641,-0.01854,0.00488,0.0053,-0.00125,-0.00459,-0.00164,-0.00953,0.00864,-0.0074,-0.00421,0.00147,-0.00672,0.0108,0.00788,0.00147,-0.01304,0.00441,-0.01236,0.0005,-0.00013,0.01246,0.00597,-0.00059,0.00572,-0.00264,-0.00234,-0.00347,0.00643,-0.00915,-0.00918,-0.00624,-0.00357,-0.00928,0.00119,-0.00677,0.00716,-0.00466,0.00248,-0.00313,0.00274,-0.00387,-0.00344,0.01025,-0.00121,-0.00666,-0.00307,0.00685,0.00345,-0.00656,0.00809,0.01845,0.01188,-0.00078,-0.01522,-0.0012,-0.00356,-0.01512,-0.00188,-0.00058,-0.00382,0.00241,-0.01089,0.00491,0.0082,-0.00255,-0.00108,0.0061,-0.00491,0.00933,0.01522,-0.01822,0.0093,0.01493,0.00732,0.01708,-0.00792,-0.01395,-0.00489,0.00295,0.0127,0.00958,-0.01522,0.0136,-0.01025,0.00109,-0.00922,0.00788,0.00287,0.00411,0.0059,0.00755,-0.00583,-0.01123,-0.00465,0.00221,-0.0046,0.00879,0.0067,-0.01545,-0.01189,0.0045,-0.00855,-0.00983,-0.00424,-0.00178,-0.0042,-0.00208,0.03427,-0.00626,-0.00364,0.01976,0.02385,-0.01167,0.00021,-0.00831,0.0006,0.00563,0.00922,-0.00216,0.00827,-0.00088,0.00883,-0.00189,-0.00791,-0.0002,0.00285,-0.02564,-0.01027,-0.0041,0.00513,0.01214,-0.00134,0.00401,-0.01948,0.01034,0.00282,-0.01491,-0.01086,0.00796,0.00786,-0.00671,-0.00915,0.00534,0.00507,-0.00688,0.0033,-0.00806,0.00513,-0.00961,-0.00164,0.00117,-0.00582,0.00155,0.00246,0.00066,0.00412,-0.00142,-0.0127,-0.00174,0.00929,0.00391,0.00297,0.00465,0.00765,-0.00145,0.0053,0.01003,-0.01024,-0.00403,-0.00233,-0.01464,-0.0137,-0.00322,-0.00044,0.01569,0.00239,0.00608,0.00252,-0.00361,0.00727,-0.01012,0.00092,0.01157,0.01249,0.00457,-0.00399,0.01101,0.00143,-0.00071,0.00321,0.00168,-0.01468,0.00019,0.00876,0.0001,-0.00307,0.00879,0.00983,0.00501,0.00158,-0.00783,-0.00744,-0.01424,0.00268,0.00889,0.00439,0.01672,-0.00699,-0.00505,-0.00011,-0.01665,-5e-05,-0.02099,0.0022,0.0119,0.00108,-0.01469,0.01463,-0.01649,-0.01348,0.00496,-0.00274],[0.00113,-0.00439,0.01546,8e-05,0.00522,0.00668,0.00066,0.00282,-0.00045,-0.0106,0.0101,-0.0042,-0.00504,0.0039,0.0047,-0.00773,0.00257,-0.00357,-0.00369,-0.0113,-0.01,-0.00647,-0.00961,-0.0004,-0.0023,-0.00603,-9e-05,-0.00147,0.01112,0.00077,-0.01128,-0.00076,-0.00288,0.01628,0.00526,0.01149,-0.00206,0.00474,-0.00054,-0.01169,-0.00111,0.00212,-0.01057,-0.0086,0.00081,0.0162,-0.00461,-0.00955,0.00241,0.00502,-0.00193,0.0025,-0.00693,0.00766,-0.00241,0.00178,-0.01767,0.00234,0.0021,0.00102,-0.01006,0.00551,-0.00498,-0.01547,-0.00572,0.00138,0.01899,0.01676,0.00147,-0.00793,0.00547,0.00129,-0.00415,-0.01685,0.00464,-0.0022,0.00463,-0.00872,0.00397,-0.00746,-0.00624,-0.00667,0.00535,0.00159,-0.01177,0.00752,0.00493,0.00128,0.00189,0.00462,0.00319,-0.00901,0.01407,-3e-05,-0.00704,-0.01128,0.00471,-0.00017,-0.00211,4e-05,0.00456,-0.01371,0.00506,0.01319,-0.01092,0.00796,0.00964,0.01967,0.00294,0.00929,0.00132,0.00139,-0.01145,0.00111,-0.00365,0.00054,-0.00059,-0.00209,0.00768,0.00878,0.00782,0.00084,-0.00679,-0.00054,0.00051,-0.00787,-0.00678,-0.00161,-0.00038,-0.00216,-0.0045,0.00675,-0.00363,-0.00066,0.0092,0.00058,-0.01808,0.00457,0.0132,-0.00224,0.01438,-0.01824,-0.00424,0.00334,-0.00591,-0.00632,0.00968,-0.03102,-0.0126,0.00021,-0.00963,0.00188,0.00121,-0.00287,0.00613,0.00719,-0.00538,-0.00314,-0.00032,-0.00178,0.01274,0.00215,-0.0071,-4e-05,0.00603,0.00015,-0.00728,0.00514,-0.00228,0.00802,-0.00678,-0.00101,-0.00242,-0.00278,0.00479,0.00707,-0.00504,-0.00277,-0.0003,-0.01161,0.00084,-0.00062,-0.00186,0.00211,0.00586,-0.0008,0.00068,0.01207,-0.00173,-0.01413,0.00461,0.00541,0.00201,0.01196,0.00379,0.00054,0.00094,-0.00159,0.00147,-0.00154,-0.00475,0.01102,-0.00082,-0.01317,-0.0042,0.0044,-0.00181,0.00359,-0.00327,0.00447,0.00016,-0.00212,-0.01615,0.00227,-0.0026,0.00035,0.00903,-0.00141,-0.01023,-0.00211,0.00485,-0.0035,0.0018,-0.01787,-0.00935,0.00672,0.00743,-0.00025,0.0058,0.00903,-0.00985,0.00491,0.00856,0.00391,0.00893,-0.00592,-0.00742,0.00191,-0.01346,0.00047,-0.00341,0.00378,0.00541,0.01056,0.00996,-0.00368,0.00781,0.00061,-0.00661,0.00298,9e-05,-0.00095,-0.00278,-0.00491,0.00495,-0.00589,-0.00144,0.00978,-0.01352,-0.01923,0.00544,-0.00775,-0.00993,0.01291,0.00609,0.00011,-0.01193,0.00036,0.00668,0.00311,-0.00256,0.00471,-0.00428,-0.01101,0.00729,-0.00327,0.00275,-0.01266,0.00465,-0.00495,0.00076,-0.00983,0.01695,0.00151,0.00772,-0.00522,-0.00949,0.00163,-0.00106,-0.00309,-0.01013,0.01178,0.00335,0.00099,-0.00111,-0.01117,0.00057,-2e-05,0.00762,0.00158,0.00035,0.00179,0.00709,-0.00162,-0.01367,0.00015,0.00637,-0.00274,0.00062,-0.00144,-0.00275,0.00351,0.00502,0.00835,-0.00399,0.0013,0.00123,0.00311,0.0096,0.00481,0.00912,0.00573,0.00469,-0.00819,-0.00372,-0.00178,0.00264,0.0129,0.00014,0.00145,0.00802,0.00092,0.00293,0.0099,0.00438,-0.00624,-0.00762,0.01879,-0.00752,-0.01095,-0.00783,0.00758,-0.01055,-0.00726,0.00336,0.01111,0.01661,0.00357,-0.00721,-0.00156,-0.00522,0.00281,-0.00044,-0.00058,0.01279,0.00338,-0.0183,0.00328,-0.00257,0.00949,-0.00922,-0.0092,-0.00122,0.00985,-0.00956,-0.00495,-0.00996,0.01246,-0.00509,-0.00871,0.00417,0.00777,-0.01133,-0.00065,0.00152,-0.00219,-0.00388,-0.00478,-0.00322,0.00166,-0.00572,0.0065,-0.01469,-0.00543,0.00393,-0.00694,0.00636,0.00898,0.0055,-0.00316,0.0057,0.00152,-0.0008,0.00864,-0.00686,0.00575,-0.00148,-0.00673,0.01808,-0.00305,-0.00294,0.01338,0.01276,-0.00035,-0.00362,-0.01261,-0.00267,0.01001,-0.02114,0.00932,0.00996,-0.00849,0.00505,0.00123,-0.01121,-0.00259,-0.00112,0.00596,0.0045,0.00045,-0.0052,-0.00206,0.00123,0.01777,0.0073,0.01459,-0.00328,0.00035,-0.00473,-0.00609,-5e-05,0.00846,-0.00229,0.00557,-0.01352,-0.0069,-0.00159,0.00122,-0.00444,0.01214,-0.01127,0.00042,-0.02196,-0.00227,-0.00707,-0.00221,0.00048,-0.01656,0.00615,0.00213,0.00461,-0.00481,-0.01697,0.01353,-0.00766,-0.00666,-0.00344,-0.00829,-0.00521,0.00215,-0.01202,0.006,0.01373,-0.00669,0.00533,0.00733,-0.00581,0.00171,-0.00062,0.0002,-0.00309,0.00459,-0.01121,-0.01188,0.00312,0.00719,0.00785,0.01049,0.00036,0.0073,0.00859,-0.00505,-0.00841,0.00283,-0.00273,0.00281,-0.0068,0.00752,0.00745,0.0077,0.00886,0.0113,0.00231,-0.00757,-0.00676,0.00212,-0.00069,-0.00371,0.00164,0.00674,-0.00731,0.00157,0.01183,-0.00583,-0.00985,-0.00142,0.01311,-0.02199,0.00204,-0.00516,-0.01409,0.0024,0.00338,-0.00255,0.01497,-0.00759,-0.00844,0.00209,-0.01091,0.00316,0.01026,0.00142,0.00081,0.00972,0.00966,0.0057,-0.00133,0.00684,0.00535,-0.00663,0.00332,0.0075,0.00678,-0.00389,-0.01563,-0.00361,-0.00555,-0.00188,-0.00345,0.00348,-0.01038,-0.00749,-0.00742,-0.00475,0.0099,-0.0005,-0.00016,0.00653,-0.00128,-0.01457,-0.00391,0.01114,0.01455,0.00393,-0.0047,-0.01838,0.00317,-0.00909,-0.00047,0.00682,0.00072,-0.0053,-0.00664,0.00487,-0.0007,0.00177,-0.00623,-0.00583,-0.00758,-0.00189,0.00551,-0.00631,0.00478,0.00471,0.00499,0.00885,-0.00369,-0.00198,0.01085,0.00422,-0.00109,-0.0026,-0.00475,0.00291,-0.0115,0.02002,0.0108,-0.00104,-0.002,-0.00335,0.00345,0.00355,-0.00299,0.00498,-0.00494,0.00033,-0.00665,0.00117,-0.00018,-0.00676,-0.00695,0.00843,-0.00191,0.00385,-0.00776,-0.01003,-0.00606,0.00077,-0.00068,0.03957,0.00734,-0.00099,0.00486,-0.00476,-0.00128,0.00044,0.00881,0.00795,0.00863,-0.00516,0.0012,-0.00099,-0.00385,-0.00742,0.00294,0.01084,0.00536,-0.00298,-0.01669,-0.00286,-0.00475,-0.00905,0.00418,-0.00591,-0.02555,0.0132,-0.00628,-0.01411,0.00491,-0.00859,-0.00128,0.00478,-0.01074,-0.00917,0.00038,-0.00168,0.00395,0.00378,-0.00281,-0.00028,0.01006,0.00466,0.00232,-0.00444,0.01271,0.01361,-0.01169,0.00619,-0.00257,0.00202,0.01658,-0.00622,-0.00279,0.00621,0.00084,0.00397,-0.01047,0.00538,-0.0025,0.0094,-0.00372,0.01064,0.00068,0.00237,0.00657,-0.0101,-0.00653,-0.01047,-0.00558,-0.0003,0.00762,-0.00232,0.00776,0.01036,-0.0088,-0.00322,-0.00776,-0.00475,0.00989,0.00027,-0.00481,-0.01072,0.00236,-0.0048,0.00076,-0.00193,0.00819,-0.0046,-0.00629,-0.0059,0.00335,0.01535,0.00534,0.00296,-0.00362,0.00897,-0.00687,-0.00363,0.00637,0.00372,0.0198,0.00524,0.00379,-0.00298,-0.00105,0.01202,0.00119,0.00271,-0.0118,0.00107,0.00988,-0.01563,-0.00859,-0.0007,-0.01077,-0.00795,-0.00753,0.00517,0.00133,0.01159,0.00857,0.00534,-0.0042,0.00024,-0.01377,-0.00398,-0.00849,-0.00527,-0.00324,0.00253,-0.00199,0.00332,0.02555,0.00855,-0.00671,0.00598,0.00999,0.00288,0.00465,0.00809,-0.00167,-0.00869,-0.01215,0.01185,0.00178,0.00388,-0.01212,-0.00767,-0.00504,-0.0053,-0.0043,0.00106,-0.01161,0.00053,-0.00828,-0.01285,-0.00715,0.01053,0.01003,-0.00246,0.00797,0.01101,0.00476,0.00486,0.00562,0.00029,-0.00467,0.00799,-0.00579,-0.00249,0.00425,0.00194,-0.01368,-0.00179,-0.01011,-0.00448,0.00283,0.00699,0.00862,-0.00129,-0.00786,-0.01634,0.00041,-0.00808,-0.01156,0.00187,-0.00232,-0.00401,-0.01021,0.01255,0.00308,-0.00188,-0.00277,-0.00299,-0.01129,-0.00339,0.00029,-0.00878,-0.00048,0.00573,-0.0068,-0.00365,0.01331,-0.00272,-0.00485,-0.00192,0.00395,-0.00759,-0.00814,-0.00122,0.00465,0.00075,0.00651,0.00347,-0.00133,-0.00314,0.00062,0.00247,0.00453,-0.0174,0.006,0.00575,0.00434,-0.00791,1e-05,0.00636,-0.01316,-0.00588,0.00316,0.00708,-0.00787,0.00797,-0.01449,0.01082,-0.00327,-0.01152,-0.01019,0.00423,-0.00902,-0.0045,-0.00075,0.00596,0.00295,0.0076,-0.00041,0.00124,-0.00357,-0.00537,0.00874,-0.00089,0.01268,0.00969,-0.00931,-0.00786,0.00423,-0.03274,0.0003,0.00022,-0.00474,0.00646,0.00152,0.00442,-0.00619,0.00323,0.00982,-0.00637,-0.0016,0.00826,0.0167,-0.00191,-0.00702,0.01217,-0.00203,-0.00032,0.00265,0.00753,0.0013,-0.00214,-0.00439,-0.00283,-0.00237,0.00605,0.00392,0.00234,-0.00833,0.0041,-0.00259,-0.00122,0.00969,-0.0045,-0.00527,-0.00833,-0.02322,0.00624,-0.01336,0.00856,0.00178,-0.00191,0.00228,0.00925,0.00926,-0.00263,-0.00022,-0.0122,0.00304,0.0012,-0.00037,0.00048,0.00368,-0.00559,0.00268,-0.0027,0.0028,0.00946,-0.00359,0.00204,0.00988,0.00357,0.00072,0.00869,0.00238,0.00185,-0.00243,0.00554,0.00732,-0.01223,-0.00783,0.00342,-0.0053,-0.00146,0.01091,-0.00779,0.00237,0.00685,-0.00111,-0.00776,-0.00559,0.00118,0.00956,0.00575,0.00411,-0.00575,0.00085,0.00187,0.01225,-0.00529,-0.00115,0.01123,-0.00312,0.01419,0.00383,-0.00033,-0.00582,0.00401,0.01659,0.00631,0.00106,0.01522,0.00179,-0.00444,0.00256,0.00014,0.00355,-0.00019,0.00473,0.00206,-0.00058,-0.00624,-0.00437,0.00732,-0.01864,0.01094,0.00122,-0.00082,0.00454,0.00303,-0.00471,-0.00156,-0.01079,0.00674,0.00112,-0.00824,0.00889,-0.00376,-0.02008,-0.01232,-0.00977,-0.00693,0.00548,-0.01072,0.00274,-0.00546,0.00814,0.0123,0.00017,0.00527,-0.0048,0.00345,0.00728,0.00809,0.01358,-0.00273,-0.00071,-0.00246,0.00194,-0.00713,0.0111,-0.00196,0.01414,0.00349,0.00819,0.00557,0.01124,-0.0022,-0.00039,-0.01196,-0.00352,0.00545,-0.00235,0.00122,0.00025,0.0038,0.00192,0.00143,-0.00268,-0.00888,0.00673,-0.00692,-0.00961,-0.00494,-0.00234,-0.00684,0.01441,-0.00254,0.00218,-0.00644,-0.00589,0.00535,-0.00423,-0.00522,-0.00509,0.00839,0.00321,-0.00154,0.00592,0.00481,0.00329,-0.01381,-0.00297,0.00193,0.01189,-0.0035,0.00471,0.00574,0.00696,0.00631,-0.00837,-0.00107,0.00146,0.00232,0.00666,-0.0056,-0.00204,-0.03789,0.01677,0.00048,0.0046,-0.00193,-0.00716,0.00218,-0.00772,-0.00258,-0.00718,-0.00437,0.00846,0.00025,0.00996,-0.0074,-0.00176,0.00677,-0.01346,9e-05,-0.00654,-0.00024,-0.00709,-0.00897,-0.01009,0.00559,-0.00995,0.00044,-0.00248,-0.01282,-0.00354,-0.00136,-0.00252,-0.00258,0.0037,-0.00666,-0.01041,-0.00386,0.00749,-0.00065,-0.00567,0.00293,0.01209,-0.00849,-0.0055,0.00528,0.00166,0.0012,-0.00389,8e-05,0.01086,0.01164,0.0034,0.00304,-0.0049,-0.00666,-0.00013,0.00565,0.00517,-0.01275,0.00896,-0.00844,-0.00136,-0.00239,0.00092,0.0116,-0.0033,0.00638,-0.01467,-0.01295,0.00451,0.00565,0.00319,-0.00145,-0.00069,0.0092,0.00265,-0.0121,-0.0049,0.0066,0.00975,0.00584,-0.00405,0.00441,-0.00173,0.00699,-0.00306,0.00711,-0.00691,0.00834,0.00163,0.00119,-0.00046,-0.00985,-0.00394,-0.00441,-0.00602,-0.00716,-0.00106,-0.0037,0.00513,-0.00273,0.00924,-0.00693,1e-05,-0.01329,-0.00532,-0.00182,-0.00435,-0.00201,0.02337,-0.0025,0.00308,-0.00421,-0.00731,0.00268,-0.00193,-0.00657,-0.00436,0.00308,0.00707,-0.0016,-0.00441,0.00394,-0.00774,-0.00943,-0.02356,-0.00034,0.01388,-0.00221,-0.00041,0.00142,0.01382,-0.00722,0.01241,0.00695,-0.00436,0.00662,0.00136,0.00059,0.0059,0.00714,-0.0137,0.01402,-0.00077,-0.01104,0.00346,0.00148,0.00245,-0.00516,-0.00374,-0.00257,0.00234,0.00047,0.0112,-0.00611,0.01396,0.03039,-0.00258,-0.00148,-0.00294,0.00942,-0.00575,0.00406,0.00556,-0.0075,-0.00452,0.00051,-0.00835,-9e-05,-0.00513,-0.00123,0.00495,0.00374,-0.00393,-0.00498,-0.00895,-0.00133,0.00677,0.00093,-0.01392,-0.01638,0.00584,-0.0002,-0.00718,0.0024,0.0107,0.00804,0.00087,0.00288,0.00436,-0.00651,0.0001,0.01057,0.00067,0.00401,-0.01341,-0.0092,-0.01346,-0.00565,-0.00034,-0.00722,0.00289,-0.00119,0.00435,-0.01228,0.01594,-0.00579,-0.00903,-0.00175,-0.00509,0.00061,0.00156,0.00431,-0.00227,-0.0001,-0.00146,0.01441,0.00847,0.00205,0.00712,-0.011,-0.00893,-0.02511,0.00606,0.00698,-0.00454,0.01218,0.00361,0.00421,0.00737,0.00589,-0.00435,0.00412,-0.01107,-0.00669,-0.01034,-0.00105,0.01008,-0.00426,-0.00612,-0.02048,-0.00389,-0.01302,-0.0062,0.00381,-0.00019,0.00333,-0.01039,-0.00938,-0.00322,-0.00488,0.00379,0.00175,-0.00751,0.00896,-0.00544,-0.00501,0.00204,1e-05,0.00307,0.00549,0.00837,0.01517,-0.00303,-0.00241,-0.00749,0.00734,-0.00072,0.00744,7e-05,-0.00019,-0.00362,0.00376,0.00414,0.00611,-0.00092,0.00926,-0.00783,0.01041,0.0055,0.00509,0.01762,-0.00148,-0.00143,-0.0107,-0.0013,0.00265,0.00198,0.0003,0.00181,0.00776,-0.00527,-0.00557,0.00044,-0.00462,0.00102,-0.00093,-0.00426,0.00734,0.01513,-0.00484,0.0038,-0.00314,0.00347,-0.00904,0.01055,0.011,-0.00545,0.00354,0.00628,-0.00755,0.00368,-0.00062,-0.01264,-0.00182,-0.00391,-0.01002,0.00189,-0.00843,0.00204,0.00728,0.00408,-0.00583,0.00236,-0.00248,-0.01097,-0.00168,0.00287,-0.00231,-0.00144,0.00019,0.00054,-0.00775,0.0019,-0.00365,-0.00088,-0.01479,-0.00304,0.00494,-0.00796,0.00155,0.00199,-0.00928,0.00473,-0.00661,0.00758,0.01019,0.00133,-0.03391,0.0067,0.00953,-0.03704,0.00316,-0.00599,-0.00414,-5e-05,-0.00279,0.00125,0.00357,-0.01143,0.00187,-0.00123,-0.00392,0.00091,0.00906,-0.00543,-0.00447,0.00514,0.00058,-0.00024,-0.00026,-0.00351,0.00091,-0.01214,0.00895,0.00912,-0.01044,-0.01609,0.00233,0.01763,-0.00181,0.00915,0.00487,0.011,0.01764,0.00207,0.00022,-0.00581,0.00738,-0.00322,-0.00064,-0.01085,0.01046,0.01251,0.00628,0.00046,-0.00332,-0.00022,-0.00147,0.00482,-0.00611,0.00316,0.00514,-0.00336,-0.00466,0.00018,0.0044,0.00876,0.00328,-0.00245,0.00372,0.01803,-0.00121,0.00337,-0.00729,0.00192,0.01353,0.00814,-0.00105,-0.01044,-0.00379,0.01094,-0.0048,-0.00125,0.00045,-0.00511,-0.01276,-0.00149,-0.00557,0.00196,0.00759,0.00576,0.00161,-0.01999,-0.00437,-0.00227,0.00336,-0.01083,-0.00842,0.00841,-0.00228,-0.0076,-0.01223,-0.01185,-0.00612,-0.00518,0.0008,0.00195,-0.00458,0.01793,-0.014,0.01869,-0.00592,-0.01015,0.00509,-0.00082,-0.00203,0.00722,-0.00233,-0.01658,-0.00137,-0.00362,0.01355],[-0.00591,-0.00662,0.01047,-0.00614,0.00491,9e-05,0.00152,0.01114,-0.0104,-0.00767,0.00652,0.00899,-0.00752,-0.00304,0.00833,0.00279,-0.0091,0.00417,-0.00677,-0.00445,-0.01293,0.01948,-0.00753,-0.0052,-0.00377,-0.01537,0.00206,-0.01116,0.00893,-0.00341,-0.00318,0.00255,8e-05,0.01009,-0.00051,0.0081,0.01212,-0.00825,0.00151,-0.0093,0.00754,-0.00172,-0.01357,0.0036,0.01455,-0.00039,-0.00109,-0.00497,-0.00558,0.00603,0.00177,-0.0068,-0.00572,0.00026,0.0008,-0.00308,-0.00827,0.00232,-0.01212,0.00446,-0.00147,0.00228,-0.00422,-0.00589,-0.00089,-0.00636,0.00774,0.01262,0.01778,0.00062,-0.00049,-0.00409,-0.00925,0.00179,0.00355,0.00998,-0.00163,0.00079,0.01122,-0.0044,0.01928,0.00221,-0.01727,0.00077,0.00133,-0.00242,-0.00978,0.01079,-0.00749,0.00236,0.00397,-0.00364,0.00687,-0.00903,0.00044,0.00114,-0.00408,-0.00432,-0.00788,0.00111,-0.00259,-0.00965,0.00023,0.0075,0.00743,0.00901,0.00144,0.00532,0.00362,0.00791,-0.00264,0.00306,0.00491,-0.00287,-0.0001,0.00547,-0.00707,-0.00526,0.00067,0.01241,0.00546,-0.00293,-0.00161,0.00114,0.01204,-0.00567,-0.00497,-0.0034,0.00257,0.00037,0.00545,0.00651,1e-05,-0.00404,0.00032,0.01125,-0.00264,0.00265,0.00774,0.0023,-0.00546,-0.01417,0.00563,-0.01074,-0.00312,0.0047,0.00419,-0.0012,-0.01072,0.01018,0.00652,-3e-05,-0.00513,0.00661,0.00382,-0.00083,-0.0033,-0.00594,0.00463,0.00548,-0.00981,0.00077,-0.00688,0.00755,0.00677,0.01497,-0.01178,0.00328,-0.00226,-0.00207,-0.00749,-0.00631,0.00372,-0.00786,0.0112,-0.00721,-0.00343,0.0075,0.00743,-0.00684,-0.00592,0.00135,0.00071,-0.0062,-0.00222,0.00371,-0.00115,0.00096,-0.00447,-0.00657,0.00177,-0.00242,0.00247,0.00656,0.00049,0.00259,-0.00046,0.01237,-0.0013,-0.00214,-0.00183,0.00261,-0.00138,-0.0064,-0.00262,-0.01381,-0.00202,0.00028,0.0001,-0.00741,0.0065,0.00774,-0.00444,-0.01031,-0.00558,-0.00591,0.00183,-0.00456,-0.00372,0.00295,-0.00502,0.00288,-0.00245,-0.01237,-0.01477,0.00955,-0.00322,0.00569,-0.01167,0.00646,-0.01467,0.00779,0.00561,0.00222,0.00724,-0.00946,0.00023,-0.00446,-0.00107,0.0004,-0.00436,-0.00058,0.00272,0.01224,-0.00961,-0.00507,0.00869,0.00637,0.00886,0.00189,-0.00368,0.00101,0.00061,0.00853,0.00184,0.01176,-0.00631,0.0109,-0.00101,-0.0187,0.01561,-0.00403,-0.01158,0.00814,-0.00156,-0.00482,0.00255,-0.00173,-0.00497,0.0044,-0.00507,0.00051,-0.00377,-0.01337,0.00633,-0.00267,0.00797,0.01238,0.0037,-0.00665,-0.00487,0.00276,-0.00702,-0.00699,0.00669,-0.00688,-0.02199,0.00201,0.00099,0.00463,0.00143,0.00417,0.0084,0.00615,-0.01023,-0.00293,-0.00659,-0.01567,-0.0019,-0.01523,0.00675,0.01218,0.00281,-0.00634,-0.00071,-0.0107,0.01384,0.00165,-0.00508,-0.0015,-0.00869,-0.00287,0.00319,-0.00053,-0.00702,0.0001,-0.00918,0.00414,0.00942,-0.00829,0.00249,0.00294,-0.00272,0.01434,-0.00267,0.00573,0.00451,-0.00141,0.00827,0.00159,0.00553,0.00049,0.0122,0.00656,-0.0087,0.00707,0.00936,0.00032,0.00309,-0.00267,-0.00311,0.00574,0.00399,-5e-05,-0.00062,-0.00445,0.01129,-0.00482,-0.00747,0.00593,-0.01051,3e-05,0.00731,0.0002,-0.0061,0.00213,0.00707,-7e-05,-0.00062,0.00514,0.00338,0.00437,-0.00308,0.01291,0.0054,0.01249,0.00015,0.00186,0.00586,-0.00427,-0.00456,-0.00029,-0.00627,-0.00534,-0.00738,-0.00715,0.00683,0.00153,-0.00408,-0.00315,-0.00151,0.00336,-0.00631,0.00417,-0.00841,-0.00987,0.00703,-0.00795,-0.00209,-0.00147,-0.00346,-0.01205,0.00926,0.01153,-0.00147,0.00452,-0.00397,0.00757,0.00502,0.00532,-0.00263,0.00738,0.01539,0.00154,-0.00231,-0.00489,0.00305,-0.002,-0.0192,0.0034,-0.01989,-0.00287,-0.00461,0.00174,0.00638,-0.00922,-0.00589,0.00157,0.01625,-0.00199,-0.00185,0.00542,0.00551,-0.00193,0.01647,0.01139,9e-05,0.00248,0.0072,0.00849,-0.00669,0.00194,-0.00356,-0.00522,-0.00533,-0.00752,0.00826,0.003,-0.01133,0.0074,-0.01726,-0.0124,0.00279,0.00121,-0.00209,0.00397,-0.00144,-0.01109,0.00617,0.00584,-0.00159,0.00134,-0.00983,3e-05,0.00775,0.00454,0.00216,0.00512,-0.0066,-0.00211,-0.00313,0.00594,0.00437,-0.01758,-0.00408,0.00765,-0.0065,-0.00517,0.00376,0.00292,0.0058,0.00194,-0.00279,0.01105,-0.00334,0.00807,-0.00562,-0.01139,0.00206,-0.00341,0.00277,-0.00532,-0.00802,-0.00243,0.00672,0.00041,-0.0135,0.00447,0.01281,0.0036,-0.00764,0.0092,-0.00494,0.00047,-0.01119,-0.00063,-0.00106,0.00314,-0.00181,0.00579,-0.0063,-0.00387,-0.00629,-0.00471,-0.00871,0.00358,-0.00287,-0.00873,0.01769,-0.00179,0.00067,-0.01012,0.00152,-0.00261,0.00129,-0.00739,0.0016,-0.00577,-0.00571,-0.0215,0.00494,0.01096,0.00896,0.00805,0.00203,-0.0002,-0.00019,0.00666,0.00367,-0.00262,-0.01459,-0.01246,-0.00581,0.00299,0.00048,-0.00343,-0.00067,-0.00544,-0.00318,0.00192,-0.005,-0.00678,-6e-05,-0.00459,0.00112,0.00091,0.01267,0.00093,-0.00168,-0.00549,-0.00431,0.00367,0.00606,-0.0105,0.00531,-0.00432,0.01184,-0.00667,-0.00512,-0.00023,-0.00502,-0.00473,-0.00751,-0.00052,0.00503,-0.0051,-0.00655,0.00698,-0.00143,-0.00013,-0.01034,0.00207,0.00057,0.0067,-0.01218,0.00398,0.00788,0.01431,-0.00137,-0.00371,-0.00211,-0.00114,0.00407,-0.004,0.00071,0.01032,-0.00533,0.0016,-0.00212,0.00386,0.00752,-0.00601,-0.0148,0.00759,-0.00168,0.00043,-0.01514,-0.00495,0.00153,0.00341,0.00627,-0.00447,-2e-05,0.00336,0.00282,-0.00113,0.00329,-0.00267,0.01203,0.01172,0.01149,0.00989,0.00375,-0.00346,-0.00893,0.00691,0.00334,0.00554,-0.00529,-0.00102,-0.00477,-0.0017,0.01156,0.00517,0.00076,0.00485,0.01333,0.00526,-0.00231,0.00369,-0.00355,0.01163,0.00839,-0.00112,-0.00027,0.00575,0.00017,0.00154,0.00692,-0.00461,0.00074,0.00067,-0.0048,0.0064,0.00821,0.00108,0.00545,0.00282,-0.00761,0.00327,-0.00609,0.00546,0.00237,-0.00755,0.00095,0.0108,-0.00605,0.00129,-0.00465,-0.00825,0.00284,0.02092,0.00029,0.00639,0.00204,0.00255,0.01585,0.01193,-0.01177,0.01345,0.00372,0.00633,-0.0007,-0.01334,-0.00455,0.00557,-0.00492,-0.00465,0.00105,-0.00117,0.00015,-0.00458,0.00335,-0.00464,-0.00864,-0.0017,-0.00444,-0.00139,0.01016,0.00226,0.00106,0.00923,-0.0077,0.00127,-0.00147,-0.00874,0.0016,-0.01056,-0.00497,0.00556,0.00355,0.00969,-0.00429,0.00508,0.00014,-0.00465,-0.00749,0.01393,-0.00304,0.00235,0.01469,-0.00927,0.00564,-0.00429,-0.01052,0.00742,0.0039,-0.00335,-0.00248,-0.00196,0.0066,-0.00408,-0.01075,-0.00695,-0.00418,-0.00234,0.00031,0.0112,0.0006,0.01047,-0.00111,0.0099,0.00226,-0.01253,-0.00279,-0.00452,-0.00566,0.00019,0.00604,-0.00192,-0.00533,-0.00748,-0.0065,-0.00157,-0.00933,0.00476,-0.01255,0.00145,0.00246,-0.00574,0.00129,0.00523,-0.00321,2e-05,0.00838,0.00367,0.01155,-0.0073,-0.01565,-0.00377,-0.00442,-0.0042,-0.006,0.00397,-0.01397,-0.00184,-0.00624,-0.00296,0.00744,-0.00765,0.00033,0.00786,0.00073,-0.00454,0.00039,-0.00712,-0.00566,-0.00305,-0.0045,-0.00645,0.00497,0.00192,0.00816,0.00459,-0.00266,0.01067,0.00315,0.01287,0.00592,0.00514,-0.01543,0.00068,0.00079,0.00377,0.00089,0.00349,-0.00676,-0.00388,-0.00027,0.00857,0.01334,0.00326,-0.01154,-0.00241,-0.00159,-0.00902,-0.01118,-0.00984,0.0022,0.00997,-0.00607,-0.00115,-0.00048,-0.01044,-0.00091,-0.00255,-0.00972,-0.01615,0.00408,0.0065,0.00254,0.00461,0.00055,-0.00576,0.00048,-0.0016,0.00292,0.01713,0.00849,-0.01087,-0.00157,-0.00156,0.0079,-0.00759,-0.01578,0.00201,-0.00033,4e-05,0.0085,0.0066,-0.00134,-0.00323,-0.01925,0.01101,-0.00693,0.00286,0.00246,0.00588,-0.01741,0.00668,0.01257,-0.00265,0.00071,0.00329,0.0104,0.00571,0.00229,-0.0054,0.00894,0.00825,0.00754,0.00016,-0.00566,-0.00891,-0.00761,-0.01142,0.00424,9e-05,-0.00564,0.01042,0.01035,0.00172,0.0074,0.00569,0.00701,-0.0075,-0.00233,0.01411,0.00223,0.00051,-0.01257,-0.01051,-0.00146,0.00906,0.00621,-0.00631,0.00554,0.00279,-0.00264,-0.00331,0.00423,0.00125,-0.00466,-5e-05,-0.00519,0.00463,-0.00154,-0.01052,-0.00129,0.00654,0.0061,-0.01736,-0.02089,-0.00098,0.00886,0.01248,-0.01264,-0.0041,-0.00067,-0.00531,0.00961,-0.01013,0.00497,-0.00555,-0.00353,0.00554,-0.00171,-0.0119,-0.00797,0.01067,0.00778,-3e-05,0.01258,-0.00083,-0.01299,-0.00527,0.00198,-0.00152,0.00191,0.00674,0.00427,0.00287,0.00375,-0.0066,0.0037,-0.00346,-0.00515,-0.00301,-0.0074,-0.00253,0.00823,-0.01434,0.00114,0.00777,0.00448,0.00536,-0.00954,0.00841,0.00679,0.0015,-0.00205,0.00549,0.00637,-0.00147,0.00466,0.00018,-0.01832,0.00289,0.00093,0.01006,0.00337,-0.00085,-0.00876,0.00667,0.0022,0.00101,0.00063,-0.00176,-0.00661,6e-05,-0.00314,0.00124,-0.00253,0.00068,0.00613,-0.00351,-0.00028,-0.00605,0.00059,0.00342,-0.00803,0.00262,0.00108,-0.00895,0.00043,0.00371,-0.00018,-0.00186,-0.01223,0.00953,-0.00032,0.00505,0.01136,0.00032,-0.01903,-0.00794,-0.00363,-0.00738,-0.00084,0.00473,0.00323,0.00191,0.00025,0.00327,0.0003,0.00452,0.00765,-0.00142,0.00891,0.00241,0.0046,-0.0059,0.00111,-0.00843,0.00553,0.00036,0.00716,-0.00012,0.00691,0.00428,0.00485,0.00209,0.00613,-0.00965,-0.00266,-0.0009,-0.01045,0.00721,-0.00578,0.00044,-0.00146,0.00674,0.00025,-0.00547,0.00118,-0.00514,0.00252,-0.01176,0.00549,-0.00729,0.0088,0.00758,-0.00363,0.00064,-0.00138,0.0035,0.00115,0.01231,-0.0023,-0.00127,-0.00158,-0.00079,-0.002,0.01118,-0.00151,-0.0074,0.00512,-0.00799,-0.00801,-0.00523,0.01125,0.00293,-0.00403,-0.00557,-0.00656,0.00952,-0.00108,0.00292,0.00558,-0.00738,-0.00429,0.00294,0.00399,-0.02288,0.01275,-0.00493,-0.00083,-0.00201,-0.00667,0.00333,-0.00204,-0.00428,0.00751,-0.0045,-0.00481,0.00555,0.00904,-0.00124,0.00011,0.00782,-0.0123,0.00021,-0.00092,-0.00078,-0.0006,-0.00011,0.0003,0.00668,-0.00133,0.00609,0.00094,0.00093,-0.00182,0.00762,-0.00793,0.00044,-0.00473,0.00397,0.00224,-0.0079,-0.00323,0.00487,-0.00823,0.00559,0.00958,0.00702,0.00487,0.00156,0.00774,-0.00501,-0.00446,0.00562,0.00382,-0.003,0.00018,0.01343,0.00162,0.00459,0.00226,0.00951,-0.00831,0.00532,0.00414,0.00837,0.01104,0.00396,-0.00462,0.01536,0.00416,-0.00118,-0.00258,0.01692,0.00455,0.01955,0.00648,0.00511,-0.00808,0.00028,0.00229,-0.00302,-0.00678,-0.00408,-0.00045,-0.00782,-0.00281,-0.00117,-0.00476,0.00216,0.00697,-0.00175,-0.01435,-0.02324,-0.00464,-0.00344,-0.00258,-0.00563,-0.02314,0.00095,0.01242,0.005,0.00799,0.00617,-0.00784,0.00684,0.00345,-0.00802,0.00719,-0.00309,0.00624,0.00352,-0.00288,3e-05,0.0045,0.00078,0.0151,0.00207,-0.00295,0.00025,0.00061,-0.00399,-0.00065,0.00442,0.00717,-0.01099,-0.00169,-0.00106,-0.0103,-0.01051,-0.00591,0.00123,0.00106,-0.00471,0.00243,0.00446,0.0037,-0.00346,0.01363,0.00125,-0.00714,0.00104,0.0019,-0.00384,0.00588,0.00037,-0.02966,-0.00503,-0.00476,0.00701,0.00971,0.00198,0.00643,-0.00137,0.00488,-0.00804,0.00222,0.0053,0.00395,0.01585,0.00237,0.00275,0.00139,0.00055,-0.0054,0.00333,0.00139,0.00687,-0.0019,0.00226,-0.00298,0.00301,-0.00777,0.0009,-0.00875,-0.00111,0.01301,0.00597,0.01965,-0.00528,0.00416,0.0009,0.00187,0.00273,0.00114,-0.01322,2e-05,0.0052,0.00581,0.00486,-0.00275,-0.00324,-0.01011,0.00128,-0.00408,0.00365,0.00221,-0.00229,0.00042,0.00471,0.00649,-0.00051,0.00098,-0.00013,-0.01001,0.00118,0.0018,0.00763,-0.00636,-0.00149,0.00775,0.00394,-0.00656,0.00275,-0.00698,-0.00803,-0.00396,0.00391,0.00527,0.00304,-0.00144,-0.00351,0.00121,0.00648,0.00751,-0.00052,-0.01195,-0.01321,0.0022,0.00452,0.00154,-0.00871,-0.00147,0.01148,-0.00052,-0.00643,-0.00259,-0.00382,-0.00671,-0.00977,-0.00431,-0.00675,0.00862,-0.00349,-0.00575,-0.0147,-0.0109,-0.00038,0.00402,-0.00218,0.00165,0.00486,-0.00184,-0.00028,0.00059,-0.00876,0.01094,0.00115,-0.00696,0.00684,-0.00741,-0.01639,0.00244,-0.00013,0.00191,-0.0064,-0.00503,0.01221,0.00558,0.00537,0.00606,0.00571,-0.00773,-0.0008,-0.00443,-0.00977,-0.00608,-0.00208,0.00086,-0.00678,-0.00935,0.00178,-0.00578,0.00883,0.00787,0.00666,0.00047,-0.01455,0.00293,-0.00684,-0.00225,0.0012,0.01513,0.00377,-0.0068,0.01461,-0.00377,0.00946,-0.0055,0.00033,-0.0012,-0.00157,-0.00518,0.00178,0.00518,-0.00719,-0.01455,-0.00643,0.00464,-0.00056,-0.00731,0.00375,0.00906,0.00362,0.01064,0.00596,-0.00503,-0.00243,-0.00678,-0.00083,0.00065,-0.00318,0.00627,-0.00667,-0.00211,0.00326,0.0076,-0.00925,0.01215,-0.00317,0.00084,0.01112,0.00476,-0.00627,0.00489,0.00137,-0.006,-0.00014,-0.0028,-0.00307,-0.00096,-0.009,-4e-05,0.00541,-0.00888,0.00275,-0.0023,-0.00169,-0.0054,-0.00667,-0.00372,0.00577,-0.00137,0.02741,-0.01603,0.00242,0.03076,0.00511,-0.00472,-0.0001,-0.00355,-0.00196,0.00733,0.00153,-0.01298,0.00826,0.0045,0.00089,-0.00226,0.00611,0.00543,-0.00879,-0.0051,-0.00097,0.0029,0.00486,-0.00881,-0.00772,0.01023,-0.00465,0.00407,0.00117,-0.00376,0.00185,-0.00745,-0.002,-0.00294,0.00672,0.00072,0.00526,0.00149,0.0023,-0.02197,-0.00121,-0.00092,-0.00172,-0.00977,-0.00333,0.00207,-0.00326,0.00121,-0.00103,-0.00576,-0.00099,-0.00496,0.00751,-0.00409,0.00781,-0.0132,0.01785,0.00897,0.00756,0.00281,-0.01118,-0.00078,0.0009,0.00882,-0.01036,0.00296,-0.00238,0.00206,0.00232,0.00554,0.00367,-0.00232,0.0009,-0.00545,0.00251,0.00321,0.00884,-0.00233,-0.00014,-0.00144,0.00031,-0.01265,-7e-05,-0.00461,-0.00647,-0.00559,-0.00306,-0.00107,0.00416,0.00174,-0.01226,0.00692,-0.0049,0.00117,0.00012,-0.00835,0.00176,-0.00697,0.00088,0.00285,-0.00631,0.01412,0.00166,-0.00238,0.00114,-0.01316,0.00643,0.00058,-0.00961,0.00175,0.01707,0.00032,8e-05,0.00067,0.0069],[-0.00403,-0.00695,-0.00397,-0.01194,0.00708,0.01689,-0.01058,0.00632,-0.00943,-0.00737,-0.00334,-0.00468,-0.00095,-0.00526,0.00159,0.00408,0.00219,0.00206,-0.0055,0.00081,0.00736,0.00469,-0.01003,0.00833,-0.00094,0.0005,-0.00911,0.00108,-0.00459,-0.00695,0.00194,0.0003,0.00277,0.00067,-0.00211,-0.00543,0.00518,0.00886,-0.01735,0.01104,-0.00176,0.00842,-0.00277,0.00674,0.00711,0.00787,0.00684,0.00594,0.00668,0.00409,0.00504,0.01547,-0.0064,-0.00328,0.00369,0.00233,0.00045,0.00968,-0.0028,-1e-05,0.00929,0.00283,0.00445,0.01113,-0.0049,-0.0008,-0.00746,-0.00828,-0.0023,-0.00742,0.00407,0.00155,0.00158,-0.00332,0.00675,0.00731,0.00078,0.00836,-0.00024,-0.00464,0.01324,-0.00691,-0.00032,-0.00459,-0.01067,-0.00318,0.00088,-0.00622,0.00151,0.0038,0.00306,-0.00557,0.00389,-0.00373,0.00442,-0.00274,0.00207,-0.0036,0.00448,0.0085,-0.00453,-0.00035,-0.01639,-0.00135,-0.0104,0.00638,0.00256,0.00338,-0.01044,0.00457,0.00463,0.00269,0.00041,-0.00441,-0.01245,0.007,-0.00241,0.00355,0.01185,-0.00482,0.0042,0.00242,0.00558,-0.00553,-0.0055,-0.00534,0.00456,0.00301,-0.0142,-0.00261,0.00367,0.00344,0.00021,-0.01266,-0.00012,-0.00018,0.0043,-0.00455,-0.0072,-0.01057,-0.00585,-0.00993,-0.00471,-0.00112,-0.01006,-0.00886,-0.0017,-0.0142,0.00949,0.0028,0.01237,-0.00901,-0.00557,-0.00116,0.00024,0.00439,-0.00746,0.01146,0.00377,-0.00895,-0.00322,-0.00615,-0.00196,0.00151,0.0012,0.00641,-0.01306,-0.00087,0.0016,0.00174,-0.00107,0.01326,0.00449,0.00165,0.01143,-0.0077,0.00894,-0.00265,0.01345,-0.00262,-0.00459,0.01159,-0.00449,-0.00333,0.00146,0.01178,-0.00651,-0.01546,-0.00024,0.00013,0.00553,-0.00708,0.01846,-0.00524,0.00863,-0.00751,-0.00365,0.00854,-0.00196,0.00834,0.00509,-0.00706,-0.0177,0.00606,0.009,-0.00895,-0.00274,0.00174,0.01218,-0.00498,-0.00064,0.01861,-0.00331,-0.00297,-0.00194,0.00115,0.01076,-0.00052,-0.00194,-0.00202,0.01094,0.00159,0.00092,0.0064,0.00104,-0.00379,-0.0057,-0.01043,0.00469,-0.00498,0.0,-0.00559,-0.01233,-0.0088,-4e-05,0.00656,-0.00253,-0.01072,-0.01774,0.00527,0.00589,-0.00102,0.0059,-0.0049,0.00202,-0.00456,-0.01044,-0.0031,-0.00622,0.00447,-0.00381,0.00849,-0.00529,0.00756,0.00385,0.00072,0.00229,0.01309,-0.00401,-0.00208,-0.00591,0.00635,-0.006,-0.0031,-0.00371,-0.00133,0.00187,0.00831,-0.0053,-0.00236,-0.01052,-0.00034,0.01654,0.0116,0.00392,0.00139,-0.00132,0.01112,-0.00346,-0.00806,-0.00397,-0.00427,0.00012,0.01348,-0.00586,-0.00385,-0.01671,-0.003,-0.0009,-0.00217,-0.00903,-0.00151,-0.00367,-0.00464,-0.00383,0.00221,-0.00937,0.00459,-0.00235,-0.00326,0.00069,0.01241,-0.00734,0.01258,0.00526,-0.00155,-0.00307,-0.0021,-0.00485,0.00056,0.00655,0.00276,-0.02049,-0.00554,-0.00684,0.00615,-0.01047,-0.00788,-0.00566,-0.0049,0.00324,0.00474,-0.00208,0.01477,-0.01547,-0.00863,-0.01057,-0.00135,-0.01677,0.01602,0.00278,-0.00356,-0.00551,-0.00496,0.00324,-0.00272,0.00248,-0.00744,0.00082,0.00856,-0.01627,0.00299,-0.00354,0.00177,-0.00143,-0.01508,-0.01226,-0.00143,-0.00498,-0.00046,-0.01158,-0.00926,0.01251,-0.00116,-0.00398,-0.00115,0.0048,-0.01807,-0.00803,-0.00404,-0.00505,-0.00098,-0.00026,-0.00331,-0.00182,0.01259,-0.00658,0.00319,-0.00211,0.00071,-0.0054,-0.01113,0.0007,-0.01042,-0.00226,-0.00073,0.01036,0.00653,0.00887,0.00426,0.00689,0.00852,0.00289,0.00281,-0.00688,-0.00885,0.00449,-0.0019,-0.00338,-0.00509,0.01576,-0.00078,-0.00318,0.01197,0.00276,0.02102,0.02316,-0.00823,0.00853,0.00693,-0.00933,0.00414,-0.01267,0.0058,0.00139,-0.00513,0.00068,-0.00402,0.00611,0.01279,-0.008,-0.00078,-0.00592,-0.00816,0.00356,0.00582,0.00339,0.00204,-0.00565,-0.00968,-0.00125,0.00414,-0.0053,-0.01611,0.00307,-0.00426,0.00162,-0.01614,-0.00537,-0.00997,-0.00363,0.00791,-0.00634,-0.00272,0.01004,-0.00144,6e-05,-0.01048,0.01387,0.00269,0.00394,0.00639,0.00023,0.00042,0.00582,0.01461,-0.00369,0.0042,0.00479,0.01245,-0.00098,0.00619,-0.00241,0.00329,-0.01091,0.00689,-0.00533,0.00448,-0.002,-0.01338,0.00968,-0.00023,0.00079,-0.00375,0.00401,0.00446,-0.00179,-0.00929,-0.01058,-0.01335,0.00988,-0.00842,-0.00738,-0.00463,-0.0049,0.00622,0.00184,0.0001,0.00027,-0.01523,-0.00351,0.00887,-0.00536,-0.00299,0.01621,-0.00244,-0.00583,0.0041,0.0054,0.0014,0.01402,-0.01156,0.00147,-0.00504,0.00659,0.00231,-0.00433,-0.00045,0.00177,0.00262,0.02059,0.00247,-0.00031,-0.00443,-0.00794,0.00351,0.00035,-0.00989,-0.00245,-0.00877,0.00902,-0.00978,0.00138,0.00993,-0.00054,-0.01381,-0.00137,-0.00037,-0.00975,0.01435,-0.02745,0.00875,0.00594,-0.00478,-0.01083,0.00518,-0.00927,0.00997,-0.00676,0.00935,0.00929,0.01323,0.00433,0.00734,-0.00581,0.00215,-0.00042,-0.00275,-0.00523,0.00948,1e-05,0.01236,0.00236,0.01042,-0.0005,-0.00179,0.00092,-0.01136,0.00253,0.00449,0.00865,-0.00654,0.00081,0.00161,0.00907,0.00311,-0.00796,0.00652,0.00494,-0.00707,0.00678,-0.00749,-0.00421,0.00788,0.00647,0.00971,-0.00784,-0.00061,-0.0039,-0.00368,-0.0152,0.00683,-0.0027,-0.00281,-0.00542,-0.01478,0.00082,0.00806,0.00329,-0.01528,-0.03918,-0.00286,-0.00436,-0.00767,0.00695,0.00743,0.00507,-0.00259,-0.00932,0.00482,-0.00491,0.00207,-0.01163,-0.00312,-0.00812,-0.00473,-0.01518,-0.01123,0.00936,0.00072,0.01478,-0.0053,0.00119,0.00906,-0.00657,0.00076,0.0088,-0.00491,1e-05,0.02217,-0.00281,0.01089,-0.00945,0.00643,0.01624,0.00549,0.00452,0.00365,0.00236,0.00443,0.00076,-0.00354,0.00147,0.00694,-0.00388,-0.0027,-0.0065,0.00643,-0.00146,0.00986,-0.00648,0.0113,-0.00365,0.00719,0.00378,-0.00588,0.0027,0.0091,-0.00236,0.0115,0.00321,0.00483,-0.00223,0.00145,-0.00768,0.0021,-0.00594,-0.0076,0.0059,0.00523,-0.00644,-0.00764,-0.00535,0.00602,0.00638,0.00617,0.00468,0.00704,-0.00526,-0.0053,0.01704,-0.00275,0.00361,0.00143,0.00339,0.00883,0.0076,-0.01388,0.00526,0.01037,0.00764,0.00073,0.00797,0.00012,-0.00316,-0.00132,-0.00639,0.00265,-0.0025,-0.00317,-0.0014,-0.00151,0.00015,-0.00531,-0.00242,0.0132,0.00128,-0.00822,-0.00834,-0.00095,0.00594,0.01581,0.00055,0.00755,-0.00161,-0.01778,0.01067,0.01304,0.00669,0.00674,-0.00253,0.00175,-0.00725,0.01804,-0.01064,-0.00247,-0.00228,-0.00113,-0.00923,0.00065,-0.00286,0.00038,0.00278,-0.00443,0.01828,-0.00698,0.00361,-0.00795,-0.01307,-0.00323,-0.01497,-0.00881,-0.00084,-0.0027,-0.00018,0.01565,0.00021,-0.00289,0.00434,0.01289,-0.00376,0.00567,0.00423,-0.00601,0.0065,-0.01151,0.00872,-0.00751,0.00067,-0.00415,-0.00619,-0.02393,0.00974,-0.01243,-0.00712,0.01031,-0.00113,-0.00026,-0.01197,0.00027,0.01716,0.00446,0.00587,0.01961,1e-05,-0.01192,-0.00232,0.00916,-0.00952,0.00086,-0.01275,-0.00632,0.01439,0.0017,-0.00816,0.00694,0.00325,0.01051,-0.00113,-0.01073,0.00752,0.00074,-0.00466,-0.01342,-0.00084,-0.00778,0.01015,0.02073,0.00643,-0.00593,-0.007,0.01085,-0.00815,0.00344,-0.00023,0.00087,-0.00404,0.0001,-0.01305,0.00718,-0.01073,0.00418,-0.00708,0.02316,-0.00328,-0.00799,-0.00178,0.00335,0.00383,-0.00863,0.00768,0.016,-0.0105,0.00858,-0.0046,-0.00579,-0.00362,0.0106,0.01206,-0.00846,-0.00456,0.01464,0.00254,0.00491,-0.00323,-0.00635,0.00463,0.00305,-0.00556,-0.01199,-0.00183,0.00614,0.01346,-0.00205,-0.00893,0.0006,0.00045,0.01631,0.00248,0.01538,0.00277,-0.00867,0.0121,0.00029,-0.0049,-0.00335,0.00103,-0.01083,-0.00928,0.00344,0.00829,-0.0163,0.0123,0.00636,0.00511,0.00035,-0.00432,0.00542,0.00368,-0.0146,0.00023,-0.00036,-0.00084,-0.00823,-0.00363,0.00217,-0.00111,0.00392,-0.00038,0.00334,0.00622,0.01204,-0.00654,0.0043,-0.00319,-0.02922,-0.00773,0.00345,0.00291,0.0028,-0.0036,-0.00038,0.00538,-0.00545,-0.00704,-0.00484,0.00287,-0.01577,-0.01185,0.0003,0.00074,-0.00212,-0.00849,0.00892,-0.00889,0.00161,-0.00343,-0.01309,0.01604,-0.00089,-0.00761,-0.00913,0.00883,-0.00639,0.00085,-0.00326,-0.00784,0.01045,-0.00293,0.00577,0.01101,-0.02561,0.00111,0.00093,-0.00941,0.00357,-0.00801,-0.00938,0.00265,0.00391,0.0077,-0.00372,0.00507,0.00295,0.00181,0.00829,-0.00314,-0.00257,-0.00686,0.00261,-0.00766,-0.01777,0.00711,0.02168,0.01155,0.00207,0.00759,0.00358,0.00528,0.00065,0.00521,0.00381,-0.0017,0.00066,-0.00598,0.00221,0.00093,-0.00443,0.00043,0.00101,-0.00366,-0.00744,-0.01218,0.00611,0.0002,0.00582,0.00178,0.00429,-0.00163,0.01734,0.00112,-0.00835,-0.0117,0.0045,0.00141,-0.00772,-0.0089,-0.01725,0.00103,0.0014,0.00582,0.00515,0.00675,-0.01872,0.00103,-0.00986,-0.00558,0.01365,0.0076,0.00611,0.00692,0.0079,-0.011,0.00111,0.00389,-0.00504,0.00783,0.005,0.00685,0.00424,0.00472,-0.00528,-0.0006,0.00899,0.00187,-0.00612,-0.00959,0.00533,0.00104,0.00141,-0.0021,0.01031,-0.00643,-0.00231,0.00575,-2e-05,0.01374,0.00163,0.0052,-0.00908,0.01069,0.00312,-0.00815,-0.00253,-0.00314,0.00182,0.00038,-0.00456,0.00016,-0.00919,0.00543,-0.00656,0.01071,0.00087,0.00331,0.00179,0.00333,-0.00377,0.00357,-0.00183,0.00034,-0.00333,0.00617,-0.01305,0.00685,-0.00056,0.0058,0.00598,-0.00155,0.00105,0.00657,-0.00173,-0.00308,-0.01324,0.00174,0.00581,0.00211,-0.01426,-0.00066,-0.00949,0.00157,0.00675,-0.00764,0.01031,0.00778,0.00958,0.01447,0.01467,0.01201,0.00665,-0.00472,-0.00746,-0.00868,-0.00234,0.00256,-0.00593,0.00816,-0.00524,0.00171,-0.00359,0.00239,0.00649,-0.00472,0.00553,-0.01678,-0.0013,-0.00863,-0.00483,0.00364,-0.0022,0.0021,-0.00294,-0.00655,0.00012,0.00423,-0.00417,-0.00338,0.00395,-0.00389,0.00254,-0.00344,0.00589,0.01502,-0.00443,0.00416,-0.00788,0.00325,-0.00531,0.00082,0.00974,-0.00841,0.00653,0.00635,-0.00623,0.01052,0.00322,-0.00184,-0.00436,0.00173,0.00243,0.00541,-0.00517,0.0073,0.00834,0.00117,-0.00271,-0.00494,0.00827,0.0045,-0.001,0.00828,-0.00251,-0.00175,-0.00113,0.01467,0.00596,0.00146,0.00151,0.00194,-0.00442,0.01306,-0.00445,-0.00808,-0.00181,0.00161,0.00459,0.00974,-0.00323,-0.00263,0.0047,-0.00529,0.00488,0.00601,0.02087,0.00764,-0.00091,-0.00711,0.00762,-0.00835,0.01143,0.01153,-0.00367,-0.00617,0.01668,-0.00114,0.00567,0.01072,0.00103,0.00025,0.00791,0.00294,0.00554,0.00072,-0.01059,-0.00217,0.00903,0.00036,0.00568,0.00664,0.00141,0.00786,-0.00176,0.00481,-0.00157,-0.0058,0.00181,-0.0039,0.00575,0.01743,0.00038,0.00353,0.01286,-0.01049,0.01221,-0.00029,-0.00127,0.00463,0.01172,-0.00637,-0.00673,-0.01031,-0.00556,0.00681,0.00356,0.01051,-0.00074,-0.00881,0.00173,-0.00163,-0.00112,-0.00185,0.00056,-0.01289,-0.00143,-0.01079,0.01227,0.00851,-0.00067,0.00549,-0.00304,0.00886,-0.00669,-0.00221,-0.00217,-0.01157,-0.0159,-0.00691,0.00868,-0.00101,-0.01191,-0.00029,-0.00413,0.00781,0.01399,-0.00067,0.00019,0.00609,0.01805,-0.0011,-0.00205,0.0081,0.0129,-0.0264,0.00528,-0.00048,0.00172,0.00557,0.0042,-0.01221,0.00956,-0.01032,0.0056,0.00244,-0.00451,0.00224,0.01115,-0.01022,0.01993,0.00819,0.0011,-0.00676,-0.00978,0.01016,-0.0064,-0.00177,-0.00429,-0.00361,0.00906,0.00133,-0.00374,-0.01232,-0.00322,-8e-05,0.00509,0.00215,0.00808,-0.01039,-0.00585,-0.01504,-0.00868,0.00337,-0.00872,-0.00034,-0.00678,-0.00077,-0.00824,0.0071,0.00842,0.00763,0.0042,0.00375,-0.00247,0.00018,0.00683,-0.00768,0.00451,-0.00214,-0.00154,-0.00785,-0.00125,-0.00803,0.01095,0.005,0.01111,0.01561,0.02105,0.00913,-0.00685,-0.00987,-0.01061,0.00583,0.00961,-0.00256,-0.00599,-0.01214,-0.00013,0.00079,-0.00745,-0.0005,0.00639,0.00669,-0.0018,0.00794,-0.00615,0.00768,0.00902,0.00039,-0.0042,-3e-05,-0.00581,0.00071,-0.01005,-0.00296,0.00277,0.00871,-0.00028,-0.0011,0.00245,0.01032,-0.0,-0.00019,-0.00149,0.00662,0.00155,-0.00241,-0.00851,-0.01278,0.00874,0.00093,0.00744,-0.00111,-0.00612,-0.0025,-0.00961,0.00201,0.01106,-0.00709,0.00096,-0.00082,-0.00307,-0.01124,-0.00562,0.00861,-0.01968,0.0027,-0.00085,-0.00655,-0.0005,-0.00335,-0.0134,0.00177,-0.00544,0.0014,-0.00727,-0.00586,-0.01038,0.00569,-0.00907,0.00719,0.00746,0.01119,0.00569,-0.00495,-0.01273,-0.02532,-0.00385,-0.00486,-0.00983,0.01003,0.01109,-0.00649,-0.00223,-0.00384,0.01511,-0.01102,-0.00213,-0.0018,-0.0101,-0.00933,-0.00976,0.00807,0.00478,0.0016,0.00691,-0.00783,0.00434,0.0011,-0.01175,0.00973,0.00587,0.00105,-0.00549,0.00334,0.00481,0.00344,-0.01446,-0.00819,-0.00812,-0.00275,-0.00836,0.00129,0.01319,0.00862,-0.0033,0.01211,0.0037,0.00439,-0.00512,-0.00115,0.01173,0.00065,-0.00155,-0.00167,0.00378,0.00284,0.01853,0.01123,-0.00779,0.00361,0.00278,0.00413,-0.03681,0.00083,-0.00016,-0.04496,0.00657,0.00039,-0.00925,0.00205,-0.00026,0.0029,0.01074,0.00176,-0.00639,-0.0009,-0.00107,0.00397,0.00162,0.00684,-0.00019,-0.00247,0.01403,0.00025,0.0018,-0.00106,0.00463,0.0013,0.00202,0.00527,-0.00881,0.00679,0.01241,-0.01899,0.01342,0.00658,-0.00616,0.00339,0.00353,-0.00537,0.00423,-0.00931,-0.00965,-0.01136,-0.00524,-0.00766,-0.01834,-0.00449,-0.00607,-0.00269,-0.00542,0.00792,0.00709,0.00875,-0.00554,0.00231,0.00881,-0.00176,-0.00358,0.00787,0.00577,-0.00679,-0.01189,-0.00484,0.00448,-0.0009,0.00392,0.00651,0.00695,0.00676,-0.00891,0.00222,0.0122,-0.00302,-0.01255,0.00588,0.01151,0.00443,-0.0029,-0.00143,-0.00376,0.00948,0.00627,-0.00809,0.00348,-0.00236,-0.00017,-0.00083,-0.01943,0.00649,0.00591,-0.00616,-0.00619,-0.00747,0.0087,0.00432,0.00601,0.00765,0.00771,0.00374,-0.00294,-0.01407,-0.00407,0.01347,-0.00354,-0.00748,0.01426,0.0043,0.00245,-0.00685,-0.01497,0.01185,0.00295,0.00367,0.00177,0.00565,0.00237],[-0.01133,0.0011,0.00233,-0.022,-0.00328,-0.00589,-0.01169,0.01515,-0.01253,-0.01692,0.00863,-0.01021,0.00055,-0.00132,-0.01143,0.01567,0.00889,-0.00775,0.00159,0.0168,-0.00158,-0.01064,-0.00539,-0.00473,0.00291,-0.01477,-0.00563,-0.00165,-0.01595,0.00737,0.00799,-0.00884,0.00814,-0.00148,0.00985,0.02643,0.0125,0.00045,-0.00843,0.00367,0.00696,0.00534,-0.03189,-0.0192,0.02784,-0.00016,-0.01365,0.02324,0.00992,0.02349,-0.00339,0.00107,-0.0115,0.0147,-0.03784,-0.01848,-0.00813,-0.00114,-0.01232,-0.00329,0.01356,0.00197,0.00551,-0.0018,-0.00366,0.00275,-0.00797,-0.00344,0.01342,-0.01264,-0.00691,0.00819,-0.00521,-0.007,0.0097,0.00224,-0.00958,0.00646,-0.00783,0.00056,-0.01935,0.00389,0.00364,-0.0091,-0.00529,-0.00428,0.01595,-0.00036,0.00337,-0.0015,0.00371,-0.00286,-0.00252,-0.02427,-0.01108,-0.03639,-0.01959,0.00346,-0.0119,0.02349,-0.0192,-0.00719,0.00666,0.02126,0.01319,-0.00823,-0.01485,0.00023,0.0023,0.00031,0.00131,0.00114,-0.00027,0.00582,-0.0059,0.00773,0.00235,-0.00511,-0.01435,0.00795,0.00236,0.00153,-0.00633,0.00581,-0.01537,-0.01207,-0.01523,0.00481,-0.01159,0.00708,0.02391,0.00403,-0.00967,-0.00703,0.03045,0.00356,-0.00457,-0.00613,0.01889,-0.00178,-0.02197,-0.01383,0.01579,-0.00915,-0.01343,-0.03327,0.01193,-0.005,-0.01195,0.03466,-0.00231,-0.0053,0.00965,0.01469,-0.00292,-0.00657,-0.01339,-0.00766,-0.00526,0.01403,0.00654,-0.01757,-0.01299,-0.00187,-0.00291,0.02749,-0.01485,-0.00791,-0.02208,-0.00834,-0.00637,0.00654,-0.0079,-0.00236,0.00041,-0.00413,-0.00469,0.01186,-0.0069,-0.00429,-0.01771,-0.00507,-0.01061,0.00259,-0.00929,-0.01129,-0.00728,0.00533,0.00362,-0.00832,0.01603,-0.00275,0.01093,0.0113,0.00106,-0.00306,-0.0241,0.00585,-0.01029,0.00375,-0.0038,-0.00227,0.0097,-0.00638,-0.01199,0.00588,-0.00095,-0.00741,-0.03873,-0.00079,0.01508,0.00359,-0.00383,0.00262,-0.01471,0.00817,0.00846,0.00684,-0.00588,0.00194,0.00082,-0.0093,-0.00469,-0.00483,-0.00928,0.00226,-0.00953,0.03033,-0.00796,-0.0008,-0.03443,0.02347,-0.01618,-0.01352,0.00468,-0.0098,-0.03155,-0.00493,0.02065,0.02259,-0.01186,0.00836,-0.00653,0.01953,0.01037,0.00177,-0.00835,-0.00134,0.00153,-0.01104,0.01176,0.004,-0.01386,0.00646,0.00741,0.00379,-1e-05,0.00834,-0.00448,-0.01772,0.02052,-0.01631,-0.02438,-0.0052,0.02838,0.01977,-0.00341,0.02017,-0.01942,0.0125,0.00105,0.00657,-0.0211,-0.01094,0.0028,-0.00086,-0.00327,0.00533,-0.00785,-0.00168,0.00753,-0.00852,0.01067,-0.00135,0.01289,-0.00311,-0.01844,-0.00485,-0.00794,-0.0005,0.00297,0.02275,0.01228,-0.00647,-0.01725,-0.001,-0.0257,-0.01514,-0.00949,0.0263,0.00214,0.0101,-0.00205,0.0039,0.00407,0.00607,0.00968,0.01676,-0.0202,0.00217,-0.00294,-0.02084,0.00331,-0.01297,0.00301,0.00934,-0.00426,0.00847,0.00929,0.0123,0.02195,0.01324,0.00753,0.00766,0.02009,0.00657,0.02847,-0.01688,-0.0073,-0.00792,0.00181,-0.00396,0.00219,-0.01732,-0.00474,-0.01413,0.00534,-0.00609,-0.01708,-0.00082,-0.00014,-0.02362,0.00153,0.01012,0.01071,-0.00463,0.01734,0.01304,0.00791,-0.00697,0.0122,-0.00313,-0.00068,-0.00589,-0.03088,0.00799,-0.01213,-0.00205,-0.00827,-0.01989,-0.00394,-0.00894,-0.01971,-0.01879,-0.01419,0.02979,-0.00102,0.01899,0.00798,-0.02944,-0.01984,0.0086,0.00575,-0.01047,0.01522,-0.00357,0.00308,0.00681,0.00892,0.00251,0.0141,-0.00336,-0.01187,0.00145,-0.00614,0.00572,0.01439,-0.00873,0.00258,-0.01131,0.01987,0.01294,-0.00429,0.0187,-0.00537,0.00131,-0.0052,0.01313,-0.00676,0.00929,-0.02476,0.03597,0.02094,-0.00915,-0.00416,0.01051,-0.00875,-0.01128,-0.02256,0.0022,-0.01225,-0.01252,9e-05,0.00476,0.00131,-0.01457,0.00492,-0.01797,0.02329,0.01988,-0.01245,-0.01971,-0.01681,0.00173,-0.0142,0.01401,0.00746,0.0035,-0.0138,-0.00422,-0.01543,0.00963,-0.00925,0.00036,-0.00783,-0.00219,0.0208,-0.0042,0.01059,0.00018,-0.01272,-0.01494,-0.00548,-0.01366,-0.00735,-0.00306,0.01439,-0.01215,0.01865,0.01135,-0.0035,-0.00714,-0.02217,0.02755,0.00741,-0.01564,0.00819,0.01049,0.00068,-0.00897,0.01076,0.01312,-0.00358,-0.00462,-0.00019,-0.00712,0.02118,0.0018,-0.00461,-0.00914,0.01734,0.0248,-0.01734,-0.01783,0.01337,0.00583,-0.00389,0.00691,0.01129,0.00459,0.01567,0.01629,0.00661,0.00278,-0.01124,-0.00401,-0.00817,0.01281,-0.01209,0.00243,-0.00873,0.01987,-0.00911,-0.00065,-0.00235,0.00816,-0.00339,-0.0008,-0.0106,0.01697,0.0063,-0.01534,-0.0009,-0.00306,-0.0155,0.00596,0.00297,-0.002,0.01824,-0.0016,0.01767,-0.00027,0.00708,-0.01751,-0.00178,-0.01532,0.0276,-0.03018,0.0167,-0.01713,0.00979,-0.0104,0.00297,-0.02001,0.00311,0.00522,-0.00991,0.00909,0.00635,0.00898,0.00088,-0.01561,0.00847,-0.00106,-0.00357,-0.01591,-4e-05,0.00227,0.00297,-0.00297,-0.03712,0.00082,-0.0008,-0.01235,0.0118,0.00015,-0.02089,0.008,0.0021,0.00724,0.00226,0.02696,0.02466,-0.01385,0.00201,-0.01552,-0.00873,-0.00918,-0.00347,0.00749,-0.02257,-0.02405,0.00432,0.01777,0.01171,0.00096,-0.01203,0.01441,-0.02512,-0.00581,-0.00316,-0.00196,-0.01025,-0.00934,0.02093,0.00762,0.00767,0.01285,0.00539,-0.00142,-0.01686,0.00724,-0.00645,-0.01579,0.00599,0.02744,0.00066,-0.01309,0.00229,0.00351,-0.02409,-0.00177,-0.01113,0.00286,0.01525,-0.00638,0.01648,0.0095,-0.00301,0.02315,-0.01122,0.01836,0.00942,0.01193,-0.00607,-0.01853,-0.01574,-0.0104,-0.01621,0.03337,-0.0028,0.00659,0.00024,0.01972,0.0178,-0.00044,0.00377,-0.00241,0.00693,0.00329,0.00109,0.00671,-0.00826,0.00616,-0.00051,0.01969,-0.00756,-0.01103,-0.01434,-0.01049,-0.00572,0.00478,-0.00448,-0.02916,-0.00923,0.00983,-0.00131,-0.0072,0.01284,-0.00992,0.01045,-0.00837,0.01088,0.0109,0.00752,0.00779,0.01773,-0.00972,-0.01305,0.01593,-0.01091,0.0064,0.00753,0.01108,-0.00591,0.01459,0.00602,0.02411,-0.02978,0.01217,0.00723,0.00097,0.0142,-0.01473,0.014,0.0034,-0.01994,0.00249,-0.00953,0.00162,-0.0032,-0.00417,-0.00778,-0.02522,0.00834,-0.00215,-0.00679,-0.00293,-0.01102,-0.01752,0.01045,-0.00682,0.00928,0.01195,-0.00309,0.03204,-0.00267,-0.00762,-0.00279,0.00766,-0.00681,0.00102,-0.0008,-0.0251,0.00041,0.00146,-0.00079,-0.0117,0.00583,0.00195,-0.01377,0.01547,-0.02715,0.0027,0.02031,0.00508,0.00993,0.00723,0.01359,0.00156,-0.00297,0.0002,0.00037,0.01032,-0.00537,0.01374,-0.00059,-0.00075,-0.01226,0.00029,0.01153,0.00217,-0.01255,-0.00383,-0.00581,0.01581,0.01742,-0.00153,-0.00334,0.0034,0.00599,0.00077,0.02071,-0.02201,-0.03812,-0.00953,-0.02575,-0.00079,-0.00333,-0.00017,-0.01163,0.0052,0.01152,-0.00257,0.01099,0.02676,-0.00855,0.00759,0.00309,0.01001,0.00761,-0.00825,0.01008,0.00185,0.00584,0.01042,-0.00115,-0.00043,-0.00469,-0.01196,-0.02157,-0.01408,-0.00305,-0.01042,0.00402,-0.02341,0.01623,0.00208,0.00823,0.02738,0.01068,-0.0277,-0.00125,-0.01266,0.00058,-0.02371,0.01038,-0.00551,-0.00554,-0.0151,-0.01371,0.01228,-0.00995,0.00189,-0.01534,0.0048,0.01261,0.01677,-0.0013,-0.0032,0.0071,-0.01731,-0.00544,-0.00151,-0.00997,0.01178,0.0128,-0.01981,0.00895,-0.00471,0.01308,-0.0214,-0.00283,0.01985,-0.01867,0.00405,-0.00146,-0.01095,0.01111,0.01317,0.00447,-0.00322,-0.00731,-0.0005,-0.00982,0.03013,-0.02112,-0.00156,-0.00734,0.0013,-0.00793,0.00791,-0.00316,-0.02204,-0.0024,-0.00604,-0.01245,0.00628,-0.00211,0.00886,-0.00986,-0.00409,-0.00186,0.0068,0.00519,-0.00643,-0.01524,-0.00545,0.00546,-0.02213,0.01031,-0.01166,-0.0061,0.00965,0.00036,-0.02154,-0.00176,0.02888,-0.00408,8e-05,-0.01415,0.00733,0.01237,-0.00478,0.00178,0.01806,-0.00243,-0.0078,0.00095,0.00629,-0.00222,0.0052,-0.01825,-0.00536,-0.00673,0.02457,0.00082,0.01297,0.01767,-0.00774,0.0183,0.00295,-0.00481,-0.0063,0.01373,0.00888,-0.00162,0.00152,0.00171,-0.01537,0.00079,-0.00134,0.00356,0.01142,0.0038,-0.01352,0.00238,-0.003,0.0279,0.01721,-0.01978,0.00428,-0.00596,-0.00794,0.00043,0.01629,0.00135,-0.00011,0.0026,0.00679,-0.00731,-0.00668,-0.00095,0.0136,0.02063,-0.00962,-0.00361,-0.00146,-0.0079,-0.01155,0.01276,-0.00641,0.00635,-0.00958,-0.0247,0.00367,-0.00679,0.02116,0.01202,-0.00156,0.00203,-0.0011,-0.00976,0.00269,-0.0004,0.01551,0.0057,-0.00984,-0.00684,-0.0027,-0.01183,0.00749,0.03044,-0.0044,0.00611,0.01173,-0.00812,-0.00273,0.00317,0.00177,-0.00685,-0.00972,-0.00196,0.0115,0.01294,0.00334,0.0157,-0.00739,-0.01227,0.00919,0.01116,0.00069,-0.00157,0.0242,-7e-05,-0.01792,-0.00621,0.00499,0.02813,-0.00328,0.0051,0.00424,0.01433,-0.02327,0.02253,0.01515,0.00675,0.01578,-0.01179,-0.00208,0.00839,0.00485,-0.0014,0.02383,0.00161,0.00116,-0.01735,-0.02423,0.0042,-0.01888,-0.00498,0.00055,-0.01464,0.00595,0.00619,0.01432,0.01789,-0.00717,-0.01143,-0.00046,0.01159,-0.00756,-0.01539,0.01543,-0.0149,0.00079,-0.02979,0.00066,0.00933,0.01317,0.00183,0.00407,-0.01314,-0.01751,0.00797,0.00773,-0.00473,-0.00152,-0.00982,-0.00116,0.00391,-0.01213,0.00374,-0.00252,0.01384,-0.00088,0.01397,-0.01197,0.00114,0.00391,0.01807,0.00139,0.00755,-0.00372,-0.0191,-0.03849,-0.00674,-0.00675,0.00254,0.01247,-0.01961,0.00785,0.00427,-0.00671,0.00999,-0.01221,-0.00018,0.00949,0.00999,-0.01554,0.01242,0.01041,0.00349,-0.01497,0.00682,-0.01182,0.02839,0.01381,0.01752,-0.02389,0.00802,-0.00573,-0.01731,-0.00702,0.01215,-0.0049,0.00455,-0.0262,0.00035,-0.01383,0.02438,0.01324,-0.00893,0.0123,0.01738,0.00284,-0.00842,-0.00362,0.03138,0.01015,-0.0075,0.00459,-0.01373,-0.01227,0.02867,0.00256,-0.00047,0.00451,0.00338,-0.00245,-0.00657,0.00265,-0.01451,0.00373,-0.00364,-0.00539,0.01117,-0.00016,0.00609,-0.00023,-0.0138,0.00575,0.01504,-0.01395,0.00328,-0.0301,0.00593,-0.00742,0.00542,-0.02519,-0.00398,0.00067,-0.00315,-0.03849,0.00396,-0.00491,-0.00773,0.00681,-0.01528,-0.00419,0.00675,0.01305,0.00505,0.00921,0.01325,0.01678,0.00598,0.01038,-0.03582,-0.00711,0.00244,-0.00938,-0.00275,0.02699,-0.00638,-0.0094,-0.02158,-0.01671,-0.0006,0.01035,-0.01591,-0.01402,0.0138,-0.01034,0.00611,-0.0069,-0.00238,0.00926,-0.00418,0.00232,-0.01195,-0.01149,-0.03788,0.02652,0.00318,0.00411,-0.00339,0.0075,0.00256,-0.00315,-0.00058,0.01604,-0.02404,0.00287,-0.01184,-0.00872,-0.00702,-0.0073,-0.00904,-0.00386,0.01065,-0.00886,-0.00023,0.03394,0.00938,-0.028,-0.03072,-0.0071,-0.00565,0.0085,0.00206,0.00264,-8e-05,0.00518,-1e-05,-0.02283,0.00672,0.01613,-0.02605,0.00987,-0.01445,-0.00582,0.00512,0.00925,0.00841,-0.00196,-0.00379,-0.02254,-0.00274,-0.00709,0.00481,-0.00757,0.01697,-0.03317,0.00665,-0.00044,0.01693,0.01012,-0.00201,-0.01064,-0.01559,-0.03117,0.01593,0.00623,0.00616,-0.00597,0.02546,-0.01674,0.01708,-0.0024,0.01697,-0.01507,0.00712,0.00768,-0.01176,0.00051,0.0036,0.0138,0.01795,0.00211,0.01223,-0.00737,-0.00345,0.01042,-0.01277,0.00381,-0.0007,-0.004,0.00209,-0.00718,-0.00527,-0.02214,-0.01014,0.00026,-0.02648,-0.01388,0.0213,-0.02047,0.00784,0.01408,-0.01698,-0.00778,-0.01517,0.00501,0.00856,0.01169,0.0058,0.00417,0.00957,0.01189,-0.00273,-0.01689,-0.00304,-0.01494,0.02812,0.00034,0.01027,-0.00764,0.01525,-0.01287,-0.00058,0.0072,-0.00511,0.01946,0.00346,-0.0076,-0.00619,0.02224,-0.02533,-0.01187,-0.01179,-0.00051,0.02787,-0.01166,0.00311,0.00938,-0.01245,-0.0147,0.01829,-0.00011,-0.00086,-0.00021,0.01012,0.01017,0.01295,-0.00765,0.00112,-0.0078,0.00393,-0.00493,-0.00918,0.00092,0.01048,-0.00104,0.00168,-0.01189,0.0065,0.0189,0.0106,-0.0171,0.00451,0.0011,0.0079,-0.00053,0.00132,0.00215,0.00673,-0.0181,0.02069,-0.01453,-0.008,-0.00688,0.00387,-0.01679,-0.00387,-0.01364,0.00227,0.01514,0.00201,0.01147,0.00555,-0.033,0.00707,0.01031,0.01862,0.01224,-0.01568,0.00557,-0.00042,0.01259,-0.00259,-0.00235,-0.02039,-0.01385,-0.01119,0.01127,-0.02104,-0.01544,0.01621,0.00463,0.00112,0.00209,0.01954,-0.0038,-0.01889,-0.01042,0.01903,-0.01252,-0.00857,0.01436,-0.00541,0.02627,-0.00169,-0.00202,0.0091,-0.01445,-0.00795,-0.01264,-0.00459,0.01497,-0.01267,-0.01678,-0.00027,0.01179,-0.02897,-0.02896,-0.00556,0.00339,0.01077,0.00429,0.00546,-0.00996,0.01481,0.00041,-0.00744,0.01182,-0.00066,0.00307,-0.00238,0.02283,-0.00369,0.00437,-0.0054,-0.00843,0.01053,-0.00708,-0.00338,-0.01617,0.0036,-0.00195,0.01389,0.0072,0.00526,-0.01187,-0.01002,0.01392,0.01368,-0.00031,0.02262,0.00503,0.00484,-0.00529,-0.00282,-0.01905,-0.00295,0.01541,0.0017,0.00579,-0.02242,0.00813,-0.00593,-0.01163,0.00829,0.01644,0.00686,-0.0034,0.01624,-0.01088,0.01256,0.01299,0.00518,-0.08629,0.01777,-0.01006,0.01227,-0.00946,-0.00623,-0.01723,0.00024,-0.00409,0.00375,-0.00269,0.01764,0.0231,0.00798,0.01027,-0.00215,0.00749,0.00283,-0.01355,0.01112,0.01109,0.00395,0.00558,0.00208,0.00366,0.00494,0.0218,-0.00384,0.00968,-0.00088,0.00176,-0.01542,0.00052,-0.01316,0.00365,0.01332,0.0011,0.00606,-0.00396,0.0095,-0.00467,0.00039,-0.00761,-0.01369,0.00663,0.01844,0.00645,0.01734,-0.0052,-0.0042,0.01746,-0.00518,0.00086,-0.00034,0.00254,-0.0039,-0.0035,0.02536,-0.01246,0.01618,-0.00034,0.00333,-0.00073,0.00695,0.00565,0.00133,0.00265,-0.0153,0.01021,-0.00619,-0.0029,0.02071,-0.01279,0.00604,-0.01575,-0.01493,-0.00668,-0.00442,-0.00908,-0.00555,0.00691,-0.00617,0.01797,-0.0151,0.01104,0.00096,-0.00884,-0.00634,-0.00371,-0.01687,-0.03179,0.00128,-0.0286,-0.01613,-0.00693,0.00202,-0.01262,-0.0016,-0.02076,-0.01302,-0.0108,0.00311,-0.01464,0.00373,-0.00298,-0.01153,-0.01374,-0.0079,-0.01238,-0.0048,-0.00746,0.01686,0.01265,0.017,0.03309],[-0.00558,0.01649,0.01838,-0.00766,-0.00065,-0.00509,-0.00407,0.00331,-0.02596,-0.02436,0.0147,0.01024,-0.01925,0.01911,-0.00027,-0.00523,0.01348,-0.01719,0.00655,0.01731,-0.00565,-0.01413,-0.00301,-0.00386,-0.00308,-0.01117,0.02387,0.00691,-0.01793,0.01856,-0.01278,-0.01369,0.01039,0.00245,-0.00407,0.02636,0.0247,-0.01713,-0.02198,0.00396,-0.0052,0.01772,-0.03257,-0.01531,0.0053,0.00184,-0.01601,0.02138,-0.00496,-0.00072,-0.02094,0.00311,-0.01287,0.00729,-0.02931,-0.0177,-0.00615,-0.00352,0.01323,-0.00595,-0.01106,-0.01317,-0.01174,-0.00377,-0.0121,-0.00322,-0.00224,-0.00254,-0.00409,-0.01559,0.00984,0.0211,0.00528,-0.00622,0.00072,-0.0135,-0.02301,-0.00321,0.00331,0.00198,-0.03289,-0.00014,0.00129,-0.00529,-0.01969,0.00116,-0.01389,-0.00035,-0.00662,-0.00142,0.00783,0.00052,-0.01603,0.00501,-0.01364,-0.0263,0.00662,0.00689,-0.01631,0.02213,-0.01883,0.00762,-0.00975,0.02718,0.00032,0.008,-0.00468,-0.00534,0.00103,0.00778,0.00793,0.0096,0.00417,0.00289,-0.01175,0.00537,-0.01347,-0.00015,-0.00556,0.00792,0.00407,-0.01048,-0.00515,-0.01069,-0.00514,-0.00736,-0.01658,0.00521,-0.02184,0.01196,0.01753,-0.00441,-0.00708,-0.03109,0.01845,-0.00102,0.0062,0.00775,-0.00289,0.00259,0.00731,-0.01802,0.00227,0.01234,-0.01964,-0.00326,0.03038,-0.0104,-0.00773,0.01829,-0.01821,0.00336,0.0252,0.0038,0.00976,-0.00083,-0.003,0.01651,-0.00431,-0.00105,0.00629,0.0011,-0.00771,0.0021,0.01076,0.00212,-0.0106,0.00164,-0.01429,0.0055,-0.0028,0.02562,-0.02199,-0.01233,-0.00179,0.00893,0.01132,-0.01127,-0.00704,-0.00182,-0.00918,-0.01546,-0.00434,0.01053,-0.00354,-0.0256,-0.0112,0.02575,0.00214,0.00802,0.02768,0.00413,0.01415,0.00682,0.01713,0.00783,-0.00511,0.0059,0.00296,-0.00631,-0.02478,-0.00524,-0.00218,-0.00848,-0.02791,-0.00054,0.00139,-0.0047,-0.01908,0.00288,0.01006,0.00302,-0.00108,0.00675,-0.00345,0.00223,-0.00016,-0.00237,-0.00785,-0.00588,-0.01055,0.00296,0.00214,-0.02603,0.00725,0.01753,-0.00856,0.02062,-0.02137,-0.00977,-0.03174,0.02977,-0.01864,-0.02479,-0.00077,-0.01033,-0.03202,0.00559,-0.00397,0.00438,0.0075,0.02453,-0.01237,0.03084,0.02831,-0.00817,-0.00597,-0.00811,-0.00443,-0.00408,0.01059,-0.00727,-0.00684,-0.02801,-0.00034,-0.00616,0.0028,0.01592,-0.01407,-0.00956,0.00077,-0.02289,0.0027,-0.01131,0.01484,0.0107,-0.00407,0.00628,0.00617,0.003,0.00154,-0.0081,-0.00987,-0.00169,-0.01437,-0.00589,0.00029,-0.01321,0.00923,-0.00375,0.00321,-0.01285,0.0071,-0.00985,0.01206,-0.00384,-0.01025,0.01151,-0.00304,-0.00998,-0.02831,0.00582,0.00868,-0.01539,0.00906,0.00488,-0.02331,-0.02434,-0.00907,0.03537,0.00192,-0.01978,-0.01442,-0.00171,-0.01225,-0.02464,-0.00293,-0.02101,-0.01707,-0.01523,-0.00722,-0.02303,0.01392,0.00656,-0.00742,0.00404,0.01472,0.02629,-0.00813,0.00156,0.02578,-0.00088,-0.0122,-0.00466,0.00287,0.01154,0.00772,0.00614,0.01206,-0.0214,0.00707,-0.01669,0.00718,0.00768,0.01039,-0.01647,0.00911,0.01197,-0.00696,-0.0182,-0.0094,-0.01621,0.0039,0.01442,0.01159,0.00285,0.01636,0.00417,-0.01095,-0.01919,-0.02057,0.0136,-0.01312,0.00592,-0.01022,0.00928,-0.02008,-0.00841,-0.00448,-0.01795,-0.0125,0.00113,-0.02002,-0.0122,-0.00551,0.02338,-0.00703,0.01699,-0.00242,-0.03821,0.00224,0.01351,-0.01177,-0.0116,0.02355,0.01088,-0.01652,0.00303,-0.00026,-0.01604,-0.00452,0.01407,0.00118,-0.00204,0.00488,-0.00506,0.01158,-0.00451,-0.00079,-0.01657,0.01077,0.00742,-0.01074,0.01569,0.00667,0.01778,-0.02247,-0.00977,-0.00276,-0.00926,-0.01053,0.02778,0.00433,0.00634,-0.00591,0.0107,-0.00993,-0.00547,-0.01668,0.00039,-0.00238,-0.02066,-0.0039,-0.00821,-0.0037,-0.00382,-0.00015,-0.0101,0.01142,-0.00672,0.00068,-0.02548,-0.02389,-0.00661,-0.01569,0.00818,-0.00631,0.01055,-0.02602,-0.00821,-0.01327,0.00813,-0.01098,0.0084,-0.01488,0.00558,0.00633,0.01019,-0.00964,-0.00817,-0.00708,0.00052,-0.00736,-0.02727,-0.0141,0.00691,-0.00088,0.00853,0.02483,0.00219,-0.01519,-0.01028,-0.02791,0.02524,0.00224,-0.02392,-0.00536,0.00806,-0.01154,-0.00033,-0.01351,0.02888,-0.0098,-0.00628,-0.00966,-0.01075,7e-05,0.01068,-0.00673,-0.01677,0.01273,0.0145,-0.01012,-0.02144,0.00963,-0.00641,0.00795,0.01339,-0.00793,-0.00167,0.01182,0.02589,0.0139,0.00816,-0.00463,-0.00984,-0.00396,0.00999,0.00543,0.0069,0.0013,0.01911,-0.01637,-0.00676,0.00633,-0.00155,0.00394,-0.0021,-0.0066,0.02341,0.00392,-0.01118,-0.00426,0.01245,-0.02558,0.00678,0.02307,0.01111,0.00385,-0.00078,0.00302,0.00893,0.00323,-0.01012,0.01171,-0.00244,0.014,-0.02932,0.01251,-0.00931,0.01317,-0.03943,-0.00092,0.01689,0.01776,-0.00235,0.00661,0.01187,0.00547,0.01884,-0.00995,-0.00178,-0.01317,-0.00326,0.00331,-0.01651,-0.00493,0.01391,-0.00188,0.01219,-0.018,0.00151,0.00028,-0.00389,7e-05,0.00584,-0.01157,0.00983,-0.0058,0.00756,0.0042,0.02262,0.03112,0.01979,-0.00301,-0.02957,-0.00071,-0.00451,-0.00482,0.01329,-0.01951,-0.01373,0.00197,0.01776,0.00556,0.02343,2e-05,0.00282,-0.02384,-0.00374,0.01304,-0.00404,-0.01571,-0.00728,0.01292,0.02034,-0.00399,0.00199,0.00552,0.01625,0.03323,0.00945,0.00418,-0.01125,0.00031,0.02291,0.00686,-0.00793,0.01559,-0.00784,-0.01797,0.00691,-0.00169,-0.00101,0.01774,0.00407,0.01693,0.01952,0.00507,-0.00835,-0.02595,0.00727,-0.00063,0.01111,-0.01602,0.00052,0.00243,-0.0134,-0.02245,0.03543,0.01109,0.00026,0.00643,0.00482,-0.00633,-0.00505,0.00778,-0.01169,-0.00129,0.01402,-0.00457,0.00716,0.00415,0.00733,0.00022,0.01185,0.00141,-0.0085,-0.01133,0.00936,0.00888,0.01416,0.00957,-0.03209,-0.01192,-0.00395,0.00574,-0.00727,0.02305,-0.00953,0.00291,0.00734,0.01298,0.00739,-0.0044,0.00765,0.01307,0.0036,-0.00825,0.00155,0.00373,0.00655,0.00037,0.01756,0.01087,0.01687,-0.00925,0.02814,-0.0335,0.02912,0.01672,0.0074,0.02104,-0.00898,0.00553,0.01003,-0.0127,0.00481,-0.00722,-0.01113,-0.00669,0.00375,0.01139,-0.03292,-0.00197,0.02147,-0.00359,0.01311,0.00827,-0.00802,0.0222,0.00543,0.00135,0.0159,-0.0023,0.02346,0.00381,-0.01317,-0.01869,0.00339,-0.01849,-0.00166,-4e-05,-0.02027,-0.01783,-0.00432,-0.004,0.01108,0.00838,-0.00022,-0.01579,0.01528,-0.00734,-0.00865,0.01107,0.00542,0.00309,-0.00339,0.0238,-0.0049,0.00156,0.00206,-0.00579,-0.00546,-0.0033,0.01965,-0.00567,0.0033,-0.01368,0.01471,0.01548,-0.00385,-0.01125,-0.00083,-0.00309,-0.0127,0.01364,-0.0083,0.00646,0.01333,0.02082,-0.00902,0.01004,-0.00985,-0.03631,0.00518,-0.01385,-0.01611,-0.01515,0.01394,-0.00257,0.01437,0.01796,0.00047,0.02095,0.00308,0.00074,0.00699,0.00963,0.00263,-0.00039,-0.01766,0.00874,-0.01102,0.01442,0.00023,-0.01023,-0.00096,-0.00608,-0.01316,-0.02327,-0.01151,0.00319,-0.00431,0.00135,-0.00966,0.00368,0.00792,0.00168,0.00622,0.01284,-0.00906,-0.0026,-0.00828,-0.00385,-0.0083,-0.00048,0.00087,-0.01995,0.00389,-0.00672,0.01252,0.00162,-0.00672,-0.03889,-0.01946,0.01743,-0.00497,0.01043,-0.00309,-0.00099,-0.03693,-0.00309,-0.00546,-0.03313,0.01387,-0.00247,-0.00162,-0.00699,-0.01028,0.02337,-0.01743,-0.00887,0.02008,-0.02219,0.00933,-0.00258,-0.01588,-0.00537,0.0166,0.00233,-0.02412,0.00559,0.02218,0.00367,0.01415,-0.00564,-0.01943,-0.01128,-0.01142,0.00203,0.01059,0.00349,-0.02095,0.00192,-0.00991,-0.01755,0.00773,-0.00931,0.00475,-0.0081,0.00725,0.01573,-0.01039,-0.00658,-0.00756,-0.02067,-0.00625,0.0128,-0.01123,-0.00025,0.00578,-0.01053,-0.0066,-0.00639,-0.00081,-0.02254,-0.0008,0.00707,-0.00279,-0.03069,0.02101,0.00806,-0.00794,-0.00266,-0.00735,-0.00413,-0.03234,-0.00116,0.02294,-0.00154,0.01325,-0.0263,0.00276,-0.004,0.02544,0.0008,-0.00329,0.02086,-0.00292,0.0042,0.00429,-0.01211,-0.00244,0.02215,-0.00735,0.00824,-0.00353,-0.00238,-0.01249,0.00433,0.00071,0.01246,-0.00423,0.00884,-0.01839,0.00891,-0.00093,0.00602,0.00082,0.00174,0.02317,0.00899,-0.01092,-0.00053,-0.00946,-0.01527,-0.00524,0.00735,-0.01221,-0.01162,0.01818,-0.00534,0.02044,-0.00105,0.00155,-0.00245,-0.00522,0.01903,-0.00671,0.01049,-0.01153,-0.00745,-0.00526,-0.01026,-0.00198,-0.01166,-0.01161,0.01845,-0.00428,0.00881,-0.02397,-0.01776,0.01315,0.01467,0.02658,0.00388,-0.01834,0.00731,-0.00847,-0.01202,-0.00675,0.00872,0.01077,0.01174,0.00694,-0.02546,0.00522,-0.01703,0.00586,0.01265,-0.01882,0.01595,-0.00751,0.00343,0.00373,0.01086,0.00242,-0.01243,0.00461,0.01247,-0.00537,-0.01932,0.02235,0.00476,-0.00865,0.00221,-0.00531,0.01409,-0.0099,0.00601,0.02023,0.00568,-0.01149,0.01774,0.02581,0.00494,0.00783,0.00062,0.00048,0.00157,0.00538,0.01516,0.03234,0.01635,-0.0071,-0.02483,-0.00581,-0.01355,-0.02267,0.00432,0.00391,-0.02034,0.00538,0.03341,0.00519,0.00347,-0.00087,-0.00187,0.00922,0.01732,-0.01076,-0.00821,0.00109,-0.02208,-0.02421,-0.02361,-0.00288,0.01724,0.00063,-0.00049,0.00591,0.01186,-0.00472,0.00433,-0.01273,-0.00527,0.00143,-0.01585,0.01763,0.00997,-0.0124,-0.01098,0.011,0.00874,-0.01187,0.01995,-0.00752,-0.00091,0.00577,-0.00754,-0.00077,0.0069,-0.00156,-0.00247,-0.01869,-0.01165,0.01664,0.00115,0.01949,0.00107,0.01006,0.0065,-0.00736,-0.00672,-0.0155,0.00781,0.01679,-0.01238,-0.0101,0.00818,-0.0082,0.00488,-0.01254,0.01174,-0.0019,0.01588,0.01256,-0.00274,-0.02691,-0.00515,0.00402,-0.01757,-0.01669,0.00396,0.00571,0.00723,-0.03164,0.00822,-0.01632,0.0025,-0.01352,-0.00121,0.0129,0.00536,-0.01607,-0.01744,-0.00742,-0.00255,0.01559,-0.00262,-0.00616,-0.0235,0.0256,0.02965,0.0253,-0.00136,0.01239,0.00282,-0.00175,-0.00528,-0.00079,-0.0255,0.01068,-0.01896,0.00609,0.0111,-0.0094,-0.00512,0.00256,-0.02365,0.0154,0.00031,0.00513,0.01322,-0.03282,-0.00982,0.00487,0.01456,-0.00855,-0.01248,-0.0082,0.01168,-0.03415,-0.00711,-0.00825,-0.00648,-0.00615,-0.01273,0.00214,0.02163,0.0141,0.00392,0.01448,0.02555,0.00111,0.00935,0.01061,-0.0005,0.01056,0.00314,0.00305,0.00612,0.02319,-0.01191,0.00188,-0.02149,-0.0234,0.00598,0.01103,-0.01707,-0.01246,-0.00905,-0.00464,0.01388,-0.01179,-0.00051,-0.01179,-0.01837,-1e-05,-0.01139,-0.00602,-0.02993,0.02012,0.00537,0.00101,-0.015,0.00721,-0.00417,-0.01328,0.0038,0.04003,-0.02293,-0.0005,-0.01375,0.0084,0.00106,0.00057,-0.0124,0.00057,0.00476,0.00817,0.01079,0.01418,-0.00674,-0.00977,0.00992,0.00024,-0.02799,0.0079,0.00049,-0.00855,-0.00374,0.00493,0.02352,-0.00468,-0.00326,0.01803,-0.00234,-0.00837,-0.00969,-0.00153,0.02066,0.00605,0.0109,-0.01234,0.00327,-0.00265,0.00056,-0.00789,0.00626,-0.01815,0.00921,-0.02093,-0.00207,0.00196,0.0061,0.00872,-0.0001,-0.01419,-0.01297,-0.02951,0.00729,0.00414,0.014,-0.00471,0.02097,-0.02307,0.02029,0.00675,-0.01101,-0.01789,0.01684,0.02062,-0.02189,-0.01124,0.01659,0.01267,0.00903,0.02244,-0.00947,0.00207,-0.00731,-0.00464,0.00257,0.00261,0.00956,-0.0038,0.01902,0.01987,-0.00462,-0.01451,0.00048,0.01605,-0.02457,-0.01353,0.02625,-0.00665,0.00555,0.00538,-0.00382,-0.00827,-0.00573,-0.00474,0.00262,0.01459,0.00764,0.00119,-0.0061,-0.00035,-0.00759,-0.00986,-0.01369,-0.02538,0.01176,0.00123,0.02966,0.00036,0.00517,0.0003,0.01685,-0.00025,0.00406,0.01395,0.01008,-0.01482,-0.01279,0.02885,-0.01396,-0.00462,-0.02407,0.0148,-0.00233,-0.00989,0.01564,0.03608,-0.02317,-0.00877,0.014,-0.01652,-0.00198,-0.00957,-0.00371,0.00954,0.0118,-0.00807,-0.00163,-0.01523,-0.00204,0.00346,-0.01111,-0.00011,0.02211,-0.01011,-0.02997,-0.00397,0.0124,0.03711,0.01083,0.00841,0.00288,-0.01658,-0.01357,0.00754,-0.00542,0.00558,0.00065,-0.00528,-0.00343,-0.0078,-0.01164,-0.00194,-0.00496,-0.0135,0.00339,-0.01678,-0.00832,0.01766,0.00044,-0.01056,-0.01673,-0.01706,0.0067,-0.00127,0.01698,0.01321,-0.01089,0.00372,0.0049,-0.0043,0.00197,0.0172,-0.01093,-0.00811,-0.02084,0.01796,-0.01288,-0.01629,-0.00231,0.00712,0.00703,-0.0025,0.01722,0.01649,-0.00752,0.00064,0.02214,-0.00282,-0.00401,0.01419,-0.01228,0.00451,-0.00106,-0.00298,0.01863,-0.00739,-0.01463,-0.0184,-0.00066,-0.00141,-0.03307,-0.00562,-0.00034,-0.00856,-0.01461,-0.00904,0.00432,0.00657,0.00606,0.00582,-0.0059,-0.01427,0.01026,0.00534,-0.00902,-9e-05,-0.00324,0.01206,0.00633,0.01303,-0.02337,2e-05,0.00335,-0.00719,0.00733,-0.00435,-0.01718,-0.00776,-0.004,-0.00885,-0.00765,0.01538,-0.01595,0.00278,-0.00941,0.02654,0.00566,-0.00647,0.01867,0.01011,-0.00207,-0.00195,-0.0149,-0.01664,-0.00384,0.01502,8e-05,-0.00336,-0.01238,0.00263,-0.01454,-0.02609,0.00401,0.02187,0.00087,-0.00597,0.00948,-0.00275,0.00223,0.01771,0.00465,-0.1108,0.00249,0.01784,0.04649,-0.02112,0.00178,-0.00986,0.01524,0.01158,-0.00133,0.00964,0.00487,0.01159,-0.00197,-0.00282,-0.00574,0.00874,-0.01297,0.00156,-0.00774,0.00087,-0.00625,0.00156,0.0054,0.01283,-0.00365,0.01467,-0.00088,0.01004,-0.0093,0.00386,-0.00037,-0.0182,-0.01133,-0.0067,0.01217,0.01662,0.01654,0.00091,0.00167,0.00925,0.01646,-0.01527,-0.00407,0.01229,-8e-05,0.02525,0.00837,-0.02168,-0.00426,0.01694,-0.01444,0.01496,-0.00951,-0.00233,0.00754,-0.02365,0.02869,-0.00346,-0.00303,0.00584,-0.01625,0.00677,0.0186,0.0066,0.00407,-0.00851,-0.00477,0.00303,0.00206,0.00515,0.01856,-0.01721,0.01309,0.0009,-0.01557,0.00443,0.0033,-0.01427,-0.01109,0.01876,-0.01756,0.02472,0.01113,-0.00479,-0.00347,0.01148,0.00489,-0.00101,-0.01124,-0.02214,0.01557,-0.01468,-0.0188,-0.00763,-0.01689,-0.02882,0.00723,-0.01637,-0.02475,-0.01908,0.01359,-0.01735,0.00625,0.01282,-0.00215,-0.01737,-0.00325,-0.00862,-0.00185,-0.00259,0.00168,0.00305,0.00819,0.02225],[0.00954,0.00221,0.00131,0.00206,0.00732,0.00773,0.0013,0.00407,-0.00271,0.00367,0.01477,0.02573,-0.0033,-0.00448,-0.00057,0.00166,0.00405,-0.00238,-0.00496,0.01627,-0.00102,-0.00055,-0.00262,0.00629,-0.01044,-0.00136,0.00239,0.00389,0.00539,-0.0016,0.00455,0.00245,0.00295,0.00765,0.00029,0.01309,0.0049,-0.00426,0.00241,-0.00183,-0.00157,0.00235,-0.00791,-0.00944,-0.01015,-0.00448,0.00043,-0.00141,0.00225,-0.00051,-0.00522,-0.00065,0.00735,0.012,-0.00423,-0.00451,0.02315,-0.0036,0.00258,0.00162,-0.00695,-0.00348,-0.00581,-0.00552,-0.0031,-0.01046,1e-05,0.00345,-0.00779,-0.00113,0.00616,-0.00506,-0.00214,0.00207,8e-05,0.01374,0.00819,-0.00873,0.0007,0.00142,0.00251,0.00818,0.00212,0.00275,-0.00249,0.01363,0.00087,-0.00044,-0.00053,-0.00075,-0.00164,0.00878,-0.00581,0.00045,-0.00314,0.00334,-0.00436,0.00058,0.01154,-0.00406,-0.01237,0.00517,-0.00818,-0.00062,0.00912,0.00554,-0.00219,0.00012,-0.00651,-0.00354,0.00185,-0.00463,0.00809,0.00339,-5e-05,0.00074,0.00217,-0.00341,-0.00953,-0.00567,-0.00921,-0.00217,0.00423,-0.01036,-0.00426,-0.00516,-0.00255,0.00165,0.01411,-0.0108,-0.01005,-0.00627,0.01006,0.00421,0.00263,-0.00109,-0.00231,0.0063,-0.00693,-0.00897,-0.0002,0.00924,-0.00133,-0.00387,0.00117,-0.00049,0.01027,0.00927,-0.00384,-0.00606,0.01036,0.01013,-0.00711,-0.01079,-0.00153,-0.00123,-0.00808,0.00204,-0.00565,0.01017,0.01314,0.00319,0.00446,0.00026,-0.01775,0.00415,0.00869,5e-05,-0.00767,0.00244,0.00058,-0.00301,-0.01412,-0.00758,0.00507,0.00237,0.0056,-0.00204,-0.00862,-0.00103,-0.01444,-0.00329,0.00645,-0.00291,0.00698,0.00783,-0.00032,0.00092,0.00019,-0.00342,-0.00536,0.00411,-0.00638,0.00356,-0.00212,-0.00156,-0.01633,-0.01066,0.001,0.00769,-0.00203,0.00531,0.00656,-0.00488,-0.00662,-0.00429,-0.00702,0.00491,2e-05,0.00376,-0.00802,-0.00222,0.00722,0.01015,0.00326,0.00663,0.0063,-0.00049,-0.0124,-0.00609,0.00114,-0.00331,0.01162,0.0214,0.00177,0.00816,-0.0046,0.0055,-0.00153,0.0034,-0.00438,-0.00843,-0.00813,-0.00489,-0.00968,0.00152,-0.00421,-0.00019,0.00408,0.0027,0.00162,0.00385,-0.00712,0.00405,-0.00704,0.00027,0.00227,0.00731,0.00631,-0.0066,0.00642,0.01915,-0.00943,-0.00464,-0.00294,-0.00384,-0.00243,-0.00858,-0.0001,-0.01252,0.00549,-0.00497,8e-05,-0.00745,-0.00687,-0.00214,-0.00042,0.005,-0.01466,0.00818,0.00581,-0.00546,-0.01167,-0.00807,-0.00034,-0.01204,-0.00255,0.00341,-0.00118,-0.00151,-0.00174,0.00407,0.00493,-0.0058,-0.00442,0.0003,0.00346,-0.01748,-0.00119,0.00053,0.00467,0.00333,0.00442,0.00321,0.00756,0.00119,-0.01226,-0.00189,-0.00736,0.00086,0.00507,-0.00828,0.00047,0.00137,0.00485,-0.00222,-0.0022,-0.00646,0.00415,0.00453,-0.00983,-0.00129,0.00996,-0.00085,-0.00475,-0.0099,-0.00079,-0.00676,0.01115,0.00788,0.00991,-0.00306,-0.00027,0.00583,0.00337,0.00219,0.00204,0.00625,0.00221,-0.01038,0.00527,0.0016,0.00136,0.00367,0.00879,-0.00215,0.00787,0.00148,0.01176,0.00456,-0.00074,-0.00018,-0.00073,0.00719,-0.00176,0.009,-0.00141,-0.00114,0.00491,-0.00595,-0.00991,-0.00142,-0.00236,-0.00043,0.00019,0.01483,-0.00145,0.0034,-0.01319,-0.00473,-0.00893,0.00594,-0.00301,0.00249,0.00247,-0.00069,-0.00531,6e-05,-0.01079,0.00281,0.00607,0.006,0.00427,0.00013,0.00725,0.01223,0.01059,0.00547,0.00478,-0.00696,-0.002,0.00281,0.00428,0.00752,0.00308,0.0052,0.00154,0.00654,0.00437,0.00174,-0.00172,0.00375,0.0101,-0.00216,-9e-05,0.01071,-0.01748,0.01224,0.00521,-0.00786,0.0091,0.00787,-0.00624,-0.0195,0.0028,0.00213,0.00446,-0.00449,-0.003,0.00442,-0.00434,-0.0008,0.0075,0.00531,0.00194,0.00453,0.00686,-0.00073,0.00032,-0.00046,0.00469,-0.00175,-0.00543,0.00528,-0.0,-0.00435,0.00102,0.00135,-9e-05,-0.00293,-0.00218,-0.0038,-0.00017,-0.0002,-0.00183,0.0014,-0.00446,-0.0051,0.00432,-0.00699,-0.006,0.00304,-0.00743,-0.00084,0.00415,0.00568,0.00566,0.00849,-0.00078,0.00252,-0.0029,0.00291,-0.00587,0.00839,-0.00896,-0.00207,0.00163,0.00922,-0.00302,0.00859,0.01057,0.00259,0.00545,0.0124,-0.00196,-0.0034,0.00412,0.00502,-0.00945,-0.00645,-0.00245,0.00019,0.00431,0.00525,-0.00047,0.00499,-0.00505,0.00309,-0.00751,0.00244,-0.00146,0.00406,0.00623,-0.00278,-0.0015,0.00217,-0.0025,-0.00387,-0.00011,0.01098,-0.00366,0.00137,-0.00623,-0.00484,-0.00064,-0.00318,-0.0042,0.0065,-0.00099,-0.00794,-0.0015,0.00623,0.00757,0.00158,0.00775,-0.01156,0.0088,-0.00027,-0.01481,0.00132,0.00571,-0.00297,-0.00434,0.0069,0.00251,0.00127,-0.00135,0.00523,-0.0085,0.00018,0.02381,-0.01517,-0.00104,-0.00733,0.00989,0.00685,-0.00564,0.00167,0.00842,-0.00335,-0.0055,0.00275,0.00033,-0.00139,0.00043,-0.0045,0.00563,-0.00964,-0.0005,-0.00488,-0.00598,0.00308,-0.00063,-0.01002,-0.00241,0.00567,-0.00464,0.00218,-0.0146,0.00798,-0.00627,0.00568,-0.00335,0.01724,-0.01069,-0.0061,-0.00031,-0.00866,-0.00524,0.00027,-0.00228,0.00017,-0.00073,0.00594,-0.01141,-0.00053,0.00274,-0.00828,-0.0063,-0.00189,0.01186,0.0059,-0.0067,0.00206,-0.01108,2e-05,0.00393,0.00294,0.00051,0.00275,0.00892,0.00597,0.00148,0.00224,-0.00324,-0.01087,0.00898,0.00103,0.00283,-0.00701,0.00415,-0.01225,-0.00487,0.00211,0.00734,0.00248,0.00313,-0.00244,0.00777,-0.00343,-0.00119,0.00533,0.00378,-0.00422,-0.00816,-0.01743,0.00344,-0.00516,0.00144,-0.00482,0.01036,0.00482,0.00333,0.00772,-0.0052,-0.00598,-0.00621,0.00679,0.0042,0.00543,0.00537,-0.00376,-0.00396,0.00287,-0.00138,0.0016,-0.00746,0.00569,-0.0008,0.00203,-0.00175,-0.01175,-0.00446,-0.00345,0.00017,-0.00507,-0.00411,-0.00117,0.00616,0.00401,-0.00082,-0.00025,0.00496,0.00495,-0.0,0.00662,0.01157,0.00473,-0.00109,-0.00132,0.01044,0.00838,0.00345,-0.00862,-0.00584,0.00132,0.0013,-0.00015,-0.00528,0.00728,0.01113,0.01966,-0.01343,-0.00305,0.01739,0.02401,-0.01639,-0.00792,-0.00735,0.00817,0.00977,0.0105,0.00032,-0.00042,0.00264,-0.00386,0.01237,0.01212,-0.00478,-0.00485,-0.00448,0.00153,-0.00526,-0.00359,-0.0067,0.00722,-0.00615,-0.00551,0.00267,0.00345,-0.00976,0.00322,-0.00952,-0.0146,0.01313,0.00102,0.00386,0.00736,0.00397,0.00446,0.00259,-0.00569,0.00177,0.00323,0.00253,-0.00425,-0.01044,0.00225,0.00612,-0.003,0.00206,-0.0023,-0.00447,0.00257,0.0046,-0.00519,-0.01187,0.00097,-0.00745,-0.00519,0.00311,-0.00129,-0.00867,-0.00894,0.0035,-0.02272,-0.00199,0.00508,-0.00503,0.00688,-0.00281,0.01363,-0.01041,0.00645,-0.01776,-0.00256,0.00265,-0.0043,0.00877,0.00473,-0.00073,-0.0,-0.00132,0.00223,0.00259,-0.00189,0.00225,0.0096,0.00129,0.00787,0.00266,0.00755,-0.0049,0.00359,-0.00521,-0.00192,-0.01036,-0.01386,-0.00682,-0.0088,0.01082,-0.00596,0.00513,0.00013,0.01042,-0.00246,0.01056,-0.00648,0.00176,0.00104,-0.00697,-0.00822,0.00579,-0.00755,-0.00663,0.00227,0.00228,0.00106,-0.00499,-0.00751,0.00418,0.0009,0.01175,-0.00256,-0.01433,-0.0118,0.00049,0.00655,0.00475,0.00443,-0.00917,-0.00918,0.00696,-0.01924,-0.00259,0.00476,-0.00706,0.00715,-0.0098,0.00289,-0.0085,-0.01077,0.00159,-0.0001,-0.00453,0.01584,0.00881,0.00262,0.00047,-0.00666,0.01673,0.00466,-0.00471,0.01221,0.00261,-0.01067,0.00267,-0.00446,-0.00737,0.0009,-0.00876,0.00248,0.00402,0.00531,-0.00059,-0.00407,-0.00346,0.00453,0.00123,0.00894,-0.00805,-0.0072,0.01394,0.00265,0.01477,-0.0101,-0.00154,-0.00791,-0.00416,-0.0125,-0.00436,0.0157,-0.00381,-0.0097,0.00408,-0.0006,-0.00848,0.00759,0.01435,0.00396,-0.00914,0.0086,0.0077,-0.00586,-0.0023,-0.00783,-0.00119,0.00792,0.00034,-0.00505,-0.00682,0.00045,0.00938,0.00351,0.01045,0.02218,0.00031,-0.00355,-0.01171,-0.00298,0.00427,0.00606,-0.01754,-0.01004,0.0028,-0.00191,0.00368,-0.00266,0.00648,0.00024,0.00012,0.00013,0.00999,-0.00733,0.00705,0.00899,-0.00032,0.00217,0.00052,-0.01081,0.00655,0.00196,0.0048,-0.00098,0.00882,0.00302,-0.00292,0.00013,-0.01898,0.00548,-0.0078,0.02103,0.01191,0.00011,-0.02057,-0.0064,0.00909,-0.00323,-0.00806,-0.00643,-0.00238,0.00109,-0.00317,0.00504,0.00214,0.00059,-0.005,-0.01438,0.00429,0.00856,0.00044,-0.00577,0.00584,-0.00555,-0.0036,0.00856,-0.00863,0.00107,0.00081,-0.00269,0.00282,-0.00309,-0.01018,-0.0105,-0.00351,-0.00624,8e-05,-0.00252,-0.00247,-0.00475,0.00435,0.0153,-0.00419,0.00419,0.0028,-0.00135,0.0082,0.00221,0.00205,-0.0032,0.00856,-0.00163,0.00587,-0.00894,-0.01288,-0.00647,0.00194,0.00572,-0.00859,-0.00424,-0.0097,-0.00668,0.00881,-0.00575,-0.00114,0.00326,0.004,-0.00396,-0.01542,-0.00341,0.00632,0.00611,-0.00472,0.00188,-0.00038,0.00557,-0.0064,0.00417,-0.00964,-0.00606,0.00155,-0.00152,-0.00402,-0.006,0.00175,0.00119,0.00059,0.0017,-0.00266,-0.00108,0.00067,0.00397,-0.00151,0.00453,-0.00143,0.00226,-0.00666,0.00331,0.00266,-0.00229,-0.00511,-0.0105,-0.00019,-0.00185,-0.00256,-0.01051,-0.00349,-0.008,0.00329,-0.00402,0.00141,0.00618,0.00282,0.00625,-0.00563,-0.01466,0.00922,0.0092,0.00845,0.00387,0.00085,0.00189,0.00352,0.0033,-0.00178,0.00259,-0.00268,-0.00097,0.00165,-0.00271,-0.00632,0.00592,-0.00668,0.00684,0.00435,0.01427,-0.01079,-0.00438,-0.00098,0.00678,0.00843,-0.01377,0.00536,-0.01245,0.00151,0.00497,0.00195,-0.0044,0.00337,0.00193,0.00081,0.0028,-0.00059,-0.00067,0.00154,-0.00283,0.01271,-0.00952,-0.00154,0.00343,-0.01132,-0.01019,0.01746,0.00749,0.01198,-0.00966,-0.00367,0.00319,-0.00164,0.01161,0.00657,-0.00692,0.00512,0.03853,0.0123,0.00678,0.00188,0.0036,0.00841,-0.00349,-0.00567,-0.00906,-0.01139,-0.00372,0.00847,0.00494,-0.00329,-0.00637,-0.00101,-0.01067,0.00228,-0.0001,-0.01346,0.00756,0.00437,-0.00485,-0.00114,0.00313,0.00188,0.00493,0.00459,-0.00773,-0.00292,-0.00031,4e-05,-0.00202,0.00922,-0.00289,-0.00435,0.00493,0.00291,0.01083,-0.00294,0.00289,-0.01487,-0.01129,-0.0068,0.0071,-0.01663,0.00174,0.0038,0.00113,-0.00378,0.01272,-0.00915,0.00149,0.00281,-0.00431,-0.00842,0.00034,-0.00283,-0.00826,-0.01623,-0.00094,-0.0019,0.00252,0.00212,0.00185,-0.00107,-0.01381,0.00816,0.00312,0.00207,0.00111,-0.00169,-0.00071,0.00036,-0.0087,0.00586,-0.0009,-0.00133,0.00941,0.00246,-0.00495,-0.00302,0.00432,-0.0095,-0.00414,-0.00121,-0.00076,0.0059,0.02011,0.00241,-0.00731,0.00348,-0.00486,0.02514,0.00692,-0.00243,-0.0052,0.00278,0.0034,-0.00926,-0.01047,0.00227,0.00473,0.0015,-0.01497,0.00247,0.006,-0.00111,-0.00491,0.00131,-0.00274,0.01355,0.00049,0.00139,-0.00496,0.00254,-0.00401,-0.00119,-0.02166,0.00196,-0.00052,0.00404,0.00786,0.00833,0.00277,-0.00366,0.0083,-0.01417,-0.01558,-0.01166,0.00507,9e-05,0.00169,0.00402,-0.00351,0.00408,-0.00569,0.0012,-0.00485,0.00378,-0.00623,0.02057,-0.00172,0.00028,0.01091,0.00619,-0.0061,-0.00993,-0.00268,0.00645,-0.00244,-0.01003,0.00955,0.00562,-0.00207,0.00852,-0.01949,-0.00824,-0.00197,-0.00656,0.0003,-0.00368,-0.00695,-0.00173,-0.00342,-0.00525,0.00435,-0.00176,0.00588,0.00696,-0.00414,-0.00289,0.00253,0.01229,-0.00274,0.00674,-0.00754,0.00714,-0.00301,0.009,-0.00171,-0.0036,-0.0008,-0.00482,-0.00479,0.00362,0.01273,0.00919,0.00373,0.00104,-0.00544,-0.00532,-0.00496,9e-05,-0.00186,-0.01188,-0.0049,-0.01143,0.00134,0.00276,-0.00407,-0.00071,0.00054,-0.01102,-0.00048,-0.00195,-0.01095,-0.0043,0.00027,0.00433,-0.00799,-1e-05,-0.00815,0.00181,0.00476,0.00039,-0.00013,0.00091,-0.00663,0.00047,0.00607,-0.00193,0.00802,-0.00139,-0.00396,0.00298,0.00777,0.00537,-0.01005,0.00874,-0.00294,-0.00249,0.0002,0.00373,0.0058,-0.00654,-0.0055,-0.00892,0.0068,-0.00312,0.00085,0.0024,-0.00909,-0.00024,0.00104,0.00014,-0.00231,0.00341,-0.00116,-0.00095,-0.00337,-0.00611,0.00547,0.00814,0.00157,-0.00531,0.00911,-0.00793,-1e-05,-0.00111,0.00716,-0.00049,0.00598,0.00053,-0.01238,0.00163,-0.00027,0.0017,-0.00357,0.00036,0.00505,0.00533,-0.00483,0.00062,-0.00625,0.00296,-0.00682,0.00088,-0.00179,-0.0008,-0.00186,0.00549,-0.002,-0.00176,0.013,0.00478,0.00533,-0.00708,-0.00839,0.00267,-0.00185,-0.00804,0.00283,0.00431,0.00499,-0.0036,-0.00628,0.00982,-0.00228,0.01396,-0.00052,-0.0041,0.00791,0.00724,0.00298,-0.00449,0.00589,-0.00406,-0.00355,-0.00507,0.0071,0.00269,0.00366,-0.00164,-0.00305,-0.00238,-0.00269,0.00282,-0.00908,-0.00775,-0.00605,-0.00107,0.00718,0.00363,0.00991,-0.00428,0.01026,-0.01031,-0.00591,-0.01073,-0.00135,-0.01331,-0.00301,-0.00044,-0.00398,-0.00615,-0.00224,-0.00112,0.00125,-0.00369,-0.00145,0.00529,-0.01679,-0.00269,0.01017,0.00132,-0.00339,-0.00353,0.00314,0.00066,0.01975,-0.01085,-0.01069,-0.00366,0.0155,-0.00172,-0.00967,-0.00382,-0.01477,-0.00443,-0.00475,0.00086,-0.00642,-0.00078,-0.00502,-0.01097,-0.00626,-0.00826,-0.00441,0.00845,-0.00946,-0.00387,0.00196,0.00135,-0.00319,-0.01318,-0.00712,-0.00186,0.01241,-0.00315,0.0064,-0.00382,0.00472,0.00312,-0.00726,0.0003,-0.00137,0.0026,-0.00194,0.00223,-0.01397,-0.0029,0.00606,-0.00362,0.00198,-0.02231,-0.00636,-0.00033,0.00239,0.00807,-0.00387,-0.00265,-0.00363,0.00558,-0.00289,-0.00943,-0.00556,-0.00623,0.01166,-0.01029,0.00599,0.00459,-0.00461,-0.01481,0.00992,-0.00511,-0.0037,0.00451,-0.00748,-0.00085,-0.00621,0.00442,0.00424,-0.00493,0.00448,0.00475,0.00244,0.0039,0.00594,0.00779,-0.00257,0.00019,0.00503,0.00074,0.00593,0.00103,-0.00466,-0.00338,-0.00445,0.0117,-0.00255,0.00596,0.00399,0.00533,0.00295,0.00283,-0.00462,0.0043,-0.00041,-0.00427,0.00205,0.01549,0.00115,0.01009,0.00879,-0.00047,-0.00112,0.0016,-0.00876,-0.00724,-0.01191,-0.00634,0.00422],[-0.00227,0.00698,-0.00845,-0.00467,0.00151,-0.00294,-0.00575,0.00424,0.00309,0.00253,0.0078,0.00414,-0.00809,0.00371,0.00551,0.00275,-0.00911,0.00212,0.00349,0.01437,-0.01308,0.00615,0.0063,-0.00263,-0.00631,-0.00384,-0.00529,0.009,0.00377,0.00489,0.00353,-0.00433,0.01188,-0.0021,0.00585,0.01337,0.01015,-0.00723,0.00358,0.01239,-0.00513,0.00485,-0.02123,-0.01381,-0.00435,2e-05,-0.00274,0.00479,-0.00299,-0.00892,-0.00086,0.01212,-0.00062,0.00888,-0.01723,-0.00641,0.00483,0.00951,0.00487,-0.01741,-0.01013,-0.00364,-0.00033,-0.00343,-0.01544,-0.0078,-0.00273,-0.00374,-0.00822,-0.00075,0.00085,0.00314,0.01215,-0.00847,0.00231,0.00078,-0.00737,-0.00235,0.00117,0.00095,-0.01825,0.00227,-0.01723,-0.00314,-0.0058,0.00582,0.00313,-0.0041,-0.00956,-0.00147,0.00274,-0.0042,-0.0133,0.00502,0.0024,-0.00567,-0.01008,-0.00668,-0.00204,-0.00492,-0.01329,0.00091,-0.00472,0.01469,0.00962,8e-05,-0.0008,-0.00205,-0.00443,0.00336,0.00399,0.00495,0.0036,-0.00078,-0.00159,0.00946,0.00413,-0.00666,-0.00501,0.00733,-0.005,-0.01289,-0.00466,-0.00724,0.00404,-0.01111,-0.00158,-0.00705,-0.00435,-0.00629,-0.00766,0.00072,-0.00151,-0.01597,0.01127,-0.00043,0.01369,-0.00532,0.00057,-0.00719,-0.00983,-0.0013,-0.00906,-0.00849,-0.00581,0.00333,0.01859,0.00779,-0.00792,-0.00206,-0.00083,0.00563,0.01031,0.00724,-0.00622,0.0023,-0.01591,0.00367,-0.00881,-0.00183,0.00032,-0.00042,-0.01276,0.00372,-0.01516,-0.0078,0.00553,-0.00382,-0.01759,-0.0017,0.00264,-0.00395,-0.01046,0.00478,0.00381,0.01671,0.0036,0.00721,-0.00939,0.0018,-0.02215,-0.01265,0.00354,0.00377,0.00984,0.00083,-0.00045,0.0029,0.00395,0.01028,-0.00179,0.00171,0.01234,0.01467,0.01211,-0.00545,-0.02513,-0.00489,-0.00893,-0.01315,-0.00577,-0.01095,0.00692,-0.01212,-0.01405,0.00854,0.00109,-0.00694,-0.00963,-0.00155,-0.00393,0.00683,0.00823,0.01584,0.00997,-0.00234,0.00588,0.00629,-0.00181,-0.01638,0.01198,-0.0152,0.00056,0.00394,-0.00622,0.0061,-0.00741,0.02169,-0.01406,0.00033,-0.01328,-0.00665,-0.0145,-0.01244,0.00441,-0.00417,-0.00031,-0.01421,0.00081,0.01022,-0.00296,0.00775,-0.01723,0.02023,0.00717,-0.00914,-0.00219,-0.00613,-0.00486,-0.0035,-0.00017,-0.00071,-0.01086,-0.0075,0.0173,0.00035,0.01145,0.00222,-0.01168,-0.00323,-0.00126,-0.00246,-0.00247,-0.0115,-0.00507,-0.00277,0.00315,0.00585,-0.00226,0.00697,0.00207,-0.00104,-0.00766,-0.0076,0.005,-0.0049,-0.00935,0.0016,-0.00322,0.00996,0.00265,-0.01049,-0.00166,-0.00711,0.00589,-9e-05,0.00909,-0.00327,0.00051,-0.00897,0.00298,0.01811,-0.01047,0.00336,0.00037,0.00013,-0.01923,0.00214,0.0021,0.00487,0.00109,-0.00945,-0.00631,-0.00851,-0.00768,0.01374,0.00766,-0.0038,0.00642,0.00959,-0.0055,-0.00762,0.00905,-0.00281,-0.00464,0.00067,0.002,0.00552,0.00197,0.0085,0.00838,0.00239,0.00447,-0.00182,0.00209,0.00117,-0.00133,0.00333,-0.00817,-0.00312,0.00894,0.0001,0.00037,0.00889,0.00107,-0.00785,-0.00267,-0.01152,-0.00234,0.00329,-0.00273,0.0021,0.00889,0.00899,-0.00102,-0.00504,0.01825,-0.00335,0.00049,-0.00368,-0.00502,-0.00123,-0.00658,0.00069,-0.00839,0.01268,-0.02785,-0.00666,-0.01075,-0.00436,0.0078,0.00166,-0.01076,-0.00541,0.00175,0.00344,-0.00118,0.00928,-0.00378,-0.01389,-0.00923,0.01052,-0.00164,0.00262,0.00089,0.00884,0.00586,-0.00207,0.01035,-0.00299,0.00763,0.00273,0.00726,-0.005,0.00313,0.00646,-0.0037,0.00289,0.01223,-0.0096,0.00992,0.00754,-0.00602,0.00708,0.00667,0.01697,-0.00916,0.00786,0.00053,0.00729,0.00146,0.01281,-0.0148,-0.0108,0.00518,-0.00193,0.00338,-0.00686,-0.0007,0.01426,-0.00811,-0.006,-0.00242,0.00616,-0.00535,0.00048,0.00163,-0.00672,0.00506,0.00651,-0.0132,-0.00867,-0.00118,-0.00496,-0.00824,0.01325,0.00325,0.00235,0.00097,-0.01211,-0.01519,-0.00109,0.0021,0.0024,-0.00121,0.00733,0.0022,0.00401,0.00452,0.00269,-0.01401,-0.01221,-0.00611,-0.00274,0.00361,-0.00544,-0.00725,-0.00228,-0.00197,0.0079,-0.0053,-0.00506,-0.01618,0.0173,-0.00375,-0.0222,-0.00293,0.00516,0.0073,0.00292,0.00378,0.0081,-0.01391,0.0091,-0.01243,-0.00569,0.00233,-0.01492,-0.00578,-0.00259,0.00775,0.00325,0.0073,-0.00593,0.00864,-0.00121,-0.00253,-0.00218,-0.00195,0.00419,0.00105,0.01163,0.00493,0.01168,-0.00079,-0.00264,-0.00388,0.00642,-0.00713,-0.00356,0.01134,-0.00259,-0.00313,-0.00971,0.01067,-0.00151,0.00292,-0.00059,0.00389,-0.005,-0.00965,-0.0115,-0.00714,0.00582,-0.00275,0.00123,0.01239,-0.00462,-0.0036,-0.00399,0.02187,0.00277,-0.00472,-0.00359,0.00965,0.00109,0.00285,-0.00093,0.00649,-0.01776,5e-05,-0.01064,-0.00561,-0.01598,0.01119,0.01725,-0.00047,0.01986,0.00669,0.00373,-0.01105,0.00371,0.00235,-0.01465,0.00309,0.00218,0.01074,0.00415,-0.01182,-0.01047,-0.01178,0.00196,-0.01288,-0.00911,0.00448,0.01626,-0.01039,0.01077,-0.01399,-0.00624,-0.00109,0.02984,0.0019,0.00028,-0.00698,-0.00749,0.00659,-0.0116,-0.00036,-0.00813,0.00384,0.00053,-0.00137,0.0128,0.00349,-0.00023,0.005,0.00142,-0.00659,-0.01681,0.01089,0.00176,-0.00776,-0.00758,-0.00037,0.00669,-0.01306,0.0027,0.00021,0.00145,-0.00384,0.00585,0.01146,-0.00836,0.00133,-0.00094,0.00436,-0.0148,-0.00436,-0.0127,-0.00748,0.00106,-0.01139,-0.0076,0.00604,-0.00533,0.01615,-0.00265,0.00664,0.00222,0.00191,0.02052,0.00328,0.01134,-0.00622,-0.00473,-0.00427,-0.00987,-0.0157,-0.00816,0.00301,0.0011,0.00302,0.01075,0.00092,0.00157,-0.01729,-0.00025,0.00887,0.01643,0.0104,0.00376,0.00436,0.00304,-0.01265,0.01042,-0.00201,-0.00956,-0.00493,-0.00591,-0.00288,-0.0077,0.00357,-0.01972,-0.00648,0.00086,-0.00285,0.0024,-0.00083,-0.00089,0.00768,0.00261,0.01358,0.00438,-0.00867,0.0046,0.00417,-0.00632,0.00655,-0.00402,-0.00869,0.00547,-0.00698,-0.01102,0.00486,0.02066,-0.00861,0.01184,-0.00744,0.01155,-0.00111,0.00415,-0.00342,0.00145,0.00345,0.01871,-0.01834,-0.00104,-0.00116,-0.01106,0.01449,0.00796,0.00591,-0.01922,0.01462,-0.00244,-0.00958,0.00502,0.00138,-0.006,0.00318,-0.00346,-0.00133,0.01787,0.00083,0.00185,0.00151,-0.00149,-0.01676,0.00672,-0.01189,-0.01135,-0.00606,-0.01718,-0.00234,-0.00381,-0.00296,0.01362,0.00483,0.00655,-0.0154,0.00328,-0.00469,0.003,-0.00139,-0.00465,-0.0064,0.00119,0.00686,-0.01022,0.00679,0.00067,-0.01116,0.00066,-0.00282,0.00261,0.00054,0.00582,-0.01284,0.00701,0.01092,0.00648,-0.01754,0.00607,-0.00459,-0.00646,-0.00319,0.01182,0.01573,0.00634,0.00038,0.00751,0.0075,0.00638,-0.01712,-0.00806,-0.00556,-0.01125,-0.0001,0.006,0.00088,0.00335,0.00386,0.00624,0.01256,0.00583,-4e-05,0.01566,-0.0046,0.00135,-0.00059,0.00206,-0.01262,0.00981,0.00259,-0.00618,0.00315,-0.00448,-0.00247,-0.0079,-0.00256,-0.00981,0.00053,-0.00411,0.00378,-2e-05,0.0013,-0.00617,0.00441,0.00749,-0.00634,0.00087,0.00418,-0.01369,-0.00156,-0.00518,0.00951,-0.00612,-0.01216,0.00939,0.00055,0.00409,-0.00316,0.00729,-0.00923,-0.01532,0.01009,0.00257,0.01905,0.00023,-0.00496,-0.01334,-0.0006,-0.01602,-0.01282,0.02057,-0.00054,-0.0109,-0.00893,0.0012,-0.00134,-0.01741,0.00705,0.0026,-0.01008,0.0084,0.00573,-0.01273,-0.00796,-0.00256,0.00642,-0.0092,-0.00272,0.02465,-0.00152,0.0066,0.01293,0.01054,-0.00844,0.00454,-0.00325,-0.00153,0.00129,-0.01622,-0.00426,-0.00475,0.00287,-0.00628,0.00206,0.00751,-0.00878,0.00822,0.00464,0.00075,0.00179,-0.0044,-0.00501,-0.01119,-0.0016,-0.0017,-0.0042,0.00295,-0.00735,0.0027,-0.00349,-0.00038,0.00093,0.01179,0.01125,-0.00186,-0.00868,-0.00071,0.00799,-0.00963,0.00262,0.00151,-0.00128,-0.00838,-0.0098,0.00163,-0.009,0.00488,-0.00221,0.00077,-0.00073,0.015,-0.00896,-0.00032,-0.0048,-0.01003,0.01324,0.00342,-0.00498,-0.00785,0.00646,-0.00672,0.00206,-0.00448,0.01396,-0.00366,0.00122,-0.00731,0.01239,0.01825,0.00475,0.00385,-0.00203,-0.01649,0.00902,-0.00186,0.00167,0.00568,1e-05,-0.0143,-0.01309,0.00389,-0.00833,0.00419,-0.00511,-0.00725,-0.00319,-0.00264,0.01071,0.01215,-0.00212,-0.01217,0.0088,-0.01547,0.01282,-0.00014,-0.00137,-0.00857,-0.00194,0.00595,0.00139,0.00836,-0.00529,0.00131,0.01207,-0.00127,0.01135,-0.00776,-0.00912,-0.00721,-0.00163,0.00533,-0.0132,-0.00703,0.01564,-0.00208,0.00041,-0.00059,-0.00038,0.00783,-0.0008,0.00136,-0.00315,-0.00849,0.00286,-0.00461,-0.001,-0.01001,-0.00648,-0.01157,0.0016,-0.00877,0.01037,-0.00169,-0.01546,-0.00254,0.00356,-0.00863,0.0022,-0.00066,-0.01129,-0.00432,0.00099,-0.01453,-0.00259,0.00337,-0.00452,0.0051,0.00135,0.00138,0.0028,0.0177,-0.00075,-0.00768,0.00197,-0.0,0.00508,-0.00199,0.00085,0.01383,0.0001,0.00346,-0.00516,0.00042,-0.00733,-0.00677,0.00434,0.0046,-0.01391,0.00776,0.00125,-0.00201,-0.00341,0.00519,0.00032,-0.0047,-0.00066,0.00132,-0.00675,0.00527,-0.00791,-0.00713,-0.01184,0.00148,0.0035,-0.00547,0.00335,0.00388,0.00596,-0.00396,-0.0047,-0.00143,-0.01379,0.00327,-0.00783,0.00888,0.00248,0.00325,0.00308,0.00407,0.00542,-0.00869,0.00707,-0.00214,0.01528,-0.0044,-0.00503,-0.00021,-0.00171,-0.00304,-0.00056,0.00179,0.00308,0.00235,0.00276,0.00294,0.00201,0.0062,-0.00154,-0.00831,-0.00469,0.00118,0.00339,0.00636,-0.00814,0.00677,0.00242,-0.00307,0.00429,-0.00038,0.01062,-0.00056,0.00911,0.00251,0.00639,0.00347,-0.01224,-0.00015,0.0004,-0.00997,0.01221,-0.00694,0.00596,-0.01488,0.00301,-0.00839,-0.00149,-0.01481,7e-05,0.00399,0.00634,-0.00933,-0.01571,-0.00163,0.00695,0.00887,-0.00861,-0.00282,0.00087,0.02612,0.00286,0.02309,-0.00242,0.00569,0.00865,-0.00791,-0.01125,-0.00393,-0.01926,0.00177,-0.00023,0.00938,0.00516,-0.01476,-0.00837,0.0157,-0.00503,0.00499,0.00023,0.00226,-0.00466,-0.00514,0.0049,0.00215,0.00877,-0.00398,-0.01209,0.00206,-0.01313,-0.00942,0.00317,-0.00556,0.0151,0.00314,0.00718,-0.002,0.00916,0.01277,-0.00588,0.0008,0.00835,-0.01029,-0.00367,-0.00035,-0.01477,0.00062,-0.00245,-0.00885,-0.00572,0.01953,-5e-05,-0.00435,-0.00802,-0.00715,0.00238,0.00736,-0.01173,0.00436,-0.00089,0.00062,0.00258,0.00166,0.00646,-0.00234,-0.01336,-0.01412,-0.00227,0.00462,-0.01483,0.0027,0.00917,-0.00039,-0.0014,-0.0053,0.00255,-0.00265,-0.00026,0.01249,-0.00056,0.01237,2e-05,0.00495,0.00333,0.00014,-0.00245,0.00124,0.00171,0.01963,0.01086,-0.001,0.00243,-0.00805,0.01328,-0.00512,0.00233,-0.00208,-0.01644,0.00277,-0.00329,0.00582,-0.00059,-0.00244,-0.00566,0.00483,-0.0082,0.01276,-0.01291,-0.01274,0.01241,0.00059,0.00878,-0.01084,-0.00649,0.00448,-0.00196,-0.00979,0.01072,-0.01265,-0.00532,-0.01175,-0.00168,-0.00073,0.00163,-0.00106,-0.00769,0.00904,-0.01346,-0.01086,0.01272,0.00477,0.00307,0.00055,0.01638,-0.0017,0.01098,-0.00441,-0.00494,-0.01565,0.01002,0.01273,-0.00326,-0.01189,-0.00385,0.009,0.00737,0.00996,0.00384,-0.00359,0.01312,-0.00152,-0.01014,-0.00534,0.00182,-0.01239,0.01503,-0.00694,-0.01024,-0.01329,0.00926,0.0013,-0.00057,-0.00182,0.01197,-0.0129,-0.00371,0.01822,-0.00785,0.00695,-0.0048,-0.00566,-7e-05,0.00178,-0.00592,0.0088,0.01516,-0.00479,-0.01285,0.00342,-0.00078,-0.00751,0.01493,0.0012,0.00625,0.00475,0.00449,-0.00011,0.01386,-0.00363,-0.0012,0.00546,-3e-05,-0.00072,0.00286,0.00234,-0.01498,-0.00859,-0.00543,-0.01071,0.00465,-0.01587,0.00714,0.00428,-0.00304,-0.00577,0.00744,-0.01071,-0.00249,0.00064,0.0025,-0.00544,0.01133,0.00874,-0.00152,-1e-05,0.0018,0.00589,-0.01051,-0.00693,0.00134,0.0109,0.00654,0.0079,0.00939,0.00883,0.0047,-0.00022,-0.00266,-0.00644,0.00332,0.0025,0.0062,-0.00061,0.00987,-0.00284,-0.00597,0.00023,-0.01363,-0.00171,0.00555,-0.00911,0.00822,-0.00045,-0.0157,0.00185,0.007,0.0008,-0.00094,-0.01555,0.00071,0.00436,-0.00566,0.0144,0.00696,0.00055,-0.00539,0.00463,-0.00489,0.00515,-0.01215,-0.00531,-0.01568,0.00701,0.00225,-0.01261,0.00313,0.01279,-0.00019,-0.01107,0.01509,0.00588,-0.0001,-0.00845,0.01018,0.00616,0.00044,-0.00948,0.01222,0.01511,-0.003,-0.00093,0.01673,0.00413,-0.0059,0.00395,-0.00106,0.00802,-0.01165,0.00484,0.00428,-0.01059,-0.01374,-0.01774,-0.00258,0.00491,0.00233,-0.00225,0.00527,-0.00516,0.005,0.00208,-0.00105,0.00096,-0.00225,0.00656,0.00719,0.00996,-0.00929,0.00156,-0.00961,-0.00496,0.00268,-0.01256,-0.0161,0.00869,-0.00889,0.0066,0.00301,0.00856,0.00036,-0.00823,0.00407,0.01209,0.00071,0.00533,-0.00059,0.01736,0.00294,-0.00947,-0.00322,-0.00089,-0.00428,0.01051,0.01318,0.00577,-0.00482,0.00402,-0.00193,-0.0166,0.00546,0.0191,-0.00629,-0.01199,-0.00146,0.00274,0.00642,0.00783,0.01245,-0.02019,0.01565,-0.0075,0.01349,-0.01025,0.00937,-0.00505,0.0103,0.00174,0.00375,-0.01022,-0.0004,-0.00394,0.01012,0.00368,0.00566,-0.00234,-0.00826,0.0038,0.01662,0.00705,0.00789,0.00311,0.0013,0.00379,-0.00494,0.00666,-0.01286,-0.00293,-0.00141,0.00099,-0.00738,-0.00494,0.00452,0.00327,0.0059,-0.00355,0.00429,0.00091,-0.01303,0.00268,0.00048,-0.00339,-0.00638,0.00399,0.00573,0.00619,-0.00832,-0.01297,-0.00996,0.00228,-0.00047,0.00839,-0.00569,0.00229,0.00225,-0.01096,0.00637,-0.00077,-0.01301,0.01203,-0.01586,0.00911,-0.00573,0.01649,0.00308,0.00749,-0.00458,-0.00244,-0.00056,-0.00879,0.0063,-0.0132,0.00945,-0.00989,-0.00571,0.00798,-0.00037,-0.00353,0.00177,0.00474,0.00806,0.00718,-0.00715,0.00468,0.00108,0.00639,0.00385,-0.00877,-0.01889,-0.01219,0.00233,-0.00244,-0.00255,-0.00304,0.00701,-0.00025,0.00109,-0.01745,-0.01377,-0.00911,-0.00245,-0.00638,0.0101,0.00023,0.0167,-0.00967,-0.00015,-0.01206,0.0043,-0.00425,-0.00468,0.00133,0.00112,0.01808],[-0.00374,0.01323,-0.01092,-0.01232,0.00724,0.00333,-0.01071,-0.00171,-0.00082,0.00702,-0.0014,-0.02077,-0.0091,-0.00789,0.00685,0.00366,0.0035,-0.01521,0.00629,0.01851,0.01372,-0.02107,0.0032,-0.00492,0.0005,-0.00659,-0.00733,0.01275,-0.01803,0.01003,0.00787,-0.00696,0.01007,0.00233,-0.0029,0.00372,0.00581,-0.00638,-0.00674,0.01229,-0.00639,0.00485,-0.01573,-0.00361,0.00123,-0.00341,-0.01542,0.01855,0.00056,-0.00909,-0.01027,0.01608,-0.00779,0.00648,-0.01908,-0.01349,0.01813,0.00166,0.00825,-0.01281,0.01321,0.00571,-0.01491,0.00332,-0.00602,0.01037,-0.0023,-0.00112,-0.00751,-0.01227,0.00331,0.00899,0.0059,-0.00451,0.00653,-0.00141,-0.00641,0.00712,0.00137,-0.00427,-0.0206,0.0119,-0.00376,-0.00793,0.00123,-0.00635,-0.0067,-0.00551,0.007,-0.00478,0.00087,0.01219,-0.01416,0.0087,0.00246,-0.00679,0.00328,0.003,-0.00786,0.01005,-0.01158,0.01485,-0.00959,0.00829,0.00393,-0.00058,-0.00961,-0.00744,-0.00434,-0.00056,-0.00406,0.01601,0.00054,-0.00392,-0.00663,0.00735,0.00305,-0.0016,-0.00891,0.00404,0.01338,-0.01507,-0.00817,-0.0026,0.00208,0.00455,-0.00727,0.00062,-0.01885,0.00712,0.01329,0.01074,-0.00379,-0.01023,0.01834,-0.00799,0.00177,-0.00989,0.00551,-0.00855,-0.01633,-0.00368,-0.01454,-0.00555,-0.00311,-0.00831,0.00888,0.01042,-0.00251,0.00676,-0.00185,0.00278,0.00511,0.00406,-0.00876,-0.00079,-0.00045,0.00055,0.00911,-0.00603,0.00403,-0.01106,-0.00182,-0.00772,0.00013,0.01231,-0.00484,-0.01107,-0.00697,0.00313,0.00498,0.00944,-0.01137,0.00049,-0.00141,0.0157,0.00563,-0.00496,0.00205,0.00979,0.00529,-1e-05,-0.00907,0.02,-0.00588,0.00417,-0.00028,-0.00102,0.01955,0.00884,0.01378,0.00607,0.00904,-0.00019,0.01524,0.00205,-0.01019,-0.00225,-0.01311,0.0002,-0.01484,-0.01183,0.0088,0.00503,-0.01208,0.00526,0.00219,-0.00853,-0.01729,0.01399,0.00465,0.00081,0.00904,0.01145,0.00484,0.0017,0.00794,0.00968,-0.00825,-0.01056,0.00332,-0.00466,-0.00129,0.00486,0.01272,-0.00238,-0.00415,0.02147,-0.00228,-0.00585,-0.01985,0.0037,-0.0214,-0.01585,0.0077,-0.01286,-0.01303,-0.01039,0.00461,0.01604,0.01383,0.01077,-0.01235,0.01062,0.01273,-0.00628,-0.02201,0.00338,0.00101,-0.00412,0.00057,-0.00024,-0.01364,0.00672,0.01308,-0.00259,0.0048,0.00296,-0.0037,0.00873,0.0081,-0.00129,0.00827,-0.01483,0.01087,0.01472,0.00564,0.01206,-0.00108,0.00706,0.00467,-0.00725,-0.00492,0.00106,0.00714,-0.00189,-0.00857,-0.00932,-0.00683,-0.00258,0.01191,-0.01121,0.01099,0.00658,0.00217,0.00192,-0.01543,0.00608,9e-05,-0.00247,0.00278,0.00776,0.00217,-0.01103,-0.00329,-0.00587,-0.01216,-0.00217,0.00043,0.02981,-0.00237,-0.00519,-0.01144,-0.00411,-0.00272,0.00811,0.00046,-0.00356,-0.01763,-0.00108,0.00711,-0.01466,0.00714,0.0052,-0.0052,0.0101,0.01851,0.00778,-0.00178,-0.00947,0.01129,0.01383,0.0024,0.00443,-0.00295,-0.00295,-0.00171,-0.00918,-0.00148,-0.00137,-0.00354,-0.00416,-0.01046,0.00199,-0.00154,-0.00728,-0.00483,-0.00734,0.00344,0.00312,0.00084,-0.02172,0.0071,0.00778,0.01073,-0.0029,-0.01478,0.00313,-0.0055,-0.01412,-0.00117,-0.00163,0.00549,-0.00738,-0.02424,0.00086,-0.01849,-0.0052,-0.00933,-0.00617,-0.01198,0.01528,-0.01291,-0.0215,-0.00386,0.02272,0.00084,0.00634,-0.0013,-0.03036,-0.00907,0.00164,-0.00207,-0.01245,0.01255,0.0087,0.00608,0.00702,0.00945,0.00118,0.00611,-0.0041,0.00263,-0.00333,0.00473,0.00612,0.00844,0.00174,-0.00122,-0.00785,0.01585,0.00201,-0.02117,0.01241,0.01053,0.01502,-0.00438,0.01041,-0.00893,-0.00205,-0.00907,0.01113,-0.00644,-0.00153,0.00694,0.00569,-0.00226,-0.00474,-0.00345,-0.0057,-0.00171,0.00281,0.00888,-0.00368,0.00322,-0.00124,0.0115,-0.01949,-0.00178,0.00407,0.00186,-0.00633,-0.01347,-0.01162,-0.00853,0.00258,0.01389,-0.00081,-0.0189,-0.01384,-0.01291,0.00852,-0.01467,0.00123,0.00898,0.01392,0.01619,0.01807,0.01791,-0.00059,-0.00091,0.00214,-0.00096,-0.00951,-0.00336,6e-05,0.0239,0.00824,0.00844,0.01168,-0.00743,0.00562,-0.01614,0.02013,0.00476,-0.01212,-0.00175,0.01998,0.00259,-0.00501,0.00806,0.01809,-0.00829,0.00081,-0.00523,-0.00717,0.02738,0.00411,0.00098,-0.01414,0.01207,0.00099,0.00045,-0.0208,0.01926,-0.00264,0.00292,0.00339,-0.00819,-0.00025,0.00385,0.0238,-0.00065,0.00195,-0.01059,-0.01175,0.01132,0.00266,-0.00356,0.00043,0.01077,0.00055,0.00038,-0.0052,0.00891,0.00706,0.00645,-0.00039,-0.00991,0.01604,-0.00906,-0.01794,0.00278,0.01149,-0.00815,0.01131,0.00984,0.0087,-0.00303,0.00354,0.01639,-0.00085,-0.00462,-0.0126,0.0032,-0.00694,0.01545,-0.0257,0.02272,0.00861,-0.0164,-0.02533,0.01148,0.00841,-0.00495,0.01032,0.00608,0.00813,0.00123,0.01371,-0.00809,0.00216,0.00173,-0.00358,-0.0024,-0.00703,-0.00225,-0.0069,0.0054,-0.01699,-0.00628,0.00607,0.00524,0.00136,0.01363,0.01169,-0.00876,0.01307,-0.00394,0.01007,0.00848,0.00659,0.01822,0.00899,-0.00022,-0.0132,-0.00321,0.00047,0.00355,-0.00542,-0.00267,-0.00033,0.0015,0.01983,0.00209,0.0192,0.00556,-0.00034,-0.02085,-0.00424,0.00933,0.00374,-0.0022,-0.01752,0.00703,0.00263,-0.00028,0.00084,-0.00579,0.00401,-0.00315,0.00532,0.00769,-0.00156,0.00556,0.00339,-0.00872,-0.01474,0.00513,0.00297,-0.0066,0.01276,-0.00372,-0.00168,0.01083,0.00145,0.01452,0.01667,0.00951,-0.0029,0.00429,0.01037,0.01693,0.01596,-0.00959,-0.00364,0.00255,-0.00135,-0.01221,0.01685,-0.0021,-0.00546,-0.00595,0.01825,0.01206,-0.00071,-0.00347,-0.01132,-0.00305,0.00375,0.00971,0.00949,-0.00537,-0.00161,0.0036,0.00084,-0.00866,-0.0152,0.00026,-0.0084,0.00153,0.00226,0.00148,-0.01802,-0.00033,-0.00109,0.01297,0.01232,0.00276,-0.02134,0.00054,0.00821,0.01497,0.00055,0.00099,-0.00363,0.00259,-0.00652,-0.00027,0.00029,-0.00879,-0.00314,9e-05,0.00894,-0.00387,0.01306,-0.00093,0.01808,-0.01642,0.01422,0.00043,-0.01164,0.00656,-0.01269,-0.00654,-0.006,-0.00277,-0.00856,0.00323,-0.02014,0.00206,-0.01067,0.00232,-0.01855,0.00631,0.00012,0.00097,0.0024,0.00199,-0.01122,0.0157,0.00334,0.00224,0.01093,0.0004,0.01638,0.00905,-0.00274,-0.01792,0.00797,-0.00878,-0.00403,-0.00041,-0.01114,-0.00132,-0.01449,-0.00472,0.01244,0.01025,-0.00126,-0.01068,0.00262,-0.0163,0.00484,0.01386,0.00076,0.01093,-0.01652,0.01213,-0.00326,-0.01605,-0.00153,0.0112,0.01211,-0.00127,0.00883,0.00015,0.0062,-0.01892,0.00407,0.01367,0.00997,0.00031,-0.00387,0.0124,0.00176,0.0115,-0.00626,0.00926,0.00171,0.01844,0.00114,0.01485,0.00716,-0.01776,0.00062,0.00189,-0.00641,-0.00558,0.00754,-0.00439,0.01602,0.01401,0.00876,0.01227,0.00682,-0.00548,0.00438,0.00286,0.01841,-0.00571,-0.00287,0.00367,-0.00685,0.0143,0.01499,-0.00334,0.00722,0.00737,-0.00167,-0.02067,-0.01316,0.01379,-0.01289,0.00652,-0.00682,0.00354,-0.0027,0.00066,0.02529,-0.01014,-0.01857,0.00454,-0.00051,0.00371,-0.00322,0.00709,-0.00949,0.00625,-0.00713,-0.00955,1e-05,-0.00172,-0.0012,-0.02264,-0.01567,0.00961,-0.00738,0.01126,-0.00018,0.00676,-0.0182,-0.00047,-0.00159,-0.02203,0.01201,0.0089,-0.00727,0.00038,-0.00364,0.00225,-0.01111,0.00927,0.02101,-0.01698,-0.00345,-0.00619,-0.00612,0.00259,-0.00331,0.00747,0.00292,-0.01208,0.01159,0.00072,0.01893,-0.00902,0.00935,-0.00678,-0.01301,-0.00808,0.00126,-0.00245,-0.00679,-0.00538,0.00411,-0.00045,-0.0015,-0.00748,0.01576,-0.00278,0.00581,0.00387,0.0055,0.01237,-0.01058,-0.014,-0.00564,0.00553,-0.01754,0.00521,-0.01564,0.00544,-0.00807,0.00252,0.00372,-0.00835,0.00355,0.01374,-0.00504,-0.00594,0.0141,-0.00271,-0.02174,0.00106,-0.00032,0.00227,-0.00988,-0.0168,0.00338,-0.00193,0.00411,-0.02019,0.01389,-0.00156,0.01656,-0.00965,-0.00077,0.01408,-0.0121,0.00166,-0.00113,-0.00281,0.01009,0.00557,0.00801,0.00098,-0.00944,-0.02248,0.00339,0.01007,-0.002,0.01013,0.00861,0.00566,0.00656,0.00393,-0.00855,0.01628,0.00781,-0.00643,0.00892,0.00227,-0.00456,-0.00243,0.00685,-0.00499,-0.0018,0.00429,-0.00868,0.00114,0.00434,0.01189,0.00382,-0.00021,-0.01345,-0.0002,-0.00061,0.00961,0.00291,0.00198,-0.0045,-0.01426,-0.00641,-0.01389,-0.00034,-0.00607,0.00659,0.01037,-0.00351,0.00262,-0.0063,-0.01965,0.00102,0.01587,0.02066,-0.01283,0.00042,0.00499,-0.0085,-0.00857,0.0034,0.00587,0.01302,0.00389,0.0061,-0.00533,0.00613,0.01373,0.01252,0.00012,-0.01007,-0.01022,-0.00577,0.01383,-0.00039,0.01357,-0.01414,-0.01098,0.00421,0.01316,0.00912,-0.007,0.0083,-0.01056,-0.00784,0.00678,-0.01147,0.00557,-0.00272,0.00728,-0.00067,0.01654,-0.01175,0.01118,0.00971,6e-05,0.00505,0.00609,0.0051,-0.00585,0.00479,0.00783,0.01606,0.00319,0.00269,-0.00865,-0.00642,-0.00539,-0.02778,0.02256,0.00726,-0.0211,-0.00059,0.00875,0.00654,-0.01442,0.00668,0.00893,-0.00638,0.00954,-0.01066,-0.02073,0.00813,0.00097,0.00368,-0.00792,0.0043,0.0012,0.00224,0.00851,0.01071,-0.00113,-0.01603,0.00344,-0.00211,-0.00937,-0.01325,-0.00988,-8e-05,-0.01595,-0.00545,0.00166,0.00606,0.00485,-0.01022,-0.0067,-0.01036,-0.0046,0.00507,-0.00145,-0.00219,0.00574,0.00455,-0.00373,-0.00326,0.00105,0.00135,0.00497,0.00831,-0.00169,-0.00158,0.00095,-0.00664,-0.0094,-0.00316,0.00508,0.01382,-0.00647,-0.00374,0.00127,-0.00615,0.00141,-0.0109,0.00694,-0.0029,0.02137,0.00033,0.0091,-0.00957,-0.00623,0.00247,-0.00556,-0.01167,0.00154,0.00425,0.0016,-0.01754,0.01916,-0.00244,-0.00016,-0.0081,0.00343,0.00613,-0.01461,-0.01289,-0.00981,-0.00753,0.01979,0.01565,-0.00894,0.0018,-0.01905,0.01249,0.00077,0.00466,-0.00494,0.00086,0.00387,0.00038,-0.00328,0.01366,-0.01053,0.00969,-0.00535,0.01173,-0.00289,-0.00689,-0.0021,0.00837,-0.00157,0.00418,0.00988,-0.00113,-0.0043,-0.01205,0.00153,-0.00512,0.01338,-0.02162,-0.00651,-0.00185,-0.00563,-0.02653,0.00183,-0.0104,-0.00657,0.00516,0.0044,0.00346,0.0068,0.01606,-0.00137,0.00052,0.01577,0.00623,0.00528,0.00186,-0.00218,-0.0113,0.00233,0.00654,0.00526,0.00823,-0.01033,-0.0198,-0.01205,-0.02136,-0.00452,0.00158,-0.00571,-0.00464,-0.00317,0.00429,0.00517,0.00522,-0.00354,-0.01469,-0.00789,-0.01512,-0.005,-0.01524,-0.02303,0.01599,-0.00045,-0.00987,-0.00359,-0.00404,-0.006,8e-05,-0.00107,0.02004,-0.01302,0.01061,-0.01025,-0.00364,-0.00144,-0.00489,-0.00978,0.00833,0.01383,-0.00326,0.00764,0.01102,-0.00314,-0.00342,0.00605,0.00757,-0.00342,-0.00379,-0.01795,0.00261,-0.00274,0.02363,0.0039,-0.00919,-0.0063,0.00786,-0.01641,0.00682,-0.01729,-0.00999,0.00903,0.0125,0.00079,-0.00978,0.01694,0.0015,-0.0056,-0.00617,0.00811,-0.019,0.00099,-0.01199,0.00924,0.00863,0.02029,-0.00726,-8e-05,-0.001,-0.01954,-0.01544,0.01887,0.00703,0.00309,-0.00911,0.013,-0.0048,0.01346,-0.00219,0.00023,-0.01725,0.00383,0.00954,0.00387,-0.0226,0.00508,0.00356,-0.00708,0.00595,-0.00184,0.00135,0.00224,-0.00259,-0.00292,0.0015,0.00094,-0.00486,0.00143,0.01646,-0.00118,0.00223,0.00571,-0.00466,-0.00868,-0.00094,0.02091,-0.00577,0.01674,0.00568,-0.01395,0.00024,-0.0031,0.00709,-0.00841,0.00068,-0.00591,0.00989,0.014,0.00584,-0.01648,-0.01546,-3e-05,-0.01078,0.02506,-0.0052,0.01691,9e-05,0.00583,0.00365,0.0024,0.00081,0.00093,0.01548,0.00618,-0.00378,-0.01412,0.01113,0.00143,-0.00613,-0.00896,-0.00033,0.01951,-0.01541,0.00126,0.00286,-0.00636,0.0013,0.00324,-0.00248,0.00187,-0.0086,0.00463,0.00971,0.00262,-0.00812,-0.00428,-0.00776,0.00025,-0.00374,-0.02215,0.00125,0.01451,0.00173,0.00799,0.00174,0.00999,0.01986,0.00639,0.01119,0.00511,0.00361,0.00311,-0.00146,0.00542,0.0042,0.00958,-0.00704,0.00429,0.00407,-0.00524,-0.00188,0.00279,0.00815,-0.00131,-0.0124,-0.00994,0.00769,-0.00162,-0.0006,-0.00391,-0.01747,0.00426,0.0029,0.00818,0.0155,0.00296,-0.00874,0.00513,0.00974,0.00166,-0.01014,-0.01877,-0.0101,-0.00634,0.01111,-0.02079,-0.02244,0.00531,-0.00048,-0.00281,-0.00129,0.01554,0.01281,-0.00517,-0.00779,0.00746,0.00677,-0.0062,-0.00762,-0.0148,0.00555,-0.01395,-0.00507,0.00822,0.0009,0.00215,-0.0072,0.00298,-0.00196,-5e-05,0.00081,0.00418,-0.00932,-0.01488,-0.00972,-0.00711,0.00186,0.01889,-0.00141,0.00636,-0.01268,-0.00073,-0.0003,0.0065,0.01432,-0.00038,0.00401,-0.00281,0.01208,-0.01356,-0.00129,-0.003,-0.00387,-0.00179,-0.01594,-0.00829,0.00965,0.00323,-0.00274,-0.00432,-0.00242,-0.00436,0.00404,-0.00249,0.01808,-0.00294,-0.00849,0.0045,0.01798,-0.00736,0.00263,-0.00101,-0.01291,0.00028,0.01181,0.00813,-0.00089,0.00055,0.00754,0.00139,-0.02233,0.02257,0.0253,0.00182,5e-05,0.00486,-0.00591,0.00486,0.00995,0.00166,-0.09773,0.01049,-0.0147,0.0145,-0.00895,0.02777,-0.01524,0.00929,-0.00123,-0.00543,0.0021,0.01504,-0.00157,-0.00536,0.01381,0.00047,-0.00038,-0.00527,0.00735,0.00821,0.00862,-0.00023,0.00072,0.00907,0.01477,-0.00171,0.01157,-0.02175,0.00651,-0.00597,-0.00333,-0.00117,-0.00067,-0.00155,-0.0122,0.0022,-0.00183,0.00671,-0.00627,0.01026,0.00089,-0.00025,-0.00078,0.0016,-0.00466,0.01227,0.00806,0.01582,-0.01497,-0.00055,-0.00218,0.00048,0.00969,0.00211,0.00323,0.01644,-0.00708,0.00987,-0.00204,-0.01156,0.00096,-0.00154,0.006,-0.0004,0.01677,-0.00521,0.00069,-0.0096,0.00386,-0.00325,0.00231,0.00686,-0.01483,0.0056,-0.00184,-0.01041,-0.01368,-0.00056,-0.00056,-0.00076,0.01285,-0.00101,0.01291,-0.00455,0.00351,0.00635,0.00972,-0.0023,-0.01181,-0.01193,-0.01242,0.00915,-0.00959,-0.01882,0.00031,-0.00325,-0.00565,0.00111,-0.0154,-0.01361,-0.00731,-0.00803,-0.00941,-0.01052,0.01307,0.01689,0.00035,-0.00749,0.00636,-0.00444,-0.01346,0.00158,0.00472,0.02268,0.02553],[0.00389,0.01068,0.00264,-0.00246,9e-05,0.00224,-0.00637,0.0008,-0.00823,0.00176,-0.00091,0.00766,-0.00457,-0.00439,-0.00177,0.01134,0.00557,-0.00306,0.00445,0.01973,3e-05,-0.01103,0.00202,-0.00843,-0.00483,0.00685,-0.00265,-0.00087,-0.01309,0.00743,0.01523,-0.0024,0.01108,0.00856,0.002,0.00654,0.00442,0.00372,-0.01642,0.00955,-0.0082,0.00523,-0.01961,-0.00774,0.01417,-0.00659,-0.00968,0.0158,0.0013,0.00026,-0.0175,0.00486,0.00449,0.00507,-0.00938,-0.00498,0.01709,0.00968,0.00479,-0.01084,-0.00059,-0.00622,-0.00444,0.00911,-0.00159,0.00546,-0.01074,-0.00531,-0.00348,-0.01474,0.00722,0.0143,0.00159,-0.00776,0.00279,0.00905,-0.01212,0.00505,0.00166,0.00275,-0.00955,-0.00499,-0.00252,-0.0052,-0.00837,-0.0026,-0.00417,-0.00407,0.0035,-0.00212,0.00091,0.00345,-0.01213,0.00786,-0.01342,-0.001,0.00218,-0.00783,-0.00824,0.00443,-0.01065,0.00495,0.01876,0.00106,0.0124,-0.01292,-0.00556,-0.00335,-0.00238,-0.003,0.01391,0.00358,0.0069,-0.00227,0.00571,0.00054,-0.00209,0.00537,-0.00257,-0.00673,0.00361,-0.00443,-0.01473,0.00345,0.00271,-0.00051,-0.00856,0.00295,0.00264,0.00427,0.01025,-0.00435,-0.0007,-0.01062,0.01246,-0.00217,0.00912,0.00323,-0.00237,-0.00477,-0.00448,0.00249,-0.00545,-0.00966,0.01436,-0.00303,-0.00298,0.01732,-0.00595,0.00106,-0.01138,-0.00163,0.01244,0.00656,0.00075,0.01237,-0.00282,0.00506,-0.00039,0.00084,0.00392,-0.00713,-0.00953,0.01195,-0.00245,0.00683,-0.00486,-0.01826,-0.01147,0.00388,-0.00322,0.00395,-0.00714,-0.00215,-0.00201,0.00959,-0.00246,-0.00359,-0.0017,0.01343,0.00361,-0.00432,-0.00495,0.01152,0.00171,0.00844,-0.00283,0.00988,0.00729,0.01013,0.00457,0.00064,0.00825,-0.00371,0.01686,-0.00479,-0.01436,0.00298,-0.00812,-0.00217,-0.00605,-0.01378,0.00211,-0.00176,-0.00441,0.01293,0.00056,-0.00295,-0.01829,0.00155,0.00685,-0.00211,0.0048,0.01362,0.00328,0.00159,0.00712,-0.00038,-0.00534,-0.01145,-0.00223,-0.0081,-0.00128,-0.00987,0.00673,0.00012,-0.00485,0.01518,-0.01466,-0.00515,-0.00478,0.00104,-0.00844,-0.00564,-0.00029,-0.00201,-0.01362,0.00725,0.00606,0.00189,-0.00035,0.0072,-0.00255,0.00551,0.00838,0.00654,-0.01425,0.00082,-0.01312,0.00115,-0.00671,0.01363,-0.00463,-0.01055,0.00962,0.01042,0.00737,0.00378,-0.00219,-0.00444,0.00532,-0.00838,0.0017,-0.01503,0.01056,-0.00023,-0.0005,0.0092,-0.00022,0.00584,0.00514,-0.00198,-0.00663,-0.00344,-0.00281,-0.01158,0.00974,-0.00024,-0.01284,0.00353,0.00273,-0.00858,-0.00479,0.00676,0.0034,0.00603,-0.00086,0.00858,0.0064,-0.01196,-0.01579,0.00906,0.00062,0.00335,-0.00495,0.00331,-0.00523,-0.00766,-0.00139,0.01946,-0.00246,-0.01014,-0.01005,0.00541,-0.00047,0.00222,0.00072,-0.0086,-0.00601,-0.00876,-0.00545,-0.01759,-0.00543,-0.00782,-0.00213,0.00135,0.01846,0.00731,0.00181,0.00467,0.0111,0.01019,0.00465,-0.00771,0.00302,0.00052,0.0085,-0.00458,0.00318,-0.00442,-0.00643,-0.00867,0.00647,-0.0016,-0.00716,-0.01416,0.00909,-0.01241,-0.0088,-0.00071,-0.01308,-0.01446,0.00471,0.0121,0.00552,9e-05,-0.00219,-0.00498,-0.00384,-0.0079,-0.00208,-0.00486,-0.00543,0.0135,-0.02752,0.00341,-0.03708,-0.00697,-0.01258,-0.01201,-0.01429,-0.00334,-0.01406,-0.01204,-0.00633,0.01424,0.00074,0.01735,0.01286,-0.01753,0.0026,0.00237,-0.00679,-0.01628,0.00149,0.01113,0.0031,-0.00218,0.01193,-0.00518,-0.00101,9e-05,0.00462,-0.00892,-0.0041,0.00375,-0.01249,0.00117,0.00117,-0.00658,0.00313,0.00735,-0.00704,0.00906,-0.00633,0.02008,-5e-05,0.02136,-0.00113,0.00281,-0.01332,0.01186,-0.00591,-0.01586,-0.00282,0.00698,0.00078,-0.00531,-0.008,-0.00789,0.00361,-0.01313,-0.00417,0.00115,0.00645,0.00114,0.00365,-0.01256,0.00276,0.00839,-0.00156,-0.02095,-0.01323,-0.00759,-0.02327,0.00218,0.01384,0.00734,-0.00502,0.00066,-0.00112,0.00562,-0.01311,-0.00825,-0.00063,0.00754,0.0142,0.00521,0.00421,-0.00552,-0.00395,-0.00809,-0.00418,-0.00597,-0.00073,0.00594,0.00688,0.00194,-0.00094,0.00405,0.00306,0.00085,-0.01639,0.0236,0.00189,-0.02055,-0.00524,0.01024,0.00993,-0.00714,0.01264,0.01363,-0.00652,-0.0044,0.01254,-0.00067,0.0117,0.00281,0.00592,-0.00609,0.01668,0.00028,-0.00645,-0.01264,0.00387,0.0048,0.00519,0.00096,0.00566,0.00292,0.00118,0.01406,0.01162,0.00813,-0.00824,-0.00649,0.00862,0.00255,-0.00181,0.00062,0.00011,0.01396,-0.00348,0.00189,0.00507,0.00649,0.00192,-0.00071,-0.00016,0.01041,-0.00429,-0.01153,-0.00968,0.01533,-0.00426,-9e-05,0.01046,0.00931,0.00102,0.00067,0.0187,0.00082,-0.00314,-0.00341,-0.0178,0.00157,0.0007,-0.01334,0.02128,-0.01827,0.02705,-0.00469,-0.00047,-0.00059,0.00672,0.00618,-0.00269,0.01341,-0.0007,0.01599,-0.00466,-0.00532,0.00083,-0.00459,-0.00012,-0.0084,0.00436,-0.00015,0.00172,0.00092,-0.01191,0.00643,-0.00872,-0.00314,-0.00275,-0.00511,-0.01088,0.01017,-0.0046,0.0034,0.01016,0.0034,0.01959,0.00652,-0.00439,-0.00936,-0.00904,-0.00108,-0.00742,0.00068,-0.01309,-0.00582,0.01124,0.00761,0.0004,0.00959,0.00559,0.00181,-0.01185,-0.00942,-0.00812,0.00065,-0.00829,-0.00637,0.01368,0.01563,-0.00118,0.00541,-0.01299,-0.00186,0.00712,0.00624,0.01208,-0.00986,0.00103,0.01408,-0.00011,-0.00762,-0.00304,-0.00746,-0.01578,0.01175,-0.00645,-0.01018,0.00027,-0.00103,0.00658,0.00111,0.00115,0.00978,-0.00526,0.00805,-0.00699,0.01383,-0.00924,-0.01313,0.00068,-0.00176,-0.01079,0.01569,0.00316,-0.00586,-0.00833,0.00999,0.00592,-0.0137,0.00456,-0.01226,-0.01229,-0.00515,-0.00876,0.00274,-0.00355,0.0062,-0.00028,0.00878,-0.01145,-0.01427,-0.00029,-0.00701,0.00572,0.01014,-0.00302,-0.00708,-0.00778,-0.0046,0.00433,-0.00075,0.00614,-0.00296,-0.00443,-0.00052,0.00924,-0.00643,0.00759,-0.00649,0.00558,-0.00084,-0.01104,-0.00703,-0.0029,-0.00172,-0.0079,0.0094,0.00557,0.01378,-0.00641,0.00254,-0.00908,0.01128,0.0068,0.01019,0.01421,-0.02022,-0.00478,0.00132,-0.01791,0.00112,-0.00409,-0.00986,-0.01381,0.00401,0.00584,-0.01389,0.00937,0.0106,-0.01534,0.00309,-0.01253,-0.00278,0.01089,-0.00746,0.00161,0.00607,0.00361,0.00617,0.00641,-0.00227,-0.01708,0.00359,-0.00175,-0.00223,0.00298,-0.00614,-0.00714,0.00262,-0.00964,-0.00142,0.00125,0.00415,-0.01156,-0.00657,-0.01697,0.00333,-0.00235,0.00143,0.0077,-0.00662,0.00573,-0.00012,-0.00149,-0.00591,0.0024,0.01114,-0.00899,-0.00272,0.00134,0.00527,-0.00454,0.00016,0.01145,0.00901,-0.0061,0.00467,-0.00078,0.01143,0.0055,-0.00139,0.01698,0.0121,0.00301,-0.00746,0.01041,-0.00883,-0.0212,-0.00142,0.00349,0.00374,0.0013,-0.0035,-0.00527,0.0131,0.01331,-0.0077,0.0063,-0.00122,0.00143,0.00497,0.00159,0.01174,-0.00673,-0.00575,-0.00076,-0.00369,0.00755,0.00406,-0.00491,0.0055,-0.00297,0.00257,-0.00967,-0.00672,0.01315,-0.00141,0.00056,-0.00988,0.01396,-0.00539,0.00019,0.01278,-0.00383,-0.01073,0.0044,-0.0062,-0.0012,-0.00716,0.00571,-0.0013,-0.00342,0.00882,-0.00251,-0.00269,-0.00389,0.0003,-0.00664,-0.01113,0.00634,0.007,0.00213,0.00018,0.00144,-0.01441,-0.00903,-0.01658,-0.0061,-0.00058,0.0067,-0.00651,-0.0002,-0.00923,0.00454,-0.01456,0.01098,0.00846,-0.01101,0.00733,-0.00968,-0.0122,0.00715,0.00831,0.00347,-0.00235,-0.00494,0.01088,-0.00583,0.00858,-0.00318,0.00653,-0.00016,-0.00326,-0.01208,0.00458,0.00533,0.00084,0.00347,-0.00198,0.00317,0.02013,-0.00788,0.00339,-0.00686,0.00242,0.00837,0.00285,-0.00304,-0.00851,-0.0049,-0.00191,-0.00152,-0.00948,0.00775,-0.0065,0.00506,-0.00884,-0.00429,-0.00776,-0.00919,0.00448,-0.0008,0.00081,-0.01484,0.0071,0.00034,-0.01065,-0.01228,0.00182,0.00457,-0.01711,0.00307,-0.00048,-0.00523,-0.00526,-0.01053,0.01252,-0.00898,0.02613,-0.00481,-0.00758,0.00478,-0.00484,0.00487,0.00917,-0.00719,-0.00256,0.00402,0.00855,0.00125,-0.01158,-0.00407,0.00407,0.01145,0.00914,0.00328,0.01068,0.00349,0.001,0.00589,-0.00077,0.01212,0.00375,-0.00312,0.00412,-0.00355,-0.00787,-0.00583,0.00019,-0.00368,0.00647,-0.00118,-0.0136,-0.00348,0.01128,0.00812,0.00408,0.01898,-0.01101,-0.00115,-0.00257,0.00272,0.00243,0.00449,-0.00605,-0.00034,0.00195,-0.00284,0.00382,0.00243,-0.00125,0.01525,0.00107,0.00835,-0.00869,-0.01779,2e-05,0.01166,0.00084,-0.00356,0.00532,0.01512,-0.01123,-0.01236,0.00937,0.01071,-0.00536,-0.00263,0.0056,-0.02112,-0.00736,0.0179,0.00351,-0.0008,-0.00557,0.00173,-0.01174,0.00422,0.00021,0.01934,-0.0127,-0.01273,0.00204,-0.00572,0.00418,-0.01704,0.00448,-0.00609,-0.01242,-0.00381,-0.01382,0.00879,0.00481,0.0007,0.00114,0.01355,-0.01024,0.00598,0.00738,0.01285,0.00433,0.00388,0.01842,0.00662,-0.01058,0.01103,0.02171,0.00692,-0.00159,-0.01563,-0.01907,-0.00304,-0.02147,0.0105,0.00113,-0.00769,0.00793,0.01505,-0.0019,-0.00362,0.00505,-0.00108,0.00213,0.00569,-0.00085,-0.01231,0.01292,-0.0027,-0.00592,-0.00431,0.0002,0.00632,0.01845,0.00422,0.00962,-0.0088,-0.01914,-0.00545,0.00063,-0.01043,-0.01269,-0.02207,-0.00963,0.00432,0.00546,0.00535,-0.00341,0.01146,0.00309,0.00376,-0.00409,-0.00227,0.0136,-0.00143,-0.0001,0.00534,-0.00651,-0.00653,-0.0115,-0.00473,-0.00363,-0.01108,-0.00077,-0.00506,0.00051,0.00629,-0.00595,-0.00037,0.01414,0.0007,0.00699,-0.00323,-0.00022,0.00236,0.00373,0.00365,0.00205,0.00763,-0.01038,0.01097,0.00046,0.00699,-0.01009,-0.00428,-0.00116,-0.00469,-0.00536,-0.00094,0.00147,-0.00126,-0.00556,0.00267,-0.00641,-0.00034,-0.01369,-0.00515,0.00726,-0.00107,-0.00715,-0.00832,-0.01113,0.02117,0.00565,-0.00295,0.00331,-0.00339,0.01588,0.0066,0.00643,0.00396,-0.0093,0.00408,-0.01338,-0.00963,0.00221,-0.01572,-0.00025,-0.00549,0.00397,0.00263,-0.00406,0.00146,0.00356,-0.01116,0.0066,0.00485,0.00831,0.01327,-0.00808,0.00855,0.00421,0.00423,-0.01922,-0.00898,0.00225,-0.00353,-0.03307,0.01423,0.00186,0.00335,-0.00276,0.00038,-0.00951,0.00354,0.01942,0.00866,-0.00458,0.00619,-0.00628,0.00323,0.01655,-0.00406,-0.00195,-0.00049,-0.00081,-0.00446,0.0112,-0.00904,0.00058,-0.00548,-0.01243,0.00241,-0.00027,-0.00538,-0.00138,0.00373,0.00356,-0.00498,-0.00219,0.00221,-0.0102,-0.01023,0.0022,-0.00208,-0.01196,-0.01539,0.01697,0.0021,0.00136,-0.00271,0.0005,-0.00174,0.00199,0.00067,0.01455,-0.00879,0.00204,-0.009,-0.00048,0.00084,-0.00411,-0.00836,-0.00143,0.01088,-0.00117,0.00725,0.00767,-0.00829,-0.01471,-0.00572,0.00311,-0.00387,0.00435,-0.01427,0.00783,0.00128,0.00589,-0.00576,-0.00868,-0.00913,0.00732,-0.01431,0.00183,-0.00232,-0.0079,0.01242,0.01328,0.00018,-0.0067,0.00239,-0.00584,0.00208,-0.00184,0.00335,-0.02193,0.01001,-0.00423,0.00264,0.00012,-0.00619,0.00329,0.0029,-0.00743,-0.00793,-0.0087,0.01029,0.00508,-0.00215,-0.00199,0.00984,-0.01417,0.01107,0.00136,-0.00065,-0.01103,0.0066,0.0184,-0.00183,-0.01542,0.01174,0.0146,0.00423,0.01163,-0.00606,0.00767,0.01017,0.00452,0.00325,0.0095,0.00393,-0.01073,0.00769,-0.00347,-0.0107,0.00044,0.02023,0.00457,0.00323,0.00106,0.01696,-0.00986,0.00701,0.01224,-0.014,-0.00414,-0.00224,-0.01103,-0.00549,-0.00226,0.00748,0.00327,0.02035,0.00305,-0.00703,-0.00918,-0.00291,-0.00131,0.01593,0.00419,0.01502,0.00273,0.00633,-0.00241,0.00143,-0.00399,-0.00303,0.00983,-0.00013,-6e-05,-0.0089,0.01185,-0.00966,-0.01318,-0.00454,-0.00218,0.00388,-0.01425,0.01265,0.02436,-0.00999,0.00401,0.0006,-0.01161,-0.00391,-0.00355,-0.00824,-0.00513,0.01112,0.00384,0.01004,-0.01375,0.00621,0.00536,-0.00855,-0.01066,0.00593,0.00589,0.00582,0.00107,0.00303,0.01065,0.00112,-0.00831,0.00148,-0.00044,0.00535,0.00606,-0.0003,-0.00331,-0.00025,-0.01163,0.01208,-0.00103,-0.0084,-0.00556,0.00411,0.00093,-0.00197,0.00432,0.00124,0.0031,0.0054,-0.00525,-0.00536,-0.01479,0.0017,0.00401,0.01088,0.01125,0.00209,0.00588,0.0042,0.01643,-0.00708,0.00098,-0.01484,-0.0058,-0.01013,0.0021,-0.01126,-0.01327,0.00879,0.00449,0.01244,-0.00412,0.00697,-0.00202,0.0048,-0.01381,0.00805,-0.00261,0.00143,0.00013,0.00053,0.01438,-0.01109,-0.0062,0.0043,-0.01164,0.00038,-0.00484,-0.00083,0.00728,-0.0146,-0.00382,0.01488,-0.01293,-0.01205,-0.00793,0.00073,0.00144,0.00995,-0.00183,0.01613,-0.02003,-0.00501,0.00581,-0.00328,0.00423,-0.00783,0.00349,0.00745,0.00539,-0.0049,-0.00356,-0.00259,0.00174,-0.00721,0.00071,-0.0104,-0.01152,-0.00558,0.00795,0.0098,0.01542,-0.00674,-0.00427,-0.00715,0.01642,-0.00709,-0.00585,0.00885,0.00887,-0.01319,-0.00247,-0.00588,-0.00623,-0.00743,0.00212,0.01891,0.00378,0.00921,0.00684,-0.00674,-0.02109,0.00936,0.01502,0.002,0.00104,0.00192,0.00067,0.00949,0.00109,0.01233,-0.0521,0.00474,0.00279,0.01266,0.00872,0.01098,-0.01044,-0.00386,-0.00039,0.00417,-0.01148,0.00607,0.01565,0.00861,-0.00228,-0.00495,0.00211,-0.00353,0.0004,0.00358,0.00509,0.00171,0.01088,0.00836,0.01086,0.00199,0.00605,-0.01513,0.00638,-0.00822,-0.00958,-0.01623,0.00262,-0.01272,-0.00712,-0.0003,-0.00323,-0.00222,-0.00446,0.01023,-0.00278,-0.0005,-0.00742,0.00258,-0.00109,0.00457,-0.00334,0.00434,-0.01069,-0.00453,0.01517,0.00297,0.00844,-0.00306,0.00115,0.01212,-0.00435,0.01289,-0.01151,-0.00113,0.0024,-0.00154,0.00237,0.01241,0.01426,0.00067,-0.0037,-0.00926,0.00586,0.00186,0.00137,0.01336,-0.00995,0.00537,-0.00534,-0.01209,0.00409,-0.00189,0.00281,-0.01663,0.01468,0.01249,0.00674,-0.00539,0.00187,-0.00155,0.00616,0.01403,-0.02014,-0.01142,-0.00556,-0.00091,-0.00999,-0.00491,0.00662,0.0027,-0.01786,-0.00045,-0.012,-0.01639,0.00037,-0.00497,-0.0029,-0.00874,0.0087,0.01449,-0.00962,-0.01838,0.00696,-0.00433,-0.00183,0.01235,-0.00277,0.01644,0.00688],[0.01142,-0.00567,-0.01118,0.0139,-0.01492,-0.00394,0.00575,-0.00918,0.01503,0.00787,-0.01763,-0.03316,0.01604,0.00441,-0.01062,0.00261,0.00254,-0.00044,0.00697,0.00019,0.00849,0.00922,0.01063,0.01043,0.00607,0.02028,0.00182,0.00548,-0.00669,-0.013,-0.0084,0.0038,-0.01884,-0.01426,-0.00553,-0.0174,-0.01912,0.01369,0.0127,0.00452,0.00358,-0.00432,0.04535,0.00827,-0.01279,-0.00239,0.00874,-0.00953,-0.00275,-0.00163,0.01505,0.00263,0.01235,-0.01233,0.01931,0.01494,-0.00945,-0.00418,0.00071,0.00856,0.00365,-0.00247,0.02008,0.01046,0.01216,-0.00868,-0.00766,-0.00801,-0.00959,0.01782,-0.0099,-0.00418,-0.00187,0.01824,-0.01686,-0.00408,0.0085,0.00355,-0.01644,0.0107,0.00263,0.00536,0.01743,0.00304,0.01313,0.0004,0.00733,-0.00332,0.00957,0.00014,0.0017,-0.00527,0.00077,0.00355,0.01165,0.01681,-0.00034,0.012,0.02472,-0.01488,0.01531,-0.00218,-0.01011,-0.02849,-0.00827,-0.00299,0.00388,-0.00709,0.00383,-0.01564,-0.01247,-0.01484,-0.00287,0.00511,0.00454,-0.01546,0.00277,-0.00396,0.00042,-0.00701,-0.00773,0.00786,0.02105,0.00701,-0.0099,0.01682,0.0175,-0.00126,0.00485,-0.00016,-0.01429,-0.01019,0.00786,0.02792,-0.02567,-0.00231,-0.00154,-0.00955,-0.00663,0.01228,-0.00981,0.01863,0.01852,0.00792,0.00322,-0.00029,-0.01968,0.01081,0.02455,-0.0141,0.01151,-0.00526,-0.00907,-0.00639,0.00717,-0.01262,0.01731,-0.00336,-0.00833,-0.00621,-0.01209,0.0112,0.01085,-0.00937,-0.0039,-0.02366,0.01448,0.01598,0.01143,-0.00497,0.01157,-0.00142,0.01163,0.01015,-0.00511,-0.01567,0.00526,-0.00868,0.00219,0.00629,0.00655,0.00843,0.00743,-0.01475,-0.0051,-0.01291,-0.00313,-0.01613,-0.01047,0.00331,-0.01575,-0.00763,-0.02238,-0.01195,-0.02429,-0.00037,0.01556,-0.00718,0.00851,0.00473,0.0165,0.00099,-0.00357,0.01005,0.0167,0.00184,-0.00159,0.00361,0.0192,-0.0001,-0.0084,-0.00805,0.00387,-0.01449,0.00406,0.00013,-0.02149,-0.00093,0.02155,0.01277,-0.00131,0.00639,0.00586,0.02725,0.00077,-0.01685,0.00817,-0.02502,0.02185,0.00027,0.03642,-0.01009,0.01492,0.0088,-0.01743,0.01509,0.0252,0.00775,0.00454,-0.0179,-0.00101,-0.01206,0.00396,-0.03849,-0.01358,0.01586,0.00714,-0.01316,0.00275,-0.00018,0.00713,-0.01059,0.01447,0.00958,-0.01764,-0.00412,-0.00276,-0.01995,0.01327,0.01326,-0.02468,0.01326,0.01407,0.0018,-0.0139,-0.00661,0.00352,-0.01542,-9e-05,-0.01543,0.00292,-0.00569,0.01431,0.0206,-0.01008,0.01286,-0.00379,0.01055,0.00022,0.00193,-0.00997,0.0123,-0.00284,-0.00111,-0.01441,0.009,0.01933,-0.00389,-0.00402,0.00962,0.01624,-0.02165,-0.01027,-0.00052,0.0154,0.00901,0.01856,0.0229,-0.0021,-0.01944,-0.00204,-0.00642,0.01194,0.01017,0.01229,0.00999,-0.01736,0.01785,0.01139,-0.00134,0.00676,0.01457,-0.01217,-0.0036,0.01378,-0.00431,0.00678,-0.00868,-0.00917,-0.00266,-0.02654,-0.02201,-0.00445,-0.00796,0.00505,-0.00232,-0.00922,-0.00305,-0.0017,0.00682,-0.00622,0.00511,-0.00559,-0.00372,0.00359,0.01628,0.00016,0.00013,-0.00057,0.02092,0.0158,0.00772,-0.0085,-0.01278,-0.00861,0.00021,-0.01403,-0.00266,0.01727,0.01332,0.0089,0.00765,0.00084,-0.00173,0.02682,-0.01403,0.03121,0.00461,0.02595,0.00832,0.01593,-0.00544,0.01961,0.00234,0.00577,-0.03377,0.0079,-0.02106,-0.00453,0.03469,0.0104,-0.01607,0.01735,0.02176,-0.01464,-0.00488,-0.00932,-0.00173,-0.0058,0.00746,0.00049,-0.00726,0.00745,0.00699,0.0067,0.00884,-0.00848,-0.00195,-0.00109,0.01123,-0.00876,0.00125,0.00396,-0.02405,0.00946,-0.02589,0.01232,-0.01062,-0.01073,0.00762,0.01537,-0.03883,-0.00958,0.01405,0.00087,0.0065,0.00897,-0.00152,0.03711,-0.0014,0.0177,0.01641,-0.00724,-0.00851,-0.00802,0.01011,-0.00716,0.01131,-0.01404,-0.00365,0.00649,0.01081,0.00731,-0.00267,0.00798,-0.01433,-0.01491,-0.00402,0.01433,0.00624,0.01953,-0.01167,0.02726,-0.00465,0.00806,-0.00084,-0.01368,-0.01016,-0.00041,0.00678,0.02344,0.01455,0.02624,0.01144,0.0137,-0.00418,-0.01398,0.00898,-0.01945,-0.01794,-0.00028,-0.00182,0.03716,-0.03561,-0.00899,0.02404,0.00771,-0.01257,-0.00122,0.01074,0.00433,-0.0346,0.00146,0.01818,0.00339,0.00062,-0.00534,-0.00198,-0.00612,0.02023,-0.015,0.00487,0.01252,0.02638,-0.01506,-0.00906,-0.00419,-0.00267,-0.0012,0.00037,-0.01186,-0.0176,0.00766,-0.00723,-0.00092,0.0036,0.00501,-0.00965,-0.00864,-0.01335,-0.00095,-0.02376,0.00555,0.01127,0.00665,-0.00516,-0.00243,-0.00167,-0.0068,-0.02247,0.01487,0.02337,-0.00057,-0.00149,0.02785,-0.00486,-0.02092,0.01318,-0.00242,0.00748,-0.00601,0.00729,0.00313,0.01411,-0.00601,0.0138,-0.00684,0.03248,-0.00217,0.02111,-0.03283,0.01573,-0.00887,-0.01199,-0.00638,-0.01248,0.00426,-0.02676,-0.0033,-0.00474,0.01934,0.00688,-0.00147,0.00025,0.00463,0.0126,-0.00486,0.00203,0.00576,0.003,0.02309,-0.00326,0.00315,0.00773,-0.01598,-0.00399,0.00223,-0.0201,0.00596,0.00173,-0.00323,-0.01076,-0.03601,0.01784,0.00356,0.02629,-0.00766,0.01471,0.00717,-0.00421,0.01003,0.01829,-0.00331,-0.01866,-0.00252,-0.01783,0.0007,0.00195,0.02515,0.0132,0.0072,0.00751,0.00189,0.0005,0.00286,-0.01567,-0.00113,-0.02255,0.00894,-0.00021,-0.00872,-0.00859,-0.00654,0.01502,0.00357,-0.02487,-0.00838,0.01175,0.00637,0.00011,0.01169,-0.0102,0.02139,-0.00374,-0.00182,0.01113,-0.00068,-0.00179,-0.01804,0.0119,0.0085,-0.00849,0.00178,-0.00535,0.01594,0.01894,-0.00081,0.00459,0.01412,-0.03463,-0.01977,-0.00031,0.00198,-0.01672,-0.00812,0.00029,-0.00107,0.00556,0.00701,-0.00581,0.00646,-0.00278,-0.00368,-0.00613,-0.00589,-0.01724,0.00207,0.01417,0.01995,0.0078,0.00284,-0.00984,-0.00363,0.01959,0.0283,-0.00596,-0.00203,0.00664,-0.01124,0.02008,-0.00423,-0.00372,-0.00387,-0.00532,-0.00726,0.00014,-0.01619,-0.00438,0.00814,0.0026,0.00164,0.00063,0.00598,0.00816,-0.00448,-0.02847,0.01282,-0.02276,0.02312,-0.01339,-0.02464,-0.00377,-0.012,0.01038,-0.00933,-0.00806,0.00547,-0.00541,0.01157,-0.00446,-0.00084,-0.00531,-0.01313,0.02844,-0.0135,-0.00987,-0.00312,-2e-05,0.008,0.01297,-0.02311,0.00413,-0.01256,-0.02532,0.01056,-0.00482,-0.0065,0.01104,0.01302,-0.01059,0.00671,0.0128,0.00828,0.02025,0.00999,0.02137,0.00681,0.0037,-0.00101,-0.00407,0.014,-0.01795,0.02027,-0.0115,-0.00397,-0.00695,0.00269,0.0025,-0.01803,-0.00704,-0.00851,0.0087,-0.01316,-0.00921,0.00484,-0.02122,-0.00235,0.00177,0.00633,-0.00342,-0.02891,-0.00412,0.02727,0.00605,0.01118,0.00476,-0.00585,-0.00677,-0.01494,-0.0249,-0.01957,-0.01,-0.01246,0.01387,0.03323,-0.00437,0.00365,0.0054,0.00179,-0.00843,0.0114,-0.00789,-0.01624,-0.00694,0.00445,-0.0116,0.00832,-0.00929,-0.00653,-0.01417,0.00161,0.00256,0.01029,-0.00412,-0.02343,-0.00716,0.01093,0.00577,0.01467,0.00678,0.02535,0.01421,-0.00263,0.00597,0.01006,0.0208,-0.00119,0.00158,-0.00737,-0.00736,0.00602,0.00585,-0.01322,0.00137,0.0012,0.01311,0.00182,-0.00519,0.01235,-0.00079,-0.00936,-0.00724,0.00633,0.00538,0.02849,0.0088,-0.01146,-0.00867,-0.01694,-0.01007,0.0166,0.04073,-0.00318,0.01026,0.02038,-0.00673,0.00257,0.00153,0.00582,-0.01244,-0.01136,0.01594,0.00394,-0.01767,0.02855,0.00461,0.008,0.02137,-0.00786,-0.00874,-0.00329,0.00815,0.00047,-0.00754,0.00111,-0.01039,0.01331,0.00654,0.01139,0.00332,0.00812,-0.01359,-0.00302,0.01118,0.0036,0.0046,-0.00775,-0.02462,-0.00684,0.00217,0.00235,-0.00606,-0.02224,0.00119,0.00758,0.00695,0.02122,0.00705,-0.00669,0.00651,-0.00372,0.00501,0.02367,-0.0068,0.03091,0.00609,0.01688,-0.01078,0.00832,-0.00471,0.01288,-0.0272,-0.00781,0.01465,-0.00143,-0.01096,0.00033,0.02477,-0.00783,-0.00325,-0.00767,-0.00573,0.02451,-0.00655,0.00236,-0.00676,0.011,0.00452,-0.00448,0.00043,-0.01256,-0.00169,0.01738,-0.00285,-0.0239,0.00986,-0.00186,-0.00388,0.00248,1e-05,0.01001,0.0001,-0.01244,-0.01997,-0.00894,0.00196,-0.01364,0.00834,-0.01624,-0.00101,0.00302,-0.01136,-0.00186,0.0135,0.01015,-0.01168,0.01259,0.01185,-0.0054,0.00614,-0.00054,0.01981,0.0226,-0.01232,-0.01046,-0.00194,0.00321,0.0147,-0.0068,0.00073,-0.01364,0.02013,0.00379,0.00513,0.00949,-0.00869,0.0112,0.00344,-0.00735,-0.00906,-0.01237,0.0144,0.00784,-0.00467,-0.00065,-0.02057,-0.00031,0.00904,-0.01736,-0.00022,0.00817,-0.00865,-0.00685,-0.00482,-0.0095,0.00379,0.02894,-0.00259,0.0003,0.00048,-0.01509,0.01595,0.00057,-0.0039,-0.01539,-0.00073,-0.01876,0.01019,-0.00088,-0.01015,-0.00706,0.00349,0.00779,-0.00587,-0.00599,0.02021,0.01317,-0.01047,-0.0069,-0.00297,-0.00836,-0.0,-0.00829,0.0137,-0.01569,-0.02871,-0.00658,-0.00274,-0.00281,-0.01622,-0.00272,-0.00638,-0.00898,-0.03374,-0.01153,-0.00074,0.01164,0.01806,0.00139,0.01599,0.00383,-0.01059,0.01671,0.00419,-0.025,-0.00792,-4e-05,-0.00467,0.01301,-0.00887,-0.01397,0.00219,0.00593,-0.01599,0.02705,0.02387,0.01664,0.00863,-0.02008,-0.00122,-0.00783,-0.00878,-0.0043,0.01178,0.00674,0.00071,0.00898,0.00133,0.01088,0.00178,-0.01166,0.00107,-0.00161,0.00037,-0.01232,0.01401,-0.01351,0.01519,-0.00848,-0.01776,-0.00448,0.00173,-0.01223,0.0206,0.01137,0.01379,0.01461,-0.00659,0.00526,-0.00942,-0.00227,-0.00669,-0.00736,0.01388,0.01075,0.01274,-0.01378,-0.00246,0.00785,0.01201,-0.00612,0.00321,-0.0085,0.00599,-0.01464,0.0156,-0.02075,-0.01726,-0.01182,0.01235,0.01146,-0.00474,0.01039,0.00786,0.00272,0.00462,-0.01068,0.02982,-0.01045,0.01022,-0.0125,0.00817,-0.00522,-0.00903,0.00635,-0.00013,0.01984,0.00452,-0.0117,-0.01313,0.00032,-0.00393,0.01644,0.02053,-0.03108,-0.00706,0.00119,-0.00282,0.00465,0.00509,0.01811,0.00143,0.03222,-0.00023,0.00786,-0.00256,-0.02331,0.00927,0.00129,-0.01293,0.03142,-0.00634,-0.00035,-0.0057,-0.00867,0.02187,0.00432,-0.00691,-0.0035,0.01324,0.00888,0.00753,0.01156,0.02766,-0.006,0.00138,-0.00022,-0.00764,0.00729,0.00466,-0.01856,-0.02211,0.01023,-0.00171,-0.02023,0.00322,-0.00729,-0.01129,-0.00458,0.00824,0.00197,-0.00079,-0.00994,-0.02195,0.00862,-0.01116,0.01552,0.02297,-0.00448,-0.0133,0.01648,0.0229,0.00169,-0.01954,-0.00553,0.00028,0.00715,-0.00221,0.01374,0.01402,0.02118,0.00791,0.00974,-0.03515,-0.01372,-0.00077,0.01122,-0.00318,0.01433,0.01214,0.00852,-0.02774,0.00813,-0.00652,0.01877,-0.00316,0.00421,-0.00799,0.00881,-0.01269,0.00247,0.01628,-0.01149,-0.00882,0.00372,0.0266,0.01379,-0.00552,0.00234,-0.00194,0.01385,0.00101,0.01725,-0.00682,-0.01801,0.02641,0.00216,0.00572,0.02021,-0.01221,0.02054,0.00979,-0.02893,-0.01467,-0.01552,0.00559,-0.00165,-0.00317,0.00848,0.01182,0.00367,0.02427,-0.01614,0.02542,-0.00756,-0.00912,0.00512,-0.00547,0.0124,0.00739,0.0019,0.02309,-0.02127,-0.00957,-0.01516,0.01126,-0.03672,0.01349,-0.00375,-0.01123,-0.00098,0.01595,-0.02501,-0.03045,0.01474,0.03204,-0.00698,-0.00465,-0.01069,-0.01852,-0.00456,-0.00699,-0.00681,0.01409,0.00874,-0.01357,-0.02279,0.01793,-0.0202,0.00258,-0.00216,-0.00529,-0.00784,-0.00943,0.01657,0.00163,-0.02482,0.01475,-0.00843,-0.01721,0.03176,0.00486,0.01153,0.00641,-0.00388,-0.01061,-0.01757,-0.00266,-0.00851,-0.00583,0.01098,0.00686,0.00473,0.03038,-0.02746,-0.00077,-0.02079,-0.01019,-0.00783,-0.01085,-0.0003,0.00174,0.00063,-0.01525,-0.00535,-0.00152,0.01075,-0.02139,0.00731,0.02076,0.01239,0.00549,-0.00202,0.01868,-0.0088,-0.02307,0.00771,0.00629,-0.02368,0.00514,0.01093,0.0049,0.01085,-0.00514,-0.00274,0.00562,-0.00455,0.00388,-0.00142,-0.01104,0.0083,-0.00466,-0.02129,0.00252,0.00706,0.01795,-0.0075,-0.02367,-0.00559,0.0003,-0.00417,-0.00159,-0.00956,0.00613,0.00043,-0.00219,-0.00096,0.02117,-0.00327,0.00568,0.00174,0.01095,0.00205,0.01613,0.01313,0.00748,0.00649,-0.00146,0.00209,-0.00274,0.01395,0.0265,0.00818,0.00752,-0.01632,-0.01575,0.00219,-0.01182,0.00919,0.00127,0.0032,0.00941,0.02102,0.01235,0.0104,-0.03124,0.0189,0.02634,-0.00205,-0.01106,-0.00078,-0.00012,-0.01167,0.00564,0.01358,0.01902,-0.01092,0.00026,0.01449,-0.00689,0.01234,-0.03023,0.00741,0.00587,-0.02329,0.02116,-0.00564,0.01071,0.00685,-0.00639,0.01068,-0.00184,-0.01207,-0.00069,0.02154,0.01565,0.00672,0.01385,-0.01396,0.00168,-0.00345,0.02292,-0.01848,0.00475,0.0058,-0.00435,0.00095,-0.00392,-0.0029,-0.02501,0.00902,0.00063,-0.00767,-0.00478,0.0086,0.00593,0.01274,0.0062,0.0008,0.00858,-0.00487,-0.00422,0.01326,-0.01169,-0.00868,-0.00966,-0.01007,0.01402,-0.00626,-0.02514,-0.00904,0.00734,0.00282,0.00832,-0.00245,-0.00762,-0.01194,0.00426,0.00206,0.006,0.00596,0.02039,-0.00598,-0.02768,0.00439,0.01707,-0.00126,0.01299,-0.0136,-0.01849,-0.01366,0.05709,-0.00563,0.01081,-0.01434,-0.01178,-0.01012,0.01911,-0.00021,0.00085,-0.00921,-0.00525,0.00481,-0.01323,-0.00913,-0.01107,-0.00204,-0.00292,0.00964,0.00089,-0.0007,-0.00513,-0.00148,-0.0098,-0.01341,-0.01156,0.00209,-0.01133,0.00487,-0.00318,0.03033,0.00694,0.00069,-0.0058,0.00254,0.00671,-0.01575,-0.01424,0.00524,6e-05,0.01266,-0.00235,0.00368,0.00534,0.01258,0.00013,-0.01304,-0.01353,-0.00527,0.02184,0.01321,-0.00667,0.0002,-0.01634,0.00647,-0.00804,-0.00452,0.00213,-0.02547,-0.00438,-0.00144,0.01018,0.01632,-0.01047,-0.01418,-0.00367,0.00987,0.00246,0.00825,-0.01031,-0.00721,-0.00725,-0.00852,0.0148,-0.01154,0.006,0.00315,-0.00839,0.003,0.01119,0.00091,-0.02108,-0.00138,-0.02294,0.00372,0.00141,0.01088,-0.00639,-0.00533,0.01201,0.02113,0.02557,-0.01785,0.01201,0.02099,0.00858,0.01402,0.01775,-0.00235,0.02577,0.01197,0.02252,-0.00997,0.01845,0.00366,-0.00724,0.00578,0.00324,0.00532,0.00827,0.00268,0.00374,0.0064,0.00566,-0.02728,-0.03944]],"anchors_sha":"dfb115cded2e1cd3","dhat":{"UNCERTAINTY":[0.01647,-0.00042,0.0151,-0.00075,-0.01835,0.05049,-0.01913,-0.00706,0.01061,-0.01457,0.0025,-0.09488,0.00209,0.02072,-0.0158,0.01078,-0.01063,-0.02881,0.02371,0.06064,0.0194,-0.03458,0.01845,-0.01701,-9e-05,0.0175,0.00023,0.01396,-0.00945,0.00451,-0.01432,-0.0122,-0.01919,0.04866,-0.031,0.03583,0.00516,0.01209,-0.03019,0.02806,0.00953,0.00344,-0.00407,-0.02251,-0.02521,0.03707,-0.04319,0.02499,-0.0337,-0.01066,0.00085,0.05231,-0.01759,0.03643,-0.03722,0.00723,-0.02912,0.00858,-0.00739,0.01043,0.02811,-0.00667,-0.02022,-0.00408,-0.00639,-0.0118,0.02139,0.02078,-0.01686,-0.03311,0.0229,0.04597,-0.00501,-0.01082,-0.04053,-0.02033,-0.02075,-0.00445,-0.02297,-0.00685,-0.03306,0.01066,0.00232,-0.00303,-0.00256,0.01154,0.00838,-0.00626,0.04353,-0.00843,0.02384,-0.00864,-0.01446,0.01018,0.02791,-0.01273,0.03401,0.04238,0.04141,-0.01401,-0.00122,-0.00463,-0.00527,-0.01742,-0.01856,0.0046,0.03283,0.01348,0.05022,-0.0393,-0.02284,-0.01936,-0.02493,-0.0316,-0.01264,-0.01464,0.01303,-0.02384,-0.02787,0.02907,0.01903,-0.0155,0.01457,0.02962,0.03053,0.0296,-0.01215,-0.00903,-0.02604,0.01333,-0.02991,0.04285,0.01295,0.00939,-0.00734,-0.03689,-0.01449,-0.04923,0.04467,0.00042,-0.03401,-0.01874,0.01799,0.02433,-0.02526,-0.04507,0.00319,0.0137,-0.0018,-0.00163,0.00964,-0.0132,0.00817,0.00011,0.02335,-0.02203,0.01948,-0.00056,-0.01449,-0.01955,-0.00694,0.00469,-0.02856,-0.00791,-0.01827,0.00472,-0.00143,0.04802,-0.01958,0.02041,0.03807,0.03061,-0.02211,-0.01706,-0.00598,0.03387,0.00043,-0.06527,-0.01328,0.03322,0.02599,0.0257,0.00344,-0.01815,-0.00955,-0.02796,-0.01091,0.02874,0.03033,0.00277,0.00984,0.02817,-0.02104,0.00448,0.01592,0.03122,-0.01147,-0.01606,-0.00871,0.01725,-0.02079,-0.02293,0.02081,-0.00986,-0.04056,-0.00769,-0.00722,-0.03671,-0.02105,0.06053,-0.0078,-0.00187,-0.02951,-0.01044,0.0063,-0.00898,-0.00924,0.01363,0.01968,0.01125,-0.00962,-0.03875,0.00547,-0.02949,-0.00496,-0.01675,0.03038,0.00848,0.01197,0.04502,-0.02499,0.00722,0.0056,-0.01006,-0.01305,-0.01666,-0.023,0.01364,0.00791,0.00106,0.02624,0.03317,-0.0092,-0.02218,0.03457,0.02093,-0.02798,-0.04334,-0.01471,0.00273,0.02164,-0.04129,-0.01326,0.0165,-0.00355,0.02202,-0.00699,-0.00115,0.0004,-0.03871,-0.00918,-0.00209,0.01786,0.02772,0.02503,0.01326,-0.00823,0.01027,0.02517,0.02194,-0.00537,-0.02174,0.01618,-0.00317,0.01949,0.00352,-0.00085,0.00083,-0.01218,-0.02487,0.0021,-0.00552,0.06823,0.01127,0.0106,0.01031,-0.04121,0.01245,-0.01389,0.00467,0.01196,0.02431,0.00269,-0.00848,-0.01314,-0.03083,3e-05,0.02408,-0.00388,0.04923,-0.02023,-0.02896,-0.00233,-0.02504,-0.02308,0.04794,-0.00447,-0.00735,-0.00941,-0.00662,-0.02534,-0.0337,-0.01036,0.00976,0.01083,0.03349,0.07391,0.02846,0.00825,0.0067,0.01987,0.00555,0.01905,-0.01411,-0.00124,0.01925,0.00528,0.00776,0.01117,0.00169,0.00716,-0.01859,-0.01441,0.0451,-0.00824,-0.00206,-0.00111,0.01803,-0.03364,0.04484,-0.01725,-0.01542,-0.0254,-0.02122,0.01492,0.015,-0.00529,-0.01302,-0.02986,0.00213,-0.00458,0.02213,-0.00045,0.01258,0.02558,-0.0178,-0.03743,0.02711,0.03154,0.01616,0.00104,0.01746,0.00969,0.01329,0.0075,0.01546,0.0024,0.01563,-0.00928,-0.03626,0.01818,0.02529,-0.00342,-0.00068,-0.01817,0.02151,0.00909,0.04299,0.01271,0.01418,-0.00844,0.00035,0.00326,0.00502,0.01236,0.02834,0.03941,0.01526,0.00972,-0.01757,0.01437,0.00553,-0.0047,0.04251,0.01373,0.006,0.00389,0.04496,0.03045,0.01518,-0.00683,-0.01844,0.03515,0.00288,0.02734,0.01298,0.03407,0.00936,-0.00813,-0.00667,0.0416,-0.01711,0.00927,-0.05173,-0.03033,0.00624,0.00682,-0.03784,0.02671,-0.01218,0.00313,0.00994,0.0021,-0.00646,0.00069,0.00418,-0.0224,-0.01388,-0.00961,-0.02482,0.01768,0.02762,0.02746,-0.03332,-0.03189,0.01311,0.00934,0.04971,0.00667,0.05683,-0.00813,0.00653,0.04265,0.00754,0.00044,-0.01399,0.01435,-0.0124,0.01176,-0.01155,-0.01719,-0.02489,-0.03272,0.01465,-0.04441,-0.01669,0.00027,0.01587,-0.01415,0.0096,-0.03063,-0.00034,0.02806,-0.00612,0.01406,-0.00904,0.04659,0.02196,-0.01183,0.05254,0.01545,0.06708,-0.02949,0.00031,0.03728,-0.00585,0.0096,-0.00962,0.00525,0.02795,0.02777,0.03464,0.00174,0.02774,-0.03138,0.00439,0.01795,-0.00698,0.00749,-0.01155,0.0524,-0.00219,-0.01319,-0.01436,0.00768,0.00839,0.01064,-0.00949,-0.0304,0.00974,-0.0336,0.01271,0.01426,0.05463,-2e-05,-0.04169,0.0359,0.0169,0.0193,0.02582,0.0052,0.04155,-0.0033,-0.02914,0.01046,0.02995,0.01057,0.02268,0.0319,0.03228,-0.03945,-0.00744,0.00377,0.0012,0.03981,-0.00486,0.04105,0.01069,0.04011,0.03463,0.03532,0.03945,-0.00993,-0.02458,-0.01153,-0.01104,-0.03481,-0.01449,-0.01597,-0.0133,0.01841,-0.01649,-0.0027,-0.01372,-0.03072,-0.00181,-0.0366,0.00671,0.01817,-0.03042,0.01148,0.04237,0.02139,0.1358,0.0072,-0.02796,-0.01955,-0.00229,0.01615,0.00209,-0.00263,0.03293,-0.01094,0.00914,-0.00114,0.01245,-0.0123,-0.01384,-0.0318,0.00878,0.04594,0.02112,-0.04253,-0.0309,0.05831,0.02517,0.00878,-0.05185,0.03272,0.01109,-0.02769,-0.01949,-0.00767,0.03436,-0.00281,0.01249,-0.02066,0.03188,0.01178,0.00497,0.00027,0.00446,0.01529,-0.0155,0.01767,0.05273,0.00198,0.0297,-0.03742,0.00412,-0.00211,0.00213,0.01533,0.04144,-0.0123,0.01539,-0.02095,-0.01104,-0.00426,0.17067,-0.0002,0.01642,-0.0198,0.02236,0.03838,-0.0081,0.04369,-0.0177,0.03526,-0.04369,0.03337,0.02419,-0.00613,-0.03037,-0.00247,0.0199,0.03192,0.0061,0.0099,0.01871,-0.00377,-0.0227,0.04335,-0.02886,-0.02952,0.02017,0.01619,0.01744,0.01499,-0.02158,0.0043,-0.00635,0.02225,-0.00901,-0.01644,-0.01692,-0.00187,-0.00672,-0.01044,0.01077,-0.01449,0.01951,0.02848,0.01384,0.01394,0.01783,-0.03643,0.01123,-0.0086,0.0064,-0.01842,-0.00484,-0.01158,-0.0155,-0.0018,0.00024,-0.01076,-0.0082,-0.00648,-0.05287,-0.01069,-0.0172,-0.02018,-0.01006,-0.0177,-0.02827,-0.04066,-0.02351,0.00273,-0.02916,-0.00965,-0.0257,-0.00774,-0.02642,-0.00101,0.01525,-0.01435,-0.00436,0.02301,-0.01757,-0.02355,0.01108,0.02591,0.03882,-0.0144,-0.00262,0.01191,-0.00568,0.01787,0.01248,0.02346,0.00196,0.01134,-0.01504,0.04301,-0.01229,0.02035,-0.01602,0.01859,-0.03139,0.02169,0.03573,-0.0036,-0.06827,-0.01893,0.00745,-0.01401,0.00643,-0.08197,0.02051,-0.00383,-0.01381,0.01418,-0.00942,0.01515,-0.00738,-0.00912,0.00221,0.01984,-0.00435,0.02743,-0.00307,0.0091,0.01062,-0.02503,-0.03153,-0.00714,-0.01558,-0.00259,-0.00901,0.00916,0.05911,0.01531,0.00085,0.03911,0.01544,0.04362,-0.01251,-0.00533,0.01384,-0.03629,-0.00379,0.05488,-0.00028,-0.00294,0.01837,0.0045,0.02047,0.03836,-0.03033,0.00339,-0.01943,-0.00874,-0.0081,-0.00384,-0.02197,-0.00864,0.02761,0.01089,0.02613,0.02919,-0.00857,-0.01367,-0.01436,0.00452,6e-05,-0.0011,-0.01365,-0.00892,-0.00297,-0.02823,-0.00371,-0.01221,-0.00054,0.00134,-0.0215,0.01501,-0.00164,0.01447,-0.05189,0.03044,-0.01484,-0.03085,-0.00992,-0.01732,0.04239,0.01884,-0.03962,-0.02395,-0.02294,0.03028,-0.01137,0.01165,-0.01634,-0.00277,-0.00319,-0.00701,-0.00902,-0.0394,-0.01243,-0.02229,0.02703,0.01353,0.01698,-0.00782,0.02591,-0.00488,-0.03851,0.0085,-0.03184,-0.01787,-0.03494,0.02022,0.00588,0.00497,0.02243,-0.02074,-0.03064,-0.02589,-0.00935,-0.01444,0.01043,0.00828,-0.00234,-0.00866,0.02661,-0.01984,-0.02737,0.03356,0.01703,-0.00528,-0.00697,-0.01209,-0.00487,0.04651,-0.02964,-0.00509,0.00558,0.00751,-0.01703,-0.00606,-0.01034,-0.0212,0.02816,-0.00477,-0.0284,0.01714,-0.0093,-0.02502,-0.00185,-0.00783,0.04249,-0.03626,-0.00656,-0.0076,-0.09827,0.01602,0.01135,0.02538,0.01182,-0.02174,0.0265,0.00427,0.04002,-0.01824,0.0349,-0.00758,0.01691,-0.00707,-0.00082,0.03528,0.03549,-0.03469,-0.01462,0.03858,0.01869,-0.03134,0.00745,0.02212,-0.03639,-0.02063,-0.0128,-0.02563,0.02854,-0.03069,-0.01622,0.02642,0.01759,0.0293,-0.03383,-0.00633,0.03963,0.00866,-0.01599,-0.02807,-0.00172,0.00168,0.01744,0.01609,0.01185,-0.00148,0.01303,0.00603,-0.01826,-0.01239,-0.02074,0.02384,0.0173,0.0187,-0.00767,-0.00116,-0.01876,-0.01513,0.02562,0.00829,0.00356,-0.01727,0.06059,-0.00195,0.0277,-0.0127,-0.01985,0.00896,0.02822,-0.01589,-0.04318,-0.02515,0.02096,-0.01923,0.03837,0.00955,-0.02576,-0.01212,-0.01491,-0.00477,0.00791,-0.0256,-0.01485,-0.02828,0.00142,0.0109,-0.00157,-0.01083,0.04146,0.01721,0.01532,0.00319,-0.02267,0.00451,0.04722,-0.01413,0.00636,-0.02008,0.03939,0.03106,-0.0453,-0.00733,0.03144,0.01617,-0.02968,-0.00524,-0.034,-0.00685,-0.01529,0.01231,0.01745,-0.02183,-0.01212,-0.0377,-0.00412,0.02452,0.02091,-0.01498,-0.02802,-0.00869,-0.00298,-0.04707,-0.00122,0.02477,-0.01008,-0.02476,-0.03659,-0.00187,-0.0335,-0.00136,0.03025,-0.03004,0.01573,-0.04192,-0.00946,-0.00962,0.00117,0.0024,0.03242,0.01013,0.02636,-0.02485,-0.01535,0.01593,0.00687,-0.00251,-0.03,-0.00106,0.02216,0.01377,-0.00089,0.00692,-0.01106,0.01236,-0.01312,-0.00133,0.03213,0.00711,0.03443,0.02595,-0.02741,0.03376,0.00324,-0.00117,-0.00793,-0.01203,0.00386,0.02941,0.01872,0.01027,-0.00052,-0.00698,0.00703,-0.03029,-0.0081,-0.02944,0.01617,0.0085,-0.00443,-0.01383,0.03658,-0.00775,0.00665,-0.03047,-0.04235,-0.01799,0.01636,0.01236,-0.00582,0.00531,-0.00018,-0.00297,-0.03165,-0.01834,-0.01323,-0.01599,-0.02217,-0.00433,-0.01106,-0.01297,-0.01129,0.00717,-0.01384,0.05039,0.00958,-0.0134,-0.00447,0.02626,0.06968,-0.03086,0.03063,-0.00387,-0.01777,0.00858,0.03738,0.02724,0.01325,0.04256,-0.00709,0.01204,0.03567,-0.00087,-0.01421,0.00669,0.03862,0.01695,0.01971,0.00264,0.01881,-0.01637,-0.01012,0.00829,-0.00081,-0.00055,-0.02698,0.005,-0.01245,0.00267,-0.00762,-0.00931,-0.02108,-0.01114,-0.00433,0.00886,-0.0176,-0.0292,0.01278,0.00566,0.01079,0.03039,0.00266,-0.02077,0.04149,-0.00349,-0.00119,-0.01418,0.02166,0.02526,0.02361,-0.01176,-0.04319,0.00191,-0.00695,-0.01656,-0.00141,0.00655,0.01161,0.01887,-0.05128,0.00034,0.00012,-0.00763,0.00517,-0.00909,0.01306,0.00279,-0.01342,-0.05368,0.0002,-0.02493,-0.00665,-0.01104,0.00174,0.04033,-0.02995,-0.00541,-0.00213,0.00415,0.00232,-3e-05,-0.00863,0.0025,-0.01078,0.00478,-0.01357,0.00635,0.04102,0.00142,0.01007,-0.03421,0.00767,0.02539,-0.02201,-0.01383,-0.02756,-0.0084,0.02147,0.04494,0.0321,-0.01623,0.03416,0.02404,0.01908,-0.01041,-0.0352,-0.00251,-0.0351,0.024,0.00945,0.00576,-0.01316,-0.00281,-0.00224,0.02072,-0.02656,0.04591,0.02565,0.01061,0.01147,-0.03104,0.00011,0.02728,-0.0581,-0.0609,0.00028,-0.01255,-0.02024,-0.0036,0.01258,-0.00949,-0.02053,-0.00186,0.02838,0.03548,0.00564,-0.00044,-0.02508,0.0032,-0.01498,-0.05298,0.03669,0.00345,-0.00487,-0.01331,-0.03031,-0.02279,-0.03404,-0.05042,0.0228,0.05487,-0.00576,-0.00657,0.02023,-0.0391,0.14532,-0.02482,-0.02108,-0.0038,-0.00474,0.00928,-0.00151,0.01203,-0.02065,-0.01036,-0.00913,0.03566,0.01956,-5e-05,0.01403,0.00683,0.01348,-0.0184,0.00545,0.05175,-0.02481,-0.00582,-0.02753,-0.03723,-0.00212,0.03888,0.00827,-0.03519,0.00846,0.01099,-0.05242,0.01847,0.02247,0.01656,-0.00395,0.00634,0.00714,-0.01428,0.0,-0.01895,-0.01347,-0.02237,-0.00817,-0.00988,-0.02372,-0.00581,-0.02384,-0.00013,-0.03421,0.00727,-0.01304,-0.0,-0.03222,0.00518,-0.04441,0.01872,0.01941,-0.02959,-0.00576,-0.00582,-0.00696,-0.0213,-0.01119,-0.00366,-0.03597,-0.02484,0.00304,0.01848,0.03475,-0.04928,0.0268,0.01314,0.02729,-0.01386,-0.00835,-0.00035,0.01902,-0.02593,0.02206,-0.0204,0.02618,0.00787,0.00172,0.02625,-0.02639,-0.00364,-0.05598,-0.03093,0.04607,-0.00521,-0.00495,-0.01199,-0.00943,0.065,0.02716,0.02898,0.01797,0.01693,-0.00806,0.01725,-0.01254,0.02535,0.02449,0.00822,0.01528,0.01154,-0.01108,-0.01317,-0.02052,-0.00012,-0.0018,0.00709,0.03124,0.01119,0.02695,0.02328,0.02737,0.00771,0.03881,0.00722,0.0034,-0.05237,0.0064,0.00433,0.04701,0.03493,0.04433,-0.02061,-0.05254,0.02361,0.01953,0.01667,0.00743,-0.00146,-0.01283,0.00153,-0.01409,0.01425,0.01831,0.02051,-0.00779,0.00472,0.06085,0.0039,0.00984,0.0039,0.00494,0.00879,0.0102,-0.01537,0.01325,-0.04088,0.00931,0.00148,-0.03425,0.00827,-0.04063,-0.03294,0.01676,-0.00399,0.00727,-0.00125,-0.03039,0.01768,-0.02784,-0.02931,0.01984,0.00412,-0.00188,-0.02004,-0.0112,-0.03658,-0.01282,-0.04547,-0.02876,-0.05575,-0.02295,0.01949,-0.00824,-0.01919,0.00483,-0.00919,-0.01979,0.01635,0.00405,0.01645,-0.01582,0.00746,0.00409,0.01586,0.01599,-0.02594,-0.332,0.02191,0.10128,0.07227,-0.04997,0.02253,-0.01959,0.0125,-0.04351,-0.00893,-0.02129,-0.03026,0.0334,-0.0356,-0.00704,-0.01324,0.0132,-0.01055,-0.01746,0.03664,-0.00089,0.00908,-0.01639,-0.04644,0.0175,0.00949,0.02197,-0.03247,0.03137,0.02302,0.00165,-0.00985,-0.04963,0.00421,0.01174,-0.01342,0.01354,0.05729,-0.01696,0.02332,0.00362,0.00471,-0.02759,-0.01323,0.01008,0.04462,0.00479,0.01376,0.0049,0.02907,0.01326,-0.00789,-0.02383,0.03093,0.00609,0.03387,-0.0148,-0.00542,-0.00377,-0.01201,0.0584,-0.01999,-0.00701,0.02273,0.0208,0.02635,-0.00577,0.01469,0.00771,-0.00281,-0.01536,-0.01968,-0.00922,-0.00698,0.01108,-0.04123,-0.00481,-0.02907,-0.01081,-0.03722,-0.01366,-0.01796,-0.00095,0.02455,-0.02051,-0.00441,0.01104,-0.02496,-0.00722,0.00163,-0.00205,0.01018,-0.05484,-0.0055,-0.00015,-0.03003,0.0199,-0.01752,0.03959,-0.01128,0.04122,0.02446,-0.00658,0.06023,-0.00269,0.03607,0.03251,-0.00321,0.03388,0.00934,0.00553,-0.0351,-0.01326,-0.04407,-0.00034],"CONFIDENCE":[0.02252,-0.01323,-0.00485,0.03719,-0.032,0.0387,0.00444,-0.02132,0.04082,0.0234,-0.01053,-0.05282,0.02902,0.01628,-0.01596,-0.0057,-0.03003,-0.00584,0.01901,0.00238,0.01538,0.00078,0.02285,0.00336,0.00722,0.03843,0.0046,-0.00386,0.04128,-0.02049,-0.01336,0.01797,-0.03974,0.02238,-0.01614,-0.01852,-0.02446,0.02199,0.00971,0.00781,0.02777,-0.01924,0.05455,0.01596,-0.0452,0.01916,0.00245,-0.03073,-0.03125,-0.02193,0.0254,0.01689,0.00493,-0.00279,0.03268,0.02986,-0.00606,0.00151,-0.01561,0.02505,0.02083,-0.00889,-0.00178,0.00318,0.00723,-0.0155,0.00444,0.01061,-0.01768,0.01156,0.01173,-0.00123,0.00316,0.0286,-0.04236,-0.00427,0.01522,-0.01709,-0.03009,-0.00068,0.0377,-0.0017,-0.01366,0.01479,0.03213,0.00769,0.00344,-0.00808,0.02956,-0.00011,-0.00365,-0.00247,-0.00973,0.02047,0.03961,0.04335,0.03846,0.02858,0.06085,-0.0473,0.03201,-0.01291,0.00118,-0.06296,-0.02744,0.00369,0.03044,0.00583,0.04359,-0.03889,-0.02849,-0.02826,-0.02164,-0.01718,0.01546,-0.03701,0.00744,-0.01041,-0.01251,0.0025,-0.01955,0.00448,0.03044,0.02493,0.03548,0.04832,0.01781,-0.00449,0.01054,-0.00107,-0.05382,0.01833,0.0135,0.04366,-0.06576,-0.02397,-0.00074,-0.03105,0.007,0.00175,-0.0116,0.03358,0.00927,0.01995,0.00807,0.00532,-0.0373,0.01712,0.02166,-0.05386,0.02096,-0.01574,-0.01486,-0.01065,0.01657,-0.02866,0.04336,-0.00613,-0.01197,-0.02067,-0.02142,0.02136,0.00797,0.00438,-0.01163,-0.03279,0.02685,0.05202,0.02448,0.01042,0.04557,0.00261,0.01063,-0.01005,-0.00976,0.00856,-0.00142,-0.04287,-0.00901,0.03213,0.03959,0.02749,0.02305,-0.04697,0.00018,-0.00219,0.00134,-0.00436,-0.00891,0.00514,-0.03753,0.01834,-0.04368,-0.01242,-0.02723,0.02911,0.03139,-0.02989,0.01923,0.01469,0.02302,-0.0073,0.00527,0.01746,0.01209,-0.01373,-0.00273,-0.01722,0.03424,0.02622,-0.0291,-0.01209,-0.01234,-0.03753,0.01023,-0.02479,-0.02505,-0.00471,0.02369,0.02658,-0.01975,-0.00895,-0.00019,0.0075,-0.00664,-0.02734,0.03938,-0.03274,0.01706,0.02985,0.05543,-0.03259,0.04169,0.02898,-0.02698,0.01056,0.04737,0.02869,0.00078,-0.03888,0.02574,-0.01537,0.01108,-0.07838,-0.0054,0.02416,0.00213,-0.03252,-0.00555,0.01208,0.00514,-0.04656,0.02309,0.01335,-0.02673,0.01155,-0.02794,-0.0301,0.02654,0.0018,-0.03273,0.0304,0.03829,0.04158,-0.01775,-0.00909,0.0,-0.02143,0.01901,-0.01792,0.00056,-0.02389,0.04956,0.02867,0.01993,0.01598,0.00157,0.01583,-0.00208,-0.00535,-0.00714,0.03411,0.01766,0.00869,-0.03156,0.01674,0.01175,0.00283,-0.00405,0.01258,0.04073,-0.02491,-0.00426,0.00972,0.00778,-0.01293,0.04941,0.05518,-0.00265,-0.02438,-0.01126,-0.01357,0.02042,-0.02492,0.0008,0.03676,-0.01889,0.00096,0.02831,0.00385,-0.00783,0.01457,-0.01732,-0.00558,0.021,-0.00019,0.03868,-0.00907,0.00128,-0.00083,-0.03874,-0.01444,0.01328,-0.01184,-0.00437,0.0027,-0.01746,0.01326,0.00532,0.02614,-0.00217,-0.00029,-0.01166,0.02749,-0.00556,0.02153,0.00066,0.01555,-0.0037,0.04233,0.00628,0.0278,-0.01668,-0.04269,-0.01221,0.01475,-0.02611,-0.02232,-0.00574,0.02727,0.00361,0.01177,0.00911,0.01278,0.05972,-0.0254,0.03698,0.03896,0.05665,0.04185,0.02207,0.01922,0.04366,0.04688,0.01647,-0.04918,0.01143,-0.02263,0.00064,0.05091,0.02061,0.0013,0.01088,0.01808,-0.04304,0.01827,0.00121,0.02472,-0.00535,0.01849,-0.01053,-0.01477,0.0121,0.02313,0.00647,0.01628,0.00653,0.01781,0.00136,-0.00042,-0.0337,-0.01415,0.0186,-0.01891,0.00787,-0.03855,0.02034,0.02278,0.01397,0.00846,0.03204,-0.0783,0.00636,0.00522,0.03449,-0.00181,0.03216,0.01611,0.02938,-0.01343,0.04145,0.00994,0.00691,-0.03433,-0.02025,0.02011,-0.01246,0.00313,0.00117,-0.02645,0.025,0.03845,0.03566,-0.00225,0.03062,-0.02822,-0.03408,-0.02509,0.03251,0.00661,0.04495,0.0031,0.05761,-0.02856,-0.00806,-0.00307,-0.02547,0.0339,-0.0225,0.04199,0.01326,0.00078,0.04888,0.03066,0.011,-0.02877,-0.01691,0.01047,-0.02359,-0.03442,-0.0024,-0.00186,0.0416,-0.06339,-0.03304,0.03692,-0.00148,-0.01436,-0.01034,0.02074,-0.01382,-0.05462,0.02275,0.01516,-0.00196,-0.00478,-0.00138,0.01124,-0.0024,0.06541,-0.02917,0.02838,0.0038,0.05942,-0.00076,-0.01854,-0.00544,-0.02037,0.00037,0.01731,-0.00196,-0.02636,-0.01843,0.00454,-0.00625,0.01933,0.02953,-0.03328,0.00688,-0.01041,0.02118,-0.03456,0.00221,0.00364,-0.01008,0.0091,0.00722,0.0042,-0.01318,-0.01369,-0.01871,0.04585,0.00613,0.02392,0.04063,-0.04531,0.00587,0.02394,-0.00527,0.01479,-0.02084,0.03241,0.00559,0.00804,-0.00033,0.04447,-0.01832,0.05776,-0.00511,0.05103,-0.02652,0.02616,0.00267,-0.00792,0.00688,-0.03178,0.02815,-0.03486,0.01123,-0.0015,0.03719,0.03307,-0.02153,-0.00797,0.00129,0.01673,-0.02227,-0.01349,-0.00246,-0.00075,0.05688,-0.01307,0.0108,0.00013,-0.039,-0.00358,0.00977,-0.01316,0.02418,-0.01594,-0.00149,-0.02006,-0.0452,0.08395,0.00849,0.02678,-0.00924,0.01606,0.02168,-0.00444,0.02046,0.04779,-0.00606,-0.05307,-0.01964,-0.01709,-0.00419,-0.01886,0.03817,0.02544,0.02404,0.0176,-0.01215,-0.00941,0.0336,-0.01099,0.01235,-0.05206,0.01139,0.00364,-0.02195,-0.03375,-0.0121,0.04439,-0.00334,-0.04553,-0.02453,0.04429,-0.00181,0.00265,0.02947,-0.0126,0.03175,-0.0045,-0.0146,0.04543,-0.03004,0.00073,-0.04398,-0.00964,0.01746,-0.03683,-0.00839,0.00734,0.01818,0.03279,-0.01393,0.00974,0.04388,0.04627,-0.02265,0.00805,-0.01053,-0.00778,0.00575,0.00641,0.01072,-0.00975,0.01489,-0.03104,0.01089,0.00843,0.00323,-0.02795,-0.00308,-0.02205,0.03006,0.03519,0.04036,0.02473,0.00478,-0.02189,0.02341,0.04414,0.01018,-0.00411,0.01012,0.00769,-0.02302,0.01206,-0.00839,-0.01305,-0.00351,-0.00747,-0.00582,-0.0182,-0.01891,0.01097,0.00924,0.0017,-0.00018,0.00461,0.0186,-0.00866,-0.01136,-0.04483,-0.01651,-0.03761,0.05448,-0.02669,-0.05126,-0.01599,-0.02469,0.00435,-0.00833,-0.01333,0.01766,-0.00222,0.00773,-0.03851,-0.00774,-0.01409,-0.02769,0.05416,-0.03868,-0.02726,-0.01556,-0.01542,0.00904,0.01616,-0.04661,-0.01213,-0.01352,-0.05695,0.01271,-0.03247,-0.00251,0.00879,0.05392,-0.04582,0.00981,0.02079,0.02762,0.07825,0.00292,0.01346,-0.00053,-0.01322,0.00103,0.01268,0.03611,-0.02733,0.05603,-0.02055,0.01627,-0.02032,0.00418,0.01076,-0.02894,-0.01589,0.00202,0.01514,-0.01372,-0.05177,-0.02586,-0.04014,-0.00823,-0.01419,-0.0077,0.01384,-0.04463,-0.00169,0.04196,0.001,0.01436,-0.01085,-0.02683,0.00407,-0.01138,-0.03266,-0.02045,-0.00745,-0.02188,0.02374,0.05689,0.00158,0.02972,0.00514,0.01157,-0.01877,0.0273,0.01089,-0.03413,0.00159,0.00552,-0.02803,0.03979,-0.01852,-0.00577,-0.01723,-0.02021,0.02613,0.03676,-0.01076,-0.02995,0.00207,0.00621,0.00476,0.0197,-0.00307,0.05388,0.01369,-0.02434,0.0017,0.00677,0.03194,-0.03777,0.00525,-0.01671,-0.02291,0.01063,0.03579,-0.01564,0.00842,-0.00444,0.03956,-0.01213,-0.00774,0.01333,0.01394,-0.00136,-0.02732,0.02157,0.00064,0.06525,0.00452,-0.02685,-0.01707,-0.01114,-0.04107,0.03401,0.04755,-0.00136,0.01061,0.04476,0.00411,0.00335,-0.01106,-0.00562,-0.01799,-0.00629,0.03303,0.00885,-0.05866,0.04872,-0.01342,0.00531,0.03104,-0.04002,-0.01794,-0.02804,0.02804,0.01113,-0.01376,0.00284,-0.02264,0.01577,-0.02245,0.03329,-0.00944,-0.01166,-0.03628,0.00035,0.03397,0.01294,0.03056,-0.01495,-0.03699,-0.0081,-0.02006,0.00311,-0.00362,-0.02468,0.00502,-0.0142,0.03731,0.0366,-0.00916,0.00882,0.03218,-0.02156,0.01474,-5e-05,-0.01341,0.01219,0.00095,0.01549,-0.03672,0.00087,0.00541,0.02295,-0.0248,-0.04108,0.0413,-0.00117,-0.03065,0.02264,0.0331,-0.00898,-0.01222,0.00362,0.00191,0.02484,-0.00581,-0.01355,-0.0527,0.01449,0.00483,-0.01203,0.01767,-0.03343,0.01513,0.01228,0.02692,-0.05406,0.0242,-0.00549,0.02063,0.00033,0.01018,0.02626,0.02021,-0.04581,-0.02121,0.02447,0.01216,-0.03318,0.01119,-0.02477,-0.04311,0.00232,-0.02584,-0.02495,0.03932,-0.01039,-0.0299,0.03314,0.01166,0.01943,0.00102,0.00227,0.04395,0.02926,-0.0439,-0.01991,0.007,-4e-05,0.04066,0.0026,-0.00188,-0.03147,0.02837,0.01095,-0.00154,0.02317,-0.03342,0.03583,0.00067,-0.01714,-0.00272,-0.01916,0.03075,0.02624,-0.00208,-0.01262,-0.03239,-0.0271,0.05014,-0.01139,0.02066,0.0119,-0.02874,-0.02091,0.00578,-0.02903,-0.044,0.001,0.01841,-0.01739,0.01891,0.0126,0.02527,-0.00945,-0.01586,-0.02158,0.00654,-0.05232,0.00878,0.00033,-0.00765,-0.01669,-0.00794,0.01923,-0.00199,0.01772,0.04426,0.00078,0.00824,-0.02336,0.01796,-0.03583,-0.0131,-0.03282,0.07082,-0.02781,-0.07842,-0.02185,-0.01016,0.00494,-0.01261,-0.00985,-0.03828,-0.0233,-0.06542,-0.01667,0.00951,0.01521,0.02576,-0.0184,0.04529,-0.00093,-0.0167,0.02886,-0.0363,-0.05108,-0.02431,-0.02327,-0.01196,0.02114,-0.00814,-0.03694,-0.00684,0.022,-0.03708,0.03356,0.04841,0.02326,0.02259,-0.05793,-0.01866,-0.01611,-0.00772,0.01402,0.04508,0.00278,0.02148,-0.00779,-0.00013,0.03473,-0.00507,-0.01566,-0.00187,-0.01349,0.00449,-0.00584,0.02169,-0.02176,0.01826,-0.01286,-0.01596,-0.02177,0.02597,-0.01376,0.02499,0.03428,0.02621,0.03714,-0.01725,-0.01149,-0.02553,-0.00517,-0.00677,0.00606,0.03305,0.0165,0.00858,-0.02419,-0.01339,-0.0098,0.03003,-0.03669,0.00743,-0.02028,0.01569,-0.02932,0.04364,-0.04727,-0.02977,-0.03698,0.00969,-0.00566,-0.00265,0.02591,0.01962,-0.02375,-0.00541,-0.01564,0.03825,-0.02718,0.01239,-0.03859,-0.01401,-0.00166,-0.0224,-0.01721,-0.00339,0.04538,0.01019,-0.01832,-0.01091,-0.00072,-0.00227,0.04756,0.06779,-0.07975,0.00536,-0.01471,-0.01776,-0.00401,0.02665,0.05123,-0.01415,0.06756,-0.00531,0.01374,0.02366,-0.01168,0.01065,0.01405,0.00451,0.05285,-0.01169,-0.01037,0.01502,-0.00865,0.05188,0.00768,0.001,-0.01349,0.02922,0.01849,0.0104,0.00031,0.05664,-0.00745,-0.00346,-0.00941,0.00123,0.02426,0.00468,-0.04788,-0.03159,-0.00375,-0.00133,-0.03296,-0.00228,-0.02499,0.00133,0.02531,0.01124,-0.00622,0.01255,0.00614,-0.04174,0.0035,-0.00622,0.03313,0.03672,-0.01754,-0.0275,0.04026,0.0332,-0.00658,-0.01999,-0.01318,0.00824,-0.00295,-0.00077,0.0297,-4e-05,0.02309,0.01216,0.01809,-0.06022,-0.03362,-0.00403,0.00164,-0.01559,0.02231,-0.00318,-0.00425,-0.06326,0.03975,-0.00723,0.03397,-0.0083,0.00805,-0.00185,0.02589,-0.01716,-0.00092,0.02096,-0.02014,-0.02786,-0.01674,0.04625,0.04353,-0.01518,0.00652,-0.02375,0.00888,0.022,0.02729,-0.01759,-0.02864,0.06831,0.01726,-0.0082,0.03499,-0.04037,0.0254,-0.00249,-0.03153,-0.01175,-0.02924,0.01842,0.00365,0.00936,0.01656,0.00198,0.02479,0.05551,-0.0149,0.06405,-0.0251,-0.01604,0.00488,-0.03411,-0.01918,0.01629,0.00907,0.04364,-0.03288,-0.0059,-0.03301,0.00149,-0.05182,0.04541,-0.01281,-0.00792,-0.00859,0.0244,-0.01408,-0.05111,-0.0067,0.0225,-0.00357,-0.0274,-0.01516,-0.04499,-0.02774,-0.0251,-0.01596,0.00834,0.03921,-0.01381,-0.01771,0.02128,-0.04647,0.05441,0.00383,0.00489,-0.0049,-0.00961,0.05139,0.00084,-0.03453,0.01397,-0.02583,-0.03793,0.06081,0.02503,0.02344,0.01342,-0.00086,0.00346,-0.00603,-0.01415,0.03312,-0.02613,0.01272,-0.00967,-0.01853,0.04442,-0.02937,0.00534,-0.05814,0.00654,-0.01614,-0.04173,0.01045,0.01623,0.01615,-0.02627,-0.00378,0.00716,0.00496,-0.05088,0.01845,0.01881,0.03148,-0.01238,-0.04141,0.0134,-0.01391,-0.06378,0.02404,0.0021,-0.00929,0.01247,0.02256,-0.00999,0.01344,-0.05206,-0.00646,0.00336,-0.02064,0.02555,-0.00015,-0.01546,0.00977,-0.00877,-0.04625,-0.01156,-0.00075,0.02771,-0.01126,-0.01642,-0.05185,-0.00401,-0.01241,0.02622,-0.02665,-0.00308,0.00355,0.0066,-0.02227,0.05324,-0.01467,0.03579,0.01525,0.01752,0.03029,0.00454,-0.00139,-0.01545,-0.00982,0.00392,-0.01127,-0.01141,0.01182,0.04808,0.03663,0.01411,-0.02065,-0.01833,0.02929,-0.016,0.01595,-0.0153,0.02259,0.0087,0.03954,0.03294,0.02463,-0.05082,0.03827,0.03275,-0.01328,-0.01837,0.00051,0.02796,-0.0272,0.01242,0.03136,0.0288,-0.02755,0.0343,0.01564,-0.00334,-0.01233,-0.02911,0.00693,0.0332,-0.01086,0.0513,0.00809,0.00233,0.02314,0.00888,0.02957,0.01591,-0.01591,0.0045,0.06253,0.03601,0.01657,0.01397,-0.01498,-0.01271,-0.00351,0.07016,-0.01782,-0.00089,0.0134,-0.01086,0.00645,0.00215,-0.02072,-0.01785,-0.00295,0.00269,0.00597,-0.0063,-0.01079,-0.01451,0.01934,0.02782,0.00188,0.01201,-0.01692,-0.01745,0.02095,-0.01802,-0.01872,-0.01466,0.00022,0.00851,-0.03916,-0.03428,-0.03188,-0.00901,-0.02874,0.01167,-0.03606,-0.03572,0.00607,-0.00528,0.00545,-0.00284,0.00188,0.03127,-0.0008,-0.0603,0.01036,-0.00549,-0.02649,0.03623,-0.00993,-0.02045,-0.03582,-0.04088,-0.01276,0.05864,0.06164,-0.02308,-0.00395,0.02932,-0.00898,-0.03483,-0.00277,-0.0146,-0.03747,0.00049,-0.03573,-0.01401,-0.01793,-0.01772,0.01359,0.00281,0.0094,-0.02812,0.00747,-0.02243,-0.0434,-0.00034,0.02093,-0.02947,-0.00427,0.01368,0.03426,-0.00091,-0.00591,-0.0329,0.00713,0.01973,-0.03627,-0.01381,0.02329,-0.00985,0.01169,0.0072,-0.00536,-0.00481,0.00732,-0.00838,-0.00342,-0.01749,-0.0134,0.03925,0.02637,-0.02359,-0.00073,-0.0257,0.02239,0.00119,0.00998,0.00978,-0.05374,0.01646,-0.01082,0.04823,-0.00647,-0.0119,-0.02408,-0.00264,0.02424,-0.00442,0.03289,-0.02277,-0.0009,-0.02151,-0.04435,0.02815,-0.03658,0.02268,-0.00169,0.01354,-0.01666,0.01467,-0.00632,-0.03278,0.0051,-0.04508,0.02728,-0.03067,0.01544,0.01374,-0.01439,0.00943,0.04927,0.05079,-0.02683,-0.01463,0.03589,0.00686,-0.00825,0.04219,0.0013,0.06712,0.04036,0.05922,-0.0054,0.02975,0.02776,-0.0194,0.01795,0.04448,0.01378,0.04714,0.0135,0.02099,-0.04321,-0.02833,-0.06133,-0.06517],"TENSION":[0.0202,-0.00415,-0.01266,0.03924,-0.01786,0.03826,-0.00533,-0.03367,0.04349,0.04116,-0.00026,-0.07501,0.02483,0.0069,-0.01711,-0.01805,-0.0324,-0.02689,0.01845,0.02111,0.01807,-0.0116,0.0313,-0.00989,0.00629,0.03375,-0.00406,0.00677,0.03289,-0.00929,-0.01133,0.00174,-0.0347,0.0377,-0.02263,-0.01307,-0.01862,0.01798,0.0085,0.0229,0.0173,-0.01616,0.04188,0.02417,-0.05601,0.01971,-0.0115,-0.02065,-0.03398,-0.0422,0.02596,0.04777,0.00328,0.00341,0.00195,0.02361,-0.00553,0.012,-0.00546,0.01328,0.02038,0.00476,-0.01736,0.00108,0.00603,-0.01366,0.02583,0.01816,-0.03059,-0.00481,0.01048,0.00437,0.00858,0.01697,-0.04834,-0.02542,0.00951,-0.01026,-0.017,-0.00313,0.01064,0.01269,-0.01051,0.01032,0.02863,0.00988,0.00103,-0.01268,0.03538,-0.00054,0.00421,0.01333,-0.01135,0.03613,0.04417,0.04078,0.04369,0.04506,0.05471,-0.04296,0.03388,-0.00175,-0.0189,-0.05272,-0.0445,0.01679,0.03758,0.01183,0.05763,-0.03206,-0.02216,-0.02711,-0.0281,-0.02868,0.00226,-0.03417,0.00788,-0.01182,-0.01616,0.01339,-5e-05,-0.01027,0.02988,0.02462,0.04974,0.0469,0.01437,-0.00879,-0.0114,0.00217,-0.05715,0.02305,0.00845,0.03302,-0.05027,-0.0345,-0.0061,-0.04326,0.01451,-0.00198,-0.00035,0.02244,0.00296,0.02727,0.01046,-0.01517,-0.02184,0.01077,0.00899,-0.04711,0.01172,-0.00701,-0.02084,-0.00337,0.02005,-0.02813,0.0401,-0.0161,-0.01049,-0.0214,-0.01603,0.01899,0.00097,-0.01072,-0.00135,-0.0366,0.03044,0.0501,0.02008,0.02337,0.04858,0.005,0.00199,0.0078,-0.01442,0.03922,0.00061,-0.04325,-0.00215,0.04255,0.03478,0.02539,0.01949,-0.02967,0.00654,-0.01484,0.00693,0.00307,0.01176,0.00492,-0.02166,0.02488,-0.04189,-0.00488,0.0017,0.02484,0.0216,-0.04477,0.00302,0.0209,0.00525,-0.02188,0.01198,0.01369,-0.00961,-0.00647,-0.00731,-0.02767,0.02979,0.04749,-0.03034,-0.01295,-0.0229,-0.01454,0.0192,-0.03461,-0.02749,0.00433,0.01241,0.01004,-0.00644,-0.02632,-0.00694,0.00835,-0.0018,-0.02659,0.04686,-0.02341,0.03331,0.04546,0.02794,-0.03877,0.02475,0.01994,-0.02129,-0.01137,0.04849,0.00353,-0.01111,-0.01473,0.0299,-0.00446,0.00495,-0.0703,0.00953,0.00993,-0.0172,-0.03079,-0.01114,-0.00242,0.00166,-0.03529,0.01561,0.0231,-0.00898,0.00798,-0.01502,-0.02571,0.0164,0.00473,-0.03446,0.02396,0.03497,0.04592,-0.01862,-0.00787,-6e-05,-0.01231,0.02879,-0.00532,0.00161,-0.03024,0.03137,0.02324,0.02312,0.00682,-0.01676,0.00284,-0.00246,0.00158,0.00493,0.01921,0.03634,0.00366,-0.01592,0.00301,0.01104,0.0056,-0.0036,0.02202,0.04471,-0.0216,-0.01185,-0.00585,0.01211,-0.02136,0.04226,0.06355,0.00865,0.01011,-0.01816,-0.02507,0.01005,-0.03565,-0.01513,0.0566,-0.01652,-0.00971,0.02653,0.01269,-0.01397,0.00491,-0.01567,0.01164,0.01892,0.0058,0.05399,-0.00365,0.00761,-0.0005,-0.01829,-0.00724,0.01393,-0.01372,-0.01439,0.00362,-0.01891,0.02109,0.00173,0.03015,0.00057,-0.01078,-0.01498,0.04712,0.00639,0.01132,-0.00596,0.01906,-0.01089,0.04666,0.00219,0.01979,-0.0179,-0.04315,-0.00078,0.01929,-0.0315,-0.02423,-0.01815,0.01312,0.01025,0.02205,0.01237,0.00756,0.05988,-0.01955,0.01727,0.04401,0.03715,0.04428,0.01544,0.03414,0.03248,0.02595,0.00794,-0.0446,0.0142,-0.01609,-0.01498,0.02269,0.02215,0.01525,0.003,0.02551,-0.02053,0.0201,-0.00679,0.02456,-0.00661,0.01722,-0.00995,-0.01673,0.00394,0.01743,0.02286,0.0257,0.01329,0.02186,0.01317,-0.01348,-0.02346,-0.00396,0.01391,-0.00515,0.01891,-0.02895,0.01088,0.03398,0.02591,0.00329,0.02647,-0.07687,0.00733,0.00323,0.02631,-0.0072,0.03081,0.02559,0.04564,-0.01385,0.05396,0.01539,0.01608,-0.02915,-0.01753,0.00893,0.00294,-0.00904,0.00094,-0.02537,0.01963,0.02896,0.02424,-0.00379,0.00574,-0.01588,-0.01981,-0.01895,0.02519,-0.01793,0.03345,0.01251,0.05005,-0.02701,-0.0106,0.01985,-0.02447,0.05412,-0.0047,0.0546,0.00808,0.01864,0.0374,0.01654,0.01205,-0.03069,0.00741,0.00814,-0.03134,-0.02414,-0.00926,-0.00796,0.01401,-0.03652,-0.03771,0.01715,-0.00149,-0.01206,-0.00237,0.01938,-0.01943,-0.02997,0.02757,0.02729,0.00425,-0.00595,0.02468,0.0039,-0.01262,0.06567,-0.02631,0.04322,0.00438,0.04408,0.01761,-0.0196,0.00146,-0.0153,-0.0155,0.02931,0.00416,-0.00325,-0.03624,0.01932,-0.01254,0.00752,0.03503,-0.02574,0.01427,-0.01218,0.04609,-0.03826,0.00637,-0.02274,0.00596,0.0085,0.01116,-0.00818,-0.01806,-0.01054,-0.04567,0.03113,0.00888,0.03757,0.03493,-0.04886,0.02272,0.03115,-0.01057,0.02676,-0.02535,0.02678,-0.00433,-0.00176,0.01831,0.03175,-0.01637,0.06523,0.00099,0.04972,-0.05844,0.02577,-0.00662,0.01239,0.02413,-0.02056,0.04703,-0.02022,0.02589,0.00441,0.03881,0.04403,-0.0142,-0.02103,-0.01466,0.00979,-0.02495,-0.01328,-0.01062,-0.025,0.03846,-0.02474,0.00609,-0.00681,-0.03629,0.0153,0.00254,-0.00285,0.00551,-0.02654,-0.00152,-0.00288,-0.02823,0.10166,-0.00319,0.00121,-0.00119,0.00959,0.02732,-0.0085,0.0333,0.05695,-0.01136,-0.02857,-0.02194,0.01046,0.00457,-0.02444,0.01521,0.02537,0.04568,0.02852,-0.0186,-0.02334,0.04372,0.00357,-0.00557,-0.06191,0.01025,0.01104,-0.02126,-0.03861,-0.00839,0.04356,-0.00388,-0.04718,-0.0342,0.03272,-0.00356,0.00064,0.01599,-0.00026,0.03613,-0.01501,-0.01389,0.04002,-0.02118,0.01052,-0.03223,-0.02963,0.02207,-0.01798,0.00844,0.02248,0.01456,0.02284,-0.01597,0.00479,0.02504,0.06807,-0.01819,0.00135,-0.01789,-0.0038,0.01555,0.00148,0.01513,-0.00721,0.02099,-0.03444,0.0232,0.02365,1e-05,-0.03444,-0.01253,-0.01794,0.02447,0.01308,0.03055,0.022,-0.00658,-0.05075,0.02318,0.0254,-0.00014,0.00068,0.01789,0.02294,-0.02487,-0.00609,-0.0144,-0.01484,-0.00507,-0.01818,-0.01336,-0.02556,-0.02346,-9e-05,0.00366,-0.00553,-0.00549,0.01183,0.0241,-0.0124,-0.00573,-0.01825,-0.02633,-0.02303,0.05332,-0.018,-0.03791,-0.03216,-0.03167,0.00048,-0.00808,-0.01435,0.01836,-0.00798,0.01648,-0.05803,0.00467,-0.01558,-0.02535,0.04699,-0.02688,-0.03356,-0.01464,-0.01447,0.00588,0.01444,-0.03854,-0.00923,-0.01432,-0.03126,0.01809,-0.027,0.00382,0.01356,0.02789,-0.0366,-0.00322,0.02273,0.0243,0.06187,-0.00738,-0.00272,0.00561,0.00393,0.00884,0.01366,0.03108,-0.0205,0.06027,-0.01459,0.0255,-0.00624,-0.00359,-0.01453,-0.01169,-0.02986,0.00779,0.02372,-0.00535,-0.04913,-0.03005,-0.01864,-0.01354,0.00707,-0.04725,0.01859,-0.02547,-0.00345,0.04528,-0.0061,0.02957,-0.01611,-0.0164,0.00776,0.00315,-0.02435,0.00793,0.00275,-0.01035,0.04708,0.03104,-0.01022,0.01459,-0.01918,-0.00259,-0.00027,0.03283,0.04818,-0.00897,0.0194,0.02869,-0.0193,0.03179,-0.00886,-0.00399,-0.00327,-0.03146,0.0226,0.03407,-0.00803,-0.01591,0.0104,0.00441,0.01206,0.03967,0.0001,0.04529,0.00385,-0.01941,0.00792,0.00537,0.02437,-0.03924,0.00384,-0.0015,-0.00262,0.00959,0.02935,-0.00134,0.01827,-0.00461,0.03875,-0.00162,-0.01193,0.00944,0.00624,-0.0104,-0.03295,0.00894,0.00809,0.04318,-0.00702,-0.01466,-0.02354,0.01424,-0.05251,0.02344,0.03607,-0.01174,-0.00011,0.01544,0.0252,0.00435,-0.01349,-0.02319,-0.00523,-0.0167,0.01901,0.01485,-0.0526,0.0259,-0.0032,0.00093,0.02076,-0.05092,-0.01554,-0.02536,0.02104,0.01162,0.00023,0.0088,-0.01122,0.01244,-0.01978,0.02227,-0.01447,-0.00839,-0.04401,0.00239,0.01777,0.00645,0.03692,-0.00432,-0.06003,-0.01845,-0.01523,-0.0032,-0.00193,-0.02384,-0.00412,0.00665,0.03707,0.02332,-0.02472,0.0241,0.02864,-0.02386,0.00955,-0.00756,-0.0189,0.02486,-0.00214,0.01333,-0.04047,0.01935,-0.00496,0.0222,-0.02679,-0.04902,0.02667,0.00415,-0.0389,0.01949,0.01466,-0.04373,-0.018,0.00469,0.02169,0.00969,-0.00309,-0.00158,-0.09387,0.00033,-0.00554,-0.01568,0.01746,-0.04217,0.01235,0.00814,0.03221,-0.04689,0.03336,-0.01271,0.01787,-0.0061,0.02106,0.02697,0.02479,-0.03092,-0.03479,0.03432,0.03362,-0.03854,0.00642,-0.02109,-0.04798,0.00218,-0.02329,-0.02013,0.04847,-0.03232,-0.0152,0.03113,0.01054,0.02739,-0.02135,0.0016,0.04102,0.03073,-0.02821,-0.04131,-0.00462,0.00761,0.0283,0.01747,0.01711,-0.03207,0.03702,0.00412,-0.00444,-0.0002,-0.0204,0.04635,0.01069,-0.00296,-0.01107,-0.01234,0.01021,0.0043,-0.00024,-0.00387,-0.02434,-0.03802,0.05236,-0.00972,0.02257,0.00759,-0.02683,-0.01889,0.03568,-0.01419,-0.05356,-0.00387,0.02209,-0.01558,0.03297,0.00147,0.01208,-0.01111,-0.01649,-0.0128,-0.00645,-0.04301,-0.00538,-0.01216,-0.00503,-0.00405,-0.0066,0.012,0.01003,-0.00587,0.0359,0.01036,-0.00305,-0.0203,0.02772,-0.03388,-0.00959,-0.02959,0.06608,-0.01707,-0.06562,-0.02508,0.00205,0.01457,-0.02255,-0.01862,-0.04173,-0.01104,-0.05871,-0.01731,0.01956,0.00901,0.03041,-0.02277,0.0314,0.02325,0.0073,0.00823,-0.03202,-0.04967,-0.00906,-0.04176,0.00441,0.02414,-0.00967,-0.04904,-0.02122,0.00708,-0.03967,0.03747,0.04841,0.01411,0.02847,-0.07179,-0.03369,0.00013,-0.01167,0.01953,0.04486,0.00667,0.02329,-0.00839,0.00163,0.02205,-0.00306,-0.03166,-0.00013,0.0016,0.01542,-0.0093,0.00317,-0.0185,0.01103,0.00243,-0.0221,-0.01382,0.03841,-0.01025,0.03243,0.04748,0.02812,0.04801,-0.01795,-0.02023,-0.01685,-0.00397,-0.01039,0.01538,0.03122,0.00015,0.00785,-0.0266,-0.00886,-0.02344,0.01748,-0.04774,-0.00803,-0.0052,0.01112,-0.02229,0.04417,-0.05154,-0.03318,-0.03903,0.01347,-0.01384,0.0043,0.0392,0.0099,-0.01692,0.00694,-0.02695,0.01503,-0.02338,0.00885,-0.04934,-0.03109,0.00466,-0.02622,-0.02637,-0.01136,0.01917,0.01419,-0.01798,-0.0077,0.00295,-0.00082,0.03929,0.06393,-0.07419,0.00379,-0.01437,-0.02097,0.00873,0.02178,0.04521,-0.01008,0.06185,-0.00903,0.02081,0.03515,-0.02343,0.00223,0.00698,0.02828,0.0422,-0.00177,-0.0121,0.0222,-0.0209,0.03805,0.00626,0.00019,-0.01636,0.00701,0.00377,-0.00986,-0.0194,0.04936,-0.00979,-0.00146,0.00205,0.00367,0.0484,-0.00271,-0.04358,-0.01463,-0.0132,9e-05,-0.01627,-0.01841,-0.03115,0.00244,0.01704,-0.00064,-0.01009,0.02535,0.0183,-0.03264,-0.00295,-0.02517,0.02384,0.01021,-0.01997,-0.02567,0.04244,0.03593,0.00836,-0.02937,0.00369,0.01779,0.0036,-0.01741,0.01599,-0.013,0.01438,0.00176,0.00636,-0.05304,-0.02933,-0.01786,-0.0053,-0.01623,0.03015,-0.00768,-0.01566,-0.05007,0.03937,0.01661,0.02181,-0.00559,0.0045,0.00254,0.02259,-0.01652,-0.00147,0.03709,-0.02394,-0.02339,-0.00774,0.04607,0.04927,-0.0085,-0.00402,-0.03426,-0.02411,0.02003,0.03691,1e-05,-0.01806,0.06749,-0.00339,-0.00404,0.01431,-0.04442,0.00957,-0.02044,-0.00778,0.00116,-0.02256,0.00168,-0.00122,0.01808,0.01618,-0.0093,0.03421,0.05409,-0.01372,0.05837,-0.01677,-0.00594,0.01018,-0.04052,-0.03636,0.00968,0.00851,0.03178,-0.01783,0.00366,-0.01836,-0.01417,-0.03087,0.05138,0.00424,-0.00818,-0.0124,0.00835,-0.01985,-0.04292,-0.00707,0.02547,-0.01558,-0.03883,-0.01828,-0.04188,-0.02404,-0.03429,-0.03673,-0.005,0.04652,-0.02023,-0.02073,0.01132,-0.03753,0.10367,-0.01392,0.00365,0.00505,-0.01234,0.0452,0.0182,-0.02897,0.001,-0.01907,-0.02827,0.03759,0.0254,0.02697,0.02507,-0.00853,-0.00011,-0.03551,-0.00583,0.03292,-0.02882,-0.00549,-0.01255,-0.02818,0.03239,-0.00709,0.00224,-0.06161,0.01088,-0.00301,-0.03073,0.00924,0.01955,0.02099,-0.01633,-0.01316,0.01657,-0.00972,-0.04725,0.01787,0.00641,0.01117,-0.0186,-0.01847,-0.01663,-0.00924,-0.06576,0.0268,-0.01076,-0.00751,0.00219,0.0188,-0.02241,0.02025,-0.05329,0.00517,0.00163,-0.02909,0.02098,-0.00692,0.00024,-0.00039,-0.01336,-0.0334,-0.01533,0.00505,0.02496,-0.006,-0.00288,-0.06114,0.02596,-0.01337,0.01373,-0.02678,-0.00389,-0.00495,0.0036,-0.01971,0.05004,-0.02295,0.04623,0.01288,0.01812,0.03139,0.01059,0.00041,-0.0327,-0.02947,0.02166,-0.01049,-0.02309,0.0003,0.03012,0.03793,0.01162,-0.02266,0.00632,0.05625,-0.01824,0.01559,-0.01305,0.0294,0.00943,0.0264,0.03134,0.02694,-0.03498,0.01704,0.00457,-0.00534,-0.00772,0.00262,0.02745,-0.01701,0.0273,0.03225,0.02959,-0.02541,0.04556,0.01693,-0.01533,-0.0309,-0.02906,-0.0065,0.05018,0.00323,0.06317,0.00531,-0.01278,0.03012,0.0056,0.01815,0.02637,-0.01787,-0.01063,0.047,0.02611,0.01675,0.01203,0.00524,-0.02393,-0.00977,0.07761,-0.01594,-0.01355,0.01888,-0.01507,0.0026,0.01414,-0.00671,-0.00655,-0.01327,0.00123,-5e-05,-0.01701,-0.00745,-0.02107,-0.01259,0.0533,0.01146,0.00355,-0.01686,-0.03744,0.02991,-0.01687,-0.01262,-0.00259,0.00558,-0.00461,-0.04931,-0.02497,-0.04481,-0.00607,-0.036,-0.00038,-0.03245,-0.04098,0.01473,-0.00961,0.00204,0.00222,0.00999,0.00235,0.00309,-0.03147,0.00846,-0.01616,-0.02094,0.02978,-0.00428,-0.00734,-0.02922,-0.12686,0.01323,0.06539,0.05557,-0.01658,0.01639,0.0234,-0.00142,-0.02922,-0.01735,-0.01772,-0.02673,-0.00186,-0.04015,-0.00892,-0.01198,-0.00982,1e-05,0.01052,0.02191,-0.01979,0.00729,-0.02763,-0.03301,0.00332,0.00993,-0.02626,-0.0176,0.01032,0.01983,-0.00696,0.00632,-0.02881,0.00721,0.01055,-0.02925,-0.01835,0.03575,-0.01476,0.01184,0.0214,-0.00958,-0.01284,0.0089,0.00237,0.02073,-0.00157,-0.01197,0.0189,0.02773,-0.02931,0.00654,-0.01712,0.03802,0.01206,0.02536,-0.01024,-0.05588,0.00605,-0.01638,0.05799,-0.00864,-0.00471,-0.02082,0.00609,0.02174,-0.00932,0.03912,-0.01407,-0.00381,-0.03054,-0.04525,0.01439,-0.02877,0.01159,-0.02117,0.00711,-0.01373,-0.00196,-0.00226,-0.03649,0.01908,-0.0291,0.02492,-0.02006,0.00566,0.02806,-0.0247,-0.00634,0.02633,0.0463,-0.01436,-0.02138,0.02069,0.00068,-0.01374,0.04636,-0.00507,0.05439,0.02903,0.05372,-0.00516,0.0203,0.03231,-0.01637,0.04143,0.04177,0.00192,0.04436,0.0157,0.01069,-0.06263,-0.02177,-0.06296,-0.04153],"RESOLUTION":[0.01966,-0.01405,-0.00958,0.04179,-0.02941,0.02967,0.01281,-0.01834,0.04105,0.0303,-0.00936,-0.01454,0.03002,0.01212,-0.01449,-0.0087,-0.03251,0.00547,0.01147,-0.01783,0.00831,0.01312,0.01775,0.01061,0.00742,0.03389,0.00609,-0.01322,0.05628,-0.02399,-0.00735,0.02604,-0.03594,0.01114,-0.00451,-0.02797,-0.02596,0.01793,0.01967,-0.00338,0.03068,-0.02361,0.05555,0.02384,-0.04428,0.00799,0.01706,-0.04345,-0.02288,-0.02005,0.02771,-0.00403,0.01047,-0.01115,0.04819,0.02975,0.00606,-0.00384,-0.01669,0.02399,0.01256,-0.00994,0.00349,0.00255,0.01008,-0.01328,-0.0039,0.00545,-0.01183,0.02323,0.00714,-0.01831,0.00491,0.03601,-0.03253,0.00446,0.0232,-0.02016,-0.02538,0.00089,0.05816,-0.00709,-0.02302,0.0191,0.03557,0.0044,0.00187,-0.0051,0.01575,0.0024,-0.01366,0.0007,-0.00798,0.01807,0.03243,0.05035,0.03053,0.01453,0.0533,-0.04661,0.03258,-0.01444,0.00804,-0.06178,-0.02109,0.00272,0.02089,0.00159,0.03406,-0.02905,-0.02231,-0.02638,-0.0144,-0.00664,0.0223,-0.03607,0.00537,-0.00356,-0.00705,-0.0077,-0.03449,0.01192,0.02624,0.01504,0.03028,0.04267,0.02088,-0.0023,0.02529,-0.0063,-0.05104,0.0058,0.01037,0.04516,-0.06836,-0.01233,0.00493,-0.01588,-0.00712,0.00053,0.0004,0.04517,0.00238,0.01494,0.01623,0.02414,-0.0405,0.01358,0.02271,-0.05991,0.01969,-0.01474,-0.01746,-0.01026,0.00902,-0.02477,0.03922,-0.00731,-0.00932,-0.01289,-0.02019,0.02172,0.01721,0.01148,-0.00973,-0.03479,0.03044,0.04159,0.03321,0.00376,0.03925,-0.0085,0.01842,-0.01086,-0.01117,-0.00174,-0.00276,-0.02325,-0.00878,0.02283,0.0315,0.01981,0.02517,-0.04854,0.00518,0.01185,0.00528,-0.01267,-0.02219,0.00416,-0.04702,0.0102,-0.03949,-0.01289,-0.03683,0.02185,0.03509,-0.02792,0.0262,0.01036,0.03395,0.00075,0.00024,0.0229,0.02539,-0.01347,0.00081,-0.00598,0.04407,0.00707,-0.02945,-0.01236,-0.00258,-0.04023,0.00852,-0.02678,-0.02139,-0.01117,0.01551,0.02591,-0.01968,0.00317,-0.00245,0.01505,-0.00791,-0.02279,0.03083,-0.03547,0.00856,0.01775,0.06692,-0.03908,0.0434,0.03803,-0.02403,0.01597,0.05689,0.02881,0.00139,-0.04271,0.01833,-0.02715,0.01352,-0.07724,-0.01871,0.019,0.01444,-0.02052,0.00075,0.01279,-0.00119,-0.03844,0.02945,0.00647,-0.02878,0.00496,-0.03042,-0.03351,0.02884,0.0132,-0.02937,0.03343,0.03505,0.04053,-0.0256,-0.01315,0.00363,-0.02488,0.00648,-0.02598,0.00139,-0.01744,0.04845,0.03035,0.01786,0.01779,0.00251,0.01895,0.0024,0.00376,-0.00981,0.04162,-0.00529,0.00635,-0.03934,0.01478,0.02516,-0.00381,-0.00254,0.00987,0.04239,-0.03281,-0.0031,0.01742,0.00863,-0.00704,0.05388,0.05068,-0.00284,-0.04437,-0.00515,-0.00301,0.02397,-0.02217,0.00951,0.02289,-0.01636,0.00227,0.03498,0.00947,-0.00426,0.02798,-0.01286,-0.01368,0.01953,-0.01294,0.01226,-0.02071,0.00046,-0.0004,-0.04739,-0.01622,0.01196,-0.00459,-0.00451,-0.00303,-0.01679,0.01202,0.00215,0.02567,-0.00328,0.00664,-0.00727,0.01399,-0.00453,0.02263,0.00176,0.01109,0.00892,0.02844,0.0138,0.03546,-0.00743,-0.03949,-0.01958,0.01177,-0.02381,-0.02132,0.0044,0.03071,0.00657,0.00314,0.00786,0.0117,0.05615,-0.01906,0.05645,0.03525,0.05103,0.03948,0.02496,0.01624,0.04428,0.05149,0.01626,-0.05714,0.01155,-0.02726,0.00569,0.0699,0.01525,-0.00339,0.0134,0.01873,-0.04034,0.01689,0.00106,0.01539,-0.00983,0.0147,-0.00893,-0.01637,0.01066,0.02525,0.00034,0.00795,-0.00709,0.01554,-0.00298,0.00441,-0.04269,-0.01922,0.02585,-0.03757,0.00031,-0.04494,0.01701,0.01221,0.00419,0.00384,0.03927,-0.07624,-0.00241,-0.00072,0.0312,-0.00717,0.02273,0.01328,0.02851,-0.0142,0.02758,0.01454,0.007,-0.02022,-0.0128,0.02117,-0.0174,0.01609,-0.00168,-0.02364,0.02595,0.03786,0.0403,0.00195,0.03412,-0.03357,-0.03032,-0.02337,0.04146,0.01844,0.04391,-0.00614,0.0523,-0.02167,-0.00124,-0.01076,-0.0325,0.02113,-0.03129,0.02401,0.01397,-0.00835,0.03567,0.03001,0.01108,-0.02848,-0.02421,0.01384,-0.02764,-0.03463,0.00543,0.00654,0.05532,-0.07379,-0.02101,0.0461,-0.0012,-0.02081,-0.00739,0.01892,-0.00335,-0.05988,0.01599,0.01896,-0.00983,-0.00319,-0.02055,0.00714,0.00176,0.05379,-0.03646,0.00916,0.01322,0.06738,-0.01375,-0.01851,-0.0122,-0.01925,0.00059,0.01131,-0.00903,-0.04118,-0.01941,-0.00606,0.00226,0.02275,0.02536,-0.03689,0.00391,-0.00467,0.00098,-0.03403,0.0033,0.00983,-0.01758,0.00811,0.00314,0.01123,-0.00378,-0.01689,-0.0084,0.04611,0.00082,0.00783,0.04306,-0.03702,-0.00232,0.02044,-0.01277,0.00626,-0.02511,0.02249,0.00796,0.0175,-0.00503,0.03904,-0.02263,0.05273,-0.01779,0.04164,-0.00344,0.02874,0.00189,-0.01216,-0.00493,-0.0337,0.01527,-0.03976,-0.00073,-0.01544,0.02457,0.0197,-0.02265,-0.00088,0.00777,0.0221,-0.011,-0.01047,0.0011,0.00612,0.05591,-0.00944,0.01253,0.00057,-0.03199,-0.00424,0.0209,-0.01443,0.02173,-0.00512,-0.00606,-0.03567,-0.05492,0.04498,0.00527,0.0403,-0.00266,0.01395,0.01798,-0.00355,0.02162,0.03902,-0.00039,-0.06322,-0.02221,-0.02512,0.00105,-0.01646,0.05443,0.02332,0.01098,0.01045,0.00018,0.00103,0.01726,-0.01997,0.014,-0.03614,0.00176,0.0003,-0.01604,-0.02905,-0.01042,0.0361,-0.00317,-0.0553,-0.0183,0.04043,-0.00712,0.00046,0.03178,-0.01811,0.0269,0.00076,-0.02179,0.03157,-0.0334,-0.00996,-0.03567,-0.01368,0.01783,-0.04262,-0.01811,-0.00966,0.02256,0.02783,-0.00797,0.01441,0.05011,-0.00907,-0.02394,0.00558,-0.00342,-0.01236,-0.00528,0.0112,-0.00515,-0.00645,0.00403,-0.01618,0.00028,0.0017,0.00794,-0.01892,-0.00116,-0.02959,0.0233,0.0375,0.04102,0.02062,0.00641,-0.01575,0.01171,0.05839,0.01648,-0.012,0.00501,-0.00072,-0.02995,0.02105,-0.00858,-0.01659,-0.01051,-0.00101,0.00311,-0.01136,-0.01798,0.01753,0.01368,-0.00089,0.00487,-0.00219,0.01111,-0.01787,-0.01956,-0.05634,-0.00602,-0.04445,0.06113,-0.03028,-0.05108,-0.01322,-0.0226,0.00634,-0.00356,-0.01119,0.02113,0.00159,0.00695,-0.02237,-0.00432,-0.01114,-0.02394,0.06215,-0.03781,-0.02089,0.00191,-0.00875,0.00804,0.02652,-0.0468,-0.00645,-0.01155,-0.05463,0.01347,-0.0423,0.00173,0.00974,0.05493,-0.04572,0.01877,0.0193,0.02047,0.07229,0.00803,0.01698,-0.00732,-0.01629,-0.00446,0.01145,0.03175,-0.03086,0.05901,-0.01793,0.00769,-0.01955,-0.00199,0.02097,-0.03763,-0.00498,-0.00562,0.00186,-0.01453,-0.03533,-0.02365,-0.04674,-0.00535,-0.0229,0.02605,0.00901,-0.04851,0.00487,0.03622,0.0015,0.00843,-0.01241,-0.02672,0.00613,-0.01981,-0.03266,-0.03665,-0.0051,-0.02707,0.02019,0.06933,0.01338,0.03636,0.01341,0.01454,-0.01846,0.0268,-0.00754,-0.05144,0.00137,-0.00946,-0.03615,0.029,-0.01533,-0.00365,-0.02373,-0.00841,0.03266,0.02235,-0.01284,-0.03085,-0.0033,0.00482,-0.00587,0.00467,0.00316,0.06082,0.01879,-0.02667,0.0026,0.00944,0.04049,-0.03633,-0.00621,-0.02181,-0.0364,0.00141,0.04192,-0.00982,0.01335,-0.00844,0.04365,-0.01516,-0.00515,0.01644,0.01732,0.01325,-0.02857,0.03153,-0.00048,0.07204,0.01311,-0.0351,-0.01634,-0.01795,-0.02922,0.02828,0.05158,0.01244,0.01353,0.06043,-0.008,-0.00364,0.00012,0.0037,-0.01296,-0.01582,0.03637,0.00273,-0.0604,0.05495,-0.01498,0.00972,0.03696,-0.02967,-0.01569,-0.02142,0.02097,0.00803,-0.02281,0.00561,-0.03474,0.01715,-0.01525,0.0325,0.00123,-0.01168,-0.02419,-0.00836,0.03551,0.01325,0.02627,-0.0117,-0.02611,0.0009,-0.02122,0.00746,-0.00919,-0.027,0.00878,-0.01508,0.03135,0.04732,-0.00147,-0.00272,0.02785,-0.02138,0.0226,0.00016,-0.01223,-0.01176,0.00904,0.01893,-0.03977,-0.00221,0.01744,0.02411,-0.01911,-0.03614,0.03799,0.00058,-0.02142,0.02343,0.03819,0.00456,-0.01453,0.00596,-0.01572,0.04006,-0.00453,-0.01183,-0.01388,0.01019,0.0016,-0.02503,0.01397,-0.02518,0.00863,0.00846,0.0164,-0.052,0.01407,-0.00161,0.01949,0.0042,0.00991,0.01534,0.01,-0.03732,-0.01407,0.01714,0.00161,-0.02541,0.00997,-0.0326,-0.03776,0.00945,-0.02328,-0.01912,0.03237,-0.00024,-0.02683,0.02665,0.00491,0.01029,0.01563,0.00267,0.03606,0.02949,-0.04314,-0.0084,0.00893,-0.00015,0.04012,-0.00324,-0.01008,-0.03508,0.02461,0.01064,0.00592,0.03044,-0.03188,0.0288,-0.00665,-0.02642,0.00289,-0.02107,0.04238,0.0379,-0.01166,-0.02166,-0.03438,-0.02728,0.03144,-0.00945,0.01411,0.01832,-0.02573,-0.0257,-0.00653,-0.0276,-0.03648,0.00506,0.01485,-0.01566,0.00546,0.01323,0.03896,-0.00583,-0.01103,-0.02026,0.007,-0.04939,0.0135,0.01141,-0.00919,-0.02113,-0.00972,0.0281,-0.01793,0.01623,0.04332,-0.00517,0.02414,-0.02695,0.0008,-0.03714,-0.01934,-0.03026,0.06305,-0.04319,-0.07109,-0.02096,-0.02519,-0.00261,-0.00222,-0.00756,-0.03001,-0.02486,-0.0641,-0.02269,0.00432,0.0221,0.02948,-0.00783,0.05191,-0.01478,-0.02836,0.03608,-0.03655,-0.05306,-0.02829,-0.004,-0.01703,0.01264,-0.00445,-0.03139,0.00569,0.02643,-0.02634,0.03439,0.04176,0.03475,0.02038,-0.04764,-0.01543,-0.01774,-0.00628,0.01652,0.0386,-0.00204,0.0143,-0.0003,0.00833,0.03431,-0.01022,-0.01453,0.00981,-0.01531,-0.00461,-0.0092,0.02104,-0.025,0.02351,-0.02036,-0.01125,-0.02437,0.01658,-0.0172,0.01077,0.02443,0.03564,0.02525,-0.02182,-0.01243,-0.02618,-0.00313,-0.00715,-0.00079,0.03023,0.01599,0.00714,-0.02279,-0.01756,-0.00044,0.03723,-0.02978,0.00378,-0.0271,0.0168,-0.02603,0.03313,-0.0475,-0.03594,-0.02861,0.02312,-0.00018,-0.00822,0.02108,0.02401,-0.03078,-0.0102,-0.01368,0.05104,-0.0251,0.01724,-0.03791,-0.00843,0.00129,-0.02108,-0.01122,0.0002,0.04929,0.01803,-0.03804,-0.01327,0.00545,-0.00049,0.04406,0.05648,-0.07464,-0.00206,-0.01717,-0.01422,-0.00782,0.01746,0.04809,-0.0252,0.05473,-0.00373,0.00898,0.01218,-0.00604,0.01878,0.01443,-0.0108,0.05088,-0.02029,-0.01289,0.0092,-0.00017,0.06023,0.0061,0.00247,-0.01537,0.04474,0.01965,0.0159,-0.00321,0.06379,-0.00333,0.00426,-0.00661,0.00579,0.02275,0.01324,-0.04197,-0.03529,-0.00677,-0.00386,-0.05026,-0.00283,-0.01991,-0.01176,0.02822,0.01195,-0.00226,0.00474,-0.00143,-0.0522,0.00677,0.01405,0.03591,0.04338,-0.01423,-0.02795,0.0385,0.02762,-0.01654,-0.0011,-0.01279,0.00749,-0.00237,-0.00029,0.03614,-0.01159,0.02418,0.01956,0.0389,-0.06587,-0.02863,-0.00198,0.00169,-0.0207,0.00958,0.00284,-0.00384,-0.06881,0.04412,-0.01056,0.03811,-0.00747,0.00806,0.00178,0.02566,-0.01416,-0.00534,0.00587,-0.02198,-0.03136,-0.00745,0.04551,0.04041,-0.01099,0.01153,-0.01354,0.01474,0.01734,0.00926,-0.03759,-0.02555,0.0635,0.01324,-0.01925,0.04267,-0.03145,0.02815,0.00997,-0.04534,-0.01551,-0.03322,0.02777,0.00411,0.00955,0.01002,0.00953,0.01174,0.05085,-0.0182,0.06495,-0.01593,-0.01895,-0.00434,-0.01407,-0.00016,0.0197,0.01417,0.05235,-0.03406,-0.00969,-0.03419,0.00821,-0.05314,0.03842,-0.02703,-0.01023,-0.00924,0.03541,-0.01122,-0.05111,0.01142,0.00399,-0.004,-0.03058,-0.00859,-0.0396,-0.0242,-0.01582,0.00349,-0.00209,0.02281,-0.01326,-0.01469,0.01114,-0.03696,0.00017,0.01652,0.01224,-0.00436,-0.009,0.05272,-0.00426,-0.03867,0.01971,-0.02649,-0.03877,0.05377,0.02139,0.02485,0.01086,-0.00202,0.00376,0.00811,-0.01773,0.02266,-0.01947,0.01715,-0.00512,-0.0078,0.04753,-0.04463,0.00474,-0.05401,0.00543,-0.02195,-0.02925,0.00671,0.01119,0.01218,-0.02507,-0.00625,0.00478,0.01037,-0.05645,0.02441,0.0228,0.04523,-0.01255,-0.04487,0.02293,-0.01271,-0.06198,0.02559,0.01372,-0.00714,0.01851,0.02496,3e-05,0.01376,-0.04555,-0.01413,-0.00559,-0.01302,0.03337,0.0031,-0.01392,0.01562,-0.00519,-0.0512,0.0007,0.00582,0.03076,-0.02069,-0.02796,-0.04203,-0.01758,-0.01929,0.02204,-0.02482,-0.00194,0.00413,0.00019,-0.01484,0.05097,-0.00556,0.02932,0.01114,0.01898,0.02636,0.01225,-0.00361,0.00233,0.00073,-0.01224,-0.0104,-0.00904,0.01792,0.05552,0.01837,0.007,-0.03313,-0.02699,0.02738,-0.01352,0.00912,-0.01278,0.01503,0.00011,0.03997,0.03142,0.02285,-0.05186,0.0478,0.04207,-0.01498,-0.01976,-0.00301,0.02178,-0.03358,0.00253,0.02432,0.01814,-0.0314,0.02327,0.01243,-0.00224,0.00548,-0.02854,0.00497,0.0198,-0.02173,0.03947,0.01656,0.02367,0.01578,0.00636,0.02746,0.01346,-0.01499,0.013,0.07007,0.04466,0.01303,0.0076,-0.0245,-0.01142,-0.00392,0.0545,-0.01801,-0.00265,0.01232,-0.01245,0.00709,0.00101,-0.01809,-0.02033,0.01099,-0.00227,0.00695,0.00789,-0.01632,-0.00395,0.03555,0.02383,0.00433,0.00961,-0.01934,-0.00575,0.01598,-0.01235,-0.01033,-0.02347,0.00337,0.01149,-0.03534,-0.03154,-0.02024,-0.00897,-0.01876,0.02309,-0.02288,-0.03079,0.0,-0.00266,0.01,-0.00687,0.00294,0.04349,-0.00741,-0.07031,0.0066,-0.00693,-0.03514,0.03856,-0.01564,-0.02835,-0.03011,0.0788,-0.02183,0.02863,0.0534,-0.00884,-0.01533,0.03979,-0.01565,-0.02717,0.00389,-0.01089,-0.03376,-0.00654,-0.02689,-0.01212,-0.01547,-0.0284,0.01698,0.00667,-0.00397,-0.03184,0.00652,-0.01813,-0.03327,-0.00498,0.0228,-0.04233,0.00529,0.00666,0.02733,-0.00163,-0.00566,-0.01905,0.00392,0.01824,-0.03443,-0.01858,0.00532,-0.0048,0.00258,0.00626,-0.00623,0.00164,0.00972,-0.0114,-0.01993,-0.02029,-0.02226,0.04098,0.01771,-0.0302,0.00089,-0.0186,0.01212,-0.0012,-0.00088,0.01659,-0.0573,0.02112,-0.00751,0.03494,-0.00517,-0.00939,-0.03634,-0.01081,0.01561,-0.00116,0.02906,-0.02869,0.00137,-0.01977,-0.041,0.03289,-0.04048,0.02096,0.01293,0.02182,-0.00809,0.02174,0.00566,-0.02859,0.01229,-0.04966,0.02092,-0.03111,0.02072,0.01276,-0.00656,0.01184,0.05524,0.0539,-0.03181,-0.00202,0.04194,0.00646,4e-05,0.04009,0.00997,0.05937,0.05114,0.04969,-0.01736,0.03318,0.01117,-0.02042,0.003,0.03961,0.02033,0.04036,0.00928,0.02312,-0.03709,-0.03078,-0.05217,-0.0704],"RETRIEVAL":[0.02105,-0.01098,0.01597,0.00906,-0.02437,0.0575,-0.0046,-0.00379,0.01378,-0.02452,-0.00084,-0.06943,0.00663,0.03479,-0.00898,0.01571,-0.03354,0.00634,0.02519,0.05418,0.012,-0.01193,0.0263,-0.01556,0.00261,0.02198,-0.0027,-0.00372,0.01175,-0.01278,-0.00601,-0.00379,-0.01745,0.04507,-0.03616,0.02385,0.00695,-0.00699,-0.0237,0.01963,0.01557,-0.00744,0.00536,-0.01524,-0.03944,0.03268,-0.0303,0.01033,-0.03855,-0.0113,0.01174,0.02741,-0.01359,0.03739,-0.01322,0.02283,-0.02071,0.01277,-0.02523,0.01038,0.03516,-0.01982,-0.02191,0.00246,0.00199,-0.02239,0.0099,0.00796,-0.00385,-0.01093,0.03211,0.05226,-0.00742,0.00392,-0.04078,-0.02,-0.01746,-0.00785,-0.02589,-0.00983,-0.00389,0.01275,-0.02143,0.0063,0.00803,0.01218,0.00971,0.00874,0.02641,-0.01514,0.02189,-0.02088,-0.00556,0.00186,0.03543,0.00099,0.02332,0.02857,0.05205,-0.0376,0.00087,-0.00882,0.00948,-0.02276,-0.01167,-0.00151,0.03841,-0.00439,0.06106,-0.04912,-0.02479,-0.02548,-0.01208,-0.04022,-0.00051,-0.01514,0.0148,-0.01954,-0.03024,0.02041,-0.00338,0.00192,0.02076,0.039,0.0459,0.02581,-0.012,-0.02873,0.00614,-0.00546,-0.0339,0.03949,0.01385,0.01711,-0.03713,-0.02817,0.00631,-0.04911,0.03167,0.0067,-0.04452,-0.00776,0.02526,0.03076,-0.02432,-0.0181,-0.00348,0.05049,0.01083,-0.00491,0.0168,-0.02356,0.01365,-0.00301,0.02205,-0.02355,0.02124,-0.00067,-0.01759,-0.00219,-0.0245,0.01354,-0.03045,0.01149,-0.03533,0.00257,0.01651,0.06044,-0.01424,0.0134,0.06038,0.01257,-0.0064,-0.02759,-0.01566,0.01828,-0.00505,-0.06898,-0.02806,0.03931,0.03113,0.03926,0.01178,-0.04692,-0.00964,-0.01144,-0.00906,0.02783,0.01904,0.02656,-0.01079,0.01896,-0.0201,-0.00156,0.00043,0.03656,-0.00547,-0.00462,0.00831,0.00473,-0.00262,-0.0175,0.02722,-0.00396,-0.04258,-0.03209,0.0046,-0.03549,-0.00644,0.05752,-0.01429,-0.00043,-0.01324,-0.03581,0.00509,-0.01391,-0.01916,0.00649,0.04567,0.02905,-0.01395,-0.02347,0.01204,-0.05564,-0.00178,-0.0124,0.02356,0.0,-0.02026,0.04443,-0.00407,0.00995,0.01768,0.01427,-0.02047,-0.00612,-0.01097,0.02917,0.02622,-0.01887,0.02755,0.02806,-0.00731,-0.02724,0.01605,0.01718,-0.01377,-0.03936,-0.00952,0.01886,0.0185,-0.0501,-0.00762,0.0221,-0.01333,0.03435,-0.00698,-0.00728,0.01593,-0.03452,-0.00985,0.00517,0.02505,0.03115,0.00092,-0.00527,0.00239,0.00387,0.02826,0.0161,-0.01771,-0.00687,0.03325,-0.01122,0.02194,0.01402,0.01557,0.02071,-0.01637,-0.02153,-0.00343,0.01773,0.05443,0.00812,0.00461,0.01854,-0.03975,-0.00115,-0.0143,0.00023,0.02208,0.01915,0.00462,0.0084,-0.01929,-0.0207,0.00557,0.01479,-0.01064,0.01067,-0.02281,-0.02737,-0.00209,-0.03962,-0.01631,0.03042,0.01056,-0.01474,0.01205,0.00236,-0.04044,-0.02462,-0.02066,-0.00631,0.01174,0.02934,0.05624,0.02132,0.01195,0.01681,0.0015,-0.00646,0.02277,-0.0129,0.00114,0.02207,-0.00344,0.0105,0.01661,0.00439,0.00934,-0.01082,-0.00658,0.04983,-0.01524,0.02045,0.00461,0.00268,-0.02583,0.04013,-0.01183,0.00691,-0.02495,-0.0239,0.00347,0.0041,-0.00592,-0.01769,-0.03134,0.0215,-0.00417,0.01954,-0.00799,0.02455,0.0322,-0.02062,-0.0135,0.03791,0.03734,0.0257,0.02611,0.01466,0.02553,0.04189,0.0374,0.0268,0.01793,-0.00547,-0.00556,-0.0104,0.0224,0.02609,0.0047,0.00667,-0.04984,0.02374,0.03389,0.05617,0.00799,0.00453,-0.01364,-0.00378,0.02043,0.00486,-0.00409,0.02429,0.02864,0.02088,-0.00503,-0.00084,0.00039,-0.00418,0.01011,0.04065,-0.00408,-0.00249,-0.0013,0.05068,0.02511,0.02937,0.00991,-0.04147,0.02446,0.00154,0.0354,0.01996,0.04555,0.01279,-0.01548,-0.00985,0.01854,-0.02435,-0.00214,-0.05589,-0.03269,0.01081,-0.00217,-0.03649,0.03771,-0.01732,-0.00409,0.0326,0.03178,-0.01545,0.02814,-0.00986,-0.02044,-0.02379,0.01526,-0.00353,0.02245,0.00826,0.03732,-0.04196,-0.02948,0.0056,-0.00416,0.03586,-0.02833,0.04321,-0.01145,-0.01699,0.07496,0.03035,0.00218,-0.02027,-0.00624,-0.01704,0.00165,-0.013,-0.01131,-0.01571,-0.01328,-0.00768,-0.04246,-0.00594,0.00135,0.00707,-0.01082,0.0097,-0.04521,-0.0221,0.01966,-0.00293,0.00084,-0.00563,0.02601,0.03418,-0.01271,0.06668,0.0113,0.06697,-0.0234,0.03198,0.03038,-0.00154,-0.00479,-0.03974,0.01857,0.03231,0.02924,0.01378,0.0074,0.02804,-0.03021,0.0223,0.01009,-0.02753,0.00207,-0.03025,0.03515,-0.00928,-0.01727,-0.00354,-0.00191,-0.00115,0.01183,-0.00062,-0.01273,-0.02304,-0.02412,0.01795,-0.00014,0.06325,0.00474,-0.05153,0.01909,0.02162,0.01941,0.02604,0.00043,0.05385,-0.01146,-0.03338,-0.0086,0.05021,0.00597,0.04294,0.02032,0.00409,-0.01539,0.01554,0.00138,-0.02295,0.04269,-0.00601,0.04675,0.00578,0.04764,0.02498,0.02488,0.03767,-0.0283,-0.01656,0.01153,0.00643,-0.01983,-0.01641,-0.02554,-0.00392,0.04482,-0.01625,-0.0021,-0.02103,-0.06157,-0.01688,-0.03661,-0.0004,0.02883,-0.03212,0.01413,0.0298,-0.00903,0.1317,0.0239,0.00779,-0.00972,-0.00432,0.0123,-0.00782,0.00453,0.04588,0.0009,-0.01818,-0.00241,-0.01371,-0.00456,-0.00953,-0.0003,0.00709,0.02833,0.0144,-0.04429,-0.01891,0.03522,0.02787,0.02056,-0.04912,0.03102,0.00183,-0.01469,-0.0192,0.00293,0.02925,0.00576,-0.00637,-0.01971,0.05076,0.00675,0.00439,0.01521,-0.01946,0.01139,-0.01248,0.01741,0.05263,-0.00893,0.01362,-0.03895,0.00073,-0.00341,-0.01764,0.00393,0.01871,-0.00073,0.03855,-0.01468,-0.01416,0.00358,0.14418,0.01576,0.03079,-0.01566,0.02429,0.03464,-0.0063,0.04026,-0.02376,0.02095,-0.04074,0.0298,0.01742,0.01172,-0.02329,-0.00354,0.00682,0.04993,0.02545,0.01628,0.02955,0.00476,-0.00936,0.05763,-0.01959,-0.02231,0.02346,0.01309,0.02928,0.01249,-0.0018,0.01505,-0.01692,0.0329,-0.00527,-0.01516,-0.00581,0.00788,0.00072,-0.00627,0.00958,-0.01539,0.0105,0.02427,-0.01048,-0.00204,-9e-05,-0.03333,-0.00397,0.01135,-0.00363,-0.04092,0.02654,-0.01316,-0.00689,0.00258,0.00258,0.00727,-0.01385,-0.02327,-0.04231,0.00188,-0.01273,-0.01568,-0.01617,-0.03843,-0.02664,-0.03208,-0.02385,0.01848,-0.03327,-0.03316,-0.03756,-0.01506,-0.0521,0.00083,-0.00763,-0.02267,0.00695,0.03517,-0.02164,-0.01713,0.01332,0.02398,0.05877,-0.01411,0.0121,0.00908,-0.02001,0.01752,0.01825,0.04919,-0.01897,0.0075,-0.02487,0.03674,-0.03038,0.02684,0.01155,0.00827,-0.03042,0.03538,0.03043,-0.01939,-0.09297,-0.01626,0.00103,-0.02913,-0.00349,-0.04498,0.02332,-0.01448,-0.00815,0.00571,-0.00054,0.01152,-0.01344,-0.02597,0.01272,0.0296,-0.01012,-0.00173,-0.00364,-0.00238,0.00516,0.00323,-0.02137,0.0163,-0.00535,0.00924,-0.02084,0.00869,0.04822,-0.0359,-0.02504,0.02537,0.00356,0.05277,-0.02636,-0.01385,-0.01518,-0.04278,0.00432,0.05883,-0.00036,-0.00158,0.00535,0.01951,0.01112,0.03266,-0.0377,0.02924,-0.01504,-0.01583,-0.00931,-0.00426,-0.0154,-0.00343,0.00819,0.0073,-0.00908,0.02332,-0.00546,-0.02428,-0.02668,-0.01241,0.00116,-0.01174,-0.0212,-0.01767,0.01633,-0.01438,-0.00467,0.00312,-8e-05,0.02705,-0.00231,0.00194,-0.00223,0.00326,-0.0528,0.02903,0.00358,-0.03014,-0.00543,0.01638,0.03674,0.00812,-0.03073,-0.0209,-0.02833,0.0412,-0.00501,-0.00542,-0.02584,0.0167,-0.00737,0.00017,-0.0118,-0.04584,-0.02034,-0.03371,0.0291,0.00134,0.00593,-0.01087,0.00748,0.00235,-0.04596,0.02408,-0.01539,-0.02236,-0.0324,0.00755,0.01448,0.01954,0.02068,-0.01319,-0.0293,-0.02252,-0.01266,-0.0271,0.01081,0.00869,-0.00016,-0.04079,0.03957,0.00112,-0.0231,0.03161,0.03774,0.00383,-0.0037,-0.02985,-0.00908,0.03245,-0.01178,0.01412,0.01072,-0.00568,0.00148,0.00196,-0.01456,-0.02287,0.04058,-0.00751,-0.02356,0.01874,-0.00536,-0.01292,-0.00756,-0.02238,0.03281,-0.00809,-0.00583,-0.00097,-0.09222,0.02591,0.00583,0.01075,0.0192,-0.01646,0.02498,0.00902,0.03568,-0.03337,0.01556,-0.0079,0.02894,-0.00456,-0.00542,0.03284,0.02842,-0.04289,0.00566,0.03336,0.00539,-0.02996,0.01145,0.01911,-0.05475,-0.01275,-0.02614,-0.03917,0.03291,-0.02508,-0.02393,0.03433,0.01864,0.02257,-0.03779,0.00033,0.02084,0.00907,-0.0343,-0.01605,0.005,0.00809,0.01568,0.00952,-0.00123,-0.00751,-0.00185,0.02714,-0.00284,0.0002,-0.03075,0.01647,0.00409,0.00284,0.01337,-0.00319,-0.01216,0.00171,0.01046,-0.01096,-0.01019,-0.01805,0.05731,-0.00028,0.0351,-0.01631,-0.02789,0.0008,0.00865,-0.03015,-0.03255,-0.01884,0.02362,-0.04193,0.02155,0.01933,-0.01923,-0.01285,-0.02119,-0.01752,0.01413,-0.04086,0.00165,-0.02206,-0.0064,-0.00325,-0.00906,-0.00297,0.0287,0.03762,0.03075,-0.00934,-0.02433,-0.00514,0.04686,-0.02287,0.0003,-0.03904,0.05429,0.00424,-0.05914,-0.01403,0.00958,0.0109,-0.03016,0.00108,-0.03521,-0.02111,-0.01879,0.01456,0.01644,-0.01104,-0.01797,-0.03949,0.01868,0.0183,0.01105,-0.00761,-0.04234,-0.00934,-0.02582,-0.02288,-0.00252,0.03817,-0.00988,-0.03718,-0.01469,0.01394,-0.03307,0.00891,0.03419,-0.01934,0.0186,-0.04455,-0.00835,-0.02756,0.00258,0.00128,0.04334,0.00491,0.02886,-0.01651,0.00102,0.02727,0.00171,-0.00091,-0.0308,-0.01288,0.02071,0.02432,0.0092,-0.00103,-0.01431,0.00136,-0.01575,-0.02465,0.02597,-0.0109,0.01401,0.03547,-0.00466,0.02411,0.00116,0.00021,-0.02541,-0.0039,0.00757,0.02536,0.02413,0.02845,0.01118,-0.01019,0.00626,-0.01988,-0.00169,-0.0346,0.03512,-0.00873,0.00894,-0.01463,0.04597,-0.01392,0.02061,-0.03463,-0.03972,-0.01676,0.01221,0.01328,0.01105,-0.00744,-0.01924,0.00267,-0.01642,-0.02841,-0.0186,-0.02734,-0.01273,-0.0271,-0.03216,-0.0164,0.00906,0.02769,-0.00602,0.0478,0.00163,-0.02189,-0.00295,0.05612,0.07691,-0.04066,0.04582,-0.00843,-0.0171,-0.00265,0.04926,0.03583,-0.00213,0.06019,-0.005,-0.00079,0.02579,0.01833,-0.00843,0.00749,0.03008,0.02641,0.01739,-0.01577,0.02194,-0.00417,0.01317,0.01531,0.00893,0.00216,0.00557,-0.00017,-0.0032,-0.00067,0.01498,-0.00507,-0.01318,0.00648,-0.00293,0.01263,-0.02556,-0.03398,0.01175,0.00154,0.00497,0.01467,0.00732,-0.03104,0.0411,0.02031,-0.00249,-0.02443,0.01642,0.01346,0.0055,-0.00783,-0.02227,0.02032,0.01766,-0.007,0.00552,-0.00604,0.01502,0.00674,-0.03375,0.00471,-0.00316,-0.01096,0.00969,-0.00358,0.01651,0.01945,0.02243,-0.03091,-0.01267,-0.0211,0.0047,-0.00775,-0.00859,0.03522,-0.03263,-0.00388,-0.02204,0.01162,-0.02235,0.01735,-0.01192,0.0161,-0.00805,0.01299,-0.02252,-0.01483,0.03096,1e-05,-0.01457,-0.04309,0.0062,0.02766,-0.03145,-0.00468,-0.01196,0.00145,0.03162,0.03088,0.01982,-0.02537,0.04358,0.0473,0.01914,0.02109,-0.02893,0.00631,-0.03475,-0.00105,0.00404,0.02133,0.00976,-0.00328,0.00581,0.02611,-0.01886,0.04799,0.02647,0.01503,0.01087,-0.03952,-0.01679,-0.00043,-0.04656,-0.05302,0.01043,-0.01245,-0.00445,-0.00915,0.00828,-0.02816,-0.01677,-0.01341,0.02978,0.03101,0.00902,-0.00611,-0.0151,0.0104,-0.02348,-0.0612,0.02711,-0.00042,0.00259,-0.01364,-0.03138,-0.02148,-0.01965,-0.03416,0.01403,0.05187,-0.00662,-0.00513,0.02352,-0.06331,0.10029,-0.00847,-0.02053,0.00054,-0.01583,0.02407,-0.01385,0.00533,-0.01997,-0.02924,-0.00902,0.05682,0.03465,-0.00023,0.01624,0.02048,0.01556,0.01555,0.00506,0.06837,-0.0292,9e-05,-0.00882,-0.02405,0.0182,0.02518,0.01987,-0.04289,0.01148,-0.00489,-0.0665,0.01503,0.01862,0.00993,-0.00843,0.00802,0.00058,0.00085,-0.02015,-0.00498,-0.02025,-0.00502,-0.01322,-0.03986,-0.00751,-0.00139,-0.0272,0.00924,-0.0249,-0.0069,-0.00896,0.01275,-0.02283,-0.01006,-0.06441,0.00067,0.02327,-0.02774,0.00331,-0.01329,-0.02007,-0.01845,-0.01254,-0.00981,-0.03177,-0.02372,0.01877,0.01262,0.02296,-0.05883,0.00953,0.00971,0.03261,-0.02631,-0.01804,0.00701,0.01225,-0.0243,0.03401,-0.01996,0.02651,0.00863,0.008,0.032,-0.03006,-0.00169,-0.03544,-0.02484,0.03733,-0.00234,0.00297,-0.00438,0.01134,0.07584,0.025,0.02424,0.00168,0.01496,-0.00708,0.01628,-0.0414,0.01772,0.04126,0.02466,0.01119,0.01556,-0.02976,0.00969,-0.01193,-0.00258,-0.00011,0.00583,0.03542,-0.00464,0.00827,0.03356,0.03,-0.00525,0.03503,0.00957,-0.00327,-0.04604,0.00163,0.00556,0.04954,0.02957,0.03818,-0.02101,-0.03325,0.01899,0.02395,0.02523,0.01271,0.00158,-0.01032,0.03073,-0.00871,0.01366,0.00747,0.00844,-0.00968,0.00922,0.06174,0.01009,0.02338,-0.01209,-0.00069,0.00588,0.01924,-0.03254,2e-05,-0.03758,0.00698,0.00705,-0.02698,-0.00125,-0.04527,-0.01832,-0.00456,-0.00971,0.02919,0.00557,-0.01183,0.00788,-0.02871,-0.01996,0.00707,0.00774,0.02029,-0.01555,-0.00571,-0.01936,-0.02468,-0.03933,-0.02025,-0.06755,-0.02308,0.01726,-0.00653,-0.02034,-0.00211,-0.0142,0.00081,0.01644,-0.00918,0.00806,-0.02687,-0.01086,0.01036,0.01423,0.00921,-0.0225,-0.21185,-0.00466,0.10856,0.11212,-0.03755,0.01114,-0.01732,0.00698,-0.05509,0.00628,-0.02834,-0.04867,0.03297,-0.02388,-0.01775,-0.00602,-0.00204,-0.01442,-0.02351,0.02637,-0.00729,0.00431,-0.00731,-0.05606,0.01084,0.01988,0.00441,-0.03413,0.04884,0.03574,0.00281,-0.03212,-0.05587,-0.00835,0.01711,-0.02549,0.01141,0.05237,-0.00927,0.00862,-0.01243,0.01474,-0.02985,-0.01305,-0.00085,0.02571,-0.00935,0.00431,0.01617,0.02148,0.02204,-0.01345,-0.01957,0.01355,-0.00118,0.02479,0.0057,-0.01019,0.00278,-0.0236,0.05869,-0.04213,-0.00796,0.02846,0.00559,0.0155,0.00789,0.01906,0.00075,-0.01,-0.00585,-0.02153,0.00614,-0.02956,0.01369,-0.02011,0.00889,-0.03813,0.01082,-0.04732,-0.01727,-0.02649,-0.01131,0.02837,-0.04112,-0.00197,0.00898,-0.023,0.00632,0.02044,0.01087,-0.00253,-0.05406,0.01631,0.01025,-0.02692,0.02905,-0.02521,0.05151,-0.00111,0.04983,0.02701,0.00925,0.05621,-0.01469,0.02894,0.04288,0.0229,0.02618,-0.00593,0.02371,-0.0179,-0.03583,-0.05808,-0.01913],"CONSTRUCTION":[-0.00105,-0.00162,0.01905,-0.03582,-0.0155,0.01936,-0.01926,0.00498,-0.02417,-0.0186,-0.01917,0.0235,-0.01734,0.01137,-0.01695,0.00421,0.00659,-0.02082,-0.00533,0.0411,0.0105,-0.03518,0.01459,-0.00902,-0.01297,-0.01088,0.02058,0.00541,-0.01977,0.03186,-0.03252,-0.01757,-0.00269,0.01016,-0.02298,0.04563,0.02425,0.01177,-0.03494,0.02967,-0.0035,0.01025,-0.04758,-0.04215,-0.00645,0.02708,-0.02746,0.05533,-0.03165,-0.00623,-0.02366,0.03431,-0.01902,0.0305,-0.05076,-0.00998,-0.05986,-0.00331,0.01946,-0.00746,-0.00538,0.00018,0.0023,-0.01849,-0.00184,0.01586,0.02013,0.01232,0.00553,-0.06007,-0.01563,0.04365,-0.00299,-0.03765,-0.03397,-0.02272,-0.05039,-0.0042,0.01716,0.00649,-0.04902,-0.02585,0.0086,-0.00477,-0.02609,0.01167,0.01464,0.00511,0.01843,-0.01782,0.01127,-0.0054,-0.01005,0.00988,-0.00125,-0.02459,0.01843,0.01587,-0.01789,0.02779,-0.01852,0.00047,0.01305,0.02071,0.02443,-0.00327,-0.00329,0.01865,0.00326,-0.00723,0.02011,-0.01581,-0.02235,0.0033,-0.03352,0.01544,0.01927,-0.02964,-0.01287,0.02001,0.02581,-0.02069,-0.01201,0.00196,0.00336,0.00129,-0.00534,-0.00539,-0.03104,0.02978,-0.02799,0.03772,0.01519,-0.03217,0.03825,-0.02681,-0.00506,-0.01942,0.05037,0.00096,0.01102,-0.01937,0.00347,0.01656,-0.03765,-0.0312,0.01744,-0.00553,-0.00971,0.01485,-0.0072,0.01198,0.0235,0.00689,0.01941,-0.01416,-0.01467,0.01834,-0.01476,-0.02478,0.00314,-0.00949,-0.02952,-0.02125,-0.01169,0.00932,-0.03797,0.02495,-0.03286,0.00285,-0.01467,0.03104,-0.03253,-0.01296,0.00962,0.0385,-0.02135,-0.01753,0.01298,0.00309,-0.02092,-0.01438,-0.00628,0.0194,-0.0002,-0.04766,-0.00042,0.04111,0.01867,-0.02223,0.04454,0.03312,0.00689,0.02635,0.05664,0.00149,-0.03683,-0.00312,-0.0321,-0.00016,-0.0391,-0.02311,0.00096,-0.03001,-0.05471,0.01603,0.00594,-0.02414,-0.03451,0.01865,0.00836,0.00829,-0.04419,0.02704,0.00324,0.00753,0.00673,0.01852,0.0033,-0.01671,0.00243,-0.05234,-0.00012,-0.03141,-0.00276,-0.00458,0.01875,0.00851,0.03343,0.04241,-0.06162,0.0107,-0.01444,-0.03649,0.02754,-0.02152,-0.05149,-0.0247,-0.00696,0.02361,-0.01702,0.04285,-0.02541,0.03038,0.02888,0.00527,-0.02006,-0.03725,-0.00417,0.00108,0.01596,-0.03425,-0.00881,-0.04448,0.0145,0.01438,0.02053,0.0023,-0.03632,-0.00846,0.0074,-0.01829,-0.00822,0.01158,0.04059,0.00861,-0.02024,0.02063,0.01347,0.0468,-0.00287,0.01606,-0.01283,-0.01624,-0.01142,-0.01684,-0.01597,-0.02795,-0.00273,-0.01403,-0.00186,-0.04774,0.03997,-0.01647,0.03274,-0.00492,-0.02488,0.0428,-0.00942,-0.02081,-0.04195,0.0475,-0.01005,-0.00138,-0.02007,-0.0346,-0.02297,-0.01079,0.0031,0.06287,-0.02682,-0.00515,-0.01534,-0.00529,-0.04619,0.05182,0.00179,-0.02177,-0.02458,-0.00731,-0.01412,-0.03563,0.00231,0.03011,0.00728,0.02939,0.02829,0.03123,-0.01065,0.0045,0.07131,0.00601,0.01805,-0.0081,-0.01071,0.02781,0.03085,-0.01588,0.01311,-0.02272,-0.00236,-0.00358,-0.02002,0.01421,-0.02865,-0.01436,-0.02244,0.02501,-0.03431,-0.00011,-0.02643,-0.03332,-0.00673,0.01043,0.0103,-0.00146,0.02523,-0.00267,-0.02443,-0.01157,0.00677,0.02567,-0.03338,0.02178,-0.0097,0.00424,-0.06912,-0.01711,0.00265,-0.02726,-0.00816,-0.00038,-0.00614,-0.0068,-0.02005,0.02587,-0.0097,0.03708,0.00053,-0.0633,0.01546,0.05033,-0.01489,-0.01048,0.02205,0.00168,-0.02514,0.00895,0.02074,0.01038,0.00543,0.0164,-0.00821,-0.00275,-0.00722,0.02609,0.02532,-0.01384,0.01396,-0.02498,0.0283,0.00379,0.00356,0.0501,0.00126,0.02117,-0.01055,0.03681,0.03676,0.00919,-0.02337,0.0397,0.05765,0.01114,-0.00628,0.00481,-0.0035,-0.01114,0.01237,-0.01687,0.03002,-0.0231,-0.0013,-0.0388,-0.01888,0.00878,0.00821,-0.01978,0.03229,-0.00692,-0.01739,-0.0147,-0.01054,-0.0011,-0.04597,0.0249,-0.02524,0.02998,-0.01003,-0.02705,-0.01134,0.02084,-0.02583,-0.02068,-0.04691,0.01519,0.00048,0.0412,0.0363,0.018,-0.00333,0.01852,-0.03736,-0.00146,0.00977,0.01651,-0.00285,-0.01233,0.02923,-0.02254,-0.00953,-0.04866,-0.06046,0.03866,-0.01954,-0.06011,-0.00941,0.01863,-0.01789,0.00703,-0.04369,0.05441,0.00783,-0.04766,0.04324,-0.01365,0.0168,-0.01644,0.0171,0.01914,0.04071,0.02704,-0.02724,-0.04456,0.03874,-0.02282,0.01908,0.01561,0.02412,0.0152,0.01632,0.06835,0.02107,0.03497,-0.0112,0.0132,0.00302,0.01834,0.0093,0.004,0.03725,0.03196,-0.02165,-0.01353,0.03483,0.01612,-0.00123,-0.01762,-0.01435,0.02206,-0.05257,-0.01924,-0.0045,0.03377,-0.02501,-0.03001,0.03896,-0.01345,0.02552,0.01413,-0.00297,0.03161,-0.00724,-0.03515,0.01341,0.02312,0.012,-0.01452,0.02956,0.0161,-0.02882,-0.0513,-0.01536,0.01592,0.04547,0.01728,0.00416,0.05218,0.03001,0.0342,-0.01885,0.01245,0.01419,-0.03982,-0.00328,-0.04294,-0.03646,0.0108,-0.04424,-0.00078,-0.01236,-0.00852,-0.01878,0.001,0.01629,-0.00936,-0.05196,0.02526,0.02276,-0.02955,0.01165,0.07154,0.05465,0.09316,-0.01806,-0.05319,-0.01613,-0.0194,0.00761,0.00608,-0.02457,-0.0137,-0.01865,0.04116,0.01645,0.01559,-0.00751,-0.00283,-0.04752,-0.01301,0.04501,0.01443,-0.03984,-0.01122,0.04787,0.03035,-0.01143,-0.01348,0.03546,0.00728,0.01517,0.00568,-0.0143,0.01288,-0.00703,0.04763,0.01013,0.03254,0.00791,0.01252,-0.02277,0.03594,-0.00536,-0.03727,0.01222,0.01557,0.04225,0.02485,-0.00768,0.01199,-0.03618,0.01745,-0.01024,0.01311,-0.01426,0.01194,-0.0285,-0.00611,-0.04193,0.08554,-0.00317,0.01453,-0.01014,0.01071,0.02847,-0.02147,0.03902,-0.01551,0.04447,-0.03347,0.04805,0.03014,-0.00506,-0.01748,-0.00632,0.03703,0.01735,-0.01552,-0.00536,-0.00518,-0.01334,-0.01416,0.00873,-0.05227,-0.05628,0.00674,0.00184,-0.02246,0.03078,-0.0098,0.00446,-0.01141,0.0,0.01986,-0.03087,-0.02945,-0.0238,-0.00468,-0.01615,0.00723,-0.02377,0.02732,0.01412,-0.00045,0.04947,0.05615,-0.03473,0.03077,-0.04546,0.01725,0.01813,-0.02788,0.00149,-0.0288,0.00734,0.01298,-0.02577,0.03007,-0.00874,-0.01289,-0.00434,-0.03765,0.0075,-0.04439,0.01072,0.0068,-0.00986,-0.0096,0.0013,-0.02205,0.03441,-0.02605,-0.01219,0.02874,-0.01211,0.01687,-0.00288,-0.01282,-0.00637,0.01433,-0.00745,-0.01705,-0.0095,-0.00656,-0.05087,-0.02985,0.00096,0.00299,0.02695,0.00933,-0.02369,0.03047,-0.01829,-0.00494,0.03269,0.0071,0.00847,-0.04177,0.03249,-0.01485,0.01204,0.0441,0.00059,-0.06042,-0.00852,0.03724,0.01779,0.01125,-0.0375,0.01941,0.01747,0.01211,-0.02406,-0.00759,-0.02066,-0.00214,0.00035,-0.00247,0.0208,0.01374,0.04289,-0.00802,0.02917,-0.01311,-0.04648,-0.0348,-0.05523,-0.01748,-0.01045,0.00303,-0.0009,0.07419,0.00881,0.02603,0.03909,0.04351,0.00567,0.0056,7e-05,0.02354,-0.02851,-0.01628,0.01067,0.00234,-0.00537,0.00744,0.00696,0.0472,0.03686,-0.02781,-0.02961,-0.02846,0.01682,-0.00199,-0.02981,-0.04477,0.0099,0.05085,0.0136,0.07636,0.03003,-0.00811,0.01483,-0.00897,0.01406,-0.02511,-0.01221,-0.00653,-0.03011,-0.00574,-0.00743,0.02447,-0.02422,-0.00863,-0.02277,-0.02904,0.04647,0.00839,0.0256,-0.018,0.00503,-0.07469,-0.03151,-0.02056,-0.05681,0.0555,0.0395,-0.02168,-0.01613,-0.01773,0.02893,-0.0354,0.0157,-0.00294,-0.02724,0.00541,-0.04267,-0.02061,-0.03644,-0.01303,0.01881,0.00021,0.03428,0.02814,-0.01532,0.05575,-0.00172,-0.0188,-0.00633,-0.011,-0.00644,-0.04017,0.0268,-0.01436,-0.00882,0.01495,-0.00865,-0.00072,-0.01157,-0.014,0.00363,0.01589,0.02895,0.00878,-0.01072,0.00077,-0.04966,0.00415,0.00327,0.0186,-0.00047,-0.00866,-0.00772,0.0221,0.01047,-0.04548,-0.00382,0.02333,0.00551,-0.0055,-0.0287,0.00911,0.0196,0.0249,0.00336,-0.00621,0.01322,-0.01826,0.01105,-0.00213,-0.01799,0.03266,-0.06633,-0.01678,-0.01456,-0.05234,0.00588,0.03917,0.05234,0.00107,0.00854,0.01698,-0.01807,0.03005,0.03903,0.03383,-0.01324,0.00999,0.01081,-0.01421,0.01188,0.01683,0.00796,0.00848,0.03455,0.00195,-0.02567,0.0184,0.03755,0.00328,-0.01445,-0.01292,-0.01112,-0.01418,-0.03897,-0.00161,0.00796,-0.00121,0.01546,-0.03409,-0.01425,0.01262,-0.01029,0.01166,0.00724,0.01297,0.00157,-0.02156,0.02012,0.00906,0.01434,-0.00431,-0.0152,0.01093,-0.01304,0.00728,0.01333,0.03447,0.01173,-0.00634,0.03114,-0.03835,-0.03444,0.0199,0.00453,0.02467,-0.00552,0.03198,0.0275,0.02295,-0.01423,-0.00359,0.02173,0.02194,0.00976,-0.02983,-0.044,0.00524,-0.00697,0.03142,-0.04509,-0.03271,0.00869,-0.01281,0.01282,0.01582,0.01182,-0.02824,-0.02243,-0.00693,0.01514,0.00199,-0.01404,0.03255,-0.0005,-0.00025,-0.024,-0.01452,0.03749,0.0248,0.01544,0.01837,-0.01569,0.0021,0.04174,0.0188,0.02636,0.00753,0.01337,-0.02223,-0.01915,-0.0156,0.01933,0.02552,0.01112,0.01353,-0.04198,-0.01218,-0.01396,-0.01999,0.01571,0.02512,-0.04446,-0.02832,0.01508,0.01056,-0.04475,0.00181,0.00071,-0.01118,-0.00534,-0.0363,-0.01636,-0.02896,-0.02992,-0.03125,-0.03801,-0.02107,0.00213,0.00817,-0.0115,0.02709,-0.00369,-0.00083,-0.00079,-0.0074,-0.02286,0.01315,-0.02448,0.00391,0.02387,-0.01413,0.01124,0.03234,0.00852,-0.00892,0.03046,-0.03816,0.04586,-0.01917,0.03499,0.02409,0.00951,0.02651,-0.01599,-0.05427,0.00547,0.01509,0.0079,0.01168,0.0058,-0.016,0.04273,-0.00377,-0.02119,-0.02433,0.03072,0.01987,-0.04613,-0.02753,-0.028,0.01962,0.01806,-0.00884,-0.01804,0.0033,0.00423,0.02118,-0.00695,-0.03256,-0.0364,0.03125,0.02778,-0.03371,0.03051,-0.00091,-0.00268,-0.04617,0.00374,-0.02393,0.00057,-0.01158,0.02225,-0.00997,0.00262,-0.00314,-0.02156,-0.02363,0.01216,-0.00131,0.00107,0.00617,-0.00984,0.05114,-0.00563,0.02528,0.014,-0.0003,0.02245,0.02056,-0.01008,0.022,-0.03717,-0.011,-0.00373,0.00495,0.01604,0.00505,0.00101,0.05733,-0.00667,0.02606,0.02809,0.01556,-0.03121,-0.05738,0.00767,-0.00087,-0.00337,-0.0597,0.02533,0.00194,0.0012,-0.03114,0.00213,-0.00437,-0.01534,-0.00762,-0.01524,-0.00304,-0.00268,0.02264,0.00888,0.00082,0.04102,-0.01775,0.02371,0.05267,-0.02961,0.00733,-0.00361,-0.00047,0.02072,0.0378,0.01097,-0.02376,-0.01339,-0.0151,0.00049,0.03661,-0.02071,-0.02265,0.01763,-0.0547,-0.02042,-0.00064,-0.0139,0.00697,-0.02945,0.0137,-0.02533,-0.00887,-0.06499,0.02124,0.001,-0.01221,-0.03119,0.01449,0.02852,-0.0253,-0.00145,0.02307,-0.02273,0.03447,-0.02433,-0.00396,0.00556,-0.00141,-0.01769,-0.00949,0.02,0.03037,0.03964,0.04358,-0.03762,0.00213,0.00486,-0.03504,-0.00714,-0.01224,-0.01078,-0.03247,0.03487,0.02959,-0.01069,-0.00897,0.00732,0.01348,-0.04046,-0.0148,-0.01315,-0.02247,0.04907,0.0345,0.01478,-0.04133,-0.01306,0.00566,0.00176,-0.02637,0.0441,0.00944,0.00731,-0.02351,-0.00706,0.00574,0.03732,-0.03341,-0.05569,-0.00721,-0.0073,-0.04238,0.01724,0.01171,0.01013,0.00396,0.03403,-0.00716,0.02857,0.02341,-0.00974,-0.03326,0.00054,0.02298,-0.04476,0.0226,0.00713,-0.01917,-0.00724,0.00249,-0.01082,-0.0159,-0.03701,0.00583,0.05833,0.00197,0.01067,-0.01491,0.00196,0.09897,-0.02451,-0.02345,-0.00191,0.00945,-0.02055,-0.01619,0.02333,-0.02393,0.00093,0.03095,0.00617,0.00268,-0.03138,-0.01955,0.00079,0.02111,-0.05005,0.00945,0.00974,-0.00023,-0.01832,-0.01665,-0.04169,-0.04523,0.06203,0.00067,0.0085,0.02554,0.02124,-0.02449,0.02219,0.01322,0.01284,0.0252,0.01579,0.01746,-0.00518,0.03597,-0.02958,0.00165,-0.0298,0.01383,-0.00336,-0.00287,-0.00984,0.01865,-0.00733,-0.04214,0.0231,-0.00692,-0.03868,-0.03567,-0.01664,-0.00374,0.02016,0.02302,-0.02027,-0.00871,0.00497,0.02169,-0.03226,-0.00158,0.01403,-0.022,-0.02254,0.00173,0.02858,0.03431,-0.01862,0.01287,0.01134,0.00459,0.00492,-0.02968,-0.00611,0.02122,-0.01488,-0.01798,-0.01158,0.00412,-0.02301,-0.01768,0.01611,-0.02379,-0.02026,-0.02235,-0.04336,0.02592,-0.01035,-0.00031,-0.03318,-0.02249,0.02172,0.0308,0.04879,0.03875,0.03827,0.00749,0.00289,-0.00318,0.01829,0.01533,-0.03614,0.00671,-0.01088,0.0208,-0.026,-0.02985,-0.00363,0.0211,0.0257,0.0067,0.0372,0.03978,-0.00161,0.01016,0.03688,0.02467,-0.01124,0.01779,-0.01416,0.0433,0.00216,0.02497,0.04181,0.03403,-0.01894,-0.06931,-0.01319,-0.00344,-0.02505,0.00196,0.01172,-0.02661,-0.02227,-0.0419,0.021,0.01868,0.00139,-0.00117,0.00553,0.03549,-0.01643,0.00131,0.02603,-0.00417,0.01986,0.02001,0.00224,0.01896,-0.04262,0.00035,-0.01518,-0.0078,0.01136,-0.02215,-0.04708,0.03846,-0.01025,-0.00582,-0.02744,-0.01999,0.02945,-0.03824,-0.01273,0.01957,0.0039,-0.01763,0.00377,-0.01669,-0.01072,-0.00705,-0.02517,-0.02203,-0.02861,-0.00221,-0.00391,0.00104,-0.02424,0.01674,-0.00426,-0.05615,-0.02312,0.03437,0.00145,-0.01437,0.02858,-0.01673,0.01628,0.01376,0.00267,-0.22559,0.05076,0.07444,0.0526,-0.05304,0.01693,-0.02971,0.01903,-0.01555,-0.00274,-0.00853,-0.01797,0.05886,-0.00932,0.02712,0.01618,0.02212,-0.02194,-0.01672,0.02813,0.00375,0.01777,-0.00308,0.00054,0.01199,-0.00796,0.04342,-0.02812,0.02279,-0.00362,-0.02062,0.03504,-0.04847,0.029,0.00318,0.00872,0.02343,0.03692,-0.00553,0.01159,0.01127,0.00818,0.00543,-0.00986,0.02065,0.05056,0.023,0.01314,-0.00424,0.01697,-0.0016,-0.02967,0.00362,0.0297,-0.00829,0.03366,-0.03356,0.03204,-0.01253,6e-05,0.03824,-0.00981,-0.00958,-0.00777,0.03695,0.02672,-0.00107,0.00147,0.00956,-0.00171,-0.02739,-0.00357,-0.02872,0.01533,0.00342,-0.04233,0.01083,0.00029,-0.03803,-0.02055,0.02596,-0.0138,0.01942,0.00088,-0.005,0.00584,0.01713,-0.02052,-0.02757,-0.02905,-0.02717,0.04704,-0.02818,-0.03787,-0.00904,-0.03105,0.00751,-0.01097,0.00853,-0.0425,-0.00889,0.0182,-0.02952,0.05733,0.01302,0.00148,-0.01865,-0.00877,-0.00112,-0.01431,-0.00085,-0.02752,0.00476,-0.02061,0.03383],"SATURATION":[0.0247,-0.00955,0.01436,0.00318,-0.02858,0.05033,-0.01051,-0.00764,0.01129,-0.01346,-0.00077,-0.05896,0.00775,0.0263,-0.02223,0.01037,-0.01283,-0.0191,0.01747,0.06068,0.02269,-0.02974,0.02366,-0.0097,-0.0052,0.0272,0.01333,0.0109,-0.00739,0.00012,-0.02284,-0.01164,-0.02565,0.03279,-0.03675,0.03335,0.00069,0.00796,-0.01881,0.02843,0.00976,-0.00355,0.0127,-0.02756,-0.04413,0.03507,-0.03273,0.02246,-0.0421,-0.00771,0.00192,0.03985,-0.00846,0.03733,-0.0275,0.00834,-0.02596,-0.00135,-0.00562,0.02293,0.01801,-0.01673,-0.01241,-0.00658,0.00045,-0.01627,0.01467,0.01368,-0.01579,-0.02553,0.0151,0.03726,-0.00348,-0.00559,-0.05492,-0.01887,-0.01818,-0.01537,-0.02819,0.00301,-0.027,0.0019,0.01054,0.00256,0.00243,0.0212,0.01171,-0.004,0.04165,-0.01038,0.01243,-0.00996,-0.01714,0.00575,0.03305,-0.00614,0.03519,0.04331,0.05227,-0.01932,0.00035,0.00328,-0.00167,-0.0286,-0.01279,0.0011,0.0274,0.00871,0.04284,-0.05389,-0.02228,-0.03746,-0.01655,-0.0235,-0.00816,-0.02073,0.02273,-0.02718,-0.02967,0.01911,-0.0003,-0.00539,0.01897,0.03338,0.02223,0.03196,-0.00329,-0.0058,-0.01299,0.01108,-0.05007,0.03706,0.0243,0.0191,-0.01898,-0.04201,-0.00863,-0.0434,0.03969,-0.00059,-0.02824,0.002,0.02046,0.03232,-0.02693,-0.0391,-0.00928,0.03275,0.00781,-0.01033,0.0168,-0.00619,0.00955,-0.00838,0.02377,-0.02629,0.01954,0.00862,-0.02344,-0.02043,-0.00664,0.00883,-0.02134,-0.01003,-0.03035,-0.00474,0.00873,0.0596,-0.0191,0.0142,0.0358,0.02817,-0.02472,-0.02486,-0.00755,0.02759,-0.00493,-0.06479,-0.01651,0.03016,0.01626,0.01965,0.01486,-0.0276,-0.00451,-0.03986,-0.00353,0.02928,0.01371,-0.00416,-0.00154,0.03435,-0.03319,0.00537,0.0049,0.02476,-0.01151,-0.02051,-0.00493,0.01938,-0.01236,-0.01338,0.0221,-0.01133,-0.0374,-0.01355,-0.0051,-0.03529,-0.00825,0.05446,-0.01817,0.00063,-0.0299,-0.00596,0.0084,-0.00391,-0.01808,0.01358,0.03175,0.01699,-0.01361,-0.03678,0.0116,-0.00952,0.00256,-0.01901,0.03802,-0.00519,0.02515,0.04739,-0.01001,-0.00567,0.01267,-0.00581,-0.02019,-0.00028,-0.01659,0.0181,0.01323,-0.00785,0.01795,0.03188,-0.00965,-0.03126,0.02211,0.02885,-0.01754,-0.05069,-0.00353,0.00362,0.03015,-0.04563,-0.00661,0.00147,-0.01985,0.02042,-0.00639,-0.01852,0.00991,-0.02412,-0.01519,0.00402,0.02517,0.03392,0.01755,0.00916,-0.01009,0.00362,0.02538,0.02708,-0.0019,-0.01947,0.02418,-0.00145,0.00376,0.00198,-0.00441,0.00443,-0.00571,-0.02102,-0.00043,0.0062,0.06483,-0.00239,0.00647,0.01312,-0.01042,0.01305,-0.02164,0.00177,0.01152,0.02139,-0.0021,-0.00364,-0.00512,-0.03038,0.01212,0.02474,-0.00773,0.04039,-0.01974,-0.02744,-0.00069,-0.02344,-0.02009,0.05427,-0.01375,-0.01565,-0.00094,-0.00413,-0.0272,-0.02618,-0.00401,0.01077,0.01805,0.02836,0.06101,0.02435,0.00356,0.0104,0.01504,-0.00812,0.01943,-0.01542,0.00582,0.02775,0.00989,0.01033,0.01739,-0.00761,0.00776,-0.01336,-0.0121,0.03868,-0.01243,0.00198,-0.00326,0.01975,-0.0293,0.05022,-0.01363,-0.00911,-0.02487,-0.01999,0.00517,0.01849,-0.00083,-0.01803,-0.02167,0.00897,-0.00136,0.02228,-0.01345,0.01919,0.04027,-0.01826,-0.01428,0.02835,0.04429,0.01023,0.0135,0.01753,0.0217,0.02687,0.01577,0.00141,0.00193,0.00671,-0.00845,-0.01534,0.02742,0.03125,0.0093,0.01324,-0.01912,0.02385,0.00406,0.04466,0.01155,0.01611,-0.0096,-0.00484,0.00898,0.01543,0.00643,0.04005,0.0349,0.01452,0.00776,-0.0118,0.00112,0.00252,0.01353,0.02581,0.01663,-0.01375,-0.00342,0.04476,0.02938,0.00868,-0.00054,-0.0267,0.03281,-0.00113,0.03091,0.01906,0.03501,0.009,0.01081,-0.015,0.04385,-0.0147,0.00698,-0.05252,-0.03558,0.01743,0.0032,-0.02753,0.02271,-0.02149,0.00879,0.02015,0.01187,0.00526,-0.00728,-0.00569,-0.03266,-0.00738,0.00187,-0.01668,0.0296,0.01591,0.03497,-0.03768,-0.04029,0.00536,-0.00245,0.04302,0.00545,0.04149,0.00755,0.01803,0.04316,0.02303,0.00857,-0.01239,-0.00195,0.00034,0.00618,-0.02693,-0.0099,-0.03153,-0.01334,-0.0082,-0.05147,-0.01136,0.00582,0.0139,-0.01984,0.01728,-0.03714,-0.01263,0.03207,-0.00692,0.02154,-0.02037,0.03126,0.02384,-0.01431,0.06187,0.00707,0.0595,-0.02032,0.01044,0.02447,-0.02005,0.00605,-0.01077,0.0102,0.02908,0.01878,0.02981,0.01097,0.02691,-0.02305,0.0143,0.01922,-0.01081,0.00047,-0.01082,0.04949,-0.00866,-0.01835,-0.00947,0.012,0.00254,0.00722,-0.00478,-0.02398,-0.00386,-0.03311,0.02682,0.01743,0.05582,0.01167,-0.06345,0.03005,0.0207,0.01511,0.02571,-0.00268,0.04928,-0.00412,-0.02169,0.00639,0.05431,0.00808,0.03592,0.0208,0.04473,-0.0241,-0.01573,-0.00903,-0.00603,0.04086,-0.01097,0.03652,0.00539,0.04286,0.03,0.0391,0.03746,-0.01791,-0.02738,1e-05,-0.01214,-0.03188,-0.01134,-0.02231,-0.00432,0.03095,-0.0157,-0.00382,-0.01026,-0.03509,-0.01161,-0.04389,-0.00016,0.02376,-0.0338,0.00948,0.04259,0.00204,0.15163,0.00415,-0.02051,-0.0242,0.00072,0.01569,-0.00063,-0.00322,0.03192,-0.01073,-0.00445,-0.00806,-0.00251,-0.00962,-0.01869,-0.0223,0.01616,0.04862,0.02513,-0.04985,-0.02586,0.05011,0.01649,0.0098,-0.05763,0.03428,0.00845,-0.00445,-0.01864,-0.01607,0.04215,-0.01101,-0.00658,-0.01208,0.0519,0.01832,0.00636,-5e-05,-0.00079,0.02298,-0.02309,0.01262,0.05312,0.01241,0.02503,-0.04539,0.00461,-0.00847,-0.00574,0.00662,0.01369,-0.0038,0.02374,-0.02483,-0.00939,0.00013,0.13415,-0.00498,0.02176,-0.0213,0.01489,0.03077,-0.01655,0.04245,-0.01232,0.04256,-0.04556,0.04036,0.02087,-0.01508,-0.03312,-0.00032,0.01725,0.02929,0.01783,0.02228,0.02466,-0.00445,-0.02476,0.03545,-0.02308,-0.02704,0.01205,0.00387,0.00857,0.01312,0.0008,0.00219,-0.01923,0.02014,0.00023,-0.02487,-0.01907,-0.00701,-0.00238,-0.00284,0.01093,-0.01264,0.02508,0.02806,0.00462,0.00957,0.0015,-0.03051,-0.00457,0.00044,0.006,-0.02671,0.00217,-0.0185,-0.01389,0.01414,0.00938,-0.01364,-0.00544,-0.00829,-0.05162,-0.00772,-0.02171,-0.02121,-0.00372,-0.02284,-0.02483,-0.02542,-0.01477,0.00726,-0.03538,-0.02024,-0.02831,-0.01523,-0.03656,-0.00311,0.00211,-0.01723,-0.00649,0.03776,-0.01724,-0.01541,0.00519,0.01639,0.04341,-0.01091,0.00363,0.00768,-0.008,0.02242,0.01587,0.02653,-0.00876,0.01955,-0.02727,0.0395,-0.01831,0.01473,-0.01101,0.01599,-0.03132,0.01634,0.04226,-0.02038,-0.08626,-0.01202,-0.00014,-0.01335,-0.0001,-0.06214,0.025,-0.01572,-0.00375,0.0114,-0.01197,0.00643,-0.01771,-0.01413,0.00017,0.01122,-0.01413,0.01535,-0.00595,-0.00676,0.01053,-0.00623,-0.03336,-0.00905,-0.00639,0.00532,-0.01511,0.01344,0.05962,-0.00739,0.00153,0.04375,0.0079,0.04845,-0.016,-0.01047,-0.00125,-0.03718,-0.00015,0.06628,-0.00277,-0.01761,0.00452,0.00757,0.02302,0.04634,-0.02956,0.01773,-0.0163,-0.01241,0.00087,-0.00258,-0.01611,-0.00424,0.03413,0.00276,0.02542,0.0348,-0.00956,-0.01594,-0.0073,-0.00356,0.00282,-0.00378,-0.01581,-0.01531,0.00195,-0.01641,-0.00058,-0.00106,-0.00629,0.01887,-0.01974,0.01349,-0.002,0.00529,-0.0548,0.03694,-0.0163,-0.03018,-0.01823,0.00471,0.0423,0.01831,-0.02979,-0.02499,-0.02773,0.02349,-0.01127,0.01248,-0.03586,0.00814,0.00565,-0.00182,0.00192,-0.04645,-0.02916,-0.0097,0.03032,0.01827,0.02065,-0.00428,0.01983,0.00311,-0.03952,0.0152,-0.02481,-0.01428,-0.04553,0.01883,0.01491,0.01045,0.01968,-0.02372,-0.03635,-0.02228,-0.00773,-0.01582,0.00247,0.00428,0.0088,-0.01342,0.02862,-0.01314,-0.01624,0.02451,0.02571,-0.00952,0.00674,-0.00684,-0.00986,0.04774,-0.03482,0.00427,-0.0,0.01019,-0.00851,-0.01151,-0.01616,-0.01322,0.04053,-0.00532,-0.03242,0.01663,0.01308,-0.01279,-0.00942,-0.01739,0.03502,-0.02065,-0.01186,-0.00566,-0.09512,0.02253,0.02135,0.02637,0.01684,-0.02572,0.02834,-0.01367,0.03718,-0.02575,0.03951,-0.00961,0.02198,0.00263,-0.00495,0.03588,0.03624,-0.03585,-0.0225,0.04458,0.02188,-0.04038,0.02227,0.01892,-0.04429,-0.01394,-0.02337,-0.02667,0.03158,-0.02412,-0.02404,0.03313,0.01864,0.01763,-0.03005,-0.00975,0.05885,0.01879,-0.02256,-0.02487,0.00143,0.0118,0.01783,0.00424,0.00038,-0.01539,0.02759,0.01348,0.00349,0.00462,-0.02605,0.02665,0.01963,0.01247,-0.00566,0.0019,-0.01273,-0.01246,0.0208,-0.00232,2e-05,-0.01602,0.06763,-0.00235,0.03475,-0.00992,-0.02552,0.00201,0.01437,-0.02732,-0.04569,-0.0195,0.01964,-0.02529,0.03098,-0.0059,-0.00749,-0.00172,-0.01456,-0.00886,0.01223,-0.0269,-0.01069,-0.02167,-0.01012,0.00772,-0.00656,0.00032,0.03508,0.01941,0.02735,-0.00013,-0.01648,0.00344,0.04072,-0.02589,0.00726,-0.02699,0.0483,0.02013,-0.0467,-0.00455,0.0147,0.01128,-0.03442,-0.00123,-0.0347,-0.01472,-0.02539,0.00441,0.01748,-0.0221,-0.00318,-0.03426,0.01565,0.02401,0.0117,-0.00738,-0.03993,-0.01164,-0.01354,-0.0444,-0.00383,0.02181,-0.00994,-0.02996,-0.0325,0.00423,-0.04041,0.00929,0.02826,-0.0265,0.01485,-0.04151,-0.01169,-0.02606,-0.00082,-0.00178,0.03338,0.00658,0.01655,-0.02537,-0.00604,0.01925,0.0039,0.00412,-0.02533,-0.00575,0.03278,0.00387,0.00403,0.00556,-0.00754,0.01758,-0.02354,-0.00093,0.03647,-0.00532,0.05289,0.02097,-0.01968,0.03508,0.00148,0.00522,-0.01412,-0.0114,-0.00038,0.03167,0.02157,0.01186,0.00102,-0.01269,0.00692,-0.02886,0.00212,-0.03991,0.01742,0.00139,-0.00586,-0.02425,0.04029,-0.02331,-0.00183,-0.03687,-0.03794,-0.01308,0.019,0.02119,-0.00903,0.0025,-0.00377,-0.00202,-0.01809,-0.01914,-0.01083,-0.03018,-0.01758,0.00537,-0.0143,-0.00114,-0.00993,0.02152,-0.00785,0.02235,0.00442,-0.00937,-0.0035,0.03533,0.10868,-0.04902,0.03542,0.00297,-0.01309,0.01367,0.03927,0.03563,0.01014,0.03478,-0.00902,0.01316,0.02692,-0.00709,-0.00559,0.01053,0.03086,0.03812,0.01286,-0.00169,0.02848,-0.01761,-0.00662,0.01147,0.00204,-0.00425,-0.02054,0.01941,-0.00506,0.00672,0.01154,-0.00704,-0.01756,-0.00707,-0.00209,0.004,-0.00356,-0.03759,0.00808,0.01296,0.00876,0.00377,-0.00949,-0.02031,0.0507,-0.01041,0.01291,-0.01503,0.01618,0.02266,0.02296,-0.00284,-0.03883,0.01202,0.01317,-0.01397,-0.00153,0.00485,0.00649,0.00431,-0.06538,-0.01553,0.00158,-0.00628,0.00738,-2e-05,0.01269,0.01218,0.00193,-0.04385,-0.02659,-0.02538,-0.00223,-0.0112,-0.00436,0.04389,-0.02867,-0.00244,-0.01704,0.00081,-0.00186,0.00375,-0.01133,0.00332,-0.01329,0.00435,-0.01939,0.01419,0.05809,0.00495,0.0019,-0.03554,0.01732,0.05067,-0.03177,-0.01253,-0.02679,0.00295,0.00782,0.04135,0.01027,-0.02299,0.04667,0.03042,0.01274,0.00026,-0.03676,0.01122,-0.02409,0.01053,0.00694,0.00246,-0.00769,-0.00149,0.00483,0.02215,-0.02256,0.04924,0.02981,0.00351,0.02351,-0.02926,-0.00787,0.03311,-0.051,-0.05709,0.00858,-0.01263,-0.01165,-0.01876,0.00616,-0.0138,-0.00186,-0.01931,0.02242,0.03057,0.01036,-0.00782,-0.01511,-0.00509,-0.02928,-0.03127,0.04893,0.00667,-0.00681,-0.01474,-0.04219,-0.03163,-0.03664,-0.05523,0.02785,0.06058,-0.00433,-0.00711,0.01323,-0.04625,0.10865,-0.02154,-0.0236,-0.01119,-0.00782,0.01138,-0.0123,-0.00025,-0.01232,-0.02089,-0.01396,0.05615,0.02722,0.00344,0.00983,0.00341,0.01988,-0.02406,-0.00158,0.04431,-0.0277,0.0031,-0.02148,-0.03433,0.00884,0.02422,0.00298,-0.04475,0.01105,0.00945,-0.05374,0.0279,0.02187,0.02123,-0.01302,0.00069,0.00845,-0.00147,-0.00673,-0.01106,0.00109,-0.02014,-0.00119,-0.01872,-0.00854,-0.01095,-0.02941,0.00668,-0.0323,-0.00109,-0.01125,-0.0017,-0.03934,0.00619,-0.04588,0.01103,0.01539,-0.03412,0.0023,-0.00123,-0.01285,-0.02064,-0.01366,-0.01808,-0.03592,-0.02446,0.0223,0.01202,0.02355,-0.05157,0.01461,0.01162,0.00999,-0.01354,-0.02085,0.0003,0.0217,-0.02312,0.03505,-0.01977,0.02754,0.00412,0.01238,0.02982,-0.02221,-0.00646,-0.04426,-0.03038,0.03626,-0.01206,-0.00992,-0.00468,0.00865,0.06038,0.03588,0.02337,0.01199,0.0362,-0.01533,0.01066,-0.00939,0.02604,0.03097,0.0129,0.02491,0.01252,-0.02999,0.00014,-0.01116,-0.00737,-0.00671,0.01939,0.03336,0.00799,0.03599,0.02786,0.03407,0.00616,0.03784,0.01391,0.00877,-0.03997,-0.00139,0.00864,0.04283,0.02369,0.05727,-0.01742,-0.04192,0.01826,0.01473,-0.00129,0.00548,0.00352,-0.01338,0.01924,-0.01222,0.0243,0.02499,0.00518,-0.00615,0.00381,0.07176,-0.00283,0.01588,0.00838,0.00603,0.0162,0.01391,-0.01846,0.00495,-0.04127,0.00315,-0.00878,-0.02823,0.01257,-0.03709,-0.02121,0.02109,-0.00524,0.0116,-0.00713,-0.03362,0.02058,-0.0453,-0.03368,0.01826,0.00351,0.00646,-0.01986,-0.02454,-0.03449,-0.01749,-0.05477,-0.02294,-0.06144,-0.0268,0.00533,-0.00593,-0.02607,0.01187,-0.01256,-0.01646,0.00815,-0.01526,0.01073,-0.02417,0.00614,0.01765,0.00974,0.00127,-0.03497,-0.26304,0.01989,0.12192,0.06265,-0.06533,0.01004,-0.00556,0.01029,-0.04417,-0.0109,-0.02422,-0.03876,0.03813,-0.04281,-0.00431,-0.007,0.00188,-0.009,-0.02356,0.03444,-0.01129,0.01359,-0.02051,-0.04514,0.01254,0.01201,0.02112,-0.02746,0.0353,0.03752,0.00401,0.00228,-0.05869,0.01235,0.02005,-0.01933,0.00658,0.05381,-0.01531,0.0311,-0.00224,0.01303,-0.02489,-0.00717,0.01487,0.02982,0.00322,-0.006,0.01823,0.03978,0.01117,-0.01808,-0.03275,0.03433,-0.0089,0.03417,-0.01882,-0.01556,-0.00465,-0.00693,0.06878,-0.02285,-0.01264,0.00937,0.01238,0.02291,0.00293,0.0175,-0.00529,-0.00807,-0.02571,-0.02641,0.00286,-0.01168,0.01799,-0.03628,0.00357,-0.02545,-0.00889,-0.03413,-0.01191,-0.01554,-0.01993,0.0299,-0.02436,0.01335,0.01347,-0.02484,-0.00334,0.01293,0.0113,0.01385,-0.0461,0.00628,0.00399,-0.02487,0.02886,-0.01072,0.05327,-0.00357,0.04358,0.02095,0.00517,0.07609,-0.00787,0.03118,0.03324,0.00576,0.03876,0.00334,0.00448,-0.03371,-0.01989,-0.06704,-0.02054],"FAMILIARITY":[0.00383,0.01107,0.02076,-0.02809,0.00829,0.01365,-0.02202,0.00805,-0.02743,-0.03196,0.00949,-0.05553,-0.02168,0.00634,0.00342,0.01778,0.01333,-0.02469,0.01064,0.07159,0.00519,-0.03077,0.00775,-0.02564,-0.01042,-0.00711,0.00074,0.0196,-0.05901,0.01975,-0.00389,-0.0322,0.01265,0.0287,-0.02644,0.04828,0.02596,-0.01272,-0.03356,0.02658,-0.0184,0.01987,-0.0457,-0.03627,0.01345,0.01493,-0.04719,0.05333,-0.01629,0.00591,-0.02577,0.04106,-0.01366,0.03623,-0.06552,-0.01623,-0.01623,0.01595,0.00679,-0.01004,0.00945,-0.00377,-0.02311,-0.00138,-0.01104,-0.00527,0.01467,0.00865,-0.00342,-0.03852,0.01082,0.05191,-0.00947,-0.02976,-0.01187,-0.01361,-0.03599,0.01166,0.00312,-0.00061,-0.06902,0.01933,0.0185,-0.01775,-0.02369,0.01128,-0.00173,0.00398,0.02037,-0.01003,0.02918,-0.00405,-0.00898,-0.00268,-0.00331,-0.03914,0.00139,0.02071,-0.00584,0.0181,-0.02825,0.01177,-0.01104,0.02839,0.0096,-0.00037,0.00787,0.00131,0.01111,-0.01388,0.00103,0.00377,0.0046,-0.01931,-0.02258,0.01157,0.00307,-0.01573,-0.01968,0.02718,0.038,-0.02132,-0.00723,0.01903,0.00545,-0.00836,-0.02753,-0.00787,-0.03089,0.01218,0.01599,0.0223,0.00694,-0.02589,0.04016,-0.01811,-0.00781,-0.02481,0.03426,0.00042,-0.0332,-0.04274,0.0173,0.00689,-0.02428,-0.04898,0.02801,0.02696,-0.01783,0.04376,-0.00631,0.00717,0.02161,0.00795,0.01505,0.00324,-0.01142,0.00399,-0.00645,-0.00188,0.00391,-0.01146,-0.03752,-0.00959,-0.0098,0.02728,-0.0178,0.00743,-0.04669,0.01186,-0.00081,0.02396,-0.0313,-0.00729,0.00405,0.02616,0.00072,-0.03531,-0.0025,0.0156,-0.00456,0.00426,-0.01164,0.02148,-0.01094,-0.03684,-0.01079,0.03644,0.03579,0.00663,0.03763,0.01518,0.00625,0.01231,0.04298,0.00377,-0.03975,0.01095,-0.02797,0.00263,-0.04259,-0.02036,0.01982,-0.02671,-0.05538,-0.00157,-0.00387,-0.0244,-0.04749,0.04259,0.01498,0.00936,-0.01397,0.02584,0.00186,0.01409,0.00107,0.01737,0.00993,-0.01089,0.00473,-0.03008,0.0104,-0.03252,0.00744,0.00928,0.0012,0.03461,-0.00356,0.02226,-0.0666,0.0337,-0.02766,-0.03551,0.00249,-0.02328,-0.05412,-0.00884,0.01052,0.02731,0.00524,0.04675,-0.01843,0.03962,0.03651,0.0017,-0.03115,-0.01449,-0.00953,-0.00695,0.01632,0.00518,-0.03057,0.00756,0.01607,0.02202,0.02131,0.02068,-0.01422,-0.03997,0.01344,-0.0297,-0.00723,-0.01621,0.02734,0.01132,-0.00671,0.02281,0.01826,0.04013,0.00014,-0.00827,-0.02547,-0.02976,-0.00628,-0.01878,0.00256,-0.00519,-0.0126,-0.02044,0.00964,-0.02927,0.05356,-0.00585,0.04038,0.0022,-0.04333,0.01659,-0.00822,-0.00357,-0.0225,0.03993,0.00472,-0.01761,-0.01007,-0.0122,-0.04128,-0.02648,-0.00591,0.06832,-0.00968,-0.02844,-0.02215,-0.00186,-0.02244,0.01715,0.01131,-0.01404,-0.03139,-0.01614,-0.02389,-0.05382,0.00188,0.01635,-0.00721,0.03283,0.05638,0.0428,0.00955,0.0028,0.0486,0.01272,0.00276,-0.00991,0.00819,0.023,0.01769,-0.00186,0.01812,-0.02257,0.00959,-0.02095,0.00694,0.02807,-0.00709,-0.01492,0.0059,-0.00131,-0.0295,0.01856,-0.02382,-0.03652,-0.01088,0.01834,0.02441,0.00265,0.01475,-0.00215,-0.02786,-0.0213,-0.01469,0.01946,-0.00832,0.00656,-0.02974,0.00132,-0.07295,-0.00475,-0.0142,-0.01732,-0.01698,0.00287,-0.02836,-0.02664,0.00075,0.0566,0.00046,0.02937,-0.00773,-0.0805,0.00972,0.02288,-0.01051,-0.01103,0.00961,0.00832,0.00885,0.02291,0.01628,-0.00606,-0.00326,0.00874,0.00156,-0.01469,0.00701,0.02044,0.03055,-0.00046,0.00797,-0.01403,0.03371,0.02104,-0.01683,0.06005,0.01302,0.03973,-0.01711,0.03918,0.01704,0.0111,-0.03486,0.03628,0.02269,-0.00117,-0.005,0.02255,0.01431,-0.00452,-0.02275,-0.00107,0.00718,-0.02726,-0.00462,-0.02192,-0.00984,-0.01047,0.01695,-0.04299,0.02416,0.00639,-0.01519,-0.02198,-0.02627,-0.00976,-0.02977,0.03036,0.01191,0.01654,-0.03202,-0.02694,-0.01791,0.02068,-0.01634,-0.01512,-0.02561,0.02179,0.03714,0.02277,0.02161,0.02177,-0.01375,0.01392,0.01807,-0.01099,-0.00505,0.0139,0.02688,-0.01279,0.02298,0.01576,-0.0168,-0.02381,-0.06409,0.06509,-0.01695,-0.0512,0.00409,0.03374,-0.00616,-0.00315,-0.022,0.04308,0.00806,-0.01589,0.02268,-0.00572,0.05167,0.01431,-0.01258,0.00415,0.04072,0.04517,-0.02584,-0.04026,0.03338,0.00876,0.01483,-0.00163,0.00212,0.01749,0.02222,0.0574,0.01998,0.02957,-0.02539,-0.01828,-0.00547,0.02282,0.00296,-0.01147,0.0384,0.02558,-0.01573,-0.01751,0.02435,-0.00314,0.00757,-0.01188,-0.02109,0.01254,-0.02407,-0.025,0.00464,0.04906,-0.02765,-0.01221,0.02514,0.00682,0.02727,0.01975,0.0288,0.0158,-0.01396,-0.03526,0.00369,0.00614,0.0248,-0.01416,0.04095,-0.02054,-0.01425,-0.02575,-0.00378,0.01277,0.04273,0.02381,0.02632,0.03776,0.03716,0.04041,0.00495,0.0103,0.00024,-0.02196,-0.0086,-0.0276,-0.01349,-0.00473,-0.0135,-0.01519,-0.02636,-0.00614,-0.01552,-0.0156,-0.01032,-0.00071,-0.04409,0.01677,-0.00844,-0.02016,0.01434,0.06253,0.05033,0.08291,0.00348,-0.05175,-0.01191,-0.0097,-0.00564,-0.00012,-0.02235,-0.00111,-0.00594,0.05244,0.01295,0.02575,-0.00428,0.00392,-0.06177,-0.0074,0.02508,0.01613,-0.03887,-0.02604,0.02329,0.04088,-0.00442,-0.01512,0.0151,0.00729,0.01468,0.00714,0.01104,-0.00217,-0.00204,0.04608,-0.00579,-0.00316,0.01906,0.00014,-0.025,0.01524,-0.00737,-0.0162,0.02848,0.01953,0.0291,0.0282,-0.00378,0.01861,-0.01499,0.03432,0.02567,0.03922,-0.02121,-0.00628,-0.00723,-0.02413,-0.04011,0.12621,0.02243,0.00913,-0.01448,0.02335,0.02112,-0.02529,0.03679,-0.00853,0.01494,-0.02553,0.02388,0.01634,-0.01306,-0.00387,-0.00213,0.03535,0.0073,-0.0235,-0.0167,0.00224,-0.00361,0.00181,0.02511,-0.06809,-0.02808,0.0209,0.00811,0.02028,0.03902,-0.0274,0.00706,0.00545,0.03097,-0.00827,-0.01522,-0.00707,0.01473,-0.01734,-0.02387,0.00428,-0.01525,0.02061,0.00975,0.02174,0.02255,0.05667,-0.02488,0.03558,-0.04629,0.03047,0.01994,0.03044,0.00891,-0.01841,0.0032,0.01156,-0.02455,-0.00655,-0.01304,-0.02927,-0.00582,0.00216,0.00345,-0.06341,0.01114,0.00137,-0.03779,-0.00608,-0.00078,-0.0457,0.02018,-0.01876,-0.00286,0.01415,-0.00988,0.03874,-0.01384,-0.00723,-0.02777,0.02261,-0.03318,-0.0022,-0.00059,-0.02385,-0.01678,-0.0099,0.01131,0.00701,0.01789,0.00605,-0.00099,0.0134,-0.03895,-0.00371,0.02128,0.00057,0.01421,-0.02295,0.04254,-0.02485,0.02531,0.02197,0.00085,-0.03148,-6e-05,0.0401,-0.01017,0.02937,-0.08356,0.01067,0.03638,-0.01082,-0.01681,-0.00807,0.00707,-0.00186,0.01474,-0.00093,0.03692,0.02304,0.04242,0.00161,0.0236,-0.00733,-0.0717,-0.03757,-0.03295,-0.02032,-0.00564,0.00527,-0.01414,0.06017,0.04612,-0.00634,0.04479,0.03134,0.01098,0.00331,-0.00105,0.02108,-0.02844,-0.02933,0.02746,0.00226,0.02164,0.01406,0.00651,0.02025,0.02927,-0.02235,-0.03738,-0.03017,0.01599,-0.00177,-0.00879,-0.04366,0.02705,0.02034,0.02526,0.04736,0.02006,-0.04265,-0.00468,-0.02312,0.00464,-0.03184,0.01317,-0.01621,-0.0262,-0.01295,-0.03036,0.01421,-0.02297,0.00298,-0.04965,-0.02642,0.03885,0.01403,0.02914,-0.01968,-0.00099,-0.04646,-0.03794,-0.02916,-0.05375,0.04176,0.01377,-0.02798,-0.0265,-0.00977,0.03603,-0.03902,0.00685,0.02761,-0.04015,0.0166,-0.01708,-0.03958,-0.01205,0.00356,0.00201,0.00244,-0.00283,0.03465,-0.00906,0.04412,-0.01466,-0.01859,-0.00942,-0.01989,-0.00315,-0.01343,0.01935,-0.0209,0.00146,-0.0063,-0.00456,-0.00102,-0.02328,0.01045,-0.02903,0.01367,0.02335,-0.00864,-0.00331,-0.00348,-0.04355,-0.01816,0.03106,-0.00473,0.01011,-0.01975,-0.01009,-0.00317,0.05209,-0.02673,-0.0163,0.03155,0.00449,-0.02141,-0.02236,0.00097,0.01128,-0.00909,-0.00514,-0.01025,-0.00607,-0.0346,-0.02252,0.01073,-0.01715,0.04133,-0.05028,-0.00071,-0.00017,-0.06271,0.00908,0.00139,0.03689,0.00331,0.00186,0.01708,-0.00965,0.01702,0.02186,0.01621,-0.00848,-0.00158,-0.00889,-0.00657,0.01946,0.01834,0.00287,-0.00223,0.02311,0.01977,-0.00565,0.00692,0.03821,-0.00611,-0.01558,0.00606,-0.00921,0.00381,-0.0239,0.00521,0.00328,0.01054,0.00588,-0.04723,-0.00688,0.01436,-0.00787,0.01867,-0.01029,-0.01283,0.00363,-0.02018,0.01119,0.01332,0.02317,-0.00645,0.00329,-0.01274,-0.0294,0.00938,3e-05,0.00868,0.03721,-0.00352,0.02376,-0.05013,-0.04659,0.01924,0.02176,0.02324,0.00627,0.02998,0.00791,0.00945,-0.02895,0.00572,0.02705,0.01544,0.00568,0.00143,-0.02204,0.00047,-0.00352,0.02247,-0.0055,-0.0503,0.00225,-0.0108,0.00996,-0.00097,0.02102,-0.01862,-0.03349,0.00025,0.02087,0.01114,-0.03216,0.04294,-0.00364,-0.01821,0.00242,-0.04753,0.02365,0.04038,0.01083,0.02199,0.00519,-0.0092,0.05259,0.02031,0.01613,0.03612,0.00982,-0.02141,0.00052,-0.00755,0.01574,0.03612,0.02988,0.01058,-0.0389,-0.03588,-0.02668,-0.04238,0.03979,0.03912,-0.03964,0.00519,0.03739,0.01493,-0.03616,0.01969,0.00796,-0.00045,-0.00102,-0.02965,-0.02101,-0.00404,-0.02308,-0.00919,-0.05056,-0.00415,-0.00016,0.01003,0.00303,0.0047,-0.01658,-0.00934,0.00947,0.00743,-0.01952,-0.02308,-0.0172,0.01278,0.00924,-0.03003,0.01132,0.02542,0.02145,-0.01287,0.02473,-0.02284,0.02776,-0.00151,0.0122,0.01726,0.01514,0.02437,0.00483,-0.03894,0.00549,0.01869,0.00317,0.01063,-0.0099,0.01208,0.02204,-0.00855,-0.00164,0.00697,0.0031,0.02062,-0.01927,-0.0309,-0.00158,0.00893,0.02483,-0.01598,0.00528,0.00735,0.02558,0.03356,-0.00653,-0.04983,-0.01269,0.02039,-0.00023,-0.01695,0.02647,0.00713,0.0079,-0.06167,0.0027,-0.0275,0.00918,-0.01518,-0.01028,0.00424,-0.00177,-0.0079,-0.03018,-0.02339,0.06629,0.01416,-0.02056,-0.00127,-0.00666,0.02765,0.02719,0.02805,0.01493,-0.00692,0.01379,0.01287,-0.01539,0.02641,-0.00654,-0.002,-0.00318,0.02261,-0.00206,-0.02789,-0.00617,0.03792,-0.02241,0.031,0.005,0.01763,-0.01159,-0.05006,0.00691,0.00562,0.01228,-0.05311,-0.01587,-0.01775,0.00199,-0.05498,-0.0023,-0.01835,0.00616,-0.01022,-0.00448,-0.02778,0.00502,0.04346,0.01191,0.00962,0.05477,-0.0026,-0.00374,0.04885,-0.02294,-0.00853,-0.01449,0.01814,0.01771,0.05405,-0.01627,-0.04275,-0.02229,-0.03347,0.00048,0.02196,-0.02669,-0.00359,0.02306,-0.04371,0.00854,-0.00041,-1e-05,-0.00036,-0.03143,0.01952,-0.0084,-0.01408,-0.07049,0.04756,0.00532,9e-05,-0.01056,0.01198,0.0285,-0.02173,-0.00293,0.04629,-0.03751,0.00655,-0.0283,-0.00127,-0.00298,-0.01349,-0.01508,-0.00728,0.01136,0.03634,0.01646,0.02075,-0.02253,-0.03104,-0.00348,-0.00752,-0.01875,-0.00598,-0.01988,0.00621,0.02699,0.05334,0.00155,-0.0176,0.00684,0.03293,-0.03348,-0.00094,-0.01732,-0.03383,0.05256,0.02248,0.0363,-0.03164,-0.00097,-0.00377,0.01371,-0.02435,0.03254,-0.027,0.02631,-0.03798,-0.01086,0.00856,0.01462,-0.03692,-0.04427,-0.01184,-0.0309,-0.05383,0.02025,0.01851,0.01702,-0.0177,0.03638,-0.0141,0.05226,0.01251,0.00073,-0.04809,0.00484,0.02295,-0.05221,0.0283,0.00798,0.03175,-0.00351,0.00731,-0.00386,-0.01297,-0.03647,0.01977,0.02885,0.01106,0.0051,0.01145,-0.00292,0.10054,-0.0391,-0.02829,0.00478,0.00297,-0.0265,0.00545,0.036,-0.03042,0.00928,0.0246,-0.00862,0.00658,-0.01781,0.0009,0.00594,0.00483,-0.01516,0.01437,0.03039,-0.00534,-0.01524,-0.01146,-0.01889,-0.03349,0.05829,0.00325,0.01769,0.00394,0.02641,-0.01874,0.00761,0.00862,0.00235,0.01954,0.0059,-0.00257,-0.01673,0.0398,-0.0268,-0.02909,-0.05816,0.00525,0.02404,-0.0373,0.01241,0.03748,-0.02079,-0.03157,0.00152,-0.03237,-0.01828,-0.0281,-0.01065,-0.00893,0.02185,0.0179,-0.00891,-0.03481,-0.00476,0.00022,-0.03204,-0.01203,0.0393,-0.02278,-0.02046,-0.01321,0.03143,0.04935,-0.00455,0.02766,0.02391,-0.00698,0.00552,-0.00919,-0.00547,0.0105,-0.00789,-0.01919,-0.0128,-0.00163,-0.00222,-0.007,-0.00027,-0.02828,0.00089,-0.03792,-0.02409,0.04457,0.00494,-0.00253,-0.02073,-0.04392,0.03942,0.01534,0.05063,0.03803,0.00098,0.00027,0.00562,-0.00293,0.00602,0.03109,-0.02374,-0.01376,-0.0148,0.02242,-0.0409,-0.04717,0.01534,0.01725,0.01614,0.00506,0.03088,0.02205,0.00764,0.00831,0.03006,0.00734,0.00065,-0.00172,-0.04545,0.02124,0.00027,0.02288,0.03794,0.00273,-0.02823,-0.05444,0.00536,0.00739,-0.01994,-0.00332,0.01339,-0.02544,-0.0453,-0.03919,0.00447,0.01588,0.03309,-0.00453,0.01145,0.00738,0.01289,0.00933,-0.01389,0.01476,-0.00138,0.01296,0.00109,0.02186,-0.04103,0.00496,-0.00596,-0.03159,0.0192,-0.02725,-0.05954,-0.00691,-0.01031,0.00434,0.02081,-0.01872,-0.00442,-0.0214,-0.01545,0.0419,-0.00045,-0.00331,0.01501,0.02227,-0.02078,-0.00765,-0.02545,-0.04011,-0.03565,0.00321,0.01595,-0.00422,-0.02029,0.01222,-0.01135,-0.05456,0.01925,0.05878,0.00393,-0.00748,0.02821,-0.01909,0.02031,0.03116,0.00302,-0.28383,0.02327,0.0743,0.03052,-0.03307,0.03246,-0.03809,0.01835,-0.01109,-0.01014,-0.01497,-0.00127,0.03544,-0.00625,-0.0004,0.00076,0.03045,-0.02199,-0.02269,0.02946,0.02091,0.0009,0.00415,-0.0129,0.01549,-0.00706,0.04613,-0.03644,0.02509,0.0029,0.00246,-0.00884,-0.0297,-0.00398,-0.00174,0.0082,0.01526,0.04004,-0.01105,0.01748,-0.00592,0.01616,-0.02789,-0.01096,0.01808,0.04301,0.01666,0.02253,-0.02544,0.00752,0.03749,-0.00895,-0.00251,0.01321,0.00219,0.02728,-0.02119,0.0383,-0.02637,-0.00847,0.02288,-0.01496,0.00298,0.05301,0.02072,0.00694,-0.00244,-0.01115,0.02454,-0.00596,0.00085,0.01735,-0.03051,0.02054,-0.00721,-0.04386,-0.01554,-0.01742,-0.0207,-0.04231,0.01231,-0.02223,0.03015,0.00281,0.00814,-0.01469,0.0062,-0.01077,-0.02028,-0.03953,-0.03747,0.03476,-0.04006,-0.02796,0.00215,-0.02192,-0.01867,-0.02789,-0.01288,-0.05025,-0.00398,0.0358,-0.02018,0.03872,0.01242,0.03438,-0.00525,-0.0223,-0.00252,-0.00435,-0.00796,0.01307,0.00708,-0.00082,0.04615],"NOVELTY":[0.0256,-0.00931,-0.00422,0.02718,-0.03611,0.03376,0.00028,-0.02329,0.041,0.01882,-0.01741,-0.07531,0.02681,0.0181,-0.0217,-0.00346,-0.01585,-0.02025,0.02221,0.00292,0.02011,-0.01548,0.0206,0.00527,0.00683,0.03606,0.00603,0.01094,0.02992,-0.01487,-0.02498,0.01608,-0.04666,0.01794,-0.01173,-0.00534,-0.02991,0.03809,0.00846,0.00771,0.02458,-0.0145,0.05926,-0.00068,-0.0426,0.03103,-0.0039,-0.02357,-0.02567,-0.01558,0.02567,0.02784,-0.00114,0.00167,0.02367,0.03029,-0.02764,-0.01069,-0.01,0.02633,0.01665,-0.00215,0.00789,-0.00325,0.00747,-0.0099,0.01347,0.01531,-0.02322,0.00394,0.00577,0.00317,0.00499,0.01884,-0.04352,-0.01264,0.01542,-0.01806,-0.0356,0.00166,0.01308,-0.00758,0.00587,0.01284,0.02543,0.00826,0.01421,-0.01466,0.03798,-0.0006,-0.00025,-0.00747,-0.01088,0.01904,0.03726,0.03205,0.03809,0.03523,0.06185,-0.03719,0.03246,-0.01885,-0.00082,-0.06002,-0.02818,0.00247,0.0292,0.01906,0.03743,-0.03679,-0.03476,-0.02762,-0.03578,-0.01079,0.0081,-0.03688,0.01588,-0.01983,-0.0145,0.01004,-0.00909,-0.00387,0.02808,0.01956,0.01641,0.05041,0.01929,0.0009,-0.00269,0.00905,-0.05709,0.02627,0.01537,0.04677,-0.04679,-0.03027,-0.01792,-0.03553,0.02326,0.00505,-0.0117,0.02735,0.0127,0.01872,-0.00525,-0.00974,-0.02984,-0.00592,0.0205,-0.05123,0.02197,-0.01191,-0.0138,-0.01177,0.01865,-0.03399,0.04046,-0.00475,-0.01572,-0.03062,-0.01222,0.01954,0.00705,-0.01126,-0.01229,-0.03373,0.01779,0.0512,0.01878,0.00996,0.04027,0.01152,0.002,-0.0055,-0.00592,0.01055,-0.0004,-0.0455,-0.00879,0.02598,0.03492,0.02452,0.01695,-0.03587,-0.00039,-0.01538,-0.00309,-0.00501,-0.00432,-0.00903,-0.02825,0.02406,-0.04659,-0.00811,-0.03051,0.03023,0.02824,-0.03789,0.01361,0.01665,0.01466,-0.00515,0.00477,0.01073,0.00984,0.00322,-0.00886,-0.01715,0.01801,0.02966,-0.02405,-0.01952,-0.01897,-0.02982,0.0073,-0.01327,-0.02294,0.00347,0.02129,0.02234,-0.01586,-0.02108,0.00245,0.01917,-0.0118,-0.03277,0.04562,-0.02875,0.04323,0.03236,0.04872,-0.02657,0.04145,0.01627,-0.02099,0.00782,0.03523,0.02762,-0.00554,-0.03403,0.02394,-0.00893,0.0063,-0.07348,0.00172,0.03507,-0.00042,-0.04261,-0.00697,0.00451,0.01591,-0.05663,0.01675,0.01122,-0.02201,0.00178,-0.02607,-0.03254,0.01595,-0.00488,-0.0342,0.02844,0.03453,0.04093,-0.00182,0.00123,-0.00662,-0.02063,0.01787,-0.01394,0.00555,-0.02329,0.04312,0.03114,0.0164,0.01889,-0.0082,0.00205,0.0009,-0.01099,-0.00862,0.01656,0.03128,0.00854,-0.02893,0.01738,0.00772,0.00787,-0.00431,0.01319,0.03622,-0.01236,-0.00907,0.00515,0.00925,-0.02216,0.04525,0.0623,0.00263,-0.00668,-0.0112,-0.01548,0.02618,-0.01847,-0.00709,0.05078,-0.02711,0.01478,0.01865,0.00029,0.0033,0.01295,-0.00887,0.00215,0.02459,0.00734,0.05032,-0.00789,-0.00139,0.00083,-0.02986,-0.01419,0.01516,-0.0191,-0.00024,0.00209,-0.01297,0.00717,0.0004,0.01619,-0.00643,0.00371,-0.02272,0.02349,-0.0025,0.01473,-0.01108,0.02967,-0.01106,0.04224,0.00947,0.01554,-0.02413,-0.04529,-0.01064,0.02351,-0.01913,-0.01277,0.00142,0.02201,0.00667,0.0163,0.00358,0.00748,0.0657,-0.02436,0.02382,0.03718,0.0629,0.04036,0.01667,0.0089,0.0405,0.03465,0.00083,-0.06026,0.00242,-0.01241,-0.00349,0.03995,0.0167,0.00275,0.01269,0.01891,-0.03148,0.01266,-0.01195,0.01891,0.00134,0.02907,-0.00572,-0.00932,0.00308,0.0209,0.01821,0.02295,0.0154,0.01509,0.01087,-0.00259,-0.02243,-0.00943,0.00671,-0.01635,0.01064,-0.04092,0.02305,0.01411,0.01436,0.00661,0.03128,-0.06389,0.01795,0.00999,0.03294,-0.00759,0.02939,0.01125,0.03269,-0.0062,0.05809,0.01597,0.01122,-0.0416,-0.02885,0.02138,-0.01049,0.00197,-0.00418,-0.01859,0.02467,0.02958,0.02561,0.00498,0.02721,-0.02083,-0.04654,-0.02453,0.01983,-0.0057,0.04451,0.0089,0.05929,-0.02247,-0.01419,-0.00373,-0.02523,0.03348,-0.00306,0.0526,0.0157,0.00846,0.02902,0.02684,0.01455,-0.03004,-0.01333,0.00314,-0.0141,-0.04008,-0.00819,-0.01048,0.03326,-0.05611,-0.04007,0.03305,-0.00186,-0.01413,-0.01292,0.02507,-0.01941,-0.04907,0.02715,0.01217,0.00343,-0.00286,0.00783,0.00689,0.00547,0.06475,-0.02845,0.03752,-0.0014,0.04105,0.00675,-0.02232,-0.00018,-0.00107,0.00391,0.01866,0.00161,-0.01993,-0.009,0.0034,-0.01433,0.02136,0.02917,-0.0243,0.00308,-0.007,0.02758,-0.03022,0.00509,0.0046,-0.00438,0.01533,0.00432,-0.00288,-0.02304,-0.00591,-0.02245,0.04712,0.01872,0.01319,0.04461,-0.03944,0.01069,0.01205,-0.00112,0.01557,-0.02246,0.03648,0.01059,0.00614,0.01217,0.03917,-0.01445,0.05285,0.00346,0.07121,-0.05118,0.00981,0.00613,-0.00988,0.0062,-0.02988,0.02285,-0.03475,0.01071,0.0012,0.04028,0.04047,-0.0049,-0.01391,-0.00938,0.01029,-0.03576,-0.01261,0.0007,-0.00466,0.04815,-0.01689,0.00959,0.00681,-0.02326,0.00337,0.00167,-0.01265,0.02658,-0.01904,-0.00688,-0.00404,-0.03653,0.09444,-0.00031,0.01253,-0.02066,0.01427,0.02635,0.00062,0.01615,0.04583,-0.01445,-0.03927,-0.01438,-0.01794,-0.01519,-0.01905,0.02622,0.02601,0.04503,0.0129,-0.01331,-0.0125,0.04942,-0.01638,0.00623,-0.06124,0.02786,0.01105,-0.03953,-0.02964,-0.02337,0.05441,-0.00785,-0.0301,-0.02104,0.04054,0.00522,0.00137,0.03066,-0.00651,0.03818,-0.00167,-0.00681,0.05041,-0.01728,0.01037,-0.0456,0.00037,0.01097,-0.02624,-0.003,0.01749,0.0111,0.03198,-0.02122,0.01275,0.03707,0.06997,-0.03614,0.00234,-0.00479,-0.00685,0.01226,0.00742,0.01477,-0.00435,0.03476,-0.03413,0.02598,0.01352,-0.00673,-0.04275,-0.0028,-0.01158,0.02864,0.03277,0.03664,0.01519,-9e-05,-0.03627,0.02172,0.03351,0.00234,0.00255,0.00886,-0.0003,-0.02012,0.00167,-0.01055,-0.00386,-0.00261,-0.00703,-0.01196,-0.02151,-0.02714,0.00641,0.00982,0.00684,0.00129,0.01141,0.0264,0.00346,-0.00087,-0.03973,-0.01645,-0.03019,0.04056,-0.02146,-0.0487,-0.04281,-0.02851,0.00805,-0.01564,-0.01497,0.00198,0.0031,0.01511,-0.04004,-0.01131,-0.02114,-0.03598,0.06149,-0.02884,-0.03866,-0.02078,-0.02433,0.00737,0.01952,-0.0328,-0.0085,-0.01236,-0.04659,0.00917,-0.01738,-0.00886,0.00457,0.05826,-0.03991,0.00182,0.01089,0.03386,0.06917,0.00044,0.01154,0.00494,-0.0071,0.00141,0.00493,0.02843,-0.01333,0.06162,-0.02149,0.02468,-0.01191,0.00502,-0.00465,-0.02271,-0.01517,-0.0016,0.03013,-0.00959,-0.04765,-0.02694,-0.03723,-0.00046,-0.00996,-0.0251,0.0101,-0.04335,-0.0059,0.04598,0.0002,0.01444,-0.00861,-0.02589,-0.00334,-0.0234,-0.0386,-0.00536,-0.01208,-0.01902,0.02897,0.04525,-0.00501,0.01213,0.00434,0.00568,-0.01603,0.02877,0.01731,-0.01407,0.01598,0.00997,-0.01648,0.04272,-0.01834,-0.00172,-0.00344,-0.01916,0.01778,0.02969,-0.00753,-0.03975,0.0124,-0.00064,0.01248,0.02948,-0.00948,0.04539,0.01743,-0.02458,-0.00651,0.0075,0.02293,-0.04085,0.02132,-0.01228,-0.00056,0.01688,0.03425,-0.01609,0.00699,0.00968,0.03582,-0.01204,-0.00705,0.01602,0.00488,-0.01506,-0.0206,0.00656,0.00023,0.05485,-0.00492,-0.01681,-0.01728,-0.0078,-0.0442,0.04573,0.03732,-0.00036,0.0114,0.02325,0.01125,0.0154,-0.01996,-0.00487,-0.01734,-0.01121,0.02587,0.02258,-0.05327,0.04454,-0.01086,0.0036,0.03523,-0.04533,-0.01832,-0.02366,0.02903,0.02222,-0.00786,-0.0011,-0.01031,0.01724,-0.01856,0.02392,-0.01638,-0.00885,-0.04274,0.00984,0.03162,0.00243,0.03681,-0.02279,-0.04819,-0.00717,-0.02278,0.01421,0.00031,-0.02488,0.00554,0.00477,0.03335,0.01785,-0.00669,0.00434,0.02325,-0.02243,0.01425,0.01097,-0.00564,0.02897,-0.01048,0.01103,-0.02971,0.00779,-0.00562,0.01846,-0.02419,-0.03404,0.04771,0.0002,-0.03295,0.01811,0.03678,-0.01251,-0.01276,0.00543,0.0102,0.00991,-0.01138,-0.01373,-0.05822,0.01551,0.0192,-0.00083,0.00574,-0.03314,0.0159,0.0202,0.03027,-0.04419,0.03627,-0.00429,0.01778,0.00992,0.00526,0.02705,0.02435,-0.04522,-0.02892,0.02212,0.01586,-0.03617,0.00959,-0.02341,-0.02822,-0.00874,-0.02118,-0.02202,0.03318,-0.00902,-0.02823,0.03213,0.01783,0.02378,0.00312,-0.00306,0.05294,0.02874,-0.0414,-0.03128,0.00566,0.00066,0.04811,0.00528,0.00605,-0.02868,0.03346,-0.00162,-0.00906,0.01578,-0.03198,0.03655,0.01512,-0.00908,-0.01622,-0.01968,0.03111,0.0223,0.0065,-0.00726,-0.02653,-0.02628,0.05104,-0.01705,0.02082,0.01388,-0.02905,-0.01475,0.01916,-0.02559,-0.05152,0.00524,0.01834,-0.01052,0.02698,0.00306,0.0242,-0.01553,-0.01357,-0.01983,0.00656,-0.05325,0.00014,-0.00067,-0.00655,-0.00686,-0.00299,0.02273,0.00343,0.01458,0.04076,0.01279,0.00971,-0.01774,0.02319,-0.02737,-0.0121,-0.02862,0.06976,-0.00707,-0.07712,-0.0169,0.00303,0.0065,-0.02389,-0.01254,-0.03616,-0.01891,-0.06968,-0.0177,0.00934,0.01354,0.02868,-0.02268,0.03535,-0.00382,-0.01519,0.02492,-0.03215,-0.05452,-0.0144,-0.03259,-0.01834,0.02075,-0.0152,-0.02997,-0.02301,0.01737,-0.04431,0.02546,0.05235,0.01427,0.01817,-0.05522,-0.02001,-0.01502,-0.00897,0.01681,0.04925,0.00914,0.02096,-0.01441,-0.00541,0.03569,0.00468,-0.0165,-0.00696,-0.01086,0.00526,-0.01329,0.01718,-0.01676,0.01885,-0.00258,-0.02223,-0.00717,0.02586,-0.00811,0.04289,0.02633,0.00718,0.04385,-0.01527,-0.00283,-0.01989,-0.00781,-0.00916,0.00905,0.04093,0.01242,-0.00265,-0.01662,-0.00683,-0.01903,0.02586,-0.03297,0.00281,-0.00773,0.0059,-0.0355,0.04106,-0.04531,-0.03292,-0.03542,0.00427,-0.00924,0.00276,0.02628,0.0069,-0.00848,0.0058,-0.0142,0.03309,-0.02344,0.01745,-0.02895,-0.01075,0.01133,-0.01432,-0.00558,-0.00866,0.04276,0.00115,-0.00702,-0.00556,0.00076,-0.00664,0.03596,0.05742,-0.07684,0.00643,-0.0102,-0.01367,0.00171,0.02887,0.04626,-0.00411,0.06014,-0.00406,0.02333,0.02607,-0.01959,0.0107,0.01335,0.00493,0.05567,-0.00998,0.00043,0.00451,-0.02612,0.03615,0.0032,-0.00819,-0.01328,0.01366,0.02824,0.00734,0.00331,0.04833,-0.01432,-0.01172,-0.02206,-0.00706,0.00998,0.00821,-0.04932,-0.03669,-0.00108,0.00439,-0.02301,-0.00078,-0.02002,-0.00036,0.00807,0.0151,-0.00084,0.00634,0.01091,-0.02792,0.00719,-0.02549,0.02657,0.03262,-0.02266,-0.03089,0.04665,0.02914,0.00031,-0.04202,-0.02253,0.00467,-0.00054,0.00437,0.02745,0.00519,0.01223,-0.00532,0.00139,-0.06082,-0.03765,-0.01115,0.00353,-0.0088,0.03089,-0.00489,0.00165,-0.05456,0.0388,0.00507,0.03008,-0.00834,0.00468,-0.00601,0.01873,-0.01359,0.00811,0.03282,-0.01246,-0.01279,-0.01471,0.04892,0.04607,-0.02198,-0.00065,-0.03327,0.01377,0.01074,0.04278,-0.0093,-0.02977,0.06715,0.01861,-0.0077,0.01872,-0.0439,0.02387,-0.00299,-0.02081,-0.01573,-0.04167,0.00807,0.00065,0.00065,0.01816,-0.0017,0.02627,0.06348,-0.02001,0.06027,-0.02946,-0.01199,0.02773,-0.04589,-0.03309,0.01634,0.0082,0.03416,-0.03393,-0.00749,-0.0256,0.00653,-0.05221,0.05112,-0.00814,-0.0058,-0.00182,0.01997,-0.01974,-0.05507,-0.0047,0.04321,-0.00672,-0.03726,-0.0205,-0.0474,-0.02789,-0.03862,-0.0187,0.02212,0.04641,-0.02153,-0.02292,0.02199,-0.03656,0.09028,-0.00058,-0.00354,-0.0161,-0.00485,0.03962,-0.00299,-0.03107,0.01114,-0.022,-0.03781,0.06255,0.01833,0.01669,0.01027,-0.00535,0.00304,-0.02807,-0.01413,0.01674,-0.0272,0.01681,-0.01563,-0.02763,0.03361,-0.0212,-0.00072,-0.05552,-0.00117,-0.00476,-0.04154,0.0164,0.02182,0.01617,-0.03139,0.00065,0.01289,0.00104,-0.04317,0.00059,0.025,0.02221,-0.00788,-0.02704,0.01141,-0.02401,-0.07002,0.01731,-0.00908,0.00188,0.01558,0.01609,-0.01473,0.01998,-0.04544,-0.00141,0.01127,-0.02687,0.02563,0.00089,-0.00705,0.01113,-0.00756,-0.04536,-0.01975,-0.00295,0.01988,-0.00676,-0.0149,-0.04836,0.00961,-0.00517,0.03454,-0.02148,0.0028,0.00164,0.01459,-0.02652,0.05058,-0.01996,0.03394,0.01536,0.01269,0.02804,0.00069,0.00251,-0.02738,-0.01484,0.00915,-0.01094,-0.00619,0.00734,0.03816,0.03695,0.0216,-0.01631,-0.01528,0.02106,-0.01889,0.0197,-0.00692,0.02788,0.00386,0.03584,0.04171,0.02713,-0.04393,0.03027,0.0308,-0.01665,-0.01899,-0.00323,0.02683,-0.01921,0.02785,0.02647,0.03481,-0.01882,0.04137,0.01432,0.00414,-0.01163,-0.02317,0.01234,0.03456,-0.00071,0.06214,-0.00026,-0.00952,0.02387,0.00662,0.03492,0.01077,-0.01846,0.00624,0.05096,0.02644,0.01542,0.02402,-0.01707,-0.00858,-0.00263,0.08072,-0.01959,-0.00365,0.02969,-0.00734,0.01288,-0.00793,-0.01534,-0.01486,-0.01012,0.00763,0.00092,-0.01636,0.00042,-0.01837,0.0159,0.0395,-0.00234,0.00359,-0.02614,-0.02487,0.03416,-0.02167,-0.02536,-0.01369,-0.0025,0.0059,-0.03979,-0.0486,-0.03537,-0.00459,-0.02919,0.00967,-0.03538,-0.03318,-0.00018,-0.00276,-0.00046,-0.00486,0.00449,0.02666,-0.00714,-0.05951,0.01805,-0.0023,-0.01134,0.03153,-0.00744,-0.01886,-0.04154,-0.1281,0.00485,0.06055,0.0317,-0.04429,-0.00361,0.02544,-0.00061,-0.03635,-0.01224,-0.01659,-0.03215,0.0002,-0.0398,-0.00907,-0.01676,-0.00688,0.01166,0.00012,0.02015,-0.02505,0.01219,-0.02845,-0.04261,0.00261,0.01102,-0.0119,-0.00321,0.0051,0.03685,-0.00043,0.01544,-0.03874,0.01909,0.02085,-0.03069,-0.0064,0.03192,-0.01154,0.02297,0.01252,-0.01126,0.00361,0.00225,0.00296,0.0124,-0.01208,-0.00491,0.04042,0.03322,-0.02767,-0.0012,-0.0361,0.02925,-0.00352,0.01765,-0.00348,-0.05412,0.01434,-0.00656,0.05464,0.0059,-0.0175,-0.02781,0.00774,0.03258,-0.00521,0.02927,-0.01343,0.00118,-0.02641,-0.04663,0.02336,-0.02228,0.01984,-0.01099,-0.00151,-0.0133,0.00673,-0.00396,-0.03558,0.0004,-0.04027,0.02387,-0.02102,0.01289,0.00952,-0.02402,0.01083,0.04374,0.03921,-0.01945,-0.0196,0.02291,-0.00316,-0.01066,0.04271,0.00216,0.07001,0.03886,0.06011,-0.00658,0.01905,0.04544,-0.02034,0.01889,0.0398,0.01021,0.04815,0.01807,0.0105,-0.05472,-0.01341,-0.05864,-0.05747],"CAPTURE":[0.02949,0.01399,0.0117,0.01779,-0.00883,0.04159,-0.01109,-0.00994,0.00771,-0.00557,0.01945,-0.09021,-0.00502,0.01804,0.00588,0.01365,-0.01645,-0.02849,0.0326,0.08201,0.00792,-0.02499,0.02613,-0.02903,-0.01036,0.02951,0.00889,0.02026,-0.02715,0.00033,0.00387,-0.01743,-0.00823,0.05217,-0.03655,0.03385,0.01148,-0.01523,-0.019,0.02772,-0.00085,0.00545,-0.00905,-0.02011,-0.02723,0.01423,-0.04788,0.0176,-0.03866,-0.01889,-0.01305,0.04198,0.00076,0.03408,-0.03459,0.0034,0.02386,0.02237,-0.00206,0.00592,0.01721,-0.02295,-0.04453,0.00439,-0.01579,-0.02931,0.00576,0.01485,-0.02717,-0.01356,0.03095,0.04394,-9e-05,0.00748,-0.03654,-0.00217,-0.01553,-0.00626,-0.02644,-0.00047,-0.03307,0.03229,-0.00666,-0.00731,0.0063,0.02344,-0.01893,-0.00335,0.0381,-0.00626,0.02193,0.00279,-0.03372,0.0201,0.02414,0.00404,0.03404,0.03862,0.0486,-0.03386,-0.00867,0.01478,-0.01247,-0.02406,-0.00896,0.00343,0.02919,-0.00678,0.04478,-0.04551,-0.02523,-0.00818,0.00931,-0.0363,0.00626,-0.02474,-0.00192,-0.01121,-0.03381,0.02504,0.01193,-0.01941,0.01246,0.03794,0.04152,0.02819,-0.02373,-0.0091,-0.00839,0.00204,-0.01601,0.02076,0.01651,0.01111,-0.01774,-0.03123,-0.00035,-0.03815,0.01833,-0.00443,-0.05115,-0.00964,0.01408,0.01746,-0.00215,-0.02747,0.00341,0.06304,-0.00486,-0.00753,0.007,0.00047,0.01221,-0.00085,0.02163,-0.01003,0.0274,-0.00401,-0.01453,-0.00663,-0.01018,0.00754,-0.02817,0.0101,-0.02242,-0.00063,0.02409,0.03971,-0.03045,0.02405,0.0427,0.01712,-0.02613,-0.02258,-0.01039,0.0326,0.00983,-0.07033,-0.01914,0.04869,0.02901,0.02012,0.01498,-0.01458,-0.00564,-0.02007,-0.00914,0.03543,0.02547,0.03049,-0.00728,0.02663,-0.03041,-0.00139,0.01506,0.02624,-0.0151,-0.01564,-0.0046,0.00958,-0.02536,-0.02249,0.03343,-0.00973,-0.04154,-0.01799,-0.00615,-0.03359,-0.01681,0.06471,-0.01215,-0.00269,0.00121,0.00125,0.01573,-0.00912,-0.01833,0.00529,0.01896,0.00453,-0.01763,-0.02193,0.01369,-0.0217,0.01072,-0.00046,0.02197,0.02551,-0.02431,0.0291,-0.01529,0.00272,-0.00107,-0.00653,-0.03151,-0.01328,-0.00577,0.0259,0.02054,-0.01238,0.0366,0.02992,-0.01171,-0.02047,0.03171,0.01556,-0.02597,-0.02454,-0.0143,0.00178,0.01574,-0.01409,-0.01471,0.02433,-0.00606,0.0321,-0.00604,-0.00478,0.02006,-0.04251,-0.01234,-0.00902,0.03339,0.00188,-0.00354,0.00128,0.00492,0.00251,0.03075,0.01926,0.01174,-0.04657,0.01171,-0.01012,0.00966,-0.01131,0.00999,0.01683,-0.01081,-0.01892,0.00914,0.01638,0.05957,0.00276,0.01095,0.02337,-0.01633,0.00969,-0.00966,0.00519,0.0197,0.01095,0.00912,-0.0093,0.00731,-0.01161,-0.00351,0.01018,-0.01269,0.04325,-0.00809,-0.05829,-0.00694,-0.0249,-0.01143,0.02516,-0.00128,-0.02294,-0.00326,-0.01648,-0.03686,-0.04746,-0.00116,0.00186,0.00043,0.02125,0.09436,0.03884,0.01631,0.00057,0.00239,0.00558,0.00653,-0.01985,0.01171,0.02386,-7e-05,0.02312,0.03015,-0.00646,0.01671,-0.02637,0.01688,0.05914,-0.00144,-0.0029,0.02432,-0.00268,-0.0172,0.0533,-0.01408,-0.00908,-0.01314,-0.00939,0.01429,0.02168,-0.00578,-0.03114,-0.03261,-0.00447,-0.02857,0.02275,0.00257,0.01833,0.00645,-0.01128,-0.04532,0.03395,0.0247,0.0197,-0.00526,0.02717,-0.00346,0.01319,0.0277,0.0182,0.015,0.00918,-0.00233,-0.03618,0.02701,0.01437,-0.00137,0.00152,-0.03065,0.04334,0.02374,0.04584,0.00783,-0.00605,-0.02173,-0.00982,0.02325,0.00068,0.01535,0.02985,0.02263,0.02554,0.00688,-0.01462,-0.00771,0.01213,-0.0031,0.02978,0.02622,0.01789,-0.01524,0.06513,0.0126,0.00888,-0.00413,-0.0306,-0.00071,-0.01316,0.02941,0.02562,0.04327,0.00667,-0.01629,-0.01242,0.02424,-0.02275,0.00344,-0.03411,-0.01938,0.00606,0.00661,-0.04415,0.0221,-0.01271,0.01238,0.00146,-0.00039,-0.00892,-0.00057,0.0061,5e-05,-0.00089,-0.00544,-0.01046,0.01857,0.01883,0.03271,-0.03442,-0.0243,0.0233,0.02519,0.04969,-0.01472,0.04248,-0.00959,0.00259,0.06992,0.00845,-0.00478,-0.01259,0.02113,0.00816,-0.00316,-2e-05,-0.01682,-0.0091,-0.02309,0.0166,-0.03805,-0.01879,0.00473,0.02887,-0.0103,0.01448,-0.0135,-0.00685,0.02026,0.0201,-0.00032,-0.01562,0.05004,0.03617,-0.02594,0.04521,0.01411,0.06125,-0.00827,0.02144,0.0235,0.00199,0.00439,-0.02429,-0.01474,0.03254,0.01629,0.02749,0.00726,0.02839,-0.03165,-0.01619,0.02081,-0.0106,0.00814,-0.01391,0.04899,-0.00021,-0.01657,-0.01838,0.0057,-0.0033,0.01808,0.00387,-0.03048,0.00401,-0.03101,0.01284,0.00955,0.08004,0.00655,-0.04403,0.03193,0.04129,0.01358,0.02906,0.02912,0.036,-0.01182,-0.02104,-0.00147,0.04653,0.01134,0.02869,0.03946,-0.00089,0.00907,-0.00901,0.00262,0.00961,0.049,0.00016,0.05289,-0.0003,0.04307,0.03492,0.03143,0.02687,-0.03676,-0.02479,8e-05,-0.00484,-0.00942,-0.01878,0.00019,-0.01499,0.01714,-0.01089,-0.00705,-0.03011,-0.05067,0.00509,-0.02787,0.01258,-0.00846,-0.0193,0.01247,0.03697,0.00575,0.14024,0.01143,-0.02843,-0.0101,0.00601,0.00255,-0.00569,-0.00724,0.04128,0.00269,0.00525,-0.01592,0.01696,0.00393,-0.01377,-0.02534,0.01341,0.03566,0.03085,-0.04716,-0.03869,0.03205,0.03859,0.00811,-0.05159,0.00162,0.01531,0.02435,-0.01706,0.02151,0.02402,-0.00843,-0.00889,-0.02754,0.01752,0.02063,-0.01337,-0.0029,-0.00395,0.01352,-0.01217,0.01577,0.05559,0.00116,0.02568,-0.03502,0.00308,0.00723,0.00743,0.02359,0.04761,-0.00948,0.00693,-0.00594,-0.02089,-0.00036,0.14105,0.01576,0.00729,-0.02015,0.02256,0.0057,-0.02324,0.02538,-0.0166,0.0036,-0.03327,0.01268,0.01654,-0.01113,-0.01475,-0.00175,0.01026,0.02233,0.00211,0.01883,0.02258,0.00908,-0.00458,0.04481,-0.02877,-0.00891,0.00896,0.01543,0.03747,0.02165,-0.01976,-0.00313,-0.00056,0.04299,-0.02288,-0.00609,-0.01054,0.01818,-0.00108,-0.01689,-0.0041,-0.00566,0.02167,0.01355,0.01424,-0.00465,0.01324,-0.03868,0.00109,0.00708,0.02308,-0.02594,0.05668,-0.00583,-0.01698,-0.00585,0.01095,-0.01468,-0.01956,-0.00908,-0.06935,-0.01539,0.01477,-0.01415,-0.02543,-0.02132,-0.01562,-0.0485,-0.00445,0.00653,-0.03633,-0.02402,-0.02296,-0.01065,-0.03929,0.00347,0.00856,-0.01317,-0.00087,0.00934,-0.01595,-0.03559,0.01642,0.02144,0.02895,0.00049,0.01174,-0.00077,-9e-05,0.0118,0.01726,0.02933,-0.0224,0.00641,-0.02378,0.02316,-0.02434,0.01185,-9e-05,0.01836,-0.03841,0.02439,0.01047,-0.01797,-0.05118,-0.02651,-0.00025,-0.02652,0.02056,-0.08989,0.02377,0.00856,-0.00845,0.01426,-0.00791,0.02601,-0.02716,-0.00151,0.0067,0.0368,0.00325,0.01431,0.00029,-0.00493,0.01744,-0.03321,-0.02329,0.01134,-0.01118,0.01056,-0.00996,0.00604,0.06168,0.01949,-0.0118,0.0501,-0.01388,0.04342,-0.00496,-0.00103,0.00266,-0.04771,-0.00661,0.05516,-0.01998,0.00829,0.01535,0.00929,-0.00167,0.03238,-0.01644,0.01264,-0.02102,-0.00702,-0.00091,0.01303,-0.00705,0.00094,0.00174,0.00712,0.00944,0.01487,-0.01947,-0.01516,-0.0178,-0.00947,0.00946,0.01287,-0.03094,-0.01708,0.00489,-0.02131,-0.01885,0.01202,0.00455,-0.00316,-0.03054,0.00767,-0.00147,0.02663,-0.05181,0.02301,-0.00685,-0.02625,-0.03731,-0.0065,0.04165,-0.00554,-0.03191,-0.03539,-0.02068,0.02634,-0.0164,0.01239,-0.01891,0.00064,0.01285,-0.00031,-0.02062,-0.03409,-0.00425,-0.01774,0.01361,-0.01337,0.03377,0.00327,0.01155,-0.0015,-0.03574,0.02,-0.02636,-0.01784,-0.02641,0.00931,0.00795,0.02172,0.0041,-0.01717,-0.02132,-0.03116,0.00791,-0.04396,0.0091,0.00137,-0.00912,-0.01605,0.0178,0.00054,-0.02957,0.04459,0.01079,-0.01335,0.00499,-0.01166,-0.03388,0.05609,-0.01175,-0.01313,-0.00688,0.00935,-0.00971,-0.00828,-0.01062,-0.02348,0.00405,-0.00999,-0.03967,0.01145,-0.01743,-0.03428,0.00596,-0.01098,0.03064,-0.01279,0.0085,-0.00883,-0.06583,0.01953,-0.02109,0.00679,0.01438,-0.02248,0.03441,-0.01533,0.02796,-0.02817,0.02404,-0.00565,0.00846,-0.01178,0.0084,0.04464,0.03108,-0.02675,-0.01774,0.04704,0.03625,-0.01956,0.01166,0.00762,-0.05285,-0.0012,0.00259,-0.02391,0.03973,-0.02452,-0.01946,0.02206,0.01917,0.01078,-0.04794,-0.00601,0.07019,0.0271,-0.01204,-0.02574,-0.0213,0.01098,0.01462,0.01197,0.00411,-0.0069,0.01635,0.01415,-0.01719,-0.00453,-0.02108,0.01961,-0.0172,0.03213,0.00302,0.01054,-0.02195,-0.02857,0.00525,0.014,-0.00174,-0.02376,0.06292,0.00215,0.01464,-0.02384,-0.0162,0.00259,0.00844,-0.01964,-0.02378,-0.02249,0.01268,-0.01529,0.02454,0.02951,-0.0232,-0.00108,-0.02939,-0.007,-0.00556,-0.01277,-0.00457,-0.03364,-0.00783,0.0067,0.00625,-0.02084,0.03064,0.00795,0.01575,0.00727,-0.04432,-0.01021,0.04685,-0.03117,0.00814,-0.01206,0.05043,0.01913,-0.03465,-0.00177,0.01748,0.00712,-0.01897,-0.0006,-0.03825,0.00024,-0.0103,0.01987,0.01179,-0.02779,-0.02083,-0.04767,-0.01531,0.0432,0.02304,-0.01303,-0.01862,0.0069,-0.0127,-0.04732,0.01812,0.01871,0.00343,-0.02301,-0.02735,-0.00199,-0.01555,0.00611,0.03069,-0.03035,0.02099,-0.04259,-0.00173,-0.00734,-0.00589,-0.00175,0.01587,0.01142,0.01735,-0.02993,-0.03585,0.01024,0.01119,-0.00844,-0.02344,-0.00583,0.02336,0.01829,-0.00347,0.00326,0.0055,0.00373,0.00251,-0.02628,0.03229,0.00604,0.03798,0.03546,0.00329,0.02584,0.00321,-0.01536,-0.00762,-0.01904,0.02245,0.0152,0.01376,0.01331,0.02898,-0.0292,0.01039,-0.01872,0.01038,-0.01891,-0.00153,0.00495,-0.00923,-0.00612,0.03955,-0.00801,0.00034,-0.0351,-0.04145,-0.01011,0.01421,0.0075,0.00685,-0.00329,0.0045,0.00579,-0.02975,-0.01414,-0.01869,-0.03302,-0.04246,-0.01529,-0.00447,-0.01459,-0.02351,0.00799,-0.01102,0.05706,0.01585,-0.02486,-0.00811,0.03194,0.10375,-0.0293,0.04038,-0.00069,-0.02677,0.00475,0.01989,0.02396,0.00356,0.04124,0.00236,0.00132,0.0547,-0.01432,-0.03255,0.00327,0.02269,0.0151,0.01607,-0.01864,0.03998,-0.00083,0.00221,0.0114,0.01774,0.00868,-0.01376,-0.02073,-0.01101,-0.00456,-0.01848,-0.00337,-0.02474,0.01151,-0.00629,0.02402,-0.02423,-0.02479,0.03101,0.007,0.00967,0.01693,-0.00631,-0.03396,0.04414,0.0062,0.00033,-0.02403,0.03987,0.0185,0.0225,-0.02678,-0.03993,0.00192,-0.00762,-0.01273,-0.00878,0.00425,0.02993,0.00124,-0.04145,0.00473,0.00726,0.00323,-0.014,-0.00339,0.00104,0.02319,-0.00284,-0.0459,0.00201,-0.01805,0.00268,-0.008,-0.01144,0.03698,-0.01999,-0.00818,0.00502,-0.00715,-0.00781,0.00081,-0.00379,-0.00021,-0.01863,0.00348,-0.01869,0.01033,0.05778,-0.00689,-0.02175,-0.02936,-0.00545,0.04919,-0.00319,-0.02181,-0.01908,-0.02028,0.03798,0.03016,0.03138,-0.01341,0.0397,0.00957,0.02096,0.00826,-0.02457,0.0043,-0.03239,0.0244,0.00696,0.01519,-0.00759,0.01534,0.00468,0.0293,-0.01932,0.04736,-0.01119,0.01978,0.01782,-0.02575,-0.00517,-0.00258,-0.05502,-0.0421,0.00957,-0.04072,-0.01422,-0.00889,0.0173,-0.00799,-0.0185,-0.00313,0.01013,0.04355,0.00065,-0.0134,-0.03101,0.00169,-0.01712,-0.04753,0.01987,0.01435,0.02954,-0.0067,-0.02297,-0.03582,-0.02712,-0.02758,0.01745,0.04021,0.00705,-0.00589,0.02701,-0.02755,0.1035,-0.03784,-0.02244,0.01127,-0.00205,0.02167,0.00895,0.01613,-0.02118,-0.01034,-0.00844,0.034,0.03439,0.01167,0.01607,0.00173,0.00727,0.01244,-0.00042,0.07216,-0.02667,-0.00176,-0.02081,-0.01839,0.00756,0.02211,0.00557,-0.0177,0.0087,0.0104,-0.03757,0.01777,0.01681,0.01362,0.00058,-0.00329,-0.00864,-0.01783,-0.00521,-0.00517,-0.02531,-0.03482,-0.00869,-0.00996,-0.03961,0.02001,0.00139,-0.0068,-0.00917,-0.00975,-0.03814,0.01131,-0.03413,0.00262,-0.05994,0.01129,0.00471,-0.01776,-0.01669,0.00068,-0.01908,-0.02947,-0.02857,0.00701,-0.01848,-0.01167,0.01224,0.01627,0.04491,-0.03718,0.02131,0.01058,-0.00036,-0.0187,-0.00118,-0.00678,0.00606,-0.01708,0.02944,-0.02456,0.02665,0.01162,0.01865,0.02115,-0.02106,0.00445,-0.04703,-0.02524,0.04149,-0.00172,-0.02011,-0.00403,-0.00415,0.06226,0.01584,0.0243,0.02455,0.02087,-0.01895,0.01369,-0.01082,0.01423,0.04285,0.00922,0.00673,-0.00685,-0.02503,-0.00223,-0.02593,0.00714,0.0026,0.01734,0.02227,0.00051,0.02872,0.03815,0.01965,0.00535,0.02542,0.01682,-0.01295,-0.05816,-0.0094,-0.00068,0.04046,0.02355,0.02884,-0.01687,-0.0269,0.02951,0.01799,-0.00413,0.00657,0.01133,-0.02056,0.00967,0.00462,0.01526,0.02949,0.02743,-0.02134,0.018,0.04295,0.00848,0.00868,-0.01691,0.01536,-0.00321,0.01882,-0.01012,0.01458,-0.04206,-0.00071,0.00325,-0.03373,0.00802,-0.0387,-0.04367,0.00142,-0.01337,0.01631,0.02195,-0.02017,-0.01456,-0.03098,-0.02931,0.04798,-0.00199,0.01185,-0.01298,0.02406,-0.05367,-0.02383,-0.05918,-0.0305,-0.06975,-0.02451,0.03407,-0.01106,-0.00567,0.00373,-0.01945,-0.03011,0.03765,0.0135,0.00677,-0.02026,-0.0086,0.02082,0.00697,0.01504,-0.02111,-0.26514,-0.01035,0.11379,0.10186,-0.04309,0.03641,-0.00096,0.00592,-0.0351,-0.00811,-0.03145,-0.03299,0.0271,-0.03137,-0.02348,-0.02275,0.00478,-0.01373,-0.01474,0.03206,-0.00561,0.00353,-0.01127,-0.05102,0.02059,0.01416,0.01085,-0.04825,0.03904,0.02379,0.0073,-0.0333,-0.04822,-0.01404,0.01475,-0.02502,-0.00788,0.04959,-0.02265,0.02454,-0.00072,0.02334,-0.04787,-0.00088,0.01223,0.02039,0.00414,-0.00173,-0.00526,0.01838,0.02975,-0.00304,-0.0165,0.01903,0.00341,0.0332,-0.00966,-0.0081,-0.01218,-0.02905,0.05995,-0.03187,0.00707,0.05137,0.01243,0.01258,-0.00826,0.00346,0.00164,-0.00385,-0.0104,-0.01081,-0.00965,-0.01145,0.01335,-0.04155,0.001,-0.03328,0.00395,-0.05411,-0.00863,-0.00871,-0.00965,0.02859,-0.01596,-0.00197,0.02825,-0.00882,-0.01866,0.00335,0.00517,0.01091,-0.05305,0.01311,0.01476,-0.02246,-0.00224,-0.0248,0.02941,-0.01556,0.04208,0.02749,0.01167,0.04682,-0.00145,0.0632,0.03628,-0.01697,0.04435,0.01083,0.00766,-0.00668,-0.02672,-0.03895,-0.00855],"DIVERGENCE":[0.02127,-0.00977,-0.0142,0.04471,-0.02571,0.02897,0.01171,-0.021,0.04478,0.03574,-0.0048,-0.02951,0.03029,0.00866,-0.01252,-0.0094,-0.03127,-0.001,0.01417,-0.01453,0.0086,0.01051,0.01786,0.0084,0.00752,0.03548,0.00308,-0.00747,0.05335,-0.02472,-0.00334,0.0253,-0.03595,0.01395,-0.00271,-0.02703,-0.02792,0.01854,0.0234,-0.00281,0.02895,-0.02273,0.05736,0.02494,-0.04471,0.00639,0.01453,-0.04625,-0.01946,-0.0216,0.02907,0.00098,0.01188,-0.01143,0.04555,0.02898,0.01376,-0.00245,-0.01622,0.02282,0.01219,-0.0078,-0.00011,0.00338,0.00739,-0.01553,-0.00184,0.00694,-0.01821,0.02577,0.00883,-0.02131,0.00716,0.03799,-0.03014,0.0052,0.02841,-0.01908,-0.02855,0.00072,0.05425,-0.00052,-0.02178,0.0177,0.03775,0.00536,-0.00034,-0.00862,0.01829,0.00448,-0.01256,0.00437,-0.01138,0.02024,0.03359,0.05136,0.0307,0.01774,0.05742,-0.04913,0.03285,-0.01218,0.00193,-0.06392,-0.02524,0.00519,0.02227,0.00288,0.03483,-0.02807,-0.02629,-0.02254,-0.01351,-0.00796,0.02374,-0.03875,0.00402,-0.00219,-0.00877,-0.00551,-0.03216,0.00797,0.02721,0.01409,0.03037,0.04409,0.01934,-0.00063,0.0228,-0.00741,-0.04872,0.00412,0.00999,0.0486,-0.06636,-0.01322,0.00153,-0.0177,-0.0087,-0.00137,-0.00411,0.04448,0.00071,0.01213,0.01934,0.02069,-0.03788,0.01257,0.02046,-0.06178,0.02119,-0.01306,-0.0212,-0.01004,0.00706,-0.0245,0.04071,-0.0114,-0.00817,-0.01284,-0.01815,0.02084,0.01814,0.01009,-0.00908,-0.03571,0.03506,0.0383,0.03223,0.00606,0.04168,-0.00941,0.01701,-0.00695,-0.01158,0.00052,0.00183,-0.02386,-0.00897,0.02573,0.03217,0.02005,0.0255,-0.04596,0.00642,0.01425,0.00485,-0.01616,-0.01913,0.00531,-0.0491,0.00994,-0.04176,-0.01413,-0.03887,0.02126,0.03432,-0.03287,0.02518,0.01266,0.03279,0.00049,0.00285,0.02403,0.02671,-0.01077,-0.00269,-0.00601,0.04265,0.01103,-0.03083,-0.01412,0.00192,-0.0365,0.00974,-0.02688,-0.0213,-0.00987,0.01136,0.02304,-0.01784,0.00324,-0.00268,0.02328,-0.00749,-0.02259,0.03076,-0.03072,0.00964,0.01596,0.0667,-0.0412,0.04084,0.03679,-0.02675,0.01377,0.06103,0.02802,8e-05,-0.04028,0.02275,-0.02872,0.0132,-0.07896,-0.01702,0.01905,0.01175,-0.01764,-0.00037,0.00837,-0.00127,-0.03377,0.02625,0.0146,-0.02596,0.00273,-0.03119,-0.03394,0.03053,0.01009,-0.031,0.03317,0.03612,0.03747,-0.02807,-0.01097,0.00597,-0.02506,0.00544,-0.02756,0.00539,-0.02437,0.04572,0.03123,0.02028,0.01705,-0.00078,0.01978,0.00287,0.00449,-0.00679,0.04322,-0.0023,0.00868,-0.04043,0.01451,0.02656,-0.00742,-0.00169,0.01398,0.04984,-0.03466,-0.00323,0.01416,0.01157,-0.00584,0.05288,0.05529,-0.00208,-0.04096,-0.00169,-0.00776,0.02542,-0.02215,0.01191,0.02446,-0.01762,0.00573,0.03499,0.01034,-0.00281,0.02668,-0.01043,-0.01415,0.01823,-0.01356,0.01906,-0.02184,0.0034,-0.00166,-0.05127,-0.01341,0.0106,-0.00428,-0.00347,-0.0055,-0.01903,0.015,-0.00032,0.02673,-0.00127,0.00507,-0.0058,0.01676,0.0016,0.01985,0.00362,0.01006,0.01117,0.03301,0.01629,0.0347,-0.00694,-0.04053,-0.01803,0.01595,-0.02638,-0.02189,0.00585,0.02759,0.00437,0.00236,0.01275,0.00728,0.05568,-0.01777,0.05566,0.03857,0.049,0.04334,0.02173,0.01869,0.04047,0.04653,0.01545,-0.0613,0.0117,-0.02681,0.00343,0.06873,0.01267,-0.00769,0.01461,0.02021,-0.03884,0.0199,0.00311,0.01545,-0.01006,0.01459,-0.00924,-0.01846,0.01066,0.02456,0.00717,0.00913,-0.0066,0.01862,0.00037,0.0031,-0.04322,-0.01655,0.0217,-0.04038,0.00531,-0.04308,0.01619,0.01307,0.0012,0.00137,0.04045,-0.07765,-0.00889,-0.0041,0.0329,-0.00897,0.02438,0.01348,0.02734,-0.0109,0.02895,0.01866,0.01076,-0.01647,-0.0115,0.01929,-0.01541,0.01357,-0.00553,-0.02096,0.0283,0.03587,0.03672,0.00255,0.03476,-0.03187,-0.02659,-0.02536,0.03838,0.01498,0.04349,-0.00433,0.05605,-0.01934,0.0019,-0.00768,-0.02927,0.02259,-0.02817,0.02854,0.01199,-0.00738,0.03672,0.02713,0.01045,-0.03138,-0.01662,0.0152,-0.03063,-0.03024,0.00317,0.01065,0.055,-0.07082,-0.02251,0.04829,2e-05,-0.01876,-0.00467,0.01939,0.00331,-0.06155,0.01694,0.02876,-0.01432,-0.00291,-0.01341,0.00769,-0.00191,0.05161,-0.03924,0.01145,0.01641,0.06813,-0.01356,-0.01564,-0.01292,-0.01691,-0.00618,0.01233,-0.00968,-0.04248,-0.02235,-0.00758,0.00036,0.01723,0.02679,-0.03636,0.00356,-0.00327,0.00388,-0.03694,0.00594,0.00581,-0.0188,0.008,0.00399,0.01162,-0.00756,-0.01496,-0.00874,0.04655,0.00504,0.00764,0.0468,-0.034,-0.00133,0.02232,-0.01537,0.0073,-0.02049,0.01847,0.00784,0.02008,-0.0011,0.03484,-0.02135,0.05416,-0.01595,0.04353,-0.00693,0.02953,0.00447,-0.01113,-0.00629,-0.03238,0.01808,-0.04371,-0.00086,-0.01657,0.02825,0.02134,-0.02129,-0.0012,0.00355,0.02411,-0.00881,-0.01336,0.0071,0.00013,0.05193,-0.00999,0.01364,-0.00228,-0.03096,0.00314,0.02426,-0.0135,0.01438,-0.00224,-0.00801,-0.03574,-0.05659,0.0441,0.00411,0.03691,-0.00232,0.01507,0.01866,-0.00442,0.02441,0.04282,-0.00034,-0.05999,-0.0249,-0.0213,0.0014,-0.01775,0.05241,0.02507,0.01462,0.01186,0.00137,-0.00354,0.01672,-0.02122,0.01211,-0.03893,-0.00106,0.00249,-0.02178,-0.03044,-0.00835,0.03667,-0.00392,-0.05914,-0.02183,0.03248,-0.0064,-0.00327,0.03168,-0.0192,0.02803,0.00366,-0.02067,0.03241,-0.03442,-0.00878,-0.03402,-0.01499,0.02377,-0.03818,-0.01076,-0.00178,0.02099,0.02251,-0.00639,0.01355,0.05144,-0.00468,-0.02561,0.00115,-0.00293,-0.01003,-0.00641,0.012,-0.01017,-0.00319,0.00316,-0.01297,-0.00092,0.0016,0.00455,-0.02032,-0.00198,-0.03065,0.01971,0.035,0.04093,0.01853,0.00548,-0.02053,0.01113,0.05859,0.02013,-0.01139,0.00709,0.00447,-0.03203,0.01549,-0.0107,-0.01275,-0.00685,-0.00578,0.00555,-0.00953,-0.01614,0.01574,0.01402,-0.00208,0.00651,-0.00173,0.01101,-0.01466,-0.02329,-0.05673,-0.00549,-0.04352,0.06298,-0.02796,-0.05152,-0.01204,-0.02479,0.00796,-0.0064,-0.01026,0.01898,-0.00314,0.01114,-0.02802,-0.00337,-0.00621,-0.02673,0.06628,-0.03503,-0.02603,-0.00012,-0.0086,0.00598,0.02749,-0.0475,-0.00281,-0.0099,-0.05405,0.01507,-0.03879,0.00179,0.01,0.05185,-0.04543,0.01368,0.02082,0.0227,0.06808,0.01441,0.01748,-0.00618,-0.01197,-0.00631,0.0103,0.0313,-0.03136,0.06227,-0.01607,0.0078,-0.01892,-0.00467,0.01976,-0.03707,-0.00688,-0.00712,-0.00065,-0.01275,-0.02579,-0.02574,-0.04911,-0.00666,-0.0192,0.01478,0.00699,-0.04661,0.00254,0.04003,7e-05,0.01466,-0.01375,-0.02404,0.00645,-0.02023,-0.03279,-0.03473,-0.00189,-0.02814,0.02717,0.06495,0.01333,0.03786,0.01131,0.01422,-0.01614,0.02764,-0.00745,-0.0418,0.00487,-0.00775,-0.03843,0.0283,-0.01309,-0.00201,-0.01981,-0.0082,0.03334,0.02011,-0.01384,-0.02945,0.00081,0.00279,-0.01068,0.00539,0.00544,0.06118,0.01929,-0.02759,0.00121,0.01493,0.04347,-0.03823,-0.00992,-0.02029,-0.0365,-0.00176,0.0398,-0.00899,0.01348,-0.00778,0.04708,-0.00958,-0.00624,0.01991,0.0142,0.00875,-0.03249,0.03163,0.00235,0.06825,0.0104,-0.03674,-0.01792,-0.01434,-0.03078,0.02916,0.05561,0.0153,0.01153,0.05776,-0.00727,-0.00714,-0.002,0.00165,-0.00945,-0.02176,0.03568,0.0067,-0.05894,0.05405,-0.01284,0.01319,0.03733,-0.02835,-0.01384,-0.02184,0.02104,0.00398,-0.01964,0.00869,-0.03641,0.01659,-0.01231,0.03087,-0.00115,-0.01242,-0.02224,-0.00859,0.03442,0.01149,0.02624,-0.0119,-0.03081,0.00011,-0.01647,0.00552,-0.01012,-0.03044,0.00611,-0.00633,0.02902,0.04836,-0.00552,-0.00023,0.02178,-0.02341,0.02246,0.00338,-0.01603,-0.00561,0.01121,0.01597,-0.04167,0.00257,0.01362,0.02607,-0.01947,-0.03881,0.0317,0.00105,-0.02414,0.02177,0.03741,-0.00419,-0.01456,0.00934,-0.01426,0.04158,-0.00184,-0.00993,-0.01355,0.00759,-0.0037,-0.03078,0.01074,-0.02739,0.00914,0.00966,0.01486,-0.05475,0.01488,-0.00052,0.01601,0.00322,0.01303,0.01641,0.01008,-0.03638,-0.01881,0.01726,0.00868,-0.02434,0.00606,-0.03528,-0.03845,0.0087,-0.01913,-0.01694,0.03494,0.00041,-0.02424,0.02508,0.00711,0.00977,0.01552,0.00315,0.04154,0.03373,-0.04129,-0.01599,0.00297,0.001,0.04281,-0.00245,-0.00752,-0.03656,0.02805,0.00862,0.00042,0.02642,-0.03092,0.02946,-0.00895,-0.02207,0.00051,-0.0232,0.04356,0.03651,-0.01272,-0.01867,-0.03442,-0.03073,0.03146,-0.01285,0.01088,0.01891,-0.02502,-0.0265,-0.00258,-0.02651,-0.03702,0.00926,0.01439,-0.01196,0.0063,0.01774,0.03996,-0.00937,-0.01041,-0.01914,0.00212,-0.04805,0.01251,0.0097,-0.00764,-0.01821,-0.00758,0.02841,-0.01854,0.01046,0.04079,0.00179,0.02192,-0.03103,0.0021,-0.03922,-0.02137,-0.02569,0.0642,-0.04126,-0.07189,-0.02243,-0.0218,-0.00336,-0.00266,-0.00737,-0.02996,-0.02433,-0.06691,-0.02351,0.0042,0.0239,0.03117,-0.01048,0.04699,-0.01106,-0.02681,0.03648,-0.03174,-0.05562,-0.02528,-0.00708,-0.01435,0.01153,-0.00416,-0.03114,0.00346,0.02415,-0.02373,0.03667,0.04824,0.03462,0.02398,-0.0509,-0.0186,-0.0126,-0.01036,0.01761,0.03831,0.00073,0.01553,-0.00198,0.00119,0.03545,-0.00809,-0.02048,0.0108,-0.01349,-0.00661,-0.01154,0.01748,-0.02639,0.02876,-0.02164,-0.00952,-0.02559,0.01698,-0.01489,0.01386,0.02707,0.03921,0.02865,-0.02419,-0.01435,-0.02439,-0.00612,-0.00361,-0.0039,0.03129,0.01545,0.01022,-0.02764,-0.01806,0.00081,0.03959,-0.02655,-0.00314,-0.0238,0.01357,-0.02343,0.03367,-0.0466,-0.04041,-0.02789,0.02545,0.00146,-0.0095,0.01929,0.02462,-0.02984,-0.00555,-0.01324,0.04938,-0.02397,0.01921,-0.03905,-0.01277,0.00244,-0.01721,-0.01198,-0.00514,0.04736,0.01873,-0.03283,-0.00927,0.00468,-0.00214,0.0417,0.05594,-0.07371,-0.00357,-0.01856,-0.01547,-0.00696,0.01405,0.04735,-0.02483,0.05707,-0.00198,0.01295,0.01912,-0.01264,0.01372,0.01349,-0.01317,0.0509,-0.02151,-0.01443,0.00823,-0.00146,0.06155,0.00477,0.0017,-0.0137,0.04482,0.01512,0.01202,-0.00706,0.06256,-0.00529,0.00158,-0.00582,0.00649,0.02762,0.01209,-0.042,-0.03368,-0.00873,-0.00208,-0.04968,-0.00255,-0.02421,-0.01683,0.02554,0.00953,-0.00192,0.00869,-0.00102,-0.05045,0.00294,0.00783,0.03423,0.03888,-0.01752,-0.03377,0.04306,0.03206,-0.01571,-0.00069,-0.00951,0.00964,0.00113,-0.00309,0.03752,-0.0166,0.02541,0.01398,0.03803,-0.06547,-0.03004,-0.00363,0.00477,-0.02259,0.0105,0.00566,-0.00458,-0.06644,0.04566,-0.00841,0.03787,-0.0065,0.00505,0.0002,0.02617,-0.01259,-0.00351,0.01017,-0.02585,-0.03422,-0.00208,0.0444,0.04474,-0.00558,0.00946,-0.01655,0.01033,0.0216,0.00953,-0.03437,-0.02412,0.06547,0.00861,-0.01948,0.03997,-0.0304,0.02625,0.00801,-0.04369,-0.01836,-0.03539,0.02731,0.0062,0.0081,0.01143,0.00903,0.01068,0.04798,-0.01945,0.06703,-0.01503,-0.01648,-0.00244,-0.01647,0.00015,0.02097,0.01062,0.05183,-0.03476,-0.00827,-0.03222,0.00515,-0.05303,0.04162,-0.02489,-0.01424,-0.00768,0.03317,-0.01236,-0.05374,0.01596,0.00354,-0.00571,-0.02836,-0.0086,-0.04086,-0.02463,-0.01953,0.00465,-0.00169,0.018,-0.01424,-0.01772,0.01317,-0.03325,0.00536,0.01248,0.01139,-0.00364,-0.00922,0.05366,0.00104,-0.03895,0.01875,-0.02369,-0.04062,0.04947,0.02146,0.02899,0.01433,-0.00471,0.00091,0.0076,-0.01758,0.02369,-0.02079,0.017,-0.00733,-0.00593,0.04834,-0.04584,0.00263,-0.05491,0.00265,-0.01965,-0.02518,0.00604,0.0122,0.01156,-0.02616,-0.00893,0.00352,0.00632,-0.05781,0.0237,0.02027,0.04237,-0.01502,-0.03707,0.01459,-0.0114,-0.06513,0.02416,0.01587,-0.00696,0.01518,0.02783,-7e-05,0.01945,-0.04768,-0.0124,-0.00714,-0.01234,0.03286,0.00309,-0.01399,0.01621,-0.00746,-0.05027,0.00199,0.00987,0.0287,-0.0217,-0.02713,-0.04053,-0.01127,-0.01895,0.02101,-0.02424,0.0026,0.00276,-0.00117,-0.01344,0.05296,-0.00789,0.03058,0.01361,0.02089,0.02524,0.01447,0.00015,-0.00236,0.0006,-0.01084,-0.00878,-0.01096,0.02009,0.05191,0.01816,0.00484,-0.03794,-0.02501,0.02617,-0.0167,0.0093,-0.00877,0.01495,-0.00193,0.04031,0.03182,0.02257,-0.05107,0.04625,0.03899,-0.01363,-0.02015,-0.00634,0.02046,-0.03424,0.00441,0.02531,0.01631,-0.03309,0.02318,0.01387,-0.00665,0.00167,-0.03197,0.0036,0.0208,-0.02159,0.03948,0.01727,0.02693,0.02085,0.00758,0.03022,0.01375,-0.0159,0.01338,0.06575,0.04722,0.01042,0.00966,-0.01996,-0.01408,-0.00303,0.05452,-0.01576,-0.00599,0.01205,-0.00932,0.00551,-0.00142,-0.01516,-0.01813,0.0115,-0.00198,0.00735,0.00452,-0.01507,-0.00522,0.03372,0.02621,0.00474,0.00677,-0.01578,-0.00881,0.01472,-0.01028,-0.0108,-0.01938,0.00297,0.01119,-0.03825,-0.0271,-0.02568,-0.00835,-0.0204,0.02191,-0.02221,-0.03124,0.00389,-0.00337,0.0114,-0.0076,0.0042,0.04272,-0.00197,-0.06825,0.00782,-0.00664,-0.03547,0.03984,-0.01596,-0.02668,-0.03094,0.06095,-0.0217,0.02468,0.04544,-0.00871,-0.01054,0.04332,-0.01523,-0.02652,0.00016,-0.01293,-0.03035,-0.01294,-0.02854,-0.01412,-0.01786,-0.02806,0.01704,0.00894,-0.00092,-0.02996,0.00662,-0.02058,-0.03458,-0.00344,0.02154,-0.04332,0.0028,0.0026,0.02671,0.00096,-0.00736,-0.01496,0.00288,0.01757,-0.03488,-0.0227,0.00529,-0.00736,0.0044,0.00838,-0.0078,-0.00115,0.00937,-0.00975,-0.01842,-0.01907,-0.02286,0.03765,0.01694,-0.03033,0.00623,-0.02027,0.01333,0.00104,0.00012,0.01467,-0.06184,0.0201,-0.01053,0.03541,-0.00321,-0.00664,-0.03375,-0.01047,0.01425,-0.00272,0.02752,-0.02765,0.00169,-0.01992,-0.04136,0.03102,-0.03855,0.01908,0.01027,0.01781,-0.00824,0.02287,0.00742,-0.03207,0.01557,-0.0494,0.0196,-0.02631,0.01891,0.01376,-0.00701,0.01018,0.05339,0.05275,-0.03337,-0.0036,0.0423,0.00574,0.00263,0.0392,0.00942,0.05713,0.05292,0.05177,-0.01964,0.03361,0.00984,-0.0205,0.01077,0.04267,0.016,0.04315,0.01436,0.01887,-0.03887,-0.02814,-0.04895,-0.06832],"CONFABULATION":[0.03218,0.00662,0.0111,0.02924,-0.03297,0.03875,-0.00406,-0.01238,0.02358,0.0055,0.00351,-0.09161,0.01333,0.02492,-0.01117,0.01325,-0.02709,-0.02746,0.03885,0.05329,0.00347,-0.01221,0.02906,-0.01336,0.0006,0.03283,0.0093,0.00491,0.01073,-0.00972,-0.00696,0.00274,-0.03107,0.04302,-0.01759,0.02271,-0.00293,0.00768,-0.00687,0.01611,0.02576,-0.0067,0.01424,-0.01336,-0.02734,0.01695,-0.03479,-0.00586,-0.04017,-0.01445,0.00527,0.02961,-0.00112,0.01853,-0.00887,0.01942,-0.00014,0.01048,-0.01901,0.0149,0.02593,-0.02135,-0.02062,0.00519,-0.00169,-0.0281,-0.00163,0.01402,-0.02002,-0.00145,0.02356,0.03307,-0.00279,0.02831,-0.04537,-0.00117,-0.00562,-0.01342,-0.03777,0.00266,0.00171,0.01803,-0.02041,0.00471,0.02501,0.01116,-0.00513,-0.00181,0.03665,-0.0046,0.0142,-0.00361,-0.03173,0.02053,0.02785,0.02299,0.03195,0.03487,0.05978,-0.04127,0.00745,-0.01737,0.00552,-0.04661,-0.00831,-0.0005,0.02687,-0.00362,0.05742,-0.04649,-0.03437,-0.01831,-0.00948,-0.02415,0.01609,-0.04014,0.0006,-0.02031,-0.03811,0.02197,-0.00965,-0.01123,0.02333,0.03611,0.04046,0.04897,-0.01164,-0.00968,0.00296,0.00543,-0.03188,0.01951,0.01165,0.03248,-0.0418,-0.02465,0.00125,-0.04218,0.0219,0.00528,-0.04662,0.01352,0.02996,0.01679,0.00168,-0.01233,-0.01689,0.05218,0.00581,-0.03031,0.0117,-0.01507,0.00837,0.00316,0.0246,-0.0308,0.04537,-0.01423,-0.02026,-0.01218,-0.02223,0.01787,-0.01787,0.01503,-0.01926,-0.0172,0.02064,0.05328,-0.00895,0.01483,0.05424,0.00876,-0.00865,-0.025,-0.01394,0.01763,0.00285,-0.06085,-0.02356,0.04879,0.03884,0.02472,0.01921,-0.03955,-0.00966,-0.02162,-0.01355,0.02165,0.00082,0.02305,-0.02678,0.02243,-0.04531,-0.00443,-0.01438,0.03846,0.00582,-0.02405,0.01146,0.00813,0.00024,-0.02305,0.02392,0.00574,-0.02365,-0.01266,-0.00183,-0.02846,-0.00944,0.0484,-0.01249,-0.01526,-0.00222,-0.03387,0.0092,-0.02117,-0.02857,-0.00081,0.02241,0.02365,-0.02945,-0.02073,0.00754,-0.02064,-0.00917,-0.01348,0.03236,0.01103,-0.01489,0.0301,0.02037,-0.00058,0.02764,0.01207,-0.03172,-0.00919,0.01814,0.04141,0.02174,-0.03582,0.03436,0.0092,-0.0065,-0.05677,0.0169,0.02736,-0.00936,-0.03529,-0.01021,0.00563,0.01671,-0.04597,0.00631,0.01745,-0.01752,0.02922,-0.02462,-0.01972,0.02548,-0.03626,-0.02292,0.00427,0.03929,0.02429,-0.00189,-0.00313,0.0045,-0.01091,0.01748,-0.00415,0.01055,-0.04042,0.02927,0.0121,0.01944,0.00627,0.01278,0.0252,-0.01186,-0.01153,-0.00481,0.02476,0.03787,-0.00012,-0.01562,0.03151,-0.01908,0.01195,-0.00865,0.00856,0.03891,-0.00465,0.00997,0.00539,0.00555,-0.01224,0.01882,0.03266,-0.01703,0.01277,-0.00765,-0.04294,0.01187,-0.02881,-0.00374,0.03124,-0.00697,-0.00134,0.00674,-0.01211,-0.02817,-0.02661,-0.00394,-0.01496,0.01632,0.01105,0.08628,0.02448,0.01475,0.00237,-0.01878,-0.00557,0.0151,-0.01992,0.01638,0.01758,0.00055,0.00885,0.02615,0.00257,0.00316,-0.01498,0.00774,0.04441,-0.01122,0.00976,0.01885,0.00852,-0.01642,0.0499,0.00304,0.00267,-0.01457,-0.02966,-0.00051,0.02023,-0.00838,-0.02674,-0.0132,0.01389,-0.01336,0.02361,0.00207,0.02033,0.02764,-0.01924,-0.00544,0.04471,0.05396,0.03417,0.01262,0.01838,0.01707,0.02968,0.01548,-0.01774,0.02079,0.00061,0.00921,0.00368,0.02118,0.01319,0.00677,0.00451,-0.04222,0.03721,0.00792,0.03757,0.00261,0.00903,-0.01396,-0.01374,0.01485,0.01603,0.01207,0.0238,0.01845,0.01625,0.00326,-0.01091,-0.02302,0.00172,0.00516,0.00709,0.01109,-0.01806,-0.0034,0.05782,0.00361,0.0219,0.01706,-0.05539,0.01611,-0.00614,0.03556,0.01795,0.04195,0.00554,-0.0043,-0.0171,0.03404,-0.01333,0.00041,-0.04293,-0.02304,0.0107,-0.01104,-0.02823,0.03019,-0.01084,0.02049,0.00572,0.0161,-0.01275,0.02132,-0.00659,-0.02301,-0.00996,0.01937,0.00433,0.0337,0.01165,0.05615,-0.0382,-0.02364,0.0078,0.00898,0.0477,-0.03682,0.05179,-0.00628,-0.0173,0.07045,0.01386,0.00302,-0.03041,-0.00193,0.00555,-0.00713,-0.02724,-0.0156,-0.00874,0.00775,-0.02599,-0.03124,0.00905,0.00395,0.00925,-0.01451,0.02461,-0.0162,-0.03844,0.01826,0.01581,-0.00647,-0.00582,0.02993,0.03049,-0.00624,0.06838,-0.00598,0.0698,-0.00825,0.05454,0.01578,-0.00999,-0.00443,-0.02393,0.00233,0.0357,0.01359,-0.00496,0.00218,0.01517,-0.03586,0.004,0.02523,-0.02222,0.00276,-0.02028,0.02332,-0.00739,-0.01431,0.00357,-0.00315,0.01329,0.0149,0.0086,-0.03524,0.00121,-0.0272,0.0306,-0.00243,0.0593,0.03006,-0.04354,0.02271,0.04186,0.01508,0.02775,0.0107,0.0436,0.00299,-0.01599,-0.00737,0.0494,0.004,0.03984,0.03393,0.0103,0.00131,0.00421,0.00774,-0.00939,0.03588,-0.02384,0.04515,-0.0237,0.03134,0.02032,0.02601,0.0225,-0.03779,-0.02016,0.00448,0.0007,-0.02559,-0.01857,0.00212,-0.00611,0.03252,-0.01927,0.002,-0.02368,-0.05273,0.00183,-0.01308,0.00319,0.01301,-0.01264,0.00237,0.01853,-0.019,0.12706,0.01075,0.0003,-0.01589,0.00658,0.01223,0.00101,-0.00728,0.04779,-0.00097,-0.03199,-0.01341,-0.00814,-0.00327,-0.00717,0.01363,0.0211,0.04084,0.02407,-0.03611,-0.02941,0.05265,0.02023,0.0161,-0.0591,0.01107,0.01567,0.0008,-0.025,0.0084,0.03638,-0.0008,-0.01764,-0.03719,0.03231,0.0159,-0.01029,0.01658,-0.01374,0.02796,-0.00121,0.00976,0.06562,-0.01218,0.02134,-0.05294,0.01477,0.00595,-0.01462,0.00252,0.04603,0.00567,0.02463,-0.01602,-0.01048,0.02555,0.11409,-0.01283,0.0101,-0.00625,0.0122,-0.00017,-0.00905,0.01732,-0.02346,0.00755,-0.03738,0.01206,0.02079,0.00105,-0.02286,-0.00336,-0.00178,0.03762,0.02241,0.03625,0.02398,0.01482,-0.00665,0.04575,0.00214,0.00348,0.00441,0.02159,0.01978,0.00415,-0.0118,-0.00317,-0.00703,0.02769,-0.00952,0.00256,-0.01455,0.00446,0.00878,-0.01423,0.00284,-0.00747,0.01869,0.02373,0.01091,-0.00927,-0.02058,-0.02963,-0.0158,0.02688,0.00143,-0.05618,0.02431,-0.00803,-0.01246,-0.01078,-0.00711,-0.00968,-0.0001,-0.00492,-0.06529,-0.02211,-0.00349,-0.03429,0.01085,-0.03946,-0.02079,-0.04966,-0.01661,0.0092,-0.00088,-0.03781,-0.02348,-0.01595,-0.05772,0.01308,-0.00365,-0.01118,0.00762,0.04097,-0.04216,-0.01791,0.02847,0.03455,0.06241,-0.00728,0.02516,-0.00588,-0.01772,0.00556,0.02068,0.03681,-0.02904,0.03781,-0.03305,0.03748,-0.02387,0.0158,0.01635,-0.00732,-0.03043,0.01747,0.01095,-0.02013,-0.05836,-0.04915,-0.02454,-0.02053,0.00229,-0.04541,0.02045,-0.02099,-0.00677,0.03328,-0.00227,0.02352,-0.01824,-0.01242,0.00762,0.00759,-0.02414,-0.0118,-0.01068,-0.00608,0.0142,0.00451,-0.01142,0.01653,0.00049,0.01516,-0.01585,0.02002,0.04568,-0.01956,-0.0084,0.03543,-0.02565,0.04585,-0.00884,0.005,-0.00924,-0.03711,0.00587,0.048,-0.02718,-0.01223,0.02205,0.0099,-0.00069,0.02174,-0.02114,0.04231,-0.00151,-0.02472,-0.00679,0.01472,0.01328,-0.02015,-0.0015,-0.00322,-0.00111,0.01604,0.01165,-0.02157,-0.01389,-0.00398,0.02483,-0.00482,-0.0366,-0.00465,0.01175,-0.01906,-0.02762,0.02748,0.00686,0.03467,-0.00897,-0.0078,-0.00475,0.01124,-0.0575,0.04396,0.02681,-0.0129,-0.01337,0.02155,0.03062,0.00811,-0.03783,-0.01579,-0.03137,0.01962,-0.00071,0.01176,-0.04124,0.0334,-0.0002,-0.00398,0.00144,-0.04771,0.00639,-0.03233,0.01301,-0.00332,0.00496,-0.005,0.00105,0.00154,-0.03914,0.03433,-0.01538,-0.02082,-0.03276,0.00066,0.01478,0.02298,0.02356,-0.03063,-0.0301,-0.02373,-0.01953,-0.02825,0.00396,-0.02082,-0.00021,-0.02179,0.03403,0.02306,-0.02263,0.03127,0.0135,-0.01867,0.01475,-0.01343,-0.02412,0.03293,-0.00953,0.00162,-0.01808,-0.00187,0.0079,0.00462,-0.02061,-0.0385,0.0354,-0.00453,-0.03565,0.02519,0.00509,-0.01656,0.00594,-0.00801,0.01189,0.00415,-0.00558,-0.02677,-0.02655,0.02605,-0.00361,0.00518,0.01038,-0.02067,0.02947,0.0114,0.03239,-0.04276,0.03164,-0.00375,0.02144,-0.00287,0.00249,0.0442,0.02307,-0.04234,-0.01489,0.04524,0.00853,-0.0279,0.01578,-0.01111,-0.04624,-0.00257,-0.00684,-0.0378,0.04211,-0.0147,-0.02957,0.03232,0.01572,0.01757,-0.02213,-0.00844,0.07198,0.03965,-0.03457,-0.00721,-0.01269,-0.00237,0.04476,0.0112,-0.00669,-0.01834,0.01702,0.01492,-0.01692,0.00072,-0.03586,0.0329,-0.00862,0.01324,-0.00313,-0.00121,0.01327,0.0023,-0.00454,-0.0046,-0.01968,-0.03585,0.0557,-0.0078,0.01179,-0.01151,-0.0278,0.0041,0.00167,-0.02824,-0.03484,-0.01624,0.01586,-0.02089,0.02546,0.02732,-0.0025,-0.0038,-0.03264,-0.01527,0.00669,-0.04264,0.00012,-0.0251,-0.01268,-0.00542,0.00513,0.00471,0.02119,0.01631,0.03529,-0.00114,-0.01887,-0.00679,0.03517,-0.03662,-0.00213,-0.02611,0.07585,0.00039,-0.06851,-0.00988,0.00387,-0.00505,-0.01982,-0.00843,-0.0472,-0.01117,-0.04167,0.00599,0.01031,-0.01693,-0.01012,-0.044,0.00889,0.01466,-0.00547,0.00596,-0.03497,-0.02794,-0.01883,-0.02635,-0.00916,0.0202,-0.004,-0.03067,-0.02491,0.01048,-0.02504,0.01217,0.05078,-0.01331,0.02531,-0.05961,0.00464,-0.01868,-0.00164,0.00938,0.03443,0.01436,0.02879,-0.02272,-0.02035,0.02424,0.00874,-0.01367,-0.02112,-0.01302,0.00906,0.02072,0.01603,-0.00515,0.01465,-0.00944,-0.00466,-0.02732,0.03293,-0.00464,0.03219,0.03214,0.00152,0.02999,-0.01302,-0.01804,-0.01871,-0.02576,0.01421,0.0129,0.03574,0.02885,0.01381,-0.02979,0.00576,-0.01194,0.02413,-0.02359,0.00793,-0.01286,-0.00103,-0.02452,0.04468,-0.025,-0.01481,-0.04163,-0.02813,-0.00782,0.00097,0.01325,0.01738,-0.01233,-0.00449,-0.00581,0.00354,-0.03111,-0.00818,-0.03105,-0.03161,-0.02022,-0.01257,-0.00408,-0.01331,0.03472,-0.00264,0.03292,0.00577,-0.02395,-0.00394,0.04389,0.09118,-0.05714,0.02918,-0.01373,-0.02641,-0.00092,0.02714,0.04593,-0.01517,0.062,0.00367,-0.00247,0.04623,-0.00976,-0.00602,0.01197,0.00935,0.03563,0.00388,-0.00939,0.02039,-0.00718,0.0287,0.01659,0.01104,0.00142,0.00764,-0.00215,0.0096,-0.00244,0.00075,-0.00924,-0.01834,-0.00479,-0.00942,0.02317,-0.01427,-0.04573,0.00138,0.00287,0.01164,-0.01142,0.00522,-0.02916,0.02391,0.00994,0.00577,-0.01574,0.0224,0.00973,-0.01321,-0.0185,-0.0234,0.01361,0.0164,-0.01714,-0.01894,0.02496,0.04003,-0.00037,-0.04179,0.00021,0.00526,0.00302,-0.00487,0.02027,-0.00105,0.01702,0.00889,-0.03084,-0.0305,-0.03177,0.0004,-0.0096,-0.01874,0.03808,-0.01473,-0.00569,-0.03548,0.01381,-0.00803,0.02423,-0.01135,0.00787,-0.01682,0.01447,-0.02799,0.00712,0.03602,-0.01785,-0.01371,-0.0236,0.01626,0.04048,-0.01843,-0.01256,-0.01351,0.00267,0.03842,0.03522,0.00632,-0.03112,0.06373,0.01979,0.01568,0.02571,-0.03747,0.02051,-0.01783,-0.01051,0.00074,-0.01259,0.00725,0.01173,-0.00353,0.03029,-0.00695,0.0468,0.03009,0.01001,0.03716,-0.03026,-0.02362,0.00327,-0.05098,-0.0322,0.01173,-0.03116,0.00712,-0.01722,0.0069,-0.02587,-0.00849,-0.02704,0.02599,0.0214,-0.0063,-0.00687,-0.00554,-0.00625,-0.047,-0.04774,0.021,0.00686,0.0027,-0.00376,-0.0336,-0.03756,-0.03466,-0.00126,0.02226,0.04687,-0.00716,-0.01816,0.03189,-0.03592,0.09188,-0.01527,-0.02158,-0.00054,-0.00143,0.03991,-0.00694,-0.00024,-0.01121,-0.02275,-0.0272,0.0582,0.03117,0.01087,0.01646,0.0043,0.00892,0.01674,-0.01201,0.06909,-0.02754,0.01082,-0.02471,-0.02692,0.02583,-0.00455,0.01201,-0.0329,-0.00077,0.00215,-0.05931,0.01504,0.02543,0.01266,-0.00275,0.00191,-0.00839,-0.01151,-0.03414,-0.00865,-0.0066,-0.00115,-0.01278,-0.02763,-0.01457,0.00855,-0.03206,-0.00343,-0.00727,0.00275,-0.01276,0.02667,-0.02085,0.01369,-0.07337,0.00327,0.00316,-0.01923,0.00356,0.00426,-0.01982,-0.01409,-0.02468,-0.01965,-0.01151,-0.01085,0.02273,0.0063,0.02544,-0.04729,-0.00612,-0.00602,0.03026,-0.03393,0.00385,-0.001,0.004,-0.02491,0.04351,-0.01137,0.0296,0.00956,0.01985,0.0309,-0.0177,0.0018,-0.03726,-0.01522,0.03326,-0.00261,-0.00985,0.0065,0.01947,0.06531,0.02187,0.00936,0.00413,0.01433,-0.01774,0.0223,-0.0173,0.02067,0.03636,0.02628,0.02237,0.00059,-0.04523,0.02402,0.01034,0.00442,-0.00707,0.00771,0.02531,-0.01415,0.02728,0.0353,0.02555,-0.00172,0.02832,0.01774,-0.00178,-0.03749,-0.0117,0.00688,0.04428,0.0106,0.04133,-0.01112,-0.01221,0.02468,0.02219,0.01506,0.00616,-0.01016,0.00206,0.04745,0.02497,0.01481,0.03971,0.00241,-0.02031,0.01408,0.06486,-0.00487,0.00246,-0.00444,0.00246,0.00165,0.01613,-0.02021,0.00308,-0.0264,0.00359,0.00719,-0.02557,-0.00122,-0.03508,-0.01109,0.01721,-0.01155,0.01636,0.00129,-0.006,0.00763,-0.03531,-0.03435,0.02513,0.00867,0.01941,-0.01751,-0.00585,-0.05357,-0.02313,-0.04986,-0.01115,-0.07019,-0.02543,0.02092,-0.00335,-0.00551,-0.00276,-0.01351,0.00354,0.01818,-0.03114,0.01933,-0.01499,-0.02288,0.03638,-0.00387,-0.00159,-0.03433,-0.19303,-0.01616,0.09574,0.13787,-0.04683,0.01008,0.01802,-0.00366,-0.04455,-0.00129,-0.03511,-0.03606,0.02635,-0.03445,-0.02208,-0.03302,-0.00454,0.00192,-0.012,0.02109,-0.02245,0.00571,-0.0129,-0.05931,0.01172,0.02674,-0.01187,-0.0331,0.03619,0.03709,0.00133,-0.03208,-0.05901,-0.01688,0.02375,-0.0356,-0.01404,0.05028,-0.01937,0.0225,0.00626,0.01213,-0.03301,0.00053,0.00675,0.01563,-0.00863,0.00602,0.02364,0.01808,0.00341,-0.00779,-0.0184,0.01523,0.00288,0.02193,0.00316,-0.03104,-0.00207,-0.02197,0.0702,-0.0181,-0.00653,0.01476,0.00921,0.03001,-0.00766,0.01629,-0.00754,-0.00116,-0.0237,-0.02166,0.00765,-0.03304,0.01157,-0.02896,0.01203,-0.02695,0.01603,-0.04224,-0.02523,-0.00885,-0.03047,0.02516,-0.02996,0.00988,0.02931,-0.01431,-0.00674,0.03453,0.02217,-0.01316,-0.05529,0.02559,0.005,-0.02216,0.01284,-0.0142,0.05423,0.0181,0.05813,0.00748,0.02342,0.04131,-0.01682,0.03912,0.03898,-0.00041,0.049,0.00563,0.02721,-0.02165,-0.03027,-0.05055,-0.04077],"CALIBRATION":[0.02436,-0.01238,-0.00588,0.03955,-0.02825,0.02969,0.00598,-0.02379,0.04004,0.02648,-0.01184,-0.05985,0.03007,0.01268,-0.01193,-0.00606,-0.02724,-0.00654,0.01924,0.00062,0.01344,0.0048,0.02416,0.00154,0.00702,0.03942,0.00294,-0.00127,0.03313,-0.02312,-0.01285,0.01644,-0.04073,0.01905,-0.0158,-0.02419,-0.02834,0.02116,0.01657,0.00534,0.02379,-0.01905,0.06184,0.01928,-0.0409,0.01472,0.00464,-0.03424,-0.02873,-0.02083,0.02598,0.01674,0.00964,-0.00796,0.03548,0.03206,-0.00313,0.00406,-0.0157,0.02452,0.01925,-0.0063,-0.00114,0.00611,0.01015,-0.01689,0.00538,0.00906,-0.01786,0.01814,0.00888,-0.00393,0.00191,0.03101,-0.03969,-0.00379,0.01964,-0.01385,-0.02779,0.00145,0.03519,0.00338,-0.00689,0.01351,0.03571,0.00791,0.00149,-0.0071,0.02767,0.00092,-0.00245,-0.00077,-0.00597,0.02137,0.0381,0.04837,0.03322,0.02757,0.05987,-0.04901,0.035,-0.01279,-0.00395,-0.06477,-0.02812,0.0039,0.03019,0.0039,0.03934,-0.03642,-0.0293,-0.02519,-0.01714,-0.01624,0.01699,-0.03897,0.00493,-0.00904,-0.01032,0.00222,-0.01612,0.00373,0.03174,0.0272,0.03241,0.04731,0.01889,-0.00505,0.01271,-0.00319,-0.04851,0.01247,0.01301,0.04664,-0.06645,-0.02062,-0.00229,-0.02927,0.00289,0.00425,-0.01289,0.03444,0.01298,0.01679,0.01414,0.00577,-0.03971,0.0222,0.02213,-0.05188,0.02092,-0.01171,-0.01828,-0.01086,0.01796,-0.02605,0.04489,-0.01006,-0.01029,-0.01847,-0.02323,0.02241,0.0099,0.00257,-0.00752,-0.03576,0.03016,0.04705,0.02478,0.00965,0.04235,-0.00344,0.01425,-0.00349,-0.00858,0.00437,-0.00058,-0.03995,-0.0061,0.03275,0.04053,0.02895,0.02212,-0.04341,-0.00011,-0.00535,0.00196,-0.00783,-0.01014,0.00697,-0.03986,0.01507,-0.04775,-0.01475,-0.02954,0.02538,0.03383,-0.02884,0.01846,0.01316,0.0239,-0.00598,0.00528,0.01792,0.0154,-0.01323,-0.00382,-0.01385,0.03599,0.02485,-0.02745,-0.01435,-0.00722,-0.03532,0.01094,-0.02227,-0.03024,-0.00493,0.02625,0.02604,-0.0174,-0.00493,0.00173,0.0139,-0.00487,-0.0254,0.03982,-0.03428,0.02091,0.02703,0.06112,-0.03202,0.04339,0.03,-0.02882,0.0111,0.05494,0.02801,-0.00054,-0.04091,0.02469,-0.01902,0.01367,-0.07945,-0.00967,0.02369,0.00347,-0.02662,-0.00493,0.00949,0.00283,-0.03751,0.02398,0.01977,-0.02596,0.01139,-0.02491,-0.03065,0.02962,0.00314,-0.03442,0.03013,0.03857,0.03564,-0.0255,-0.01401,0.00089,-0.02596,0.0213,-0.02023,0.00377,-0.02517,0.04585,0.02814,0.01525,0.01341,0.00307,0.01761,-0.00217,-0.00382,-0.00645,0.03473,0.01392,0.00418,-0.03075,0.01785,0.01564,0.00267,-0.0006,0.01709,0.04232,-0.03019,-0.00586,0.00829,0.01472,-0.00731,0.05024,0.05394,-0.00179,-0.0288,-0.00793,-0.0159,0.02184,-0.02095,0.0039,0.03283,-0.01868,0.00456,0.02956,0.00203,-0.00558,0.01614,-0.01753,-0.00468,0.02003,-0.00273,0.04143,-0.01048,0.00392,-0.0038,-0.04446,-0.01726,0.00865,-0.01466,-0.00106,0.001,-0.02124,0.01296,0.00769,0.02579,-0.00371,0.00076,-0.00589,0.02633,-0.00325,0.02471,0.00255,0.01315,-0.00059,0.04272,0.00975,0.02994,-0.01705,-0.04183,-0.01298,0.01542,-0.02811,-0.02204,-0.00268,0.02648,0.00215,0.01413,0.01209,0.01026,0.05769,-0.02552,0.03922,0.03904,0.0541,0.04523,0.02118,0.01621,0.04136,0.04181,0.01629,-0.05366,0.01468,-0.02706,-0.00052,0.05335,0.02148,-0.00446,0.01236,0.02235,-0.04423,0.01378,-0.0004,0.01926,-0.00752,0.01685,-0.01089,-0.01642,0.01218,0.02093,0.01007,0.01598,0.00267,0.01672,0.00166,0.00457,-0.03748,-0.01146,0.0171,-0.02225,0.0091,-0.03997,0.02021,0.02092,0.00961,0.00874,0.0334,-0.08287,0.00024,0.00639,0.02853,-0.00179,0.03298,0.01667,0.03553,-0.01234,0.0409,0.01455,0.00345,-0.02816,-0.01606,0.01696,-0.01244,0.00545,-0.00495,-0.02395,0.02613,0.03688,0.03483,-0.00345,0.0313,-0.0261,-0.02839,-0.02208,0.03325,0.0078,0.04444,-0.00065,0.05894,-0.02576,-0.00316,-0.00122,-0.02279,0.02771,-0.02268,0.04139,0.01679,0.0056,0.05188,0.03209,0.01313,-0.02757,-0.01546,0.01172,-0.02978,-0.0323,-0.00274,0.00158,0.04589,-0.06444,-0.02924,0.03995,0.00076,-0.01509,-0.00786,0.02177,-0.01275,-0.05684,0.02191,0.02011,-0.00011,-0.00012,0.00012,0.01028,-0.00211,0.06356,-0.03311,0.02594,0.0094,0.0622,-0.00584,-0.0162,-0.0056,-0.02154,-0.00286,0.01764,-0.00693,-0.03254,-0.01943,0.00296,-0.00565,0.0146,0.02773,-0.03031,0.00651,-0.01516,0.01845,-0.03666,0.00673,0.00479,-0.00731,0.00655,0.00728,0.00364,-0.01306,-0.02008,-0.01774,0.04511,0.0059,0.02137,0.04593,-0.04139,-0.00213,0.02519,-0.00508,0.01634,-0.02093,0.02712,0.00345,0.01146,-0.00192,0.04254,-0.02054,0.0636,-0.00603,0.04505,-0.03078,0.03298,0.00158,-0.0051,0.00548,-0.0303,0.03027,-0.03948,0.01019,-0.00433,0.03695,0.03101,-0.01976,-0.00638,-0.00055,0.01828,-0.01962,-0.01463,0.00255,-0.0043,0.05473,-0.01474,0.00989,0.00167,-0.04086,-0.00134,0.01605,-0.0164,0.0174,-0.01566,-0.00419,-0.02368,-0.05192,0.07616,0.00897,0.02866,-0.00805,0.01949,0.01971,-0.00713,0.02221,0.05014,-0.00663,-0.05245,-0.01978,-0.01874,-0.00276,-0.01599,0.04179,0.02879,0.02204,0.01874,-0.00808,-0.00821,0.02578,-0.01198,0.00861,-0.05291,0.00652,0.00353,-0.01573,-0.03327,-0.00845,0.04348,-0.00416,-0.04687,-0.02757,0.03852,0.00029,0.00068,0.03188,-0.01392,0.03491,-0.0017,-0.01606,0.04287,-0.03121,-0.00145,-0.04115,-0.00666,0.02051,-0.03428,-0.00436,0.00878,0.02211,0.03322,-0.01005,0.0093,0.0458,0.03227,-0.0225,0.00387,-0.009,-0.01372,-0.00277,0.00351,0.00858,-0.00487,0.00901,-0.03152,0.00882,0.00585,0.00061,-0.02675,-0.004,-0.02554,0.02736,0.03345,0.04079,0.02353,0.00604,-0.02182,0.02057,0.04618,0.01928,-0.00362,0.00981,0.01309,-0.02385,0.01166,-0.01191,-0.00908,-0.00344,-0.01295,-0.00511,-0.01881,-0.01923,0.0087,0.00713,-0.00133,0.00268,0.00468,0.01681,-0.00709,-0.01368,-0.04601,-0.0124,-0.042,0.06015,-0.02816,-0.05146,-0.01109,-0.02579,0.01005,-0.01217,-0.01732,0.01986,-0.0029,0.01081,-0.03757,-0.00694,-0.0079,-0.02899,0.05651,-0.03722,-0.02615,-0.01888,-0.01488,0.01031,0.02046,-0.0494,-0.00896,-0.01469,-0.05637,0.01398,-0.03335,-0.00409,0.01325,0.04892,-0.04233,0.00921,0.02217,0.02632,0.07515,0.00642,0.01655,0.00128,-0.01086,-0.00216,0.0108,0.03938,-0.03157,0.05647,-0.02091,0.00878,-0.01785,0.00081,0.0108,-0.03087,-0.01593,0.00356,0.01262,-0.01347,-0.04416,-0.02527,-0.04046,-0.00919,-0.00703,-0.00933,0.00978,-0.04222,-0.00365,0.04688,0.00278,0.01863,-0.01026,-0.02488,0.00307,-0.01285,-0.03354,-0.02126,-0.00725,-0.02438,0.02769,0.0592,0.00126,0.02852,0.00545,0.01299,-0.01682,0.02681,0.00979,-0.028,-3e-05,0.00537,-0.03194,0.03594,-0.01839,-0.00446,-0.01932,-0.0199,0.02334,0.0303,-0.0121,-0.03003,0.00286,0.00783,0.0044,0.02119,0.00276,0.05571,0.01865,-0.02272,0.00511,0.00829,0.03614,-0.03668,0.00061,-0.01449,-0.02447,0.00741,0.03467,-0.0173,0.00986,-0.00346,0.04079,-0.00925,-0.0108,0.01435,0.01165,-0.00506,-0.03076,0.02221,0.00473,0.06501,0.00713,-0.02792,-0.01658,-0.00915,-0.0383,0.03055,0.05779,-0.00219,0.00887,0.04377,-0.00017,0.00105,-0.00734,-0.00688,-0.01323,-0.01208,0.03511,0.01005,-0.05688,0.05017,-0.00936,0.00513,0.03094,-0.0389,-0.01414,-0.02797,0.02595,0.00678,-0.01511,0.00372,-0.02627,0.01744,-0.0166,0.03497,-0.00563,-0.00573,-0.0365,-0.00147,0.03411,0.01375,0.02927,-0.01031,-0.03951,-0.00781,-0.01968,0.00085,-0.00427,-0.03047,0.00184,-0.0104,0.03598,0.03985,-0.00753,0.00915,0.0303,-0.02158,0.0138,0.00319,-0.01675,0.02123,0.00641,0.01684,-0.0391,-0.00033,0.00368,0.02845,-0.03052,-0.04167,0.03715,-0.00062,-0.03188,0.01638,0.0352,-0.01419,-0.01189,0.00328,6e-05,0.03169,-0.00524,-0.01202,-0.05416,0.01459,0.00021,-0.0158,0.01783,-0.03518,0.01197,0.01627,0.02333,-0.05526,0.02245,-0.00775,0.01815,0.00046,0.01277,0.0244,0.01739,-0.04379,-0.02609,0.01999,0.0179,-0.03032,0.01288,-0.03167,-0.03965,0.00533,-0.02478,-0.02388,0.04281,-0.00751,-0.02793,0.03302,0.01291,0.01478,-0.00161,0.004,0.04312,0.0308,-0.04367,-0.02254,0.00348,0.00015,0.04017,-0.00023,0.00144,-0.03145,0.03041,0.01203,-0.00214,0.0215,-0.02923,0.03731,-0.00314,-0.0153,-0.00435,-0.01731,0.03074,0.02409,-0.00741,-0.01083,-0.03781,-0.02421,0.04869,-0.01554,0.01723,0.01148,-0.02508,-0.02088,0.00358,-0.0272,-0.0363,0.01017,0.01442,-0.01346,0.01679,0.0102,0.0253,-0.00826,-0.01701,-0.02326,0.00225,-0.05115,0.011,0.00164,-0.0101,-0.01782,-0.00272,0.01889,-0.00583,0.01151,0.0428,0.00543,0.00208,-0.02519,0.02042,-0.03491,-0.01292,-0.03097,0.07183,-0.02824,-0.07676,-0.02013,-0.00887,0.00301,-0.01389,-0.0111,-0.03647,-0.02082,-0.06917,-0.01636,0.00883,0.01849,0.02664,-0.01736,0.04437,0.00338,-0.01496,0.03127,-0.029,-0.05132,-0.02191,-0.023,-0.0071,0.02063,-0.00685,-0.0388,-0.00512,0.02188,-0.03513,0.03722,0.05049,0.02583,0.0215,-0.05958,-0.01792,-0.01261,-0.01255,0.01136,0.04307,0.00472,0.02167,-0.0041,-0.00455,0.03388,-0.0031,-0.0201,-0.00113,-0.01119,0.00349,-0.00717,0.02356,-0.02403,0.02318,-0.01168,-0.01654,-0.02196,0.02582,-0.01484,0.02851,0.03716,0.03328,0.0377,-0.01771,-0.01351,-0.02631,-0.00608,-0.00582,0.00077,0.03475,0.01683,0.01555,-0.02917,-0.01311,-0.0047,0.02985,-0.03336,0.00307,-0.01796,0.01591,-0.03081,0.04367,-0.05027,-0.03037,-0.03734,0.01522,-0.00241,-0.0034,0.029,0.02361,-0.0221,-0.00058,-0.01784,0.04265,-0.02712,0.01498,-0.03748,-0.01217,-0.00495,-0.02263,-0.01701,-0.00094,0.04386,0.01121,-0.01874,-0.01381,-0.00308,-0.00224,0.04715,0.05436,-0.07882,-8e-05,-0.01087,-0.01845,-0.00404,0.02257,0.04811,-0.01373,0.07184,-0.00445,0.01417,0.02399,-0.02044,0.00903,0.01188,0.00098,0.05307,-0.01292,-0.01421,0.01461,-0.01204,0.05511,0.00762,0.00237,-0.01384,0.03056,0.01395,0.00975,-0.00148,0.05831,-0.0088,-0.00272,-0.00426,-0.00278,0.02674,0.00143,-0.0491,-0.03261,-0.00408,-0.0019,-0.03506,-0.0046,-0.02676,-0.00284,0.02589,0.00988,-0.00726,0.01366,0.0039,-0.04693,0.00353,-0.00864,0.03426,0.0377,-0.01651,-0.03024,0.04341,0.04013,-0.00496,-0.02168,-0.01229,0.01146,0.00233,-0.00361,0.03227,0.00276,0.02507,0.01415,0.02342,-0.06193,-0.03114,-0.00414,0.00592,-0.01597,0.02347,0.00239,-0.00442,-0.06542,0.03794,-0.00714,0.03454,-0.00725,0.00733,-0.00323,0.02652,-0.01859,-0.00169,0.0231,-0.02391,-0.0344,-0.0109,0.04569,0.04174,-0.01101,0.00674,-0.02223,0.00741,0.0225,0.02892,-0.01512,-0.02909,0.06769,0.01328,-0.00687,0.03676,-0.03751,0.02812,-0.00011,-0.03171,-0.0136,-0.02955,0.01779,0.00503,0.01072,0.01867,0.00659,0.02138,0.05328,-0.01467,0.06496,-0.0235,-0.0175,-0.00049,-0.03515,-0.01572,0.01627,0.00733,0.04752,-0.03414,-0.00687,-0.03064,0.00352,-0.05507,0.04489,-0.01299,-0.00935,-0.0083,0.02616,-0.02185,-0.05476,-0.00356,0.03016,-0.00725,-0.02305,-0.0174,-0.04283,-0.02499,-0.02484,-0.01128,0.00983,0.03682,-0.0138,-0.02137,0.02609,-0.04312,0.05139,0.00052,0.00596,-0.00442,-0.00972,0.05277,0.00786,-0.03923,0.01671,-0.0248,-0.03835,0.0584,0.02515,0.0246,0.01414,-0.00275,-0.00385,-0.00885,-0.0161,0.02626,-0.02563,0.0153,-0.00255,-0.01394,0.04747,-0.0362,0.00382,-0.05482,0.00204,-0.01382,-0.03583,0.00479,0.01526,0.01285,-0.02788,-0.00682,0.00626,0.0053,-0.05386,0.02228,0.01945,0.02725,-0.01132,-0.03549,0.01179,-0.01205,-0.062,0.02346,0.00574,-0.01592,0.01076,0.02394,-0.00763,0.01282,-0.05108,-0.00921,0.00357,-0.01688,0.02222,-0.00137,-0.01574,0.01346,-0.01093,-0.04265,-0.00838,0.00418,0.02672,-0.01147,-0.02116,-0.04652,-0.00202,-0.01213,0.02082,-0.02632,-0.00021,0.0019,0.00313,-0.0214,0.05256,-0.0162,0.03466,0.01935,0.01963,0.02492,0.00878,0.00307,-0.01055,-0.00621,0.00253,-0.00857,-0.01289,0.01345,0.0497,0.03357,0.00995,-0.02298,-0.01899,0.02767,-0.01836,0.01656,-0.01589,0.02088,0.01143,0.04199,0.03214,0.02476,-0.05205,0.0393,0.03528,-0.01011,-0.01715,0.00054,0.0241,-0.03104,0.01268,0.03378,0.03059,-0.03021,0.03037,0.01842,-0.00851,-0.01092,-0.03648,0.00803,0.03215,-0.01626,0.04806,0.00802,0.00693,0.02346,0.00383,0.02778,0.01674,-0.01806,0.00335,0.06323,0.04047,0.01442,0.01615,-0.01531,-0.01597,-0.00341,0.06885,-0.01967,-0.00446,0.01119,-0.01106,0.00274,-0.00025,-0.01822,-0.02312,0.0013,0.00265,0.00551,-0.00796,-0.00867,-0.00984,0.01646,0.02702,0.00127,0.01298,-0.01221,-0.01922,0.01976,-0.0164,-0.01606,-0.01464,-0.00228,0.01138,-0.03809,-0.03305,-0.03436,-0.00667,-0.02326,0.01387,-0.03342,-0.03625,0.00426,-0.00484,0.00936,-0.00372,0.00522,0.03153,-0.00192,-0.0586,0.00943,-0.0002,-0.02607,0.0373,-0.01307,-0.02181,-0.03522,-0.01411,-0.01734,0.05595,0.04698,-0.01757,-0.00264,0.03344,-0.01019,-0.02862,-0.00695,-0.01577,-0.0326,-0.00619,-0.03309,-0.01737,-0.01764,-0.01467,0.01476,0.00303,0.00718,-0.02698,0.00438,-0.02091,-0.0406,-0.0038,0.01767,-0.03099,-0.00316,0.00911,0.03605,-2e-05,-0.00359,-0.0287,0.00569,0.0189,-0.03848,-0.01888,0.02011,-0.00949,0.01209,0.0058,-0.00599,-0.00319,0.01204,-0.0072,-0.00626,-0.01985,-0.01236,0.03962,0.02433,-0.02421,0.00301,-0.02607,0.02059,0.00072,0.0076,0.01205,-0.05693,0.01176,-0.01103,0.04336,-0.0006,-0.01189,-0.01924,-0.00626,0.02233,-0.00489,0.03318,-0.02049,-0.00197,-0.01938,-0.04356,0.03075,-0.03543,0.01911,-1e-05,0.00776,-0.01506,0.01686,-0.00669,-0.03732,0.00645,-0.04666,0.02412,-0.02439,0.0131,0.01346,-0.01361,0.00979,0.04843,0.05349,-0.02822,-0.00846,0.03874,0.00859,-0.0046,0.03962,-0.00319,0.06519,0.04189,0.05915,-0.00437,0.03484,0.02277,-0.02152,0.02243,0.04262,0.00988,0.04482,0.01327,0.02089,-0.03624,-0.0252,-0.05956,-0.06819]},"true_probe":{"ABYSMAL":[0.00302,-0.03149,0.00596,0.02894,0.00327,0.01716,0.01144,-0.00831,0.02521,-0.001,5e-05,-0.00681,0.01411,-0.00027,0.00076,0.00032,-0.01204,0.01767,-0.01032,-0.06138,-0.00522,0.02615,-0.01046,0.00209,0.00692,0.03331,-0.00631,-0.02037,0.02919,-0.02837,-0.00656,0.00966,-0.02603,0.00522,-0.02016,-0.03578,-0.03452,0.0208,0.00931,-0.01376,0.01812,-0.02313,0.0681,0.03297,-0.00355,0.02297,0.03986,-0.0518,0.01403,0.00547,0.0431,-0.00457,0.00645,-0.03,0.07023,0.05355,-0.01073,-0.01409,-0.04356,0.04156,0.00952,-0.00379,0.00309,0.00604,0.01954,-0.00034,0.01744,0.01955,0.01765,0.01451,0.01259,-0.01172,-0.01195,0.01317,-0.00304,-0.0144,0.0417,-0.0058,-0.01683,-0.00673,0.06007,-0.01792,0.00028,0.00827,0.01693,-0.01041,-0.01368,-0.00848,0.00766,0.01372,-0.01062,-0.02103,0.02037,0.00284,0.02486,0.04446,0.01695,0.00866,0.03902,-0.0292,0.04431,-0.02275,-0.00273,-0.04405,-0.02948,0.00925,0.02494,0.01103,0.02839,-0.01971,-0.02832,-0.02619,-0.01048,-0.00971,0.02216,-0.02231,-0.00745,0.00337,0.01012,0.00099,-0.00661,0.03429,0.01222,0.00985,0.01084,0.02379,0.025,0.00312,0.02794,-0.01105,-0.02654,0.00252,0.00125,0.04256,-0.05405,-0.00149,-0.0095,0.00025,-0.01353,0.0081,0.02482,0.01771,0.00135,0.02758,0.01458,0.01747,-0.04334,-0.00818,0.01912,-0.04222,0.02375,-0.01538,-0.03678,-0.01051,0.01427,-0.011,0.03206,0.00781,0.00882,-0.0151,-0.00173,0.00346,0.03112,-0.01252,0.01707,-0.01658,0.02332,0.01856,0.04338,0.00664,0.01368,-0.02212,0.02384,0.00268,-0.00241,-0.02175,-0.00352,0.00915,0.015,0.0005,0.04708,0.03142,0.01523,-0.04652,0.00082,0.02629,0.0103,-0.03241,-0.0328,-0.00381,-0.04393,-0.00814,-0.02507,-0.01975,-0.04031,-0.00194,0.03733,0.01006,0.03916,0.00394,0.04373,0.02789,-0.01887,0.02204,0.04345,-0.01566,-0.00316,0.01115,0.0497,-0.01356,-0.01231,-0.00951,-0.0052,-0.0502,-0.01232,-0.02461,-0.00055,-0.01958,0.0314,0.01875,-0.00685,0.0167,0.01043,-0.02274,-0.01707,-0.01394,0.02442,-0.05874,0.00277,0.00386,0.06295,-0.02143,0.04165,0.04739,-0.01137,0.02638,0.05389,0.02975,-0.01522,-0.0341,0.00971,-0.04043,0.02585,-0.05158,-0.03982,0.01251,0.01748,0.00595,0.00685,0.02079,-0.01474,0.00137,0.02505,0.02028,-0.00639,-0.00812,-0.02832,-0.01599,0.03005,-0.01213,-0.01733,0.02767,0.0144,0.04495,-0.03842,-0.02358,0.00884,-0.01655,0.01455,-0.02533,-0.01113,0.00659,0.0542,0.01055,0.0243,0.04458,-0.00465,0.01847,0.01606,-0.00021,-0.00105,0.0373,-0.00237,0.01558,-0.01118,-0.00293,-0.00279,-0.00688,-0.00047,0.02397,0.02172,-0.04572,-0.0078,0.00714,-0.0032,0.00685,0.04485,0.03591,0.01168,-0.06711,0.00132,0.02878,0.02781,-0.01551,0.00755,0.00037,-0.01705,-0.00441,0.02369,0.01609,0.00144,0.04474,-0.03131,-0.02766,0.0092,-0.00738,-0.03331,-0.03325,-0.0008,0.00855,-0.05869,-0.01305,0.00899,0.01332,-0.01265,-0.01407,-0.02654,0.00997,-0.01251,0.02181,-0.00263,0.00299,-0.00126,-0.00453,0.00738,0.01527,-0.01087,-0.00618,0.01614,0.01793,0.00677,0.04216,-0.02675,-0.03552,-0.01202,0.00165,-0.01682,0.00399,0.00699,0.04082,0.00148,-0.02991,0.01898,0.00966,0.06222,-0.0193,0.08651,0.03518,0.04123,0.04732,0.03322,-0.01325,0.04503,0.04359,0.01917,-0.04306,0.00741,-0.02398,0.00552,0.07097,-0.003,-0.03479,0.01448,0.02206,-0.04927,-0.00559,0.01632,0.00779,-0.01308,0.00249,-0.02364,-0.00974,-0.00078,0.01002,0.00702,-0.01066,-0.00656,0.01172,-0.00769,0.01161,-0.0282,-0.02596,0.00923,-0.01619,-0.01559,-0.0328,0.03448,-0.02027,-0.0008,-0.00751,0.04173,-0.05986,-0.00272,0.01637,0.02911,-0.02823,0.00961,0.01643,-0.0322,0.00705,0.00721,0.03249,0.00624,-0.00124,0.00917,0.01481,-0.02392,0.02407,-0.02195,-0.03405,0.02344,0.06511,0.04531,0.00664,0.06124,-0.02624,-0.01865,-0.02107,0.03012,0.02784,0.03456,-0.00449,0.03778,0.00044,0.02826,-0.01007,-0.03482,-0.03145,-0.02291,0.02563,0.02091,0.00156,0.03733,0.02628,0.00696,-0.01839,-0.01647,-0.00889,-0.02563,-0.00762,0.02494,0.02481,0.05798,-0.07178,-0.01684,0.06634,-0.00069,-0.02934,0.0037,0.00715,-0.01149,-0.04597,0.02117,0.0055,0.00274,0.02582,-0.01311,-0.00199,0.01037,0.02275,-0.03149,-0.01616,-0.00093,0.04954,-0.02481,0.00515,-0.01714,-0.02006,0.02936,-0.00344,-0.00753,-0.05815,-0.0133,-0.02865,0.01464,0.0385,0.00097,-0.04455,0.00179,-0.01848,0.00102,-0.02172,0.01894,0.00898,-0.02636,0.00105,-0.00497,0.00116,0.02325,-0.02625,0.00817,0.0443,0.02128,-0.0276,0.0344,0.00173,-0.02936,-0.00985,0.00391,0.00685,-0.02692,-0.00534,0.01742,0.02961,0.01709,0.01322,-0.01454,0.05916,-0.044,0.03951,-0.00588,0.07224,0.01541,-0.00256,-0.02536,-0.0401,0.00776,-0.03849,-0.02307,-0.02691,0.04121,0.0289,-0.00193,0.01245,-0.01316,0.03336,0.00046,-0.01148,0.00732,-0.00972,0.04596,-0.02761,0.00372,0.01249,-0.02592,-0.01421,0.02826,-0.03474,0.00375,-0.00559,-0.00927,-0.05543,-0.05747,-0.01258,0.00887,0.04125,-0.0007,0.02838,0.01085,-0.00538,0.02825,0.03436,-0.00502,-0.05751,-0.01319,-0.02477,-0.01403,0.00019,0.05988,0.02752,-0.02013,-0.00701,0.02251,0.0181,-0.01317,-0.03927,0.01928,-0.02445,0.00563,-0.01743,-0.04891,-0.00744,-0.01854,0.01395,-0.00263,-0.04538,-0.00447,0.03748,-0.01222,0.00921,0.02531,-0.02912,0.01849,-0.00312,-0.02743,0.00676,-0.05533,-0.02745,-0.0245,0.00352,0.03288,-0.04226,-0.01953,-0.01351,0.02788,0.01451,0.00339,0.01624,0.05003,0.04598,-0.0083,0.00378,0.00427,-0.03165,-0.00454,0.02452,0.00333,0.02303,-0.00417,-0.01597,0.00399,-0.0059,0.00599,-0.00619,0.00741,-0.02596,0.01865,0.04216,0.01343,0.02271,-0.006,-0.00124,-0.00971,0.06687,0.01735,0.00036,-0.00761,0.00212,-0.02799,0.03124,-0.00845,0.00313,-0.01389,-0.02655,0.00329,0.00265,-0.02628,0.01136,0.02179,-0.0009,0.01667,-0.01456,0.01107,-0.01606,-0.03729,-0.05257,0.00841,-0.0482,0.05922,-0.03949,-0.0238,0.00153,-0.01078,0.02544,-0.0082,-0.0225,0.03961,-0.01064,0.00159,0.01966,-0.01204,0.011,-0.02064,0.07368,-0.03044,-0.01587,0.01565,-0.01641,0.00303,0.03903,-0.03608,-0.0113,0.01027,-0.04659,0.00417,-0.04027,-0.00708,-0.00236,0.0487,-0.02281,0.03147,0.0174,0.04358,0.0513,0.02733,0.01502,0.02428,-0.01773,-0.01,0.00787,0.03678,-0.00915,0.04447,0.00857,-0.02986,-0.01446,-0.00965,0.01842,-0.046,0.00435,0.01609,-0.00648,-0.00121,-0.01084,0.00028,-0.03422,0.01481,-0.02637,0.00781,-0.00407,-0.03922,-0.01395,0.02703,-0.00597,0.00618,0.01495,-0.02825,-0.00246,-0.03595,-0.02224,-0.02541,0.00488,-0.03225,0.01424,0.09126,0.02763,0.02003,0.03013,0.00017,-0.02517,0.0135,-0.02067,-0.0329,-0.01221,-0.04351,-0.03415,0.01278,-0.03245,-0.00385,-0.02113,0.01412,0.02296,0.0306,-0.01275,-0.01573,-0.00697,0.01429,0.00372,-0.0119,0.009,0.03412,0.02793,-0.02528,0.0017,-0.01155,0.03994,-0.04224,0.00529,0.00382,-0.05706,0.00036,0.04442,-0.01451,0.02746,-0.00524,0.0284,-0.01139,0.0155,0.0188,-0.00385,0.00329,-0.01833,0.00732,-0.00373,0.06202,0.03054,-0.02993,0.00135,-0.04054,-0.01055,-0.00121,0.0587,0.02042,0.06374,0.05281,-0.03407,-0.01141,0.01128,0.01549,-0.00083,-0.00838,0.05548,0.00827,-0.0356,0.04564,-0.0174,0.0011,0.03814,-0.00574,-0.00584,-0.03303,0.02241,0.00644,-0.03953,-0.00688,-0.04671,0.00347,-0.01554,0.0301,-0.00192,0.01574,0.00505,-0.00102,0.03526,0.01116,0.01245,0.0102,-0.00823,0.01021,0.00822,0.02622,-0.00577,-0.02256,-0.03198,0.00319,0.01933,0.04188,0.00246,-0.00281,0.03418,-0.0039,2e-05,0.00946,0.00288,-0.02633,0.02533,0.02219,-0.03225,-0.0168,-0.00738,0.04554,-0.02148,-0.03041,0.02961,-0.00135,-0.00297,-0.00219,0.03607,0.02079,-0.01108,0.02975,-0.00649,0.03179,-0.00494,0.00213,-0.12337,0.00575,0.01364,-0.029,0.02654,-0.03134,-0.00192,0.0268,0.00685,-0.04535,-0.01803,0.00108,0.0161,-0.00867,0.01926,-0.00585,0.00297,-0.03503,-0.01429,-0.01412,0.00363,-0.01096,-0.00035,-0.04154,-0.01308,-0.00132,-0.02121,0.00783,0.02508,0.00874,-0.00991,0.00138,0.0123,0.02647,0.03467,0.02903,-0.02847,-0.00285,-0.03637,-0.01707,0.02679,-0.034,0.03257,-0.01142,-9e-05,-0.01942,0.01037,0.0188,-0.00205,0.02872,-0.01923,0.02016,-0.00709,-0.04164,0.00118,-0.01921,0.03228,0.0464,0.00491,-0.01128,-0.04911,0.00999,0.02937,-0.03229,0.01377,0.01656,-0.00888,-0.03019,0.00673,-0.02521,-0.02187,0.01975,0.00478,-0.0171,-0.00807,0.04455,0.02441,-0.00573,0.01742,-0.02233,0.00055,-0.05111,0.03192,0.02468,0.00538,-0.03019,-0.00689,0.02183,-0.03121,0.02353,0.00809,-0.00329,0.00723,-0.04217,0.00622,-0.01467,-0.02417,-0.0183,0.04407,-0.03527,-0.05348,-0.0201,0.00275,0.00535,-0.00577,-0.01089,-0.00958,-0.02168,-0.07176,-0.01771,0.01952,0.04104,0.01694,0.02193,0.06051,-0.00675,-0.02779,0.04145,-0.00594,-0.04347,-0.02179,-0.00359,0.00052,-0.00368,0.01062,-0.02211,0.01872,0.03257,-0.02276,0.02661,0.02472,0.04917,0.00718,-0.03291,-0.02624,0.00613,-0.01508,0.00273,0.03057,-2e-05,0.0283,0.01686,-0.00209,0.04732,-0.00573,-0.0082,0.00043,-0.00862,-0.0156,-0.01114,0.01814,-0.01978,0.02645,-0.00647,-0.0092,-0.00242,0.0038,-0.01114,-0.00019,0.02572,0.03075,0.0171,-0.01101,-0.01139,-0.02829,0.02366,-0.0123,-0.00792,0.01405,0.0161,0.00128,-0.01794,-0.03399,0.01295,0.03043,-0.01544,0.0115,-0.01551,0.03582,-0.02087,0.02538,-0.03881,4e-05,-0.01811,0.04253,-0.00179,-0.00971,0.00721,0.02772,-0.02352,-0.00117,-0.00686,0.05764,-0.01437,0.01748,-0.01451,0.02184,0.00043,-0.01229,-0.00231,0.03573,0.02939,0.02039,-0.00995,-0.01559,0.01046,-0.01475,0.02937,-0.08973,-0.04491,-0.0238,0.0011,-0.01113,-0.02388,0.00914,0.02159,0.00036,0.06935,-0.01842,-0.00352,0.00203,-0.0033,0.01722,0.02953,-0.01352,0.02799,-0.01674,-0.01469,-0.00589,0.00952,0.04438,-0.01403,-0.00185,-0.01562,0.03365,0.02195,0.00132,0.00677,0.07274,-0.00925,0.00164,-0.01379,0.00086,0.00989,0.00678,-0.02793,-0.0501,-0.00955,0.00127,-0.02689,0.00629,-0.01756,-0.03711,0.03068,0.00603,-0.00238,0.00766,0.00421,-0.03612,0.0033,0.01629,0.03207,0.04431,0.00452,-0.03314,0.04044,0.03303,-0.00686,0.0204,-0.00257,-0.00068,0.01008,0.01641,0.03912,0.0141,0.01692,0.01925,0.06895,-0.03234,-0.02045,0.00865,0.01948,-0.00602,0.01449,0.01292,0.00778,-0.05704,0.05572,-0.02426,0.0054,-0.00922,0.00142,-0.00321,0.03154,-0.00932,-0.03012,-0.02315,-0.03006,-0.04822,0.00027,0.02974,0.00694,-0.00236,0.02433,0.00017,0.0219,0.02027,0.00394,0.00782,-0.01089,0.03984,0.02014,-0.02462,0.03137,-0.03679,0.02554,0.02735,-0.03312,-0.01199,-0.02246,0.03451,0.00053,0.00385,0.01465,0.01025,-0.02083,0.06545,-0.00792,0.05272,-0.02186,-0.00652,-0.02365,0.02522,0.01242,0.00202,0.06811,0.06491,-0.03064,-0.00814,-0.03122,-0.00747,-0.05389,0.03999,-0.02506,-0.00303,0.00257,0.03239,-0.01616,-0.04739,0.00357,0.01486,-0.00541,-0.03562,-0.01346,-0.02144,0.00102,0.00391,-0.02937,-1e-05,0.011,-0.00291,-0.01327,0.0159,-0.03749,-0.02166,0.016,0.02084,-0.0133,-0.0018,0.04364,0.02614,-0.03933,0.0366,-0.02481,-0.04,0.03174,0.00279,0.02346,0.00229,0.00391,-0.00557,0.00985,-0.0172,-0.0204,-0.01704,0.02331,0.02313,0.0054,0.03714,-0.04576,0.00048,-0.05283,-0.00104,-0.02786,-0.01534,-0.0194,-0.00231,-0.00106,-0.04998,-0.01665,0.01071,0.02441,-0.02929,0.03186,0.02025,0.04094,-0.01177,-0.03701,0.02186,-0.00313,-0.04555,0.02083,0.00454,-0.01781,0.02113,0.02327,0.01444,0.01394,-0.02169,-0.00985,0.0259,-0.0092,0.02598,-0.02105,-0.00121,0.04858,0.01114,-0.03598,-0.0197,0.00442,-0.02504,-0.0243,-0.04954,-0.03941,0.00682,-0.00389,0.01399,-0.02162,-0.00402,0.00192,-0.00525,-0.02199,0.02097,-0.01375,0.01095,0.03308,-3e-05,-0.01053,0.01627,0.0123,0.0111,0.02362,-0.02272,0.00141,0.01067,0.01896,0.04653,-0.00669,-0.00384,-0.04363,-0.04579,-0.00644,-0.0069,0.00642,-0.0222,0.01993,-0.00263,0.04445,0.01073,0.04195,-0.03092,0.03987,0.05397,-0.01036,-0.02179,-0.00578,0.01785,-0.04197,-0.03074,0.00582,0.0131,-0.02193,0.00845,-0.00154,0.00096,0.00936,-0.03576,0.01345,0.01383,-0.03113,0.00858,0.01495,0.00356,0.0043,0.00884,0.06918,0.01371,-0.02283,0.03055,0.05297,0.0359,-0.00503,-0.00416,-0.01695,0.00053,-0.0082,0.03423,-0.00797,-0.00144,0.00652,-0.02525,-0.00309,-0.0322,-0.00467,-0.02787,0.03959,-0.00438,0.01581,0.00658,0.00675,0.00158,0.03878,-0.01382,-0.01636,0.01171,-0.0052,-0.01799,0.00767,0.01468,0.00845,-0.05376,0.00589,-0.01714,-0.04079,-0.03344,0.01372,-0.0058,0.00943,0.02109,0.01267,-0.03701,-0.00966,-0.02025,0.0123,-0.00982,0.01439,0.06248,0.0037,-0.06499,-0.0037,0.02205,-0.0163,0.00072,-0.00876,-0.02999,-0.0277,0.10501,-0.03398,-0.00658,-0.03184,0.01553,-0.03166,0.032,-0.0161,-0.01292,0.00172,0.01077,-0.03605,-0.01842,-0.00557,-0.00426,-0.0077,-0.0027,0.02567,-0.00012,-0.02084,-0.00715,-0.00444,-0.00399,-0.0162,-0.03851,0.0135,-0.03022,0.03682,-0.02355,0.02002,-0.002,0.00834,0.00905,0.00509,0.00882,-0.02373,0.01166,0.00078,0.01733,-0.01188,-0.00217,-0.02237,0.02653,-0.0024,-0.03287,-0.01583,-0.02328,0.00345,0.02603,0.00647,-0.01554,0.00638,-0.03758,0.00954,0.00596,-0.02415,0.03776,-0.03561,0.03151,-0.01657,-0.01907,0.00874,-0.01923,-0.01285,-0.01749,0.00243,-0.00413,0.03343,-0.02369,0.01146,0.01482,-0.0423,0.02937,-0.0223,0.01755,0.03558,0.01196,-0.0001,0.01603,0.02373,-0.034,-0.00362,-0.04703,0.01941,-0.01817,0.00386,-0.02663,-0.00217,0.0467,0.0383,0.02958,-0.04377,0.01682,0.0366,0.00596,0.00893,0.0322,-0.00501,0.05025,0.03902,0.03957,-0.00246,0.03968,0.01737,-0.0183,-0.03037,0.04389,0.02248,0.01833,0.02317,0.02411,-0.01248,-0.00465,-0.03197,-0.06824],"ACTIVITY":[0.0033,-0.04249,0.02302,-0.05185,0.00602,0.04279,0.01137,-0.02268,-0.03448,-0.04925,-0.00626,0.06269,-0.00386,-0.00667,-0.00782,-0.00328,0.00547,0.01716,-0.00768,0.02533,0.00719,-0.01692,0.00756,0.00611,-0.01476,-0.05559,-0.04347,-0.02848,-0.03273,0.016,0.00044,0.02106,0.01509,0.00891,0.01225,0.06447,0.01828,0.00637,-0.01447,0.00912,-0.03359,0.01841,-0.0676,-0.02472,0.02611,-0.00868,-0.04317,0.04069,-0.0043,0.00873,-0.05222,0.01219,-0.05453,0.03822,-0.04639,-0.0238,-0.02392,0.00395,0.01307,-0.03483,0.00931,-0.01054,0.01674,-0.0035,-0.0028,0.02267,0.01403,-0.00213,0.01167,-0.03896,0.00944,0.02229,-0.00084,-0.03695,0.01507,0.00522,-0.0401,0.00894,0.01439,0.02356,-0.03612,-0.02941,-0.01415,-0.0189,-0.03759,0.00219,-0.00026,0.0272,-0.01349,-0.01271,0.0199,-0.00019,-0.0013,-0.00489,-0.03558,-0.03041,-0.02971,0.00496,-0.03527,0.04238,-0.07231,0.01399,-0.00176,0.042,0.01761,0.00167,-0.02461,0.01289,-0.01916,0.01573,0.0278,0.00692,-0.01068,0.0054,-0.00936,0.03314,0.03418,-0.01902,0.00227,0.01009,0.03263,-0.00823,-0.05818,-0.00025,0.01067,-0.03991,-0.01307,0.00961,-0.01199,0.03603,0.02348,0.01588,0.01797,-0.02053,0.04772,0.01448,0.01668,0.00944,0.01864,-0.00711,0.00242,-0.04347,0.00977,-0.01522,-0.03823,-0.02663,0.03092,0.00314,-0.01251,0.04433,-0.00702,-0.03021,0.03287,-0.0138,-0.00823,0.01583,-0.04332,0.01092,0.00734,0.01176,-0.01249,-0.00212,-0.05024,0.02824,-0.00375,0.04415,-0.05649,-0.0334,-0.04093,0.00819,-0.01215,0.00749,-0.03664,-0.01999,0.01094,0.00858,-0.01258,0.0125,-0.00126,-0.01934,-0.0274,-0.02307,-0.02861,0.04081,-0.00482,0.00176,0.00171,0.02445,0.01096,-0.01235,0.00665,0.02828,0.03168,0.01932,0.03456,-0.03332,-0.01462,0.00954,-0.03198,0.00362,-0.01688,0.00751,-0.00678,-0.03074,-0.03853,0.00509,0.02597,-0.00463,-0.06821,-0.00998,0.01981,0.01713,-0.022,0.03019,-0.02003,0.00317,0.01445,0.00472,-0.01587,-0.02052,0.01787,-0.00618,-0.0125,-0.00483,-0.00488,0.01337,-0.00945,0.03169,-0.01472,0.01976,-0.04491,0.02739,-0.02059,-0.03552,0.03513,-0.0077,-0.0393,-0.01353,-0.01183,0.0205,0.01048,0.02855,-0.02123,0.06451,0.01188,-0.01741,-0.01617,0.01412,-0.00805,-0.0369,0.0059,0.01345,-0.01964,-0.0417,0.031,0.00857,0.03471,0.02038,-0.01671,0.01708,0.04894,-0.06041,-0.03425,-0.02213,0.05815,0.0179,-0.01564,-0.00378,-0.0309,0.02517,-0.00564,0.05432,-0.03688,-0.01214,-0.01112,-0.02674,-0.00056,-0.00684,-0.03333,0.01436,0.02309,-0.05861,0.00242,0.00338,0.02098,-0.0304,-0.04302,0.00118,-0.01844,-0.05853,-0.03926,0.04101,0.00435,0.00634,-0.01919,-0.01336,-0.01056,-0.02068,-0.01493,0.03417,-0.01805,0.00094,-0.01859,0.0137,-0.00494,0.01256,0.01874,-0.00089,-0.01408,-0.01394,-0.02871,-0.02151,-0.00019,-0.00823,-0.00621,0.0108,0.02266,0.00021,0.00648,0.00047,0.06545,0.0153,0.0147,-0.00964,-3e-05,0.00999,0.0211,-0.02045,-0.01513,-0.03413,-0.00525,-0.00742,-0.00289,-0.00787,0.01129,-0.00784,0.00168,0.00124,-0.05424,-0.01747,-0.01138,-0.02514,0.01234,0.03516,0.01795,0.02879,0.00896,-0.00229,-0.01138,-0.02297,0.01856,0.02658,-0.03114,-0.00035,-0.0432,0.01894,-0.0515,-0.02902,-0.02746,-0.03147,-0.01824,-0.00073,-0.03991,-0.01087,0.00617,0.06951,-0.0221,0.03045,-0.00862,-0.04776,-0.01224,0.0374,0.00574,-0.0039,0.03519,-0.01936,0.02047,0.00508,0.03911,0.0075,0.01584,0.00527,0.00277,0.01213,-0.00569,-0.02523,0.02172,-0.02812,0.00823,-0.02142,0.06505,0.01639,-0.01414,0.05232,-0.00809,0.03457,-0.01451,0.05155,0.01873,0.01147,-0.04523,0.05551,0.0172,-0.00814,0.00466,0.01442,-0.02197,0.01144,-0.02511,-0.00292,-0.01641,-0.02227,-0.03386,-0.01281,-0.03196,-0.01119,0.0107,-0.0342,0.02142,0.02228,-0.01179,-0.03552,-0.00598,-0.01458,-0.04324,0.03799,0.02284,0.00478,-0.01385,-0.02036,-0.02038,-0.0122,-0.01988,-0.00473,-0.03603,-0.00599,0.00884,0.02653,0.01027,-0.02254,-0.03674,0.00952,-0.03411,-0.01907,0.01652,0.0278,-0.01355,-0.03362,0.00641,0.03355,-0.03137,-0.00366,-0.05954,0.05334,0.00866,-0.06381,-0.00609,0.05155,0.00357,-0.0196,-0.00265,0.06178,-0.01142,-0.03552,0.011,-0.00581,0.02098,0.00847,0.0147,0.00287,0.04851,0.0051,-0.03891,-0.05043,0.03225,0.00072,0.00626,0.01195,0.03249,-0.01879,0.04184,0.00132,0.03093,-0.01436,0.00269,0.03113,0.00814,0.02117,-0.01048,0.02735,0.0063,0.04592,-0.01065,0.02064,0.00067,0.00979,-0.01422,-0.00593,-0.00892,0.00337,-0.04751,-0.04478,-0.0307,0.01914,-0.01555,-0.00948,0.00044,-0.01824,0.03284,0.00411,0.02229,-0.03112,-0.02643,-0.00377,-0.00576,-0.01679,0.00356,-0.03239,0.01454,-0.03637,-0.01792,-0.04399,0.01178,-0.03331,0.01964,0.0339,-0.01261,0.04045,0.0376,0.01738,-0.04119,-0.01139,0.00148,-0.01071,-0.01638,-0.02714,-0.01414,-0.03321,-0.03621,-0.00343,-0.0571,-0.01049,-0.0175,-0.00889,6e-05,-0.01708,-0.06163,0.01899,0.02701,-0.00817,0.00814,0.06524,0.05786,0.01274,0.01039,-0.05597,-0.01885,-0.03312,0.01433,-0.01108,-0.03757,-0.03067,0.00262,0.03714,-0.00412,0.00334,-0.01558,0.00699,-0.04191,-0.00303,0.01035,0.01557,-0.02222,0.00902,-0.00223,0.04977,-0.01529,0.03622,0.01284,-0.00601,-0.00858,-0.00212,0.00804,-0.03051,-0.01002,0.06775,0.03107,-0.01111,0.01016,-0.00938,-0.00241,0.02312,-0.02832,-0.02336,0.01358,0.00179,0.0335,0.03653,0.04857,-0.01294,-0.02792,0.00611,0.01658,0.0267,-0.0325,-0.02133,-0.02143,-0.01284,-0.04673,0.04342,0.02735,0.03473,-0.0099,0.02935,0.02124,-0.02096,0.03594,-0.00733,0.00326,-0.003,0.02539,0.01625,0.00732,0.01499,-0.00261,0.0104,-0.01407,-0.03331,-0.03053,-0.0135,-0.00054,-0.01779,0.00303,-0.0649,-0.03189,0.01809,-0.00777,-0.02109,0.03409,-0.00917,0.00319,-0.00203,0.01216,0.02207,-0.02273,-0.01359,0.02804,-0.02663,-0.00679,0.0045,-0.00746,-0.00275,0.00072,-0.00484,0.03161,0.06165,-0.00911,0.02995,-0.05782,0.01363,0.06323,0.02971,0.01802,-0.02306,0.02103,0.0452,0.002,0.03467,-0.00294,0.01736,0.00793,0.00639,0.00817,-0.06591,0.04385,-0.01339,-0.02863,-0.00344,0.00571,-0.05227,0.04045,-0.01715,0.01762,0.01584,-0.03696,0.0154,-0.0282,-0.01686,-0.01749,0.04771,-0.00811,-0.00383,-0.0396,-0.05162,0.00665,-0.04182,0.03199,-0.01429,0.01532,0.014,-0.01914,0.01434,-0.06065,0.03105,0.00657,0.00135,0.02489,-0.01438,0.02153,0.00147,0.04252,0.01075,-0.01145,-0.00643,-0.00979,0.02947,0.00372,-0.01242,-0.01421,-0.00723,0.04072,-0.01774,-0.05477,0.00815,-0.01541,0.00946,-0.00746,0.00785,0.0321,0.0394,0.02269,0.01236,0.03268,-0.03967,-0.0612,-0.00707,-0.05384,-0.03093,0.01256,-0.02389,-0.00257,0.04426,0.02682,0.03891,-0.00971,0.03374,-0.0043,0.03513,0.00014,0.04578,0.01766,-0.01618,0.00216,0.05424,0.00358,0.00824,-0.02077,0.00369,-0.0157,-0.02204,-0.03537,-0.01939,0.05246,-0.00993,-0.02376,-0.02309,0.01898,0.01977,0.02112,0.04296,0.01791,-0.02683,-0.00971,-0.00937,0.00099,-0.04103,-0.00412,0.01975,-0.02673,-0.00148,0.00595,0.03831,-0.03902,0.00622,-0.04934,-0.01889,0.03783,0.02475,0.031,0.00782,-0.00527,-0.06265,-0.03834,-0.03987,-0.05944,0.04357,0.01987,-0.01475,-0.00707,-0.01393,0.03305,-0.03798,0.0152,0.04126,-0.06312,0.01929,-0.011,-0.02896,0.00238,0.02171,-0.00558,-0.00498,-0.00499,-0.00594,-0.01224,0.04822,0.01141,-0.03408,-0.01682,-0.00744,-0.02231,-0.0189,0.02766,0.01179,-0.01986,-0.01245,-0.00791,0.03603,-0.00195,-0.00224,-0.00976,0.00645,0.01986,-0.0043,-0.01951,-0.01004,-0.02364,0.00977,-0.00911,-0.0279,0.02538,-0.02607,0.0035,0.00585,0.02636,-0.03359,-0.00185,0.05441,0.00246,0.00238,-0.02528,0.0297,0.02967,-0.00361,-0.00152,0.01499,0.00235,-0.02544,0.0433,-0.01132,-0.01774,0.02195,-0.04342,-0.00607,0.01441,-0.05156,-0.00264,0.02899,0.01121,0.00218,0.02871,0.0028,-0.02686,0.01132,0.0152,-0.00618,-0.01448,-0.02276,-0.01522,-0.02631,0.00912,0.00685,0.03148,0.01927,0.01845,-0.01921,0.01632,0.00463,0.04736,0.01296,-0.01161,-0.01139,0.02386,-0.02944,-0.05642,0.03677,-0.01068,0.01306,0.02625,-0.00756,-0.00527,0.01054,-0.0284,0.02246,0.01296,-0.01506,0.02721,-0.02112,-0.01263,-0.00725,0.03318,-0.01054,0.01707,-0.00908,-0.02053,0.00289,-0.02022,-0.00851,0.00603,-0.0005,0.0219,-0.03011,-0.03023,0.01483,-0.0082,0.03477,0.02523,-0.00616,0.02015,-0.00854,-0.02808,0.02708,0.03131,0.00443,0.02644,0.02415,-0.02487,0.00768,0.01688,0.01623,-0.0044,-0.02776,0.00987,0.00214,-0.00844,0.01207,0.03614,-0.00287,-0.02122,0.01695,-0.00752,-0.01898,-0.01476,0.03228,0.00297,-0.05006,-0.01092,0.00321,0.05074,0.0247,0.02364,-0.01069,0.00823,-0.0226,0.04549,0.01411,0.01321,0.032,-0.00514,-0.00233,0.00111,0.01432,0.00287,0.06792,0.04117,-0.00959,-0.05453,-0.0552,-0.02088,-0.04346,-0.00832,0.02064,-0.04789,-0.00836,0.04932,0.03027,0.01272,0.01709,-0.00936,0.0232,0.01685,0.00627,-0.00413,-0.01053,-0.03921,-0.02171,-0.0454,-0.02387,0.01798,0.00487,-0.02224,0.02361,-0.03385,-0.00345,-0.01192,0.0058,-0.02145,-0.02278,-0.02722,0.01989,-0.00722,-0.00674,-0.00032,0.01999,0.02762,-0.00827,0.02252,-0.03565,0.02904,-0.01108,0.01677,0.00749,-9e-05,-0.02177,-0.04988,-0.02055,-0.04293,-0.00533,0.01877,0.02779,-0.01175,-0.01677,0.00705,-0.03017,0.0037,-0.00707,0.0128,0.0104,-0.00129,-0.03133,0.00612,0.01681,0.03356,-0.00912,0.01676,-0.01331,0.01364,0.02545,0.01488,-0.0415,-0.00498,0.02881,0.00909,-0.01713,0.03849,0.00392,0.01179,-0.06804,-0.00214,-0.01764,0.04586,-0.00225,-0.01681,0.00861,-0.02126,0.01462,-0.02498,-0.01248,0.03829,0.01012,-0.02766,0.01064,-0.01576,0.00275,0.03932,0.00493,-0.00113,0.02622,0.00708,-0.00488,-0.03811,0.00414,-0.06117,0.02345,0.00987,-0.01396,0.02426,-0.00049,-0.02819,0.02205,-0.04582,0.01531,0.0204,0.01965,-0.00281,-0.04795,0.01722,0.00428,0.03194,-0.06934,0.02063,-0.00588,-0.01035,-0.04214,0.00436,0.00324,-0.00428,-0.00491,-0.0136,-0.00421,0.01077,0.05237,-0.00059,0.01333,0.06632,0.00469,0.01494,0.05671,-0.01672,-0.00804,-0.0063,-0.00356,-0.01516,0.03953,0.01119,-0.00462,-0.01325,-0.01552,0.00267,0.01933,-0.03757,-0.02365,0.01071,0.00482,0.00481,0.00097,-0.00851,0.02542,-0.0321,0.04595,-0.03205,-0.03015,-0.0379,0.0417,0.02168,0.00809,-0.01915,0.00986,-0.01368,0.00893,-0.00546,0.04646,-0.04035,0.01354,-0.0381,0.01741,0.00789,-0.00906,-0.0335,0.00817,-0.00373,-0.00617,0.02998,0.03254,-0.01958,-0.05074,-0.01458,0.01081,0.00341,0.02603,-0.00931,-0.0095,0.00326,0.02644,-0.00039,-0.05417,0.02473,0.00718,-0.04704,0.01325,-0.00816,-0.0038,0.05,0.02862,0.05104,-0.01562,0.0048,0.0209,-0.01812,-0.02046,0.00696,-0.04106,0.0134,-0.04493,0.03181,0.03112,0.00559,0.00245,-0.02205,0.00361,0.01137,-0.05778,0.01266,0.0353,0.01677,-0.0154,0.0557,-0.00986,0.01598,0.00732,0.0156,-0.05126,0.03741,0.04495,-0.03124,0.00487,0.00616,0.00725,0.008,0.01267,-0.00924,0.00558,-0.03757,-0.01229,0.01173,0.03159,0.00815,-0.01189,0.00992,0.02079,0.00472,-0.04367,0.02833,0.00303,-0.02139,-0.03171,0.02465,-0.03116,-0.00459,0.04798,-0.01901,-0.011,-0.03657,-0.01152,0.00623,0.01329,-0.00707,-0.00266,-0.01087,0.02757,-0.05515,-0.01384,-0.01849,-0.05262,0.04204,0.00426,0.0263,0.02354,0.02102,-0.01537,0.00791,0.00091,0.00185,0.05719,0.02011,-0.00948,0.00476,0.03574,-0.03643,-0.00251,-0.01502,0.01506,0.02745,-0.03444,-0.00542,0.03135,-0.04663,-0.00745,0.02253,-0.02673,-0.03818,-0.00035,-0.02232,-0.0144,0.01185,0.02684,-0.00337,-0.05338,0.00605,0.00902,-0.02525,-0.01267,0.05593,-0.0071,-0.02093,0.00262,0.01426,0.02119,0.01938,-0.00812,0.02418,-0.01709,0.03153,-0.01859,-0.01614,-0.01952,7e-05,-0.0499,0.00405,-0.03038,-0.00158,-0.02615,0.00286,-0.01409,-0.01951,0.02045,0.01186,0.01338,0.0196,0.02116,-0.01818,-0.04164,0.00812,0.01287,0.05901,0.02561,0.00189,0.03062,0.00014,0.01015,-0.0046,0.02378,-0.01887,-0.01089,-0.01929,0.05036,-0.04543,-0.03818,0.0041,0.04269,-0.011,-0.02078,0.05454,-0.02522,-0.00373,-0.02534,0.02766,-0.02808,-0.00442,-0.00446,-0.01014,0.03206,0.00305,-0.01875,0.03527,-0.03786,0.01604,-0.02044,0.00893,0.01155,-0.00852,0.00818,0.00215,-0.01656,-0.07164,-0.03238,0.01975,-0.02023,7e-05,0.02357,0.00917,-0.01074,-0.01593,0.01607,-0.02207,-0.00857,-0.01103,0.00245,-0.00432,0.02511,-0.00163,0.00904,-0.00958,0.00737,-0.00814,0.01294,-0.03139,-0.03571,0.02099,-0.00318,-0.00406,0.02043,0.02014,0.00971,-0.00095,-0.01941,0.00729,0.0154,0.00381,-0.00668,0.0099,-0.00335,0.00441,-0.01864,-0.01966,0.04667,0.00413,0.02851,-0.00604,-0.00963,-0.01895,-0.02818,-0.02809,0.05502,-0.01942,0.00475,0.03846,-0.04024,0.02985,0.04244,0.02006,-0.06431,0.01003,0.01938,-0.06147,-0.00368,0.0229,-0.0388,0.00926,0.00105,-0.00646,-0.02277,0.01864,0.02126,0.02831,0.01642,0.03886,0.00649,-0.03385,-0.03393,0.00606,0.00922,0.02202,0.03137,0.01834,0.02512,-0.00637,0.02031,0.01772,0.01514,-0.02553,0.00191,-0.01484,-0.0099,-0.01712,0.01466,0.02506,0.03662,0.01199,0.00259,-0.00935,0.00235,-0.0063,-0.01614,-0.02649,0.00832,0.04809,0.02372,0.00464,-0.04797,-0.01094,0.03026,0.01882,0.03559,0.00476,-0.00489,-0.02965,-0.0317,0.03702,-0.00971,-0.04264,-0.03503,-0.00592,0.0109,0.0336,0.02717,-0.0168,0.00188,-0.03078,0.00589,-0.00539,-0.01227,-0.01717,-0.04454,0.00348,-0.0147,-0.03895,-0.00487,-0.01086,-0.00827,-0.0255,0.01235,0.01672,0.03391,0.00729,-0.02231,0.01388,-0.01106,-0.00682,-0.00773,-0.01249,-0.03934,0.02173,-0.03196,-0.03023,0.0048,-0.01162,0.00767,-0.00885,-0.03111,-0.06402,-0.02754,0.0535,-0.03429,0.02492,0.02215,0.02592,-0.00699,-0.03814,-0.0351,-0.00841,-0.0219,0.01724,-0.00469,0.02345,0.05839]},"mu14":81.875}""")
_pl_sha = hashlib.sha256(json.dumps(
    {k: PAYLOAD[k] for k in sorted(PAYLOAD)}, sort_keys=True,
    separators=(',', ':')).encode()).hexdigest()[:16]
assert _pl_sha == PAYLOAD_SHA, f'payload sha drift: {_pl_sha} != {PAYLOAD_SHA}'

W5 = np.array(PAYLOAD['W'], float)
assert W5.shape == (15, 1536), W5.shape
assert abs(PAYLOAD['mu14'] - MU14_ATLAS) < 1e-9, 'atlas mu pin drift'

# locked atlas: byte-sha gate, then true dirs for train/eval names
_atlas_bytes = open(ATLAS_FP, 'rb').read()
_a_sha = hashlib.sha256(_atlas_bytes).hexdigest()
assert _a_sha == ATLAS_FILE_SHA, (
    f'atlas_dirs.json sha mismatch on Drive: {_a_sha[:16]}... — the locked '
    'atlas is the pinned source of true dirs; do not proceed on a drifted file')
ATLAS = json.loads(_atlas_bytes)
assert ATLAS['anchors'] == sorted(set(TRAIN256) | set(EVAL64)), 'anchor split drift'
assert hashlib.sha256('|'.join(ATLAS['anchors']).encode()).hexdigest()[:16] \
    == ANCHORS_SHA_F, 'anchors sha drift'
TRUE_DIRS = {}
for n in TRAIN256 + EVAL64:
    v = np.asarray(ATLAS['inst14'][n], float)
    TRUE_DIRS[n] = v / np.linalg.norm(v)
for n, pv in PAYLOAD['true_probe'].items():
    assert float(np.max(np.abs(TRUE_DIRS[n] - np.asarray(pv, float)))) < 1e-4, \
        f'true-dir probe drift at {n}'
DHAT = {w: np.asarray(v, float) for w, v in PAYLOAD['dhat'].items()}

STIM = mint_stimuli(W5, VEC, TRUE_DIRS, DHAT)
for k, v in STIM.items():
    assert abs(float(np.linalg.norm(v)) - 1.0) < 1e-9, f'unnormalized stim {k}'
assert len([k for k in STIM if k.startswith('atom:')]) == 28
assert len([k for k in STIM if k.startswith('pairc:')]) == 84
assert len([k for k in STIM if k.startswith('full:')]) == 320
assert len([k for k in STIM if k.startswith('wperm:')]) == 16
assert len([k for k in STIM if k.startswith('true:')]) == 320
assert len([k for k in STIM if k.startswith('dhat:')]) == 13

# frame codes: derive from THIS pack, assert sha == build pin
FR_CODES = frame_codes(pack['concepts'])
assert frame_codes_sha(FR_CODES) == FRAME_CODES_SHA, 'frame codes sha drift (pack?)'
CODES64 = {n: FR_CODES[n] for n in EVAL64}
CODES_TRAIN = {n: FR_CODES[n] for n in TRAIN256}
assert set(EVAL64).isdisjoint(TRAIN256)
assert not (set(EVAL64) | set(TRAIN256)) & set(CHOICE_SET), 'wing leaked into split'
assert code_levels(MU_FRAME) == [0] * 14, 'mu_frame is not all-MID'
print(f'G1 PIN GATE: payload {PAYLOAD_SHA} ok | atlas sha ok | '
      f'{len(STIM)} stimuli minted + probed | frame codes {FRAME_CODES_SHA} ok')


In [ ]:
# ── Readout LoRA config + model builders + train loop (flown slices + E8-F) ──
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, LoraConfig, get_peft_model

LORA_KW = dict(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
               target_modules=['q_proj','k_proj','v_proj','o_proj'],
               task_type='CAUSAL_LM')          # E4's exact shape
LR = 1e-4
ACCUM = 4 if SMOKE else 8
EPOCHS_CAP = 1 if SMOKE else EPOCH_CAP_F
MIN_EPOCHS_STRAND = 1 if SMOKE else 2
PLATEAU_REL = 0.05

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def build_condition_model(cond):
    """base weights (+ merged instillation adapter for real/scrambled) + fresh
    zero-init readout LoRA. Zero-init B => directions/mu computed with the
    adapter attached equal the pre-readout model exactly."""
    m = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16,
                                             device_map=DEV, low_cpu_mem_usage=True)
    if cond != 'base':
        m = PeftModel.from_pretrained(m, ADAPTERS[cond]).merge_and_unload()
    m = get_peft_model(m, LoraConfig(**LORA_KW))
    for p in m.parameters():
        if p.requires_grad:
            p.data = p.data.float()
    m.eval()
    return m

def resolve_layers(m):
    """decoder layer modules by index (robust to the PEFT wrapper).
    hidden_states[L] is the OUTPUT of decoder layer L-1 (hs[0] = embeddings),
    so injecting at hidden-state level L means hooking layer module L-1."""
    mods = {}
    for mod_name, mod in m.named_modules():
        mm = re.search(r'(?:^|\.)layers\.(\d+)$', mod_name)
        if mm:
            mods[int(mm.group(1))] = mod
    n = m.config.num_hidden_layers if hasattr(m.config, 'num_hidden_layers') \
        else m.base_model.config.num_hidden_layers
    assert len(mods) == n, (len(mods), n)
    return mods

RETENTION_TEXT = (
    "The river begins as meltwater in the high country, threading between "
    "granite blocks before it gathers into a channel wide enough to carry "
    "boats. Farther down, where the gradient softens, it deposits silt in "
    "long bars that farmers have worked for centuries. Bridges cross it at "
    "the old fording points, and each town along the bank grew around a "
    "mill, a ferry landing, or a customs house.\n\n"
    "Bronze is an alloy of copper and tin, harder than either metal alone. "
    "Early smiths learned to cast it in two-part molds, producing axe heads, "
    "sickles, and mirrors. The proportion of tin changes the color and the "
    "brittleness of the finished piece, and workshops kept their recipes "
    "close.\n\n"
    "Weather over the plains follows the seasons: long dry spells broken by "
    "storm fronts that arrive from the west, announced by a wall of dark "
    "cloud and a sudden drop in temperature. Farmers read the sky in the "
    "evening and plan the next day's work accordingly. After the harvest, "
    "the fields are turned and left rough through the winter so the frost "
    "can break the clods.\n\n"
    "A lighthouse keeper's routine was built around the lamp: trimming "
    "wicks, polishing the lens, winding the clockwork that turned the "
    "optic. Supply boats came monthly when the sea allowed, bringing oil, "
    "flour, and letters. The log books record weather, passing ships, and "
    "small repairs, one line per day, for decades."
)

def retention_ppl(m):
    enc = tok(RETENTION_TEXT, return_tensors='pt').to(DEV)
    with torch.no_grad():
        out = m(input_ids=enc.input_ids, labels=enc.input_ids)
    return round(float(torch.exp(out.loss)), 4)

def encode_featex(ex):
    """(ids, prompt_len, answer_ids, skey) under the featural grammar."""
    msgs = [{'role': 'user', 'content': feat_prompt()}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    pid = tok(text, return_tensors='pt').input_ids[0]
    ans = tok(ex['target'], add_special_tokens=False)['input_ids'] + [tok.eos_token_id]
    ans = torch.tensor(ans, dtype=pid.dtype)
    ids = torch.cat([pid, ans]).unsqueeze(0)
    return ids, len(pid), ans.long().unsqueeze(0), \
        (ex['skey'] if ex['kind'] == 'inject' else None)

def build_eval_model(readout_dir):
    """RESUME path: base + merged E4 real + the TRAINED readout adapter."""
    m = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16,
                                             device_map=DEV, low_cpu_mem_usage=True)
    m = PeftModel.from_pretrained(m, ADAPTERS['real']).merge_and_unload()
    m = PeftModel.from_pretrained(m, str(readout_dir))
    m.eval()
    return m

def train_readout_feat(m, layer_mods, mu14, examples):
    """Six-strand SFT through ONE answer-sliced forward per example
    (full-sequence logits never materialized — E8-N OOM law; checkpointing
    engaged non-reentrantly and ASSERTED). For injected examples the
    backward runs INSIDE the Injector context (E8-N v2 law). Convergence =
    PER-STRAND plateau (E8-N v2 minted lesson, prospective); cap EPOCHS_CAP."""
    import torch.nn.functional as Fnn
    m.train()
    _cm = m.base_model.model
    DEC, HEAD = _cm.model, _cm.get_output_embeddings()
    try:
        _cm.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={'use_reentrant': False})
    except TypeError:
        _cm.gradient_checkpointing_enable()
    try:
        _cm.enable_input_require_grads()
    except Exception as e:
        print('  (input-require-grads unavailable:', e, ')')
    assert getattr(DEC, 'gradient_checkpointing', False), (
        'gradient checkpointing did not engage — refusing to train without it')
    params = [p for p in m.parameters() if p.requires_grad]
    n_tr = sum(p.numel() for p in params)
    opt = torch.optim.AdamW(params, lr=LR)
    try:
        scaler = torch.amp.GradScaler('cuda')
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    losses, micro, max_tok, hook_calls = [], 0, 0, 0
    strand_epoch_means = []
    plateaued, epochs_flown = False, 0
    t0 = time.time()

    def fwd_loss(ids, plen, ans):
        lo, hi = answer_slice(plen, ids.shape[1])
        hid = DEC(input_ids=ids, use_cache=False).last_hidden_state
        logits = HEAD(hid[:, lo:hi, :]).float()
        return Fnn.cross_entropy(logits.view(-1, logits.size(-1)), ans.view(-1))

    for ep in range(EPOCHS_CAP):
        order = np.random.default_rng(E8F_SEED + 100 + ep).permutation(len(examples))
        ep_strand = {s: [] for s in STRANDS_F}
        for i in order:
            ex = examples[int(i)]
            ids, plen, ans, skey = encode_featex(ex)
            max_tok = max(max_tok, ids.shape[1])
            ids, ans = ids.to(DEV), ans.to(DEV)
            if skey is not None:
                vec = ex['alpha'] * mu14 * torch.tensor(STIM[skey])
                with Injector(layer_mods, ex['layer'], vec, plen - 1) as injh:
                    loss = fwd_loss(ids, plen, ans)
                    lv = float(loss.detach())
                    scaler.scale(loss / ACCUM).backward()
                hook_calls += injh.calls
            else:
                loss = fwd_loss(ids, plen, ans)
                lv = float(loss.detach())
                scaler.scale(loss / ACCUM).backward()
            assert math.isfinite(lv), (
                f'non-finite loss at ep{ep} strand {ex.get("strand")}')
            losses.append(round(lv, 4))
            ep_strand[ex.get('strand')].append(lv)
            micro += 1
            if micro % ACCUM == 0:
                scaler.step(opt); scaler.update(); opt.zero_grad()
            if micro % 100 == 0:
                print(f'    training ep{ep+1} {micro} micro-steps, '
                      f'loss~{np.mean(losses[-50:]):.3f}, {time.time()-t0:.0f}s')
        epochs_flown = ep + 1
        strand_epoch_means.append({s: (round(float(np.mean(v)), 4) if v else None)
                                   for s, v in ep_strand.items()})
        print(f'  epoch {epochs_flown}: '
              + ' | '.join(f'{s} {strand_epoch_means[-1][s]}' for s in STRANDS_F))
        if per_strand_plateau(strand_epoch_means, MIN_EPOCHS_STRAND, PLATEAU_REL):
            plateaued = True
            break
    if micro % ACCUM:
        scaler.step(opt); scaler.update(); opt.zero_grad()
    try:
        _cm.gradient_checkpointing_disable()
    except Exception:
        pass
    m.eval()
    peak = round(torch.cuda.max_memory_allocated() / 1e9, 2)
    k = max(3, len(losses) // 10)
    log = {'n_examples': len(examples), 'epochs_flown': epochs_flown,
           'epochs_cap': EPOCHS_CAP, 'plateaued': plateaued,
           'plateau_rel': PLATEAU_REL, 'min_epochs_strand': MIN_EPOCHS_STRAND,
           'strand_epoch_means': strand_epoch_means,
           'micro_steps': micro, 'opt_steps': micro // ACCUM,
           'hook_calls': int(hook_calls),
           'trainable_params': int(n_tr), 'max_example_tokens': int(max_tok),
           'peak_vram_gb': peak, 'secs': round(time.time() - t0, 1),
           'loss_first_k': round(float(np.mean(losses[:k])), 4),
           'final_smoothed': round(float(np.mean(losses[-50:])), 4),
           'losses_every_10': losses[::10]}
    print(f'  trained: {log["opt_steps"]} steps over {epochs_flown} epochs, '
          f'loss {log["loss_first_k"]} -> {log["final_smoothed"]}'
          f'{" (PLATEAU)" if plateaued else " (CAP HIT)"}'
          f', peak VRAM {peak}GB, {log["secs"]}s')
    return log


In [ ]:
# ── G2 stimulus machinery (E8-J slices, verbatim) + Injector + runner ──────
import numpy as np
rng_dir = np.random.default_rng(E7Q_SEED)
CENT_NAMES = list(rng_dir.choice([c['name'] for c in pack['concepts']],
                                 size=256, replace=False))

def pooled_reps(m, names, layer, bs=32):
    """Mean-pooled hidden_states[layer] of the E4 text rendering 'NAME: desc'."""
    texts = [f"{n}: {DESC[n]}" if DESC.get(n) else n for n in names]
    reps = []
    with torch.no_grad():
        for i in range(0, len(texts), bs):
            enc = tok(texts[i:i+bs], padding=True, truncation=True, max_length=64,
                      return_tensors='pt').to(DEV)
            out = m(**enc, output_hidden_states=True)
            h = out.hidden_states[layer]
            mk = enc.attention_mask.unsqueeze(-1).to(h.dtype)
            reps.append(((h * mk).sum(1) / mk.sum(1).clamp(min=1)).float().cpu())
        if len(texts) > 64:
            print(f'      pooled_reps: {len(texts)} texts @ L{layer} done')
    return torch.cat(reps).numpy()

canon_prompt = report_prompt(list(range(len(CHOICE_SET))), DESC)
canon_enc = tok.apply_chat_template([{'role':'user','content':canon_prompt}],
                                    add_generation_prompt=True, return_tensors='pt',
                                    return_dict=True)
CANON_IDS = canon_enc['input_ids']

def compute_dirs(m, layers, names):
    """Directions for `names` at each layer: pooled rep minus the 256-name
    centroid, unit-normalized — the E7-Q/E8-R formula verbatim."""
    dirs = {}
    for L in layers:
        reps = pooled_reps(m, names, L)
        cent = pooled_reps(m, CENT_NAMES, L).mean(0)
        d = reps - cent
        d = d / np.linalg.norm(d, axis=1, keepdims=True)
        dirs[L] = {n: d[i] for i, n in enumerate(names)}
        print(f'    dirs ready: L{L} x {len(names)} names')
    return dirs

def compute_mu(m, layers):
    mu = {}
    ids = CANON_IDS.to(DEV)
    with torch.no_grad():
        out = m(input_ids=ids, output_hidden_states=True)
        for L in layers:
            mu[L] = float(out.hidden_states[L][0].norm(dim=-1).mean())
    return mu


class Injector:
    """Adds alpha*mu*dhat to a decoder layer's residual output from the final
    prompt position onward — OUT-OF-PLACE (h + masked constant), so the same
    hook is autograd-safe during training forwards and identical in math to
    E7-Q's eval-time hook (first forward: positions >= start; cached steps:
    every position)."""
    def __init__(self, layer_mods, hs_level, vec, start_idx):
        self.mod = layer_mods[hs_level - 1]
        self.vec = vec
        self.start = start_idx
        self.handle = None
        self.calls = 0
    def _fn(self, module, args, output):
        hs = output[0] if isinstance(output, tuple) else output
        v = self.vec.to(dtype=hs.dtype, device=hs.device)
        if hs.shape[1] > 1:
            add = torch.zeros_like(hs)
            add[:, self.start:, :] = v
            hs = hs + add
        else:
            hs = hs + v
        self.calls += 1
        return (hs, *output[1:]) if isinstance(output, tuple) else hs
    def __enter__(self):
        self.handle = self.mod.register_forward_hook(self._fn)
        return self
    def __exit__(self, *exc):
        if self.handle:
            self.handle.remove()


# ── G2 identity probe + featural generation runner ───────────────────────────
def g2_probe(m):
    """Wing-13 dirs recomputed in the own-13-call shape (batch-composition
    lane law) vs the shipped E8-R real bundle at L14; mu asserted vs shipped
    AND the atlas pin. Runs on the PRE-readout model (zero-init LoRA)."""
    dirs_re = compute_dirs(m, [14], CHOICE_SET)
    stab = dirs_stability(dirs_re, {'14': SHIPPED_E8R['dirs']['14']},
                          tol=DIRS_TOL_F)
    mu = compute_mu(m, [14])
    mu_ship = float(SHIPPED_E8R['mu']['14'])
    assert abs(mu[14] / mu_ship - 1.0) < 1e-3, (mu[14], mu_ship)
    assert abs(mu[14] / MU14_ATLAS - 1.0) < 1e-3, (mu[14], MU14_ATLAS)
    print(f'  G2: wing-13 resid {stab["resid"]} (tol {DIRS_TOL_F}) | '
          f'mu {mu[14]:.3f} vs shipped {mu_ship:.3f} / atlas {MU14_ATLAS}')
    return stab, mu[14]

def run_trial_feat(m, layer_mods, mu14, trial):
    """Generation under the featural grammar; greedy; 110 new tokens."""
    enc = tok.apply_chat_template([{'role': 'user', 'content': feat_prompt()}],
                                  add_generation_prompt=True, return_tensors='pt',
                                  return_dict=True)
    ids = enc['input_ids'].to(DEV)
    gen_kw = dict(max_new_tokens=110, do_sample=False,
                  pad_token_id=tok.pad_token_id, use_cache=True)
    with torch.no_grad():
        if trial['kind'] == 'inject':
            vec = trial['alpha'] * mu14 * torch.tensor(STIM[trial['skey']])
            with Injector(layer_mods, trial['layer'], vec, ids.shape[1] - 1) as inj:
                out = m.generate(input_ids=ids, **gen_kw)
            calls = inj.calls
            assert calls >= 1, 'injection hook never fired during generation'
        else:
            out = m.generate(input_ids=ids, **gen_kw)
            calls = 0
    text = tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
    return {**trial, 'response': text.strip()[:400],
            'parsed': parse_feat(text), 'hook_calls': calls}


In [ ]:
# ── Flight: G2 -> curriculum -> train -> ship readout -> eval -> ship bundle ──
def run_eval(m, layer_mods, mu14):
    rows = build_e8f_eval(SMOKE)
    out, t0 = {}, time.time()
    for i, t in enumerate(rows, 1):
        r = run_trial_feat(m, layer_mods, mu14, t)
        out.setdefault(t['block'], []).append(r)
        if i % 25 == 0 or i == len(rows):
            el = time.time() - t0
            eta = el / i * (len(rows) - i)
            print(f'    eval {i}/{len(rows)}, {el:.0f}s elapsed, ~{eta:.0f}s left')
    return out

def fly_real():
    t0 = time.time()
    bundle = {'condition': 'real', 'mode': MODE, 'stamp': STAMP,
              'payload_sha': PAYLOAD_SHA, 'atlas_sha': ATLAS_FILE_SHA[:16],
              'frame_codes_sha': FRAME_CODES_SHA}
    try:
        prev_readout = SEM / INFLIGHT / 'readout_real'
        resume_eval = bool(RESUME_STAMP) and \
            (prev_readout / 'adapter_config.json').exists()
        if resume_eval:
            print('RESUME: trained readout found on Drive — rebuilding eval model')
            probe_m = build_condition_model('real')
            stab, mu14 = g2_probe(probe_m)
            del probe_m; free_ram()
            local_ro = Path('/content/readout_real_resume')
            shutil.copytree(prev_readout, local_ro, dirs_exist_ok=True)
            model = build_eval_model(local_ro)
            layer_mods = resolve_layers(model)
            bundle['g2_dirs'], bundle['mu14'] = stab, mu14
            bundle['resumed_eval'] = True
            bundle['train_log'] = {'resumed': True}
            bundle['ppl_pre'] = None
        else:
            model = build_condition_model('real')
            layer_mods = resolve_layers(model)
            print(f'model ready — {ram_report()}')
            stab, mu14 = g2_probe(model)
            bundle['g2_dirs'], bundle['mu14'] = stab, mu14
            assert stab['pass'], f'G2 dirs-stability FAILED: {stab}'
            examples = build_e8f_train(VEC, SMOKE)
            _v1 = validate_no_eval_leak(examples)
            assert not _v1, f'FIREWALL — eval concept in curriculum: {_v1}'
            _v2 = validate_no_wing_leak(examples)
            assert not _v2, f'FIREWALL — wing stimulus in curriculum: {_v2}'
            cur = {}
            for e in examples:
                cur[e['strand']] = cur.get(e['strand'], 0) + 1
            bundle['curriculum'] = cur
            if not SMOKE:
                assert cur == EXPECT_TRAIN_FULL, cur
            print(f'  curriculum: {cur}')
            bundle['ppl_pre'] = retention_ppl(model)
            torch.cuda.empty_cache()
            bundle['train_log'] = train_readout_feat(model, layer_mods, mu14,
                                                     examples)
            torch.cuda.empty_cache()
            model.save_pretrained(str(OUT / 'readout_real'))
            ship(OUT / 'readout_real', f'{INFLIGHT}/readout_real')
            print('  trained readout shipped inflight (eval-resume point)')
        rows = run_eval(model, layer_mods, mu14)
        bundle['eval_rows'] = rows
        cnt = eval_counts([r for rs in rows.values() for r in rs])
        print(f'  eval rows: {cnt}')
        if not SMOKE:
            assert cnt == EXPECT_EVAL_FULL, cnt
        bundle['ppl_post'] = retention_ppl(model)
        bundle['secs'] = round(time.time() - t0, 1)
    except Exception as e:
        import traceback; traceback.print_exc()
        bundle['error'] = f'{type(e).__name__}: {e}'
    return bundle

BUNDLE = None
if RESUME_STAMP:
    prev = SEM / INFLIGHT / 'bundle_real.json'
    if prev.exists():
        b = json.load(open(prev))
        if b.get('eval_rows') and 'error' not in b:
            BUNDLE = b
            print(f'RESUMED complete bundle from Drive ({RESUME_STAMP})')
if BUNDLE is None:
    print(f'flight: starting — {ram_report()}')
    BUNDLE = fly_real()
    fn = OUT / 'bundle_real.json'
    jdump(BUNDLE, fn)
    ship(fn, INFLIGHT)
    print(f'flight done in {BUNDLE.get("secs", "?")}s — bundle shipped')
    free_ram()
    print(f'torn down — {ram_report()}')


In [ ]:
# ── Gates, primaries P1/P2/P3, S1-S8, fork, banner, ship ─────────────────────
ok = bool(BUNDLE.get('eval_rows')) and 'error' not in BUNDLE
if not ok:
    print()
    print('=' * 72)
    print(f'  FLIGHT INCOMPLETE — error: {str(BUNDLE.get("error"))[:200]}')
    print(f'  Fix comes home to the builder (lane law). To resume eval after '
          f'a mid-eval death: Restart runtime, RESUME_STAMP = '
          f"'{RESUME_STAMP or STAMP}', Run all.")
    print('=' * 72)
else:
    verdict = {'flight': f'{MODE}_{STAMP}', 'protocol': 'E8F_PROTOCOL.md',
               'payload_sha': PAYLOAD_SHA, 'resume_stamp': RESUME_STAMP or STAMP}
    blocks = BUNDLE['eval_rows']
    allrows = [r for rs in blocks.values() for r in rs]
    NP_PRIM = 200 if SMOKE else N_PERM_F
    NP_TEX = 50 if SMOKE else N_PERM_TEX

    # ── gates ────────────────────────────────────────────────────────────────
    g2 = BUNDLE['g2_dirs']
    ppl_pre, ppl_post = BUNDLE.get('ppl_pre'), BUNDLE.get('ppl_post')
    ppl_d = (100.0 * (ppl_post / ppl_pre - 1.0)) if (ppl_pre and ppl_post) \
        else None
    inv = invalid_rate(allrows)
    fa = sham_claims(blocks.get('sham', []))
    gates = {'g1_pins': {'payload': PAYLOAD_SHA, 'asserted_in_flight': True,
                         'pass': True},
             'g2_dirs': g2,
             'g3_ppl': {'pre': ppl_pre, 'post': ppl_post,
                        'delta_pct': (round(ppl_d, 4) if ppl_d is not None
                                      else None), 'tol': PPL_TOL_F,
                        'pass': bool(ppl_d is None or abs(ppl_d) <= PPL_TOL_F),
                        'note': ('resumed eval: ppl_pre unavailable, post-only'
                                 if ppl_pre is None else '')},
             'g4_parse': {**inv, 'max': PARSE_INVALID_MAX,
                          'pass': bool(inv['rate'] <= PARSE_INVALID_MAX)},
             'g5_sham': {'claims': fa, 'n': len(blocks.get('sham', [])),
                         'max': SHAM_FA_MAX, 'pass': bool(fa <= SHAM_FA_MAX)}}
    gates_ok = all(g.get('pass') for g in gates.values())
    verdict['gates'] = gates

    # ── G-INSTALL (spot) ─────────────────────────────────────────────────────
    spot_codes = {r['concept']: FR_CODES[r['concept']]
                  for r in blocks.get('spot', [])}
    spot = p_stats(blocks['spot'], spot_codes, AXIS_NAMES_F, NP_PRIM,
                   E8F_SEED + 22)
    install = {'pooled_acc': spot['pooled_acc'], 'p': spot['p'],
               'min_acc': INSTALL_MIN_ACC,
               'pass': bool(spot['pooled_acc'] >= INSTALL_MIN_ACC
                            and spot['p'] <= P_CRIT_F)}
    verdict['g_install'] = install
    verdict['spot'] = spot

    # ── primaries ────────────────────────────────────────────────────────────
    p1 = p_stats(blocks['p1'], CODES64, AXIS_NAMES_F, NP_PRIM, E8F_SEED + 20)
    p1_pass = bool(p1['p'] <= P_CRIT_F and p1['n_axes_sig'] >= P1_MIN_AXES)
    p3 = p_stats(blocks['p3'], CODES64, CARRIED8, NP_PRIM, E8F_SEED + 21)
    p3_pass = bool(p3['p'] <= P_CRIT_F)
    p2 = p2_stats(blocks['p1'], FR_CODES, NP_PRIM, E8F_SEED + 23)
    p2_pass = bool(p2['exact_rows'] >= P2_MIN_ROWS
                   and p2['n_distinct'] >= P2_MIN_DISTINCT
                   and p2['p'] <= P_CRIT_F)
    verdict['P1'] = {**p1, 'pass': p1_pass}
    verdict['P3'] = {**p3, 'pass': p3_pass}
    verdict['P2'] = {'exact_rows': p2['exact_rows'], 'n_rows': p2['n_rows'],
                     'n_distinct': p2['n_distinct'], 'distinct': p2['distinct'],
                     'p': p2['p'], 'pass': p2_pass}

    # ── S-blocks ─────────────────────────────────────────────────────────────
    verdict['S1'] = {'profile': s1_profile(blocks),
                     'p1_cond_speech': cond_speech_acc(blocks['p1'], CODES64,
                                                       AXIS_NAMES_F),
                     'p3_cond_speech': cond_speech_acc(blocks['p3'], CODES64,
                                                       CARRIED8)}
    verdict['S2'] = s2_stats(blocks['wperm'], VEC, NP_TEX)
    verdict['S3'] = s3_confusion(blocks['p1'], CODES64)
    verdict['S4'] = s4_titr(blocks['titr'],
                            {r['concept']: FR_CODES[r['concept']]
                             for r in blocks['titr']})
    verdict['S5'] = s5_wing(blocks['wing'])
    verdict['S6'] = s6_margin_split(p2, blocks['p1'])
    verdict['S7'] = s7_carrier(blocks['carrier'])
    verdict['S8'] = s8_density(blocks['p1'], CODES64)
    verdict['train_log'] = {k: v for k, v in BUNDLE['train_log'].items()
                            if k != 'losses_every_10'}

    # ── fork ─────────────────────────────────────────────────────────────────
    if SMOKE:
        fork = 'SMOKE — mechanics only, no verdict'
    elif not gates_ok:
        fork = 'GATES DIRTY — primaries withheld (NO_VERDICT, lane law)'
    elif not install['pass']:
        fork = ('NO_VERDICT — G-INSTALL failed (installation, not '
                'generalization); re-fly after curriculum/budget revision')
    elif p1_pass and p3_pass:
        fork = ('FF2 — the code reads TRUE substrate geometry on the '
                'carried-8: first semantics-bearing featural read')
    elif p1_pass and not p3_pass:
        fork = ('FF1/FF3 — L3 lands in bottleneck form; true-dir arm did not '
                'clear (interface-without-geography, separability family)')
    elif not p1_pass:
        fork = ('FF4 — composition wall inside the span (installed but no '
                'held-out generalization); S8 + per-axis table localize')
    verdict['fork'] = fork

    print()
    print('=' * 72)
    if SMOKE:
        n_par = sum(1 for r in allrows if r['parsed']['kind'] != 'INVALID')
        green = ok and n_par >= max(1, int(0.5 * len(allrows)))
        print(f'  SMOKE {"GREEN" if green else "RED"} — flight mechanics '
              f'{"exercised" if green else "FAILED"} '
              f'({n_par}/{len(allrows)} rows parsed; gates mechanical: '
              f'g2 {g2["pass"]}, parse {gates["g4_parse"]["pass"]})')
        if green:
            print('  Next: Runtime > Restart runtime, set SMOKE = False, Run all.')
    else:
        print(f'  E8-F VERDICT — gates {"PASS" if gates_ok else "FAIL"} | '
              f'G-INSTALL {"PASS" if install["pass"] else "FAIL"} '
              f'(spot acc {spot["pooled_acc"]} p {spot["p"]})')
        if gates_ok and install['pass']:
            print(f'  P1 spelling: pooled {p1["pooled_acc"]} vs null '
                  f'{p1["null_mean"]} p={p1["p"]} | axes sig '
                  f'{p1["n_axes_sig"]}/14 -> {"PASS" if p1_pass else "FAIL"}')
            print(f'  P3 true-dir (carried-8): pooled {p3["pooled_acc"]} vs '
                  f'null {p3["null_mean"]} p={p3["p"]} -> '
                  f'{"PASS" if p3_pass else "FAIL"}')
            print(f'  P2 decode-to-name: {p2["exact_rows"]}/{p2["n_rows"]} rows, '
                  f'{p2["n_distinct"]} distinct, p={p2["p"]} -> '
                  f'{"PASS" if p2_pass else "FAIL"}')
        print(f'  fork: {fork}')
    print('=' * 72)
    vf = OUT / 'e8f_verdict.json'
    jdump(verdict, vf)
    ship(vf, f'e8f/{MODE}_{STAMP}')
    fn = OUT / 'bundle_real.json'
    if not fn.exists():
        jdump(BUNDLE, fn)
    ship(fn, f'e8f/{MODE}_{STAMP}')
    print('shipped: e8f/' + f'{MODE}_{STAMP}')
